# SU(3) \(O(y^4)\) Complete From-Scratch Pipeline

This notebook requires **no prior files from an earlier Colab session**.

Use a **standard CPU runtime**, then select **Runtime → Run all**.

The pipeline automatically creates and validates:

\[
\text{Stage 0} \rightarrow \text{Stage 1} \rightarrow \text{Stage 2}
\rightarrow 3B/3C \rightarrow 3E \rightarrow 3G \rightarrow I \rightarrow J.
\]

Existing valid outputs are reused on rerun. Stage 3G is checkpointed.

The final output is written to:

```text
/content/Y4_STAGE3J/CERT_Y4_stage3j_verdict.json
```


In [ ]:
from pathlib import Path

PIPELINE_SOURCE = '#!/usr/bin/env python3\n"""\ny4_complete_from_scratch.py\n===========================\n\nOne command from an empty Colab runtime to the final O(y^4) verdict.\n\nRun:\n    %run /content/y4_complete_from_scratch.py\n\nThe script creates and validates Stage 0, Stage 1, Stage 2, then executes the\nfinal endgame bundle. Existing valid outputs are reused on rerun.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport gzip\nimport hashlib\nimport json\nimport subprocess\nimport sys\nimport tempfile\nfrom pathlib import Path\n\nVERSION = "2026-06-13-complete-from-scratch-v1"\nSTAGE0_SOURCE = \'#!/usr/bin/env python3\\n"""\\ny4_stage0_geometry_manifest.py\\n================================\\n\\nSelf-contained Stage-0 compiler for the O(y^4) SU(3) one-flux flat-band program.\\n\\nPURPOSE\\n-------\\nThis script freezes the finite CONNECTED geometry problem before any expensive\\nSU(3) Haar/representation contraction is attempted.\\n\\nIt does all of the following in exact integer/rational arithmetic:\\n\\n  1. Regresses the certified O(y^3) arithmetic anchors\\n       b3      =  1975/124848\\n       leak3   = -12331/249696\\n       d3      = 7/32 + 12 leak3 - 4 b3\\n               = -109151/249696\\n\\n  2. Enumerates every rooted, site-connected multiset of FOUR plaquette\\n     perturbation insertions on the cubic lattice, INCLUDING repetitions.\\n\\n  3. Quotients those supports by the 8-element proper cubic stabilizer of the\\n     rooted input xy plaquette.\\n\\n  4. Attaches every possible one-plaquette output whose boundary links are\\n     contained in the geometric support.\\n\\n  5. Applies an exact NECESSARY SU(3) link-triality/bare-link filter:\\n       total fundamental-minus-antifundamental incidence == 0 mod 3\\n     on every link, after enumerating all 2^6 orientation assignments\\n     (ket, four insertions, bra).\\n\\n  6. Expands surviving support classes into ordered fourth-order words and\\n     quotients the ordered transitions by rooted proper cubic symmetry.\\n\\n  7. Writes frozen gzip JSON manifests with SHA-256 hashes.\\n\\nIMPORTANT SCOPE\\n---------------\\nThis is NOT yet the fourth-order Haar contraction and does NOT compute w_g.\\n\\nIt is the correct finite candidate compiler for the next stage. It deliberately\\nkeeps:\\n  * repeated plaquette insertions,\\n  * site-only corner contacts,\\n  * all ordered words surviving the necessary triality filter.\\n\\nThe later des-Cloizeaux/linked-cluster stage must still add:\\n  * exact intermediate SU(3) representation channels,\\n  * electric-energy denominators,\\n  * folded/subtraction terms required by the chosen effective Hamiltonian,\\n  * exact Haar contractions and rational weights.\\n\\nRUNTIME / HARDWARE\\n------------------\\nUse a standard Colab CPU runtime. An A100 is not used by this stage.\\nTypical runtime is tens of seconds to a few minutes.\\n\\nCOLAB\\n-----\\nUpload this file, then run exactly:\\n\\n    %run /content/y4_stage0_geometry_manifest.py\\n\\nOutputs are written to:\\n\\n    /content/Y4_STAGE0/\\n\\nNo edits are required.\\n"""\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport gzip\\nimport hashlib\\nimport itertools\\nimport json\\nimport math\\nimport os\\nimport platform\\nimport sys\\nimport time\\nfrom collections import Counter, defaultdict\\nfrom fractions import Fraction\\nfrom functools import lru_cache\\nfrom pathlib import Path\\nfrom typing import Dict, Iterable, List, Sequence, Set, Tuple\\n\\nVERSION = "2026-06-13-stage0-v1"\\n\\n# ---------------------------------------------------------------------------\\n# Paths / runtime\\n# ---------------------------------------------------------------------------\\n\\ndef default_output_dir() -> Path:\\n    if Path("/content").exists():\\n        return Path("/content/Y4_STAGE0")\\n    return Path.cwd() / "Y4_STAGE0"\\n\\n\\n# ---------------------------------------------------------------------------\\n# Exact cubic-lattice geometry\\n#\\n# Plaquette representation:\\n#   (x, y, z, a, b), with 0 <= a < b <= 2\\n# where (x,y,z) is the lower anchor and a,b are coordinate axes.\\n#\\n# Link representation:\\n#   (x, y, z, a)\\n# where (x,y,z) is the lower anchor and a is the positive axis.\\n# ---------------------------------------------------------------------------\\n\\nVec3 = Tuple[int, int, int]\\nPlaquette = Tuple[int, int, int, int, int]\\nLink = Tuple[int, int, int, int]\\n\\nE: Tuple[Vec3, Vec3, Vec3] = (\\n    (1, 0, 0),\\n    (0, 1, 0),\\n    (0, 0, 1),\\n)\\n\\nPLANES = ((0, 1), (0, 2), (1, 2))\\nPLANE_NAME = {(0, 1): "xy", (0, 2): "xz", (1, 2): "yz"}\\n\\nROOT: Plaquette = (0, 0, 0, 0, 1)\\n\\n\\ndef vadd(u: Vec3, v: Vec3) -> Vec3:\\n    return (u[0] + v[0], u[1] + v[1], u[2] + v[2])\\n\\n\\ndef vsub(u: Vec3, v: Vec3) -> Vec3:\\n    return (u[0] - v[0], u[1] - v[1], u[2] - v[2])\\n\\n\\ndef smul(s: int, v: Vec3) -> Vec3:\\n    return (s * v[0], s * v[1], s * v[2])\\n\\n\\n@lru_cache(maxsize=None)\\ndef vertices(p: Plaquette) -> Tuple[Vec3, Vec3, Vec3, Vec3]:\\n    x = p[:3]\\n    a, b = p[3], p[4]\\n    return (\\n        x,\\n        vadd(x, E[a]),\\n        vadd(x, E[b]),\\n        vadd(vadd(x, E[a]), E[b]),\\n    )\\n\\n\\n@lru_cache(maxsize=None)\\ndef boundary(p: Plaquette) -> Tuple[Tuple[Link, int], ...]:\\n    """Positively oriented plaquette boundary: +a,+b,-a,-b."""\\n    x = p[:3]\\n    a, b = p[3], p[4]\\n\\n    xa = vadd(x, E[a])\\n    xb = vadd(x, E[b])\\n\\n    return (\\n        ((x[0], x[1], x[2], a), +1),\\n        ((xa[0], xa[1], xa[2], b), +1),\\n        ((xb[0], xb[1], xb[2], a), -1),\\n        ((x[0], x[1], x[2], b), -1),\\n    )\\n\\n\\n@lru_cache(maxsize=None)\\ndef link_set(p: Plaquette) -> frozenset[Link]:\\n    return frozenset(link for link, _ in boundary(p))\\n\\n\\n@lru_cache(maxsize=None)\\ndef vertex_set(p: Plaquette) -> frozenset[Vec3]:\\n    return frozenset(vertices(p))\\n\\n\\ndef plaquettes_containing_vertices(vs: Iterable[Vec3]) -> Set[Plaquette]:\\n    """All unit plaquettes containing at least one supplied lattice vertex."""\\n    out: Set[Plaquette] = set()\\n    for v in vs:\\n        for a, b in PLANES:\\n            for ia in (0, 1):\\n                for ib in (0, 1):\\n                    anchor = vsub(vsub(v, smul(ia, E[a])), smul(ib, E[b]))\\n                    out.add((*anchor, a, b))\\n    return out\\n\\n\\n# ---------------------------------------------------------------------------\\n# Proper cubic rotations and rooted stabilizer\\n# ---------------------------------------------------------------------------\\n\\nRotation = Tuple[Tuple[int, int, int], Tuple[int, int, int]]\\n\\n\\ndef permutation_parity(perm: Tuple[int, int, int]) -> int:\\n    inv = sum(\\n        perm[i] > perm[j]\\n        for i in range(3)\\n        for j in range(i + 1, 3)\\n    )\\n    return -1 if inv % 2 else +1\\n\\n\\ndef proper_cubic_rotations() -> List[Rotation]:\\n    out: List[Rotation] = []\\n    for perm in itertools.permutations(range(3)):\\n        parity = permutation_parity(perm)\\n        for signs in itertools.product((-1, +1), repeat=3):\\n            if parity * signs[0] * signs[1] * signs[2] == +1:\\n                out.append((perm, signs))\\n    assert len(out) == 24\\n    return out\\n\\n\\nROTATIONS: List[Rotation] = proper_cubic_rotations()\\n\\n\\ndef transform_vec(v: Vec3, rot: Rotation) -> Vec3:\\n    perm, signs = rot\\n    out = [0, 0, 0]\\n    for j in range(3):\\n        out[perm[j]] += signs[j] * v[j]\\n    return (out[0], out[1], out[2])\\n\\n\\n@lru_cache(maxsize=None)\\ndef transform_plaquette_cached(p: Plaquette, rotation_index: int) -> Plaquette:\\n    perm, signs = ROTATIONS[rotation_index]\\n    anchor = p[:3]\\n    a, b = p[3], p[4]\\n\\n    aa, bb = perm[a], perm[b]\\n    new_anchor = list(transform_vec(anchor, (perm, signs)))\\n\\n    if signs[a] < 0:\\n        new_anchor[aa] -= 1\\n    if signs[b] < 0:\\n        new_anchor[bb] -= 1\\n\\n    aa, bb = sorted((aa, bb))\\n    return (new_anchor[0], new_anchor[1], new_anchor[2], aa, bb)\\n\\n\\nROOT_STABILIZER: List[int] = [\\n    i\\n    for i in range(len(ROTATIONS))\\n    if transform_plaquette_cached(ROOT, i)[3:] == (0, 1)\\n]\\n\\nassert len(ROOT_STABILIZER) == 8\\n\\nROOT_SHIFT: Dict[int, Vec3] = {\\n    i: transform_plaquette_cached(ROOT, i)[:3]\\n    for i in ROOT_STABILIZER\\n}\\n\\n\\n@lru_cache(maxsize=None)\\ndef rooted_transform(p: Plaquette, rotation_index: int) -> Plaquette:\\n    q = transform_plaquette_cached(p, rotation_index)\\n    t = ROOT_SHIFT[rotation_index]\\n    return (q[0] - t[0], q[1] - t[1], q[2] - t[2], q[3], q[4])\\n\\n\\ndef canonical_multiset(ms: Sequence[Plaquette]) -> Tuple[Plaquette, ...]:\\n    return min(\\n        tuple(sorted(rooted_transform(p, i) for p in ms))\\n        for i in ROOT_STABILIZER\\n    )\\n\\n\\ndef canonical_support_output(\\n    ms: Sequence[Plaquette],\\n    output: Plaquette,\\n) -> Tuple[Tuple[Plaquette, ...], Plaquette]:\\n    return min(\\n        (\\n            tuple(sorted(rooted_transform(p, i) for p in ms)),\\n            rooted_transform(output, i),\\n        )\\n        for i in ROOT_STABILIZER\\n    )\\n\\n\\ndef canonical_ordered_transition(\\n    word: Sequence[Plaquette],\\n    output: Plaquette,\\n) -> Tuple[Tuple[Plaquette, ...], Plaquette]:\\n    return min(\\n        (\\n            tuple(rooted_transform(p, i) for p in word),\\n            rooted_transform(output, i),\\n        )\\n        for i in ROOT_STABILIZER\\n    )\\n\\n\\n# ---------------------------------------------------------------------------\\n# Support enumeration\\n# ---------------------------------------------------------------------------\\n\\ndef enumerate_connected_multisets(\\n    order: int = 4,\\n) -> Tuple[Set[Tuple[Plaquette, ...]], List[int]]:\\n    """\\n    Enumerate rooted site-connected multisets of `order` plaquette insertions.\\n\\n    Repetition is allowed. Connectivity is enforced constructively: every new\\n    plaquette contains at least one vertex already in the rooted support.\\n    """\\n    states: Set[Tuple[Plaquette, ...]] = {()}\\n    counts = [1]\\n\\n    for depth in range(1, order + 1):\\n        started = time.time()\\n        new_states: Set[Tuple[Plaquette, ...]] = set()\\n\\n        for ms in states:\\n            support_vertices: Set[Vec3] = set(vertices(ROOT))\\n            for p in ms:\\n                support_vertices.update(vertices(p))\\n\\n            for p in plaquettes_containing_vertices(support_vertices):\\n                new_states.add(canonical_multiset(ms + (p,)))\\n\\n        states = new_states\\n        counts.append(len(states))\\n\\n        print(\\n            f"[enumerate] depth={depth} classes={len(states):,} "\\n            f"elapsed={time.time() - started:.2f}s",\\n            flush=True,\\n        )\\n\\n    return states, counts\\n\\n\\ndef candidate_outputs(ms: Sequence[Plaquette]) -> Tuple[Plaquette, ...]:\\n    """\\n    Candidate one-plaquette outputs.\\n\\n    Any output link absent from ROOT + insertions would be a lone external\\n    fundamental/antifundamental factor and fail the triality test. Therefore\\n    it is necessary that all four output boundary links lie in the union.\\n    """\\n    links: Set[Link] = set()\\n    support_vertices: Set[Vec3] = set(vertices(ROOT))\\n\\n    for p in (ROOT,) + tuple(ms):\\n        support_vertices.update(vertices(p))\\n        links.update(link for link, _ in boundary(p))\\n\\n    out: Set[Plaquette] = set()\\n    for anchor in support_vertices:\\n        for a, b in PLANES:\\n            p = (*anchor, a, b)\\n            if all(link in links for link, _ in boundary(p)):\\n                out.add(p)\\n\\n    return tuple(sorted(out))\\n\\n\\n# ---------------------------------------------------------------------------\\n# Exact SU(3) triality / bare-link filter\\n#\\n# Variables:\\n#   s0      ket orientation\\n#   s1..s4 perturbation character orientations\\n#   s5      bra orientation\\n#\\n# Each s is ±1. The bra contribution is conjugated, hence carries -s5.\\n# A necessary SU(3) Haar condition is zero net link triality mod 3.\\n#\\n# We precompute, for each row in F_3^6, a 64-bit mask of sign patterns that\\n# satisfy row dot sign == 0 mod 3.\\n# ---------------------------------------------------------------------------\\n\\nMOD3_SIGN_PATTERNS: List[Tuple[int, ...]] = list(\\n    itertools.product((1, 2), repeat=6)\\n)\\nSIGNED_PATTERNS: List[Tuple[int, ...]] = list(\\n    itertools.product((-1, +1), repeat=6)\\n)\\n\\n\\ndef build_row_masks() -> List[int]:\\n    masks = [0] * (3 ** 6)\\n\\n    for code in range(3 ** 6):\\n        x = code\\n        row: List[int] = []\\n        for _ in range(6):\\n            row.append(x % 3)\\n            x //= 3\\n\\n        mask = 0\\n        for index, signs in enumerate(MOD3_SIGN_PATTERNS):\\n            if sum(row[j] * signs[j] for j in range(6)) % 3 == 0:\\n                mask |= 1 << index\\n\\n        masks[code] = mask\\n\\n    return masks\\n\\n\\nROW_MASK = build_row_masks()\\nALL_SIGN_MASK = (1 << 64) - 1\\n\\n\\ndef admissible_sign_mask(\\n    ms: Sequence[Plaquette],\\n    output: Plaquette,\\n) -> int:\\n    assert len(ms) == 4\\n\\n    rows: Dict[Link, List[int]] = {}\\n\\n    factors = (ROOT,) + tuple(ms) + (output,)\\n\\n    for column, p in enumerate(factors):\\n        external_factor = -1 if column == 5 else +1\\n\\n        for link, incidence in boundary(p):\\n            row = rows.setdefault(link, [0] * 6)\\n            row[column] += external_factor * incidence\\n\\n    mask = ALL_SIGN_MASK\\n\\n    for row in rows.values():\\n        code = 0\\n        multiplier = 1\\n\\n        for value in row:\\n            code += (value % 3) * multiplier\\n            multiplier *= 3\\n\\n        mask &= ROW_MASK[code]\\n\\n        if mask == 0:\\n            return 0\\n\\n    return mask\\n\\n\\ndef signed_patterns_from_mask(mask: int) -> List[List[int]]:\\n    return [\\n        list(SIGNED_PATTERNS[i])\\n        for i in range(64)\\n        if (mask >> i) & 1\\n    ]\\n\\n\\n# ---------------------------------------------------------------------------\\n# Classification\\n# ---------------------------------------------------------------------------\\n\\ndef ordered_word_multiplicity(ms: Sequence[Plaquette]) -> int:\\n    counts = Counter(ms)\\n    value = math.factorial(len(ms))\\n\\n    for multiplicity in counts.values():\\n        value //= math.factorial(multiplicity)\\n\\n    return value\\n\\n\\ndef support_extent(plaqs: Sequence[Plaquette]) -> List[int]:\\n    vs: Set[Vec3] = set()\\n\\n    for p in plaqs:\\n        vs.update(vertices(p))\\n\\n    return [\\n        max(v[axis] for v in vs) - min(v[axis] for v in vs)\\n        for axis in range(3)\\n    ]\\n\\n\\ndef contact_flags(plaqs: Sequence[Plaquette]) -> Tuple[bool, bool]:\\n    unique = tuple(sorted(set(plaqs)))\\n    has_link = False\\n    has_site_only = False\\n\\n    for i in range(len(unique)):\\n        for j in range(i + 1, len(unique)):\\n            if link_set(unique[i]) & link_set(unique[j]):\\n                has_link = True\\n            elif vertex_set(unique[i]) & vertex_set(unique[j]):\\n                has_site_only = True\\n\\n    return has_link, has_site_only\\n\\n\\ndef json_plaquette(p: Plaquette) -> List[int]:\\n    return [int(x) for x in p]\\n\\n\\ndef class_id(\\n    ms: Sequence[Plaquette],\\n    output: Plaquette,\\n) -> str:\\n    payload = json.dumps(\\n        {\\n            "operators": [json_plaquette(p) for p in ms],\\n            "output": json_plaquette(output),\\n        },\\n        separators=(",", ":"),\\n        sort_keys=True,\\n    ).encode("utf-8")\\n\\n    return hashlib.sha256(payload).hexdigest()[:20]\\n\\n\\n# ---------------------------------------------------------------------------\\n# JSON helpers\\n# ---------------------------------------------------------------------------\\n\\ndef write_json(path: Path, obj: object) -> str:\\n    data = json.dumps(\\n        obj,\\n        indent=2,\\n        sort_keys=True,\\n        allow_nan=False,\\n    ).encode("utf-8")\\n\\n    path.write_bytes(data)\\n    return hashlib.sha256(data).hexdigest()\\n\\n\\ndef write_json_gz(path: Path, obj: object) -> str:\\n    raw = json.dumps(\\n        obj,\\n        separators=(",", ":"),\\n        sort_keys=True,\\n        allow_nan=False,\\n    ).encode("utf-8")\\n\\n    with gzip.GzipFile(\\n        filename=str(path),\\n        mode="wb",\\n        compresslevel=9,\\n        mtime=0,\\n    ) as handle:\\n        handle.write(raw)\\n\\n    return hashlib.sha256(path.read_bytes()).hexdigest()\\n\\n\\ndef read_json_gz(path: Path) -> object:\\n    with gzip.open(path, "rb") as handle:\\n        return json.loads(handle.read().decode("utf-8"))\\n\\n\\n# ---------------------------------------------------------------------------\\n# Main\\n# ---------------------------------------------------------------------------\\n\\ndef main(output_dir: Path) -> None:\\n    started = time.time()\\n    output_dir.mkdir(parents=True, exist_ok=True)\\n\\n    print("=" * 96)\\n    print("SU(3) O(y^4) STAGE-0 CONNECTED GEOMETRY / TRIALITY MANIFEST")\\n    print("=" * 96)\\n    print(f"version       : {VERSION}")\\n    print(f"python        : {sys.version.split()[0]}")\\n    print(f"platform      : {platform.platform()}")\\n    print(f"output        : {output_dir}")\\n    print("hardware      : CPU exact combinatorics; A100 is not used at this stage")\\n    print()\\n\\n    gates: Dict[str, object] = {}\\n\\n    # ------------------------------------------------------------------\\n    # G0: exact O(y^3) regression anchors\\n    # ------------------------------------------------------------------\\n    b3 = Fraction(1975, 124848)\\n    leak3 = Fraction(-12331, 249696)\\n    d3 = Fraction(7, 32) + 12 * leak3 - 4 * b3\\n    d3_expected = Fraction(-109151, 249696)\\n\\n    assert d3 == d3_expected\\n\\n    gates["G0_o3_exact_arithmetic"] = {\\n        "b3": str(b3),\\n        "leak3": str(leak3),\\n        "d3": str(d3),\\n        "passed": True,\\n    }\\n\\n    print(\\n        f"G0 PASS: d3 = 7/32 + 12*leak3 - 4*b3 = {d3}",\\n        flush=True,\\n    )\\n\\n    # ------------------------------------------------------------------\\n    # G1: cubic group\\n    # ------------------------------------------------------------------\\n    assert len(ROTATIONS) == 24\\n    assert len(ROOT_STABILIZER) == 8\\n\\n    for i in ROOT_STABILIZER:\\n        assert rooted_transform(ROOT, i) == ROOT\\n\\n    gates["G1_cubic_group"] = {\\n        "proper_rotations": len(ROTATIONS),\\n        "root_stabilizer": len(ROOT_STABILIZER),\\n        "passed": True,\\n    }\\n\\n    print("G1 PASS: 24 proper cubic rotations; rooted stabilizer size 8")\\n\\n    # ------------------------------------------------------------------\\n    # G2: local incidence anchors\\n    # ------------------------------------------------------------------\\n    root_neighbors = plaquettes_containing_vertices(vertices(ROOT))\\n    assert len(root_neighbors) == 33\\n\\n    repeated_root = (ROOT, ROOT, ROOT, ROOT)\\n    repeated_mask = admissible_sign_mask(repeated_root, ROOT)\\n    assert repeated_mask.bit_count() == 22\\n\\n    gates["G2_local_anchors"] = {\\n        "plaquettes_touching_root_by_site": len(root_neighbors),\\n        "repeated_root_triality_patterns": repeated_mask.bit_count(),\\n        "passed": True,\\n    }\\n\\n    print("G2 PASS: local geometry and SU(3) triality synthetic anchors")\\n\\n    # ------------------------------------------------------------------\\n    # G3: enumerate all rooted connected order-4 operator multisets\\n    # ------------------------------------------------------------------\\n    support_classes, depth_counts = enumerate_connected_multisets(order=4)\\n\\n    expected_depth_counts = [1, 6, 156, 5082, 182440]\\n    assert depth_counts == expected_depth_counts\\n    assert len(support_classes) == 182440\\n\\n    gates["G3_support_enumeration"] = {\\n        "depth_counts": depth_counts,\\n        "expected": expected_depth_counts,\\n        "passed": True,\\n    }\\n\\n    print(\\n        "G3 PASS: rooted connected multiset counts "\\n        + " -> ".join(f"{x:,}" for x in depth_counts)\\n    )\\n\\n    # Freeze every support class.\\n    support_records = [\\n        [json_plaquette(p) for p in ms]\\n        for ms in sorted(support_classes)\\n    ]\\n\\n    support_path = output_dir / "y4_connected_supports.json.gz"\\n    support_sha = write_json_gz(\\n        support_path,\\n        {\\n            "meta": {\\n                "version": VERSION,\\n                "root": json_plaquette(ROOT),\\n                "scope": (\\n                    "rooted site-connected multisets of four perturbation "\\n                    "plaquettes, repetitions allowed"\\n                ),\\n                "proper_rotation_quotient": True,\\n                "reflection_quotient": False,\\n            },\\n            "supports": support_records,\\n        },\\n    )\\n\\n    # ------------------------------------------------------------------\\n    # G4: outputs + exact triality filter\\n    # ------------------------------------------------------------------\\n    candidate_output_pairs = 0\\n    raw_survivors = 0\\n    canonical_survivors: Dict[\\n        Tuple[Tuple[Plaquette, ...], Plaquette],\\n        int,\\n    ] = {}\\n\\n    process_started = time.time()\\n\\n    for index, ms in enumerate(support_classes, start=1):\\n        for output in candidate_outputs(ms):\\n            candidate_output_pairs += 1\\n            sign_mask = admissible_sign_mask(ms, output)\\n\\n            if sign_mask:\\n                raw_survivors += 1\\n                key = canonical_support_output(ms, output)\\n                canonical_survivors[key] = (\\n                    canonical_survivors.get(key, 0) | sign_mask\\n                )\\n\\n        if index % 20000 == 0:\\n            print(\\n                f"[triality] supports={index:,}/{len(support_classes):,} "\\n                f"canonical_survivors={len(canonical_survivors):,} "\\n                f"elapsed={time.time() - process_started:.2f}s",\\n                flush=True,\\n            )\\n\\n    assert candidate_output_pairs == 895524\\n    assert raw_survivors == 449\\n    assert len(canonical_survivors) == 449\\n\\n    gates["G4_triality_filter"] = {\\n        "candidate_support_output_pairs": candidate_output_pairs,\\n        "raw_survivors": raw_survivors,\\n        "canonical_survivors": len(canonical_survivors),\\n        "passed": True,\\n    }\\n\\n    print(\\n        f"G4 PASS: {candidate_output_pairs:,} candidate output pairs -> "\\n        f"{len(canonical_survivors):,} exact triality survivors"\\n    )\\n\\n    # ------------------------------------------------------------------\\n    # Build detailed support/output survivor manifest.\\n    # ------------------------------------------------------------------\\n    survivor_records: List[dict] = []\\n    distinct_hist = Counter()\\n    sign_hist = Counter()\\n    order_multiplicity_hist = Counter()\\n    contact_hist = Counter()\\n    raw_ordered_word_count = 0\\n\\n    for (ms, output), sign_mask in sorted(canonical_survivors.items()):\\n        distinct_count = len(set(ms))\\n        word_mult = ordered_word_multiplicity(ms)\\n        raw_ordered_word_count += word_mult\\n\\n        has_link, has_site_only = contact_flags(\\n            (ROOT,) + tuple(ms) + (output,)\\n        )\\n\\n        distinct_hist[distinct_count] += 1\\n        sign_hist[sign_mask.bit_count()] += 1\\n        order_multiplicity_hist[word_mult] += 1\\n        contact_hist[(has_link, has_site_only)] += 1\\n\\n        all_plaqs = (ROOT,) + tuple(ms) + (output,)\\n\\n        survivor_records.append(\\n            {\\n                "class_id": class_id(ms, output),\\n                "root": json_plaquette(ROOT),\\n                "operator_multiset": [json_plaquette(p) for p in ms],\\n                "output": json_plaquette(output),\\n                "output_plane": PLANE_NAME[(output[3], output[4])],\\n                "distinct_operator_plaquettes": distinct_count,\\n                "ordered_word_multiplicity": word_mult,\\n                "triality_sign_assignment_count": sign_mask.bit_count(),\\n                "triality_sign_assignments": signed_patterns_from_mask(sign_mask),\\n                "support_extent": support_extent(all_plaqs),\\n                "support_vertices": len(\\n                    set().union(*(vertex_set(p) for p in all_plaqs))\\n                ),\\n                "support_links": len(\\n                    set().union(*(link_set(p) for p in all_plaqs))\\n                ),\\n                "has_link_sharing_contact": has_link,\\n                "has_site_only_corner_contact": has_site_only,\\n            }\\n        )\\n\\n    assert raw_ordered_word_count == 4296\\n    assert dict(sorted(distinct_hist.items())) == {\\n        1: 6,\\n        2: 170,\\n        3: 271,\\n        4: 2,\\n    }\\n    assert dict(sorted(sign_hist.items())) == {\\n        2: 2,\\n        4: 10,\\n        8: 416,\\n        12: 20,\\n        22: 1,\\n    }\\n    assert dict(sorted(order_multiplicity_hist.items())) == {\\n        1: 6,\\n        4: 15,\\n        6: 155,\\n        12: 271,\\n        24: 2,\\n    }\\n    assert dict(sorted(contact_hist.items())) == {\\n        (False, False): 1,\\n        (False, True): 198,\\n        (True, False): 46,\\n        (True, True): 204,\\n    }\\n\\n    gates["G5_survivor_classification"] = {\\n        "distinct_operator_histogram": dict(sorted(distinct_hist.items())),\\n        "sign_assignment_histogram": dict(sorted(sign_hist.items())),\\n        "ordered_word_multiplicity_histogram": dict(\\n            sorted(order_multiplicity_hist.items())\\n        ),\\n        "contact_histogram": {\\n            f"link={k[0]},site_only={k[1]}": v\\n            for k, v in sorted(contact_hist.items())\\n        },\\n        "raw_ordered_words": raw_ordered_word_count,\\n        "passed": True,\\n    }\\n\\n    print(\\n        "G5 PASS: survivor classification; "\\n        f"raw ordered words={raw_ordered_word_count:,}"\\n    )\\n\\n    survivor_path = output_dir / "y4_triality_survivors.json.gz"\\n    survivor_sha = write_json_gz(\\n        survivor_path,\\n        {\\n            "meta": {\\n                "version": VERSION,\\n                "root": json_plaquette(ROOT),\\n                "filter": (\\n                    "necessary SU(3) link triality condition only; "\\n                    "not a sufficient nonzero Haar criterion"\\n                ),\\n                "sign_order": [\\n                    "ket",\\n                    "operator_1",\\n                    "operator_2",\\n                    "operator_3",\\n                    "operator_4",\\n                    "bra",\\n                ],\\n                "bra_is_conjugated": True,\\n            },\\n            "classes": survivor_records,\\n        },\\n    )\\n\\n    # ------------------------------------------------------------------\\n    # G6: expand and symmetry-quotient ordered transitions\\n    # ------------------------------------------------------------------\\n    ordered_transitions: Dict[\\n        Tuple[Tuple[Plaquette, ...], Plaquette],\\n        int,\\n    ] = {}\\n\\n    for (ms, output), _support_mask in canonical_survivors.items():\\n        for word in set(itertools.permutations(ms)):\\n            key = canonical_ordered_transition(word, output)\\n\\n            # Recompute exact sign mask in the ordered column convention.\\n            word_mask = admissible_sign_mask(word, output)\\n            assert word_mask != 0\\n\\n            ordered_transitions[key] = (\\n                ordered_transitions.get(key, 0) | word_mask\\n            )\\n\\n    assert len(ordered_transitions) == 4221\\n\\n    ordered_records: List[dict] = []\\n\\n    for index, ((word, output), sign_mask) in enumerate(\\n        sorted(ordered_transitions.items()),\\n        start=1,\\n    ):\\n        ordered_records.append(\\n            {\\n                "ordered_id": f"W4-{index:05d}",\\n                "root": json_plaquette(ROOT),\\n                "ordered_insertions": [json_plaquette(p) for p in word],\\n                "output": json_plaquette(output),\\n                "triality_sign_assignment_count": sign_mask.bit_count(),\\n                "triality_sign_assignments": signed_patterns_from_mask(sign_mask),\\n            }\\n        )\\n\\n    gates["G6_ordered_words"] = {\\n        "raw_words_before_rooted_symmetry": raw_ordered_word_count,\\n        "canonical_ordered_transitions": len(ordered_transitions),\\n        "passed": True,\\n    }\\n\\n    print(\\n        f"G6 PASS: {raw_ordered_word_count:,} raw words -> "\\n        f"{len(ordered_transitions):,} rooted symmetry classes"\\n    )\\n\\n    ordered_path = output_dir / "y4_ordered_transition_words.json.gz"\\n    ordered_sha = write_json_gz(\\n        ordered_path,\\n        {\\n            "meta": {\\n                "version": VERSION,\\n                "root": json_plaquette(ROOT),\\n                "word_length": 4,\\n                "proper_rotation_quotient": True,\\n                "reflection_quotient": False,\\n                "scope": (\\n                    "connected triality-admissible candidate words; "\\n                    "intermediate SU(3) channels and denominators not yet attached"\\n                ),\\n            },\\n            "words": ordered_records,\\n        },\\n    )\\n\\n    # ------------------------------------------------------------------\\n    # G7: round trip and final summary\\n    # ------------------------------------------------------------------\\n    assert len(read_json_gz(support_path)["supports"]) == 182440\\n    assert len(read_json_gz(survivor_path)["classes"]) == 449\\n    assert len(read_json_gz(ordered_path)["words"]) == 4221\\n\\n    gates["G7_round_trip"] = {\\n        "supports": 182440,\\n        "survivors": 449,\\n        "ordered_transitions": 4221,\\n        "passed": True,\\n    }\\n\\n    elapsed = time.time() - started\\n\\n    summary = {\\n        "meta": {\\n            "version": VERSION,\\n            "date": "2026-06-13",\\n            "python": sys.version,\\n            "platform": platform.platform(),\\n            "walltime_s": elapsed,\\n            "hardware": "CPU",\\n            "a100_required": False,\\n        },\\n        "scope": {\\n            "completed": [\\n                "rooted connected support enumeration",\\n                "proper cubic symmetry quotient",\\n                "repeated insertion bookkeeping",\\n                "candidate external plaquette attachment",\\n                "necessary exact SU(3) link-triality filter",\\n                "ordered fourth-order word expansion",\\n            ],\\n            "not_completed": [\\n                "SU(3) Haar contractions",\\n                "intermediate representation-channel enumeration",\\n                "electric-energy denominators",\\n                "des-Cloizeaux folded/subtraction terms",\\n                "exact fourth-order rational weights",\\n                "H4 flat-band commutator",\\n            ],\\n        },\\n        "counts": {\\n            "connected_support_multisets": 182440,\\n            "candidate_support_output_pairs": 895524,\\n            "triality_survivor_classes": 449,\\n            "raw_ordered_words": 4296,\\n            "canonical_ordered_transition_words": 4221,\\n            "classes_with_site_only_corner_contact": (\\n                contact_hist[(False, True)] + contact_hist[(True, True)]\\n            ),\\n            "classes_without_site_only_corner_contact": (\\n                contact_hist[(False, False)] + contact_hist[(True, False)]\\n            ),\\n            "classes_with_any_link_sharing": (\\n                contact_hist[(True, False)] + contact_hist[(True, True)]\\n            ),\\n            "classes_with_no_link_sharing": (\\n                contact_hist[(False, False)] + contact_hist[(False, True)]\\n            ),\\n            "pure_site_only_classes": contact_hist[(False, True)],\\n        },\\n        "histograms": {\\n            "distinct_operator_plaquettes": dict(sorted(distinct_hist.items())),\\n            "triality_sign_assignments": dict(sorted(sign_hist.items())),\\n            "ordered_word_multiplicity": dict(\\n                sorted(order_multiplicity_hist.items())\\n            ),\\n        },\\n        "gates": gates,\\n        "files": {\\n            support_path.name: {\\n                "sha256": support_sha,\\n                "records": 182440,\\n            },\\n            survivor_path.name: {\\n                "sha256": survivor_sha,\\n                "records": 449,\\n            },\\n            ordered_path.name: {\\n                "sha256": ordered_sha,\\n                "records": 4221,\\n            },\\n        },\\n        "next_stage": (\\n            "Attach exact SU(3) representation channels and des-Cloizeaux "\\n            "energy-denominator words to the 4,221 ordered transition classes."\\n        ),\\n        "passed": True,\\n    }\\n\\n    summary_path = output_dir / "y4_stage0_summary.json"\\n    summary_sha = write_json(summary_path, summary)\\n\\n    print("G7 PASS: gzip JSON round-trip and exact record counts")\\n    print()\\n    print("SUMMARY")\\n    print(json.dumps(summary["counts"], indent=2, sort_keys=True))\\n    print()\\n    print(f"SUPPORTS : {support_path}")\\n    print(f"SURVIVORS: {survivor_path}")\\n    print(f"WORDS    : {ordered_path}")\\n    print(f"SUMMARY  : {summary_path}")\\n    print(f"SUMMARY SHA256: {summary_sha}")\\n    print(f"WALLTIME : {elapsed:.2f} s")\\n    print("ALL STAGE-0 GATES PASS")\\n    print()\\n    print(\\n        "NEXT: exact SU(3) channel/denominator attachment. "\\n        "Use the A100 only after the contraction tensor dimensions are known."\\n    )\\n\\n\\nif __name__ == "__main__":\\n    parser = argparse.ArgumentParser(\\n        description="O(y^4) connected geometry and SU(3) triality manifest"\\n    )\\n    parser.add_argument(\\n        "--output-dir",\\n        type=Path,\\n        default=default_output_dir(),\\n    )\\n    args, _unknown = parser.parse_known_args()\\n    main(args.output_dir)\\n\'\nSTAGE12_SOURCE = \'#!/usr/bin/env python3\\n"""\\ny4_stage1_stage2_autobundle.py\\n================================\\nOne-file Colab driver that rebuilds Stage 1 from the existing Stage-0 manifest\\nwhen necessary, then runs Stage 2. No separate Stage-1 upload is required.\\n\\nCOLAB:\\n    %run /content/y4_stage1_stage2_autobundle.py\\n\\nRequired existing input from the completed Stage-0 run:\\n    /content/Y4_STAGE0/y4_ordered_transition_words.json.gz\\n\\nOutputs:\\n    /content/Y4_STAGE1/\\n    /content/Y4_STAGE2/\\n"""\\nfrom __future__ import annotations\\nimport argparse, base64, gzip, hashlib, json, subprocess, sys, tempfile\\nfrom pathlib import Path\\n\\nVERSION = "2026-06-13-autobundle-v1"\\nSTAGE1_GZ_B64 = \\\'H4sIAAAAAAAC/+09a3Pbuo7f/Su4OnP3yq3s+JG6rXt9Z7M5OT3d7Wk7Tc59jMerkW05UWtLriSnycnkvy8APkRSlOO03d0vm+nUtgSCIAiAAAhKP/3L0a7Ij+ZJehSn12x7W15l6bDleV7r9jgsyugy7oeLqyhN43W4jNNsk6RRmeXhJkqTVVyU3e1ta3LoX6t1dhMtSnb+uz9sszze5nERp2VUJlnaEb2wKF2yJC3jfBMvk6iMO3Ea55e3bJFttsk6ztkqy1l5Fbfe+7f/ddxmWRp3VuvdDYtXq3hRJtdx59dok6zLLE2ilG3z7DKPNt1W6827D79ftDr417q4itk5jq7TY1m+jPN42fkKX5gc17jVYvB3tMiAlLQ8+udxeH5x8vqsdwR8ES3CMo/SIkHqQ2xcdD8VWdq9/KNF+ItFnmxLFq2LjMHtzwXLdmWRLGN2mq2jOfualFes00nS7a6kUXc6AAA/OsskB3rf/36hCO4QNZKIPhKxd1YkIbVWMU5AWCTp5Touwzz+BCzjdDt6gLHnAC7G93md7MMshKXYbTZRfstRtv7+68kFu/j1zTkjQPbh4/u/nZ2zI3b6/jcY3Nm5GN6ev9YvMN/xdZzf2jPGaMaQc/w2cCSH+0D2p90lCRVAzpOSZSuWlAWjsbfKPInWSXnLiuQyZVGBHxuY4iIAqUoKMW00/0/YMkaxy4oYWkeLK7ZO0s8MBKIAmkCyljuQZkK7vkWhzYRoJzkId8H8bfClHRCiPP6yS0DcgVzegFB9TQqQEz4ZLFqB0KNksxVM55rN8whwFnGOI5FYNtk1IJFj6ETLTQIjmK9jNgcpslQqQdL5XX2chApnFkQyWq9Bs4DcNajasiJK10AmVXMblVeFoFNRVrB+MAiGHCtya1fGgtcsXoN45cmCkQ4ncP0s/MQFH8eJnVwCLpixa6AMJ+xsNBmdycEuUKkIsrzK4xiuFNkaIZkm9ICz1wG0E78/6pyNwk/toxFHsFpHlwXbZMt43Sm20YIjSKN0gYQA5KQ/AsxRKWcH0Bad03WW/BFHYFBW2XoJPEFGFCZJ19Fit9sYTBIYe3XaTzsgnmCIriLg69M+CexpJ1suxaUi/ByXT4oQ5lvQDWP9A0WF2LmOb3CmwdR8BmKIrx1NM5nSzLq2/fwedO3d+wupbQ8qW+sNsDaDrrGRmEt2uo7nxeKq8xpGBNK7yMDQJotEqAyQEYMNW8Y3TKMK+PVrFOUsAvqTcgd8DQQ7j4rdXNFu4sp06RfW/WucXF6BhWp9NEUbRZFtdusygQ4WYIORXzCDSoAXwOZkieuHlF4TOmBpVmr0vWrRUI7AkpTWWGBRuBWqx6Llkgw+UKh6AOpav558/PnvJx8rFrd+ByIiBlYxXUZgprjNP/3wO8t3IOqbuMtwkTjp93oMrA4SsytiXPqEGUJbB4hP3789+Xexan3cpbjcsfk6W3wWekg2kfXEevUnQF4tWocv4bjgt1Z5tmFhuNqVuzwOQ4bmI8elCYgjrhetlryWX26jvIjl78s/kq38DkJ9tU7m8mcCVJZZti7kBVoZxPcNTKP8vgUTBGv7Rv4ublUT5Bcnb5Gt0abQtIibp9kO9TAA7V1FMMfLBKw8Aa/UDArQX8QFcXuXLogyeXud78IFmHnRF8oYDETe/YC00o3ydgs22+ofTKz8enG7jV2QPwNlAXsDEBHY5IC9BbUN2DkYnxhsUsAudqDtrdbfzj6ev3n/jk2YN+gNRp3eqNMfdvhMdq77MFMf1hG0KUE5J7zRFExRwOr/zVpvccVqhpq13uBaVYOYtc7I7bqAxXDd0M0M/Dm44/v9gPXgXztgPnz2q6/4q91unfXCEQCCtWU/gbWevDgaBuwKx8xG8Jtugai3YALlJIbkFvlt1vkrMX5M4r0gJZrQFd/7NufMaxOqZMWxdcG8FmXht3kP+JfHoAApv00XQd1A33m33cXXJZB1xDzVq4e/DulZdkz4mjum23QRbEtZDXcDQwV7E1XjDcGDunjEoEUHiNZiN/c8Q/A8bZ4DvSa7vXYz6Q0T0zf7d/MRBZtougYj6+/GTpEL2LX7BlHtujHWe/Z3096MPWXX8BGw3bTPf/Tpx4D/GMzaQMi/KWPgb6KbAlyCyTuwvW2icA56Dlb91t+OmdJFjQL+P6qepLrb7QpKbmBCt9PxcEa/ooDN6cIQgLbTY371JoJrxIabgJ1Noxln383cuDyfGVz11WT4/g2N74YGdgMjClgEKvm0T96oAoo4VMTBIoKbO+DmHG7O4eYKX8eCq3U614EO5mqIFrzOWrSeGk+NeaUW1ay0sa+fWOfH/QG25tiV5WDkf3B/XBUokAB+bPwkHzMy1cQLYAAf/zZgX0AoktyQhC0Icr8NrqT/RX3Da/hrAEp3xAaig8UgTHeN2MEt+MhRDp+cDiiceUU+GsVBpnd/i24MR3c06qJD0UjfFujh5Dwhkrbicyiu4+eXh6RlBe4S+Cow4xjVRGtrDFwF6beufUAXDUMGcbB24WKl+Ya3HfS/5Tzv1nHDWMBijmkF553M4M6UMx7A2jNpO7fsr+idSS2BVt1ou43TpQ/AHQKmKVKr0pc9DRC4o4ClNceR+gDXPohlEXif38+2XtD/kWyTXJgdwgXBY40Th/H5e9hWsQlinuxznI5RS/axDCLuGF1XBEav3u/0ybLqNDhkuI2D4Y0mEwwX4zUEEg0z9yDpMtGjT1S4QC+18FPqecxXylRhrwamzAC4nuAyjHX3tpo9cdG/I4evPWb9ez5GytOp0T/tB2iGeKcwg8gOfkF1rCYvvSkf6KytYLGXJDdFETvkNHch+tgYjkrVZoBgYm7FpFpwgpYpwIJrMDH6UIC8I6ANIPW55dennCuzhyYKDaqK0ijP4hNFheHtoHTV3AzdUa+7G1IL+eqlpXpoaREpHWQIBpZIRZdHkrx3DOZAiG7AxIOlBxkQw/4ck3clUkCoXdWPgf5jqP84DjCpJTo4YX/EeSYkZBOD38pzMSugE8ihNARGxWW2W1ypRafLzmQqCcN9nmeJl1zjzMQUUjWgbMuwy96n61sxWLAJFJOlhLSefQMSOTrCX0ZJGi+7Bi+Faq/jVExSG3V1pN+L1mu/VGrfI83nGkEd81YtQ7vIvxFmRBd+a/5nqAd3ldsltK5SQ00F71tKEzEHhVHVMr4JKrWMYbWGaLSM5UAsJXwMTVr47QtYMUJdT6/A9GcoSg8pKY9IULFQP02z2Kusoqa8TTouMNV1W/gtk8oLGrRrQIiIE31btzJqNO4xyD+gXeM/iQUKZ8CGDQ3wT3Q6APrEVzSbnOag7WyGXHkMQieotHhTCV+3fW7jx6VZ6NNETu5lXPq+SA8o8922dQV16XN8S4o0JJ7DLxIRjs+xdIMfAnbMFwByAsj1J882S68xWxxyjvFVL84LHiat45W1xuhmdMYjlhxzj2OVoGm0twBOVrkR3bhyfRphaqsbMgHCww3uAhC9dRlDkDmAzBGEU2vr0NSPeMjLg7iIh7w8lIt4yIsBXZtPcgQr8mau8xpQCIZywxxyQ8G5yC8VGotUyCZYiLsO+n1anvgMgDVHTxDiuZYdvFfLl1wEyWPExugvzhR/hD3bmrZMkKUxKkkXyZISThM7zGyThK4pruxVahXfwEyk0TrUt5MmEM2iMnM1BlF9xg3R075qJ2yUu/0Tzo8ptZ/Bz4ouwip/ENKeibOQ/qxm7Ax9UIsKzVYEfd/+AU6j2vjxRRNcMccUMRtTZLFfTAqCiUg7y1CreV8cy9TDi55IQ9BunATg/2/5mkfTI5vIVFW1gSQRcDtd64NflkBK5Di9St441bhaIE2wXjzlBIHJ5AgCsRjhTKPFEubjjkJZRaVATxe4TIR4WUst3As8l+tsDtMrd7YO0+w7kRXV/WRK94WbjDbnxk3LrMpn1Q0F+Y40+yHOZYTpe6kySm80xQHvM6Q2y/gSY7aJEDW6TsQk6XWUJ+CYY+qB7iuNI3ahQUI2ji0RBVDTSgh2BlzIAmpV6ZgIBmAuYPEtRbxTc5IqaBkp6C06e1qo8amoo8KhgOrcgCu+dTUQuDSHhjuTk2bPvaIDdBs9WbppGmihwHe1hdgzN+pXcUT7uN6Y/RKBcQjqDVZRAqRcEtUAhluCZNXaD8CKqZJNBO2ORpyJAMe/OCEkdwlK/jAh71uaVTbFDCfV8LBwZsX6Xfe9iJ0Vk4UzYeGcHBAC69Fv25QLpzYo+ajfDcz+NXFZRGmWJgtaELBvxJOkvouEgPka8wS72xVlhsGY+jZmWudrF3G9N2mjFb9auJxWRK45umhog7JsIHrSTV6XBRrI6WtxRa22JcXWubBfuP5opkts4JcVibwHZcNg1A/C8N175+0qXGoQOrFqWKNRzmel3dVSSLtfHeh1vYsJN/9GHgt20tY2UQTlm6j4LPTB59AgyIinzf7yF/aJsHwKdETS9REoK5xisE6MvcPRVToLIzUCUNL8s5G0H6KFqfOePsECOSwLss3SBtfNqaG40EL/aTUz+Aegxm8LVuMLQGq/KjjdUq1MZKYVR56ALxR9DVWFSiiqgzwUXkzzuBqImqCQV0aEVGhCDS7yXWytEw0ybahwPfwDmgCd3Pb2B/0RrGQYDzyBjz7/GMzajxpNUeY+3G0/akS0ahkt9mkqjUpnvjY/5gAbdNlE4DAv0rAh3UK6a2tx8xqM81PJiUcpKhgu+UQo1BDM0ndNrj3Lo0BJtnwMC7a+usg2jnWnaitM0z4VQitgWzBSfjRhGip9oQlLcEhxcFOD/3dNfkKoViBoA7E8rkr7PIYafN8F71rMBfzABc+dMxQJgKJP2xmxUz2AKuCQmrk3Vlxl7FXbmc4z1zKKPBeYXLd1not10xJoIVTWpOntlC7ZCArdEFpotfa6OrpQ7FFXDYtQRxcCt6ZqbXUtNVo61DeQ6c0fvsv7OoZZLvNbrfaOXcXrLfgw/yM7vLzeLCxix443XDy8ZqBmvm4CzQnQSglcl+e1y2YTHeReUE6ivJ9uSZOKsRtC6iplV6LFpcJNf19ySUsYzbNsDQyB/0XKaJcmX9C/kSqHJMqMENfZq6ggw2usS3ixANUOM9wq0O9QoolSGFF6GVOqkvfRtlJxnyqghG9SumFlGZBkIIeYJjCsf61d/TRz5Ii1EdR8hXidGGJlYK9fb8SvM0N1ImZU9h+YkGIei6to8GwUrhJMAWEhEVX8cNEoc97bFWaheaVgl8OLbAZVB2OrbgYrtO/lc6+NtXWwni3XscnwxdWOZyOwyNBfR5v5MhoLyG4eR0u/jx7uAHPPc8+zBnrV3W2xftMnNEZG7ap7Fd8sk8sYvFOZUfua4zCxoEobVMCy+acx/hcvSnOE3PeiAqzlbrMtfAAKKHWYlpNBQOIZwnJTTMidwDR49jUEAzsh0Wt3QeazZex7u3LVeSEqqIgxnJL5bQkLtvLEqpnReYq3944lvPzje4ZTxNuIO/MT3wu8gHljr/2NY6OZx0rS7mv47xeUHhShNNrEE3Q6kUwsOoCGE+/r3AvIRsMaUazj63g9eQn3sFJ00nMKjBALGnqNbbbISkahEDn4RKzhXBpbtJPUbomXXl4CkTRU8FsmcrAu4gQZxN51BoLLASQZuB6GdJgh1EuPrmPflWBvTqcXKiqFG/4NT8/e8H2WS5mySuNLBda50UAMjmH2Aquo40tJ5CaCS1S3GRoSpUoLNd5hSKSKC9BMY49YFY3/+Xo+GBt2N5+xMhFEDR0wIVNUhBhmn+mn8OG3OY7Lm3gQ2fR7Ip0vLvLSFFFXThWHnT47+8fJ6QU7/fXk3buzt+yI/Xz27v1vb96dXLz/yH47effml7PzC6/9EO6VB3a1wAw/G7M7Ub9775kg/KAPI5CKSzaUSIITVMUAE8q7ivLlV9yfHjsKzF/VisqNxoJTKl0mCVFFnQFbefKkDacZ8awyqg0xKJfzdxn3QlAgXHI1Naog2xogABk6ZUNR2SrfPQToqUe/vZm9706XabfweDAQod6lto8O9iIQCkr75SKWB/+uNxaHU5yn0LCSj0Xp4gosWtdglKrEo3LnNt+pbIDgdc/7IPoK4kUDxED1YhQWiK1qjYjjhtsC/0vX7Qp5v2fIAzlLuK3uizLujkBzJ/dwOer7vY2e6o2wUIx3qCbh4mvGouWnaIFnebbSq4O5w1ITqiPT6pzEjg2dyEjmeCLlbHL8SmCiEg5QB3F2SR5VKvCgBm60DI+g5wHWmz87GvIJlZGxlrXUYv4+pthlvuSYqK9uvqjdBDdPXTB5H7CR3nQe5cODWvdcrUcHNeWzWjU1Jqk+7G8cdx9LF/YNrX+MhQ1N1PefV3e1+pRt5TuPLLf6i/sWedE57vhh+aCZjpKCSUUdUqX05U6VjkhRxZpTvdr2W/F1GvG1zC0PYWQs6QtxHvZtWfStqRmEgwdaDKzpGoa9B1oMSYv1FsOHW5gz/mAfo6qP+5Zz/0YxyMGhfo0Dg9oI+7URjGoUPjPkkBaQqfe6F57/PgzNlSHElcEza6+88msWKvMV2hoG+O8+jylfes1FBeKla5QWG1KmlO41Ap3MEzzxxnU+aU23yMulkae8192H1z324eT8fCyWQJBdPGHKQFDjtKDTe+w0KpJNgju3WEC3xPxLps6BqgEKoZYnyfiZk6adm0/83Nb+DR55mrF+V5SDwIKg1TQUalOH36n85OqOzNs6b3Ky4qXzJsdZS2ZpjZsSZs79qGYoCI9CBWYRsUlugLymuzp+jMDt+zQjIUYnoZqBaqerlmy8ItbrJQii5sasOMAsNE9aP6ZFJbIHNjLyz9lisQN7iod3v6VIwrkD9RjqRe61lk83yifc2VuzwuJbKy9wKmUBJ9chc6+Opjrg8dSkry2UJNqmztzx6E5fwWS5jTxbHtrNvNm9Vu5h6+HTCfnldiNt5/AndX5fb7fZFSWbx2yxzvD4K7h9cS72lcWBfqZVUHUNx4BiVhm86r2OXYu3I6ClmNeFoKJa6ZOsF9oTinNuOim6rxUqDMBBQI7xDshncDPQ5LlupwTPBYaK5gdLtbgnog7wf0/R1sGFW3raMqglGM3Ub1XDpVGpVXK1rfnZv6ZIO1hZwMo+qnuVSXPfU4rpvu0w6lq3btutAAwjXsEYss7HqdRfFzzTDAhpMFVAqlDsTOyY6CzPF+sIiwQpcpQUcr4HezGovsPDcCl4bZa5/Tit2QNWfI3kgzboCRhERo7PyBAeDCu2eEot6rpMgiRo2rTVO0O1rA+gGd4kWUwaJrdW3p3Qiz8rZVr+eXbfOb3TpnbcGyxFXkWxL8yWy5A/KmJiMZrv5FvXnllEiOqzh2mvbwM4vSOj5KBJv5xANR9Q7sLfOQvRPck+8GTl16AJUrKUdjsto7X0Zg3taFwm/2RRiiXSDQj4EqWKfFXrSoibGlbTilvT1S83/JMncv7q9+/rhwHQnCbpLm7tl/qGLddGqW+C39/LgxUJjf093NIcoNPVr4liyq101c2ejW9zaGlK3nPVcv+uuUMR+SOAloTJpW/i+TThYg1sxCdAkPvv1SDdUUFtqLQXiJ0e1JcehtS7dAUpjg5dJ1AcfdmBS72/ptDGrO3ZF6c91aeqobBi5q6zciJDehprmNwtUvFgLfPIkRVxTR+mclZntRWEaUjsUqOm1mZAZiNw1B9JRLWqGR6joS9RIXFUDzmWGDzmM9GeXWDQSG2n9YqiBnNewTuqih5o46wscrSp29qmKHUKQ+OnakQXWjHSrM5CNeUmF5tM7th5UG0pYw8CmzbVXc7ch8SaY+Tp0jXzD4TFWCddN1x72tS5/ZilwGzdrlHaGJs30ulusY/KJvV9iLYDDiDsUW8HvsbCdedA91YU1kfZaB72DbMeZSE51tVHDNMZmRko1Y3gENobsBuBnYG+uhMcIAEO9I7FFMuYfFxTHOCOKBFXGHDttVaO8ONBX/swP/u7fOVv8pPtRhg4WIWfA1erEM/aKte67wQ52Pn2LN+lqqaUV+ptmpz1+7ZR1GzFTOPalNcSt6YJrioUwwKmOK6to/0e/EGAaCmZAfOUoHQgpRc1OB2qkn4LTGWkpOBVoqwNnycznWcJHhnL8cRWBcnzXEEDzip/Nd6b3LLb81SW1kbktiw4meDCcgjcMApFXssbV7mveguVCAPwHOyH1UrdtpqamVpuEkShsEgCWQ3MwL/eQM1Y7VxFPQ0A7Sz5tRopSTGL3m2D39TIWf3uMO0N7RvLqF3m24GDD1Lqn9D3asyGGXa1bwheJIa6LbdPwFi6TeXr5qWaiCdqr1J1ZkyrfrKlWv31rTu5RijtNDWWP2fUuY54+HxS6BDruoLv0+fHD/2xUvYdAvadsvW/qoHfYVoeZcdcyxo/If8n9gzWFjxpZi5ufB+6HsF4U1m/M+ML4OSuwjYO7o/uqtov+Mk8BwrOnMldbbsEG4gHXk/uXAxtwhivo20RLwFlVaDIOrJwcdztr+4Lr+4FrNa74mpiHhXSVkGlT7ixVPECgsYJL2RmN2NrPe/cTOvaYanPTZNutVvV/xgqosIG/DifvY1IdLl2EuXRMFB13F/BL6qurt95/UxU1gFd210BzL7E4ly1baeeiFXbw5+w4fD58EUdytiVx8eEvhgZ9WzuvX0CHD7TAd37/NCvUT3n3i2fsMHALBChB+rarqnQSqtERJRH6vuo4zoLtOKNZszjOlMOKfqQRA/Ew9+5Xsvsf7Iu49wmWrFV9exitHH6SLBXNXAx3NXAyZ8B7WrsReCaKmER7cuHV8ZUO24wx0evB6JM5q42X2gqdAnu/FWzHCvPbXxOO/z7K+Y0P/dCe+QTdISt8lqa0TCUw1FPAgL9fGSqyL7yE1CU58d9o0b1gYIV0IXe4Jn9qKv6FueEPT9+YSh0Qx0LIHx+/LKJAqOkZcJGz16+MDVxGOoPYZcU09nyWrGWm2lKrWp3Hn26bh+Gx5yv24fHwW1o7biqHyd1sR4Pk7quN9BsOrENd75N04ammtVGj7oj3ifAH9hV0nNVQI8MnWtmH1/5+d2A3e1lNcJqAIb2xTdbrrX1ZG/tmWxDrNMcs+eDXmBcHwXsGVwfHGuX+/TQbATvPx8OBy+0WwN6TNgAbw37xzquIfUxgjvHsg+jkJKebuQ4v6pTXT00C7Vrz/AMRbbzlVaFemP+0QHXnD0G0zR4WYNvyuFO2MuXpmEwX19SeU22VWhiDRZvmjl6z7/7PO3N7gP46M/u269geBP4Ppjd4/HW+kliUex56BxU/vT/walxbTL2PnzhoOz9/ijpkMz6weYDJlpYDzwTg49L5w9rF2e14ep1LJ8DNBLvFpLpYlXvyka6KQGzx0/QiKcZ0M4WYI7m4BZfyc3YtnR8/47nyeQzBrtmPSy9pWKiHWQST4o/5N09nlafWjRian6fj2dEGc2k7HnJjzjgw9/o04jB8eIfz6oL5oeDzEOHBpMCXSU3cRnV1E+esIIb4nyVFbIKR5LOE3m86lo7W+QG5seVCFr+tgN2+Rgb+YYaAPboJTWv6IEsk/7IczyFhD81w2zVP/LxTTWf2pPRkXpXTb2xUXReU7IwKcI0K0P14pLa0yjubSeZVE7P8gh1ahsCZk+Rr0veD5gefMIrVtADI2QZaPUmpl1+nVxjrT8dowT9s16RxOvUpWMstMdzj1m6zeN6uZExcKUZLuE01OYHjB4jZTF6MFIL8QhaES2LqvpqjXpFuzrAMWBMvJQvjPEa55hI5cEPD91belrmJ/YRz+510GsiK8nXSMtU5QhDINYBPUNPLdY1NjL417YP79U7Vef8qiN99QN/Vp/TaujNzdSB2sIQaDocHpbxTQkL71TJDMczPDZdiWfy7RjUdYh9216EqWDemGhxTFLT/Xr8PDx+zIGOZ2IRFJW4eAgctyXxQDLL1ezzwxxc6kQFOT+NIw9yiEyXeRa3SnXpy4E1/m/SiiXfCNTecmObQ/5CQjTQt0VXoLNBxGuDAEh+7covfruW1+QHZ7HX0w+/291FsOyHYtlfup+1532FWAsZE+IsCYa5NVMuRRZXUP4eXqAeXJlklk9KVJUZdVMjZrpGjuhAbSYdnKn6nmzVnscrHZpzcqB4bBZqH4rHJqTspFSU3j4iMaX7Gjy6fWSG4sdkKX5kpuJbsxXfk7H43qyFqS8qZKzpzCOisu+NzH58dPZjIrQ9D6uiijF666o3NtIQzuLHKgexf1evEatVB7kfX53Ve/GaFZKNqJsL51yIm6FdHZgrShnnW3wpgbAqdw0VASJKKHby9Yq44I1eqBNY9rtPeZxcd7ybXW6+T6A95i+EZTVETyucx7Rn6I0dZZ1evxcMhs+VbVdJcX4CCEsf0XGLIJDHF1I4X97p2KTz/OfB8YsXaFk6UsXxXUfB8+OX3Gi8Eu+CXYDPcclfQMBzRxm9eg8dbu2BbGYP9nRfbncgcYtEODeOUeKrHc2EBL0OU5TJ4ks16NwTPe5APHG6KTcBMF3XiLGLFBxYtedWvRJ0m2ewYNxu8dEI2W69xJdXImvB4zk6RwH45e2bdxfiOSE7fNsrf1bILcz0p11RJqskXnYbeaBLJPnIwAP61K7jE0DqzoYRS3TxwT4AUfk6er4gqKJX9JXvg1b9JEnRgEeLaQNR3cZdaguLEaXU0egRYjM5OjdwPng+pCYW3htcTnguSXv8iJrxzjzCk1HGe0dX6o2ooJx6XotRObUlFZ7fD8SzNgb4MaQ3W8HHED9G9AuwoYaV+C5j58tu9TeyahLQPiQYkYmiWipBzyAFEsx4Yo/1oJ7ffjv5+E/jqTXac6dE+6n0aGfND9Vqt5v6wDfivj37x5uLxm7u/j8J63z+58MJ9mlzUl3UI9x/w5StPPk8Jnw0kmFJ7GcofTz7j7PTi3N6hpJuKWy4v7//+J9v33CEhi2wAYVEEkJdmJvgzn89AROiAYNK1Do/efv24s1vZ4hThJHj7mB1zwrzWU8App5W9frk4uycQn2viU3V7L07+8eFfNaRbWZoNTJsTUbLg77SeD9nfFmEoAXfwJSRLaI1AxaKrkqDt5IVC0O0nmGI6RMvDHG1CUNPvKIQXxSMz2mRLw3unuSXOxSND3TH157Ty1++DgRNPP11izJFRs+iqLwnSpnBQpOTpfznsXwZsqeVuvDeu9ESQjrRre91Ojw2D8iQTvhzysT7iybWG2j3ouH5ls4yyffj0l+vKtJh+WURsHCXfk6zr3ioTPRAHyFdxJ4K9QSCJPXxd5fICqh9t0Lbbv03TEB7asuBAAA=\\\'\\nSTAGE2_GZ_B64 = \\\'H4sIAAAAAAAC/9U97W7bOLb//RRcLRawWtnxR+qZ9UwGyHQznS6mH2jSnV0YHkG26ViNLXkkuUnazf99l/tW90nuOYekRFKUnXYSYG+BtpZ0SB4enm8eSn/+09Euz45mcXLEk49se1us0mTY8jyvdXsc5kV0yQchv4nmRbiKoixcx7Msym6729vWyf4/rdYZNmPrdB6t2fn79tBnP0MXnW2WfuDzIs3YPN1s4zXP2BIuihVnb9q3vx37LE14Z7ne3TAAvcyiTbfVevf+NTv96eLsHTu/OH1xxvqtjv2nxeDPX7Jdwo7maVLwpDg6NIXWz6fv/vbr6buzsrfW+5yziEGrZBFlC/Y8XUcz9vztewYdF/GGd9kFIHra7/VYnLMkLdgu5wvA8OXrt+8vWhUmJRL/Og4J5z6iM19FScLX4YIn6SZOIiBDuImSeMnzovshT5Pu5adW6837i7Kzht4GR/luGBJxxbRKsubUT1MrwGEdJ1ch3M5h7DlMMi8Hbm4jyZjvNhukHY3Q+vXn0wt28fPLc7koz9+8ArTPzutro6j7Eyw0/8izW8kWiArbpBsYjy2jTby+Zfku+xh/jJNLdo5jsn4ArAGkzudZvC2AZ5K8yHbzIm8hx9C6El+xkgAtmsf//ud/2OI9Y+/a7312At1uws9RMLtj/74Iox9Y+8Vvnzv9Ox9uwr3vL8LZv4NW63rFM84+A8QdLm+UwADbdTyPCxYnH6MsjgBRQTo2i3ICWbRehMAjJ9hJFEBHPwA3IJMs449czCrmAJjxscCs3Q/6PmMLvi4i0Uug9b4AJkvyOE2Axwl6EAwAesuzza6ICnjQOdhyIFoOgx605Ns8XsPNwyMNg+GXjjQSLUc00iLdzda8c48BnwkKyTWN1nnKNtEWiLRes/63A4DMiziBlSUe6PQFoxTpFU9YHl+C3OwyoGmRCgZoySUhboYBSZlsV7d5jEwGDAeDp9mCZ3IFrniBeOU8w1mGff1ioF8M9YvjgIHmQFF/9fbNu4vT1xfs/Pmbt2c2u8PUkF+Je1HF7QpEtWTWX948P/1FsKxAO++yl0CclAt9csuJy4sMgaFZa7uOft/xouAM7/FOnCz4DUt4cZ1mVyyaZ2meE4GIG6uO5NjAhwlQQejW1jWPL1dFDuQABsw7z9dp/IlHoGuX6XrBF4BStslhjqewErBsxWrDi3iOwiCwf0sWgv2E2AFRjmJQFpegwivYLlmP1jJLNywMlztcqjBk8WabZrDWCaBGzJW3WupedrmNspyr68tP8Vb9XkX5CjS2uowBvyJN17m6QZpI/gZCFWBKNuo6vy3BUHULlObpes0J91zh9DwF3c6zACiyjHbrYhEDSxHwUk6zBFXzFo+3UYHIqYdv4VI8KG63qMHk/b9BfwGsMc8ikI+A/QLcHbBzDquazOH6Yrdd81brH2fvzl++eQ2axBv0BqNOb9TpDztC83Y+9r0WPhq2zv5x9voifH366uwcrifE0B5wtBeInxpf128N6reG9VvH6hbwO/ycgrSi6J0ryYNxCecJrD5KiPOfaetVVGTxDQDjhCf0j6LfFPpsAbkVycM4AV5t+6zzA5FxTOPPgZ/jRYTyo2aKfxCg7X29gfX8wOirO79ewNBHzCu78vDqvt0FNmYbwArQjirUQjCLF4OvRbC5yy/tcUr/or+FvIuqsqLxuBwzXtLjLr+BNcvbfvUE/2QceCAhiJZ2XXU06dmrm+4KWN5wEWf2EsNQ5nJ6vmNYOUTDwg88v2VB1Zd0ADqp9WfWebg/0JtwcTeCz6P1JQeBeeBBiI4xrG0RF7ehGKqdjFHCiJRCyMY6ASpJKeWtDeDtmJ2csA9+wMD9QBb4gOufRcklbyf+tGyEj2L9keQcgYrAAOQV3Lict8XluNRmk/IHjMj+zSqBr2GbgFCveSK7EKNEOeogePQD65E1A4+gjUBZeu0j+gmhB1eIoGopmu4ux6xUOS4a3Ihp31RNJ/F0yp7WyevDg8P0UA/BpOiPK7bdxh/BDgPS/KZot0Wck1Wg0C5giY8yAMhPsukE7kzZn05YDxbpNYRBftmVJIzoUUYeCBEwLwdTs1uDOyGw96o20Cn2GNAvajoFbKqLoARplY0+SQi8Kx7Z3SF1b0C0PlXUVI+0frRlqhNGSj4+hzWFluYj0v0g5HGy48YDhRw0tJCTPX6qd6TgDaZwAUw+TFkHhnhSzf/D1NnCEp4BNEn8GqQkhhJKGGKSjKc6YWAcS6w2oC3j7fq2HSleBqdT/axJkGSKiCRlRv+irESggElY8GLmN6gGA18Ij6BZPJ1cTWEyM/gPqYGoXlXzFN355kwtWhAMjl9BNUoS4eo71QsAXH2pboGbpU1B8dholNJI0JPqQlcWbh1BXFNTN2I6m6DUXxEIa0l2QWzAHh72/riKwJ4CthE6Yr+GQKOttAM+MSWhFKdKnCfYOeoATTVUikE8NdWCuGdpBXHTpRTEk306YePWCTA/IiEZgErY67ItsXJqA4ncXsm35V5O8H5i3yzyJQ88PZGxtZocMcYJ25hTmWU8ujK0BQJKiRBCXhpcTIhVukHeNTREtfhSP9i6Bbsom5KisA0gsjNC+f6+fmQPASPQA/2IyWBAFebwLLlsg2QrsSO84bbhxsB1+6ab7DYQN4FPSzJw09W8XByxz/g6hxjX+6yB3h19NgDvPFO7CATyUsFoxKtCFQCaTk2/amKgfw9N8fAe59sqOyPC/Vm0jkAlLtj7o/dPalmq/KG9UW18MwrsdrtKh0u+CLVMUns71lEXQUB1LcgMUQKKag9NEHLO1i/1Zxywj9F6x5G4XK4zB4BKjqDxhEBQ3GN92QpEsg3PFRNiYiQ18cstBAP2+z0Q9jxvy6IlBPbs9y5mPWqjbie/k4/psHxbv5QKDZFwfjtf89xFr9K+5Zwj8Sc/RcD6FrHmmMsg26NIlxcRJk6soY1gDzucENy0wWpUN7B7Q619wOwqti3vXK/iNScTTB1/sDqVN5F9Msu/w762Sv2quBKHlIRSzB6KZHH7qoqDMGkjxtGXFU00CHO7zBt1jUUXJLlSPo2uW5HAmo+fx5ebCKmot9eImHwMBciJk/3pWWUvQE2MmZkQEePppqaIds0DCgKt4fZHTMc4WbpEKsC+TGsFGHSj7ZYni3bp+7xmT564eFGNQ5Fj1Q1tzcg+MDwTxJKzB5ysSJHsgGZOjAeVORK9iOS6brc/G+h7BBBSKtQDJgjMp8XtlsNtj/LX4TaKUWN7FtBylyzCIg1BW8b0W5s6NCa+2fp2oxRg1hF2vs3SxW5ehJlI5bexD3DRwHeSHU6wQ7gz9bWh70xvOGBbS6dpK2hEmlIcKjp4QggAlaXX/nx1F8BffSQvCRELeH5l3FToWQ8W/DLjSDUMZ/QHgtSKpDp/0Lz1EUvzE5a5fqQkqhx9WnbnACM2C6r7yBRwm3jDvKv4BpddOiAtQdhHyO/IrQxpTMng7pLS5IpdzXLS+WOkfuRuyhAjQjK0M/n/vNJ9euSDxP4MEj8DiDuffc+GjgioElSpJDE9FLEf2MzHdIi8mKuLGV3oXNjp41haF38BtiFP7Gm/ZSKudLXgRol8xYUuDS79TSB5wRdt2VJv5JO3OekFbDh1au4qoBuh0ppOdd3kBuwHbKQA76Gh0vl8t43BFKFgYM6eRIpiT0IXERwKknilxN1LkkV7EOdqvg1yLQnTINzVhUvKh40SLhfusFz3HXJsBlqfaxGSpbh7QR2iGY8SJF+nBY0GHADrNpg2wIT5ls9jjqDGejnANcUuB24TkuCOBuJHX/0YGArdVOrTP6zDNNkBy5VBOEWmuMyut6s4RbjflhMe2F65imLAQb0Ax3GX0FYoqK+y95yl4ASCQBBdUSBTVlzD3wziPJ6Xru0uiZVxz3khvaeRdAmk967vMMSUyah8L/BSZhiRab4XSR2soOaQLuMsx66EB62UQC9AbYR9ai5IzsFDXdiwJZodQpP609NWgKnyW8TDQHZkhruoSTFqoDCzp0su3G1cKIpB2sIRVTfH7J4rFUg1BaqQ51riC7fTgpap61HHo7usBtERLI3Gk7bsDKKQKgiJQH8+qYBKwrihZQbRN6c8ely9PpJej7EFuF8sZPyzTrOQ2MHl/AuXTfLea2C8jG95VJwoJi77Cz+KYhpzK6FpucvrgOlpAR0Z07cvG5hbcMbOwm69DqWyKEw0aqnbevbcngfmdjOMFetPPuCTlivZlZlhoz4dKwt8MCdczfG+meFaC0kXMxlFiWKDVMRCz1oyUsa9fr6QwRaKkamjKmqIqopGDCr1BMNGa9KAom/QShOrm2pe+W5Wyz5ODHRxYWSqnUhG3U9r5IzLZw7qxWbeHAb1wWPDKSgkfTsEF7eVGrTwN5LKRjdE28bkpZJvRRkpy2ihA3YcsGdyCW1vzbFX56CRuW+pBnHwkHqk7Xg/TFR6H59v5Pb5yoC24j3NEwpcjFjFhI5VNOwWJgCq3fdGZqQB1ZJ/mZ+m41qHtUYEeOvOHi9PFI2Fe5w9mmoofBEVlwub7vL6iCAWtLTuj+Ik6tj5T9RdA41mb9H/LwoIRo0BQeMSNcQFzzSICMS4YobK46yFChNaKdpLwx8zX+hE8nLqNtJ2tashpNNtdY5R7L6EvaFuDoyAOtaapRLQ0OB79KQAUD38f5P1eFluIgitxdHJESl/TIC4am5BWQGv80fJgAhmFI6WkIsx+ZSBrpWk/0l7U+qO4T6bznOyBJ0pOpso8RGKMomMJ6X4TO3QQBtKVKYswRNIImNjfr1u99j3J8Bz37PXWvGJ1tQ0D2pgwSDTiTZDgcBVTOregBOCKm0rGCMBc+LK1WkGhOyVjspknCy1fV2YuPU8WY6r59g3PCcEJvtzqVUjsYej7x5YbmaytDwV0eKJyFDJDCvODQeq8qt2hota1emhNFgtI1bFS/Wl0TuwNGE9ZlXk0E3WtB6uSjDTWE2bsdKZzYjNZGzrjOZERNfUUtrEmimKYvBe/oH0O8uyNGvj3Mt9KiXuIYr7rRjKEMmW3FsImwUxUHHabpPsBSOJVQ6hIDVYGY35XdZHEHHNl2bIVPqVdWUSoI2xUKadDoNHo4pHYTBfcy0zLKi+/2CYl63NvT7erHE8mLVGA8M2mDsFRmCIFJlEGAACMPyYzPA34Q4/D051H1p17qA6dUNXu9aSHA9tKtITmTbOwWY/RdIbjLoNNG/MvFNj2C+GrKbx0JZUHV+gkwsdPJ6A5xywNvwxDCU4H2mCRx5kaia84vdIz5iV3GbFg5mCc+Vr1OLDyu6qgZNoww+PbBd8NPu6Ld0TQDKGsMpheRJEsEh5OWZmuXrQoKfQ288y1DghbTILVFtW0kqYszSP1YbyJFZbd3aYpgYUpSpoMZ72pd9Q2sWv7KjTn2qegji3VYWTD+cVlAeb0LiYzohpmL/K/ousaGyl1bTEiE4hYy9Vu62s/r1yVmZbK2tl9qBR1hkkuwPlpmB5Ytx0bFDce6O6BKaDTOVkCBqDCPp/bxMUxxLcXQpH9NeOloCJCJh+PZtOGxvqgRoN44R00ODOSiLSJV/v99l0EbJkChpadygZU2ei1n0X/bF2sMy1pGVUvxuhcRlpBfV10X26sgvnfliZ5ajSVi5iVZmrA1kLc50aXeMHXS45CSP3tn8SX6HalDM/qXSOsR/i8vGnVn5VOvr7u7D8//8idXQoKacl5uqcLHKT7jZyyvVG4sE9RjooBQQ8vc/4B7sS0I19lRIlfx1WcEJucq04uyngcucFyZGsHJ8ywVk6DFraSMyRWiCOJq+YMx5XzpMxf7XDW1Uv6RmxMM1mcRGCL53xHMaiIrHQ9qpkWlW/5UpdOox1Ux6Tiu1rOkIHN7xRaOHwTg1quAZvGqZSg2Z20HC9ZebW6ZF/yWD+pDe9C7xGy0vqynuIcfpTM6dMAA7ZNtrpJFed1tvUxzuYQt4b4jclqlFyas/iRGJEVQ/lVgbp1eYcq1Uo7o62fWMaBcSSuADlFoHJYt5LCSFfRhDN2EUYYeGk2CZWu/4+M1fba7/4rdP3BfysLXIHVhPP5slHyQofvXmMoDVfRYNno3AZY7U0Hgul85tmfLgCmsmD4F0BL3f7r+NiJY6qpmAg214283wW5QCcLNbcDGvmq11ypfIC7XW0mS2isYTsZjxatPvs++/ZAE/WzDzPykOuurst5vXb1I1RC7fqrvjNIr7koIhVOHydwSghHr7VJhWwdPZhjP/wuR0BR9cwRzqsu9httnkbgDAex/MUJ4OASiZQqPMTrJkOMLOcXoMiS06oCNzvgnJNF7zt7Ypl51t5HpYIIzCZ3WKJBIxiYm7SFB/vnUt4+emPTCfn24jOZ+QnbS/wAuaNPf8r50Yrjy8K6L6Af35C7kEWQtV+ggdHEE3oewMNT7zrmRdQiTTYqXwN6mB98ld4hi8FOOk5GUayBU29RjabZRWhkIkcdCLSCCqNLdyJa7dESy8rAEmaKqicEzVZF3ISDSLvOgXGFQAKjZxjmUD8iasKnVoirkpuiKItvJZQZS2XUHbetNoydylDb2o1qLTj/oaaFq1XVj206noVxclj6K4N9NumNxeEhliUp901BqhOZdFhCU5Va/hCIfynXVJRNuxurvCwPMgLKAAlGHQuPkyv6FK6iNsMN0a8E489Yf1ez9dvivJk+VIlOgTfGbCzf54+v1AvITk9fcfevnvz97PnF2/esV9e/vju9N2/PP9Qz0tP1v0yBn6lfG/FnWeCEF3wBoBUNLKhxJQFVDV9E8pbgbG+BlIAUP3NTN/V3sZkNPaN0pQKkfItAwE4UCpPK3DGfpYppSoNzGX6HW+ACkA7rimCCtBXq3zJ+wBkaAUb6joVeTwBPfHour7XSLdpl/F4MOjLIhp6ZwO9XATPqgVSxaDcfb4TIA/B8rKjF71x9e4luQVcHTZTbznqPuywqls5SytTHZDeovmWqhHrSfv+uHZQqK/5au0B1ZvWYAY6DJaWA4xdxE63NbBhwHoOMLptgg0dIw6tEUdVVyNtxJEONjJGLMFG1Yhy8fnNVtQklJ5z7iRVv06ZQZ0Q/fqk+/UJjurzeVbH/ZnCU+PyCi/wBMb32OJTbh2AB+rdYSqxg5wIhnujikLvUHIc9NAEaeK96IWO0XLPZDD0LWBImIXYmS5RqI4n1scxkbnXoBNvi5RZeMY5OakPX/TY29Pz83FNDiG24TnTFrwfDIJ+MAqeef7Dq4S+QuBXDvYdjVrCjsr3jc1TvlzG8xhfvgVcv8IXXT0sCpJ11JJPBEdP646Ifc5j6E/NYtKqC5IAdxeOKkXo61swIOV1B0sdj30tQp3YzzRoaj0N9OLW68uh2schdEiuaugI1NFNBjYTTT6XvX4DRKAgxhj5uGeNPOr5pgDe6PVQ2KleLEUSpI3YQL2eE936AlgHbcwJ9+7dg6nngPFGYVXo2bBe/YFjwUzSHLjxxYvcr/HIcLT/Dtz4ZuAewYlOrbU5pOiu3uYPDGD21zTJ+9PIid89ZjnV32vl4seRm5ucXOPuZNTIku5ODP3el68e1bShbVK8Xy/DPmbd+0dDPQMHtweheqOCeP5t7XmRRUmu8moI1OkfDY4tsKHRzTdHoB9qEK6Ojh1gq4xzcTBYoDQyYJQjJJ85Ho3CslhSJegakm5O6uo5N2khx2QfdZ9Cmcm+YSZFOKTZKipmrFur/DFs5WDsrpukUpkjei/IfMXnVw9tI6n/MOP5bl3kIjS4p+ekn3iAPvCokbPAp/aOKgl+ck8nzsBwovwrFBH9bRsE5VevV/kz+5lnG2DWOXA1A8YtOkQ/lgKbsAXH91jGCb5GdC6TpeLcSrfsYX/9EWH2paeIZDmRdj4t2kAX6nxaPgEej0FzDUQeXp2kmdrwT8sGHRf8eFovdhItx7VzI7dND/WXijXXVd1S0NkEcEuFV5Jhz9SbchXxI9vxaxiumozlBZjBBFoNv/EW4Wk6CI8z5gB9TNeYHTGoof0H1WuZQyHetu63ZAq3SwyBqCvQEJld9abrPgJaVVKxH/Cg8hwYytPSWHmgy1/gWvPH0KLDMcvn+FZmeod0+XLg6l3S5TbmA6vRsl9tezMfqzfHTsx6M1xi+USln1AUMYlDvjUmc8y9CdpIVc8mXprFYjsVVpse1UoUJFvTw4knXAx8D+CaF+GSg12dYUEBJsVq71ihV4vQNjKqNRxU9iLeC25uMNcGNohRHry1j1dU/ftNSoeOlVV1bnhyqgkUy+ipW5IxkMGnfW2sqpNmRPVVm+TVOtGLbFpNWOmtxCngbwfGObfdxg3cpWQA2FDZ6pvRcNBSJ6h/36nEfLiKqc6k4hWrKnEfmFYBqL+kRlY81rbciVKijtONsrT6mtmnMxI4R1nJuJfkdGyiBO7sB5bb8WDnoVE7WQbQvHpabWYLAHdZq2xkOx9m4zixyh5EiiCQma5AprcCmcAKZNbK8bIWxYoV5prL1DLNya0ewlctKqtd54KJgffUfMOSmyMcTexFb2k0bSojaqyp1ZhJ2Xu7/9rpgdJllzOUnFtPOg579azjX3v1tONxr55kHPTqWcaBbsFKLDR8GzHpfzMcDr6tY9Mf9o8dCH0zcGB07ETp2JHmpJ226jUFFheUQugb0aVGzKa+3CzS0J9FFtNlGYpvPfQrToC1BnML92zfReJVMxnjPWo00GuZCuBd/WsUGiAWVR1Wr0ENmaoBTi2VB+pMJUAxBmZwRRgUwI9KO+5ZFWcVlEXMrxjxwNo5Rz3gvVWAL4bSjVOfVqh9RAEPJPDFd1rZiQcWKwCTJR0r/VMFjHiBy7LKR/DwjscQB0cZ+HXzNPmwuxTHDunkBG4/AWGAULcP7N2R9pvdhrpj89kMA9sIM6nV3039sUBuKT8/Ic6qgrLV1sQyzTqYMWo98lY0qDytzs1+yyqtYNnONU69e9L/ZGot0EkJMq1bWiSIM7YXL3XW+z6UBDD6rCfa7M5cp56UAqMX4gMDhRoDiXIvW3/N12lO6nXBs/Bync5AHeHMw+U63tYjJqvOcZmJ42pg5MOEQz8JRFubqJiv6k0BEvrWYkE92/TlodmxCs0oDNI+foIVPSzSk1kdkflDObbkilOLR4jSfsWimVrYKD+7REk3/XMtDyzKcphwG91iaYy54BuOh13sVwnKWgZ4ICsZrFf+XWbpDtlB1FN4jS8ErJdnVzV/KR2Pw6yR+loAX3P87FGOMRp+HAZswSq9TPFTLfVyzypNwFJ8OVPtNa9j9vbkx/aPv12wH/3fwPuGX2Yv9nsM8cCjPIcIDIEFkoIEjkmcRfMVe49fjHn/RHyZJp7RN20iOlUoXs2Py0qnCWWFoi+zbo65UAn6d+pUOafZ59GGm/uYQBb8hlBarEDPLfAlVHvnI50Wqq9A3wEMrlZr4QYW5Rt0ol7WdAROY6t8+RrrOMqKn4jN4YYarPprcqwkrHQJ7FxszQm4s5i9wBLJqn4GP3yx//tgntGBKGfRihb1ngNbqPxWFYE+pKCZdbpahboFR7FHmM9Tqvp1cOxz4S7sULywbEjzHdXXoeYr0P2Jy9dwcKxIz5j6PxdfgsOqwjXHcq4rXhwBmeBOtt3lpImL63jOXR0iBuT+Ciy/o0rej/ECuImsegctEHhl0reoLA1DS+PqUb7wHiR1xufRLhdiZaLMKkmnr5vltxtYqCyeH5At4fRseJTg4bSx60zU0746VxShVoNpSDX33nX4qUPQqizcbvHE1aRH/UekAqNZTl/0SsSX4TCCsM8qPbJ2sFK3Uj7oNEYlN11UCIfaVMNoN9xqSPgwwt/MdTWgBNGhBPZ97s/TGtsKAGvVqm4DU9ZLl+EdFsl18PhSVxZxUrGbSk0ZlW9Vb7WX7OnNJnKa06aE23xy74Mn4iDLnEoY9gxB2TltlLKuNjfUINWIAyFvCtDHE6HiaFcYvQPyUqWDYLqjz9THj7J0J896Ne8BaJaGTm/ISyNq1lZSgtFv//DJnAeIsl2utdKA7v7xbcJfNsZB9/eZdH//fv7m9RHWUbOsZETtDTUZn6elHfDKQ4vRNreLb/FFkaIoV742Tnzm8iFsGp4ZQNVVfdHNdiDFN1dxJW6BHqI7G0R+2A5PicqfXfWjbauva4j0cVohUl9O1wJR9bSI2PO3tor2on6vF6LNiTNaByrDd+skpVEtqqC8HNazhxSsKIYdaxWwDYpRrHANC7EtUpcusAPz9Q7L7EtG3idy1gEvp7Q+s8BlFsm9peKW2n2S+0BJssMi3DzQF8uxEJroRjqhrveMlSDuQ2CjJjto7QeUeTcHD9hpwdzpv/zhjKDL6TDp94WD3jcpaA2s08lReDQ+UJekH4iTQkH/68FQvHbMp+b3OL3E/b6OK97aI5IWuSv/4h7jl06P82WGzZZ1D7VVOPLZjlWkaDnfvODZBbzVt4Vh+kET/AvgdeEuY0SDFk/mwpxtrNfVawVQ+uKDLw1W9HKl6mKGT4euzrS336vSqSqyJEZW7QmvkasPTEXv/7CvyEnj9331VJHVl3UsG0Og8AC5RZKPub+oqz66Gze8sYFIv0x3WbHqUJDKLnmKAdQtw/qauNgt3Avg/NAuUiffzdSI4sO7rtY/H6Mjve7kWyyrmu9mvDNDdyei1F9eNFPlzjqfmoFNKT9vYfHp5XYHKnoe541pID3a1XcGVGJPpucxbbO+xYRXRkO581kl89KLxz9xFkH4muYFG92MuuJAD8awGccjVjnbga3KIVqM+eJArIrfURM7WA2pAe3LytJPFHPRPktP4YsSB/qUdMkvrlje8VFmlAMwUsDKKZXEkPcSyBdLoG3HVCB+IbghNzDHD/HibOn01sBnC/yUVcrkp17REDPK9CbgAWKeSaQuUFc30+fg/pHu+zaGk44vwXtGw1ouSe8xUGDG8TXr1NqrV/YpNO0kqWw/UV7ftPmYrO83jLH05FE3PHWmmy77dNrz03d/OxeH0yr7YgNJhBFIn2oT2PnPp2CENGAgmA376+kvv1y8fHUGUNKDH3cHyzuWm+fiAKo81/fi9OLsnAIjr2na1fK/PvvnharskjqxQRNilgVChIDN+GWcYAJIHF/VdguRYU02zdAE5HRAEFieY0TSLXcMW/GSheKVRCFFzWGIQh6G8p0x9CnwDD/sJz8L3j3NLneYI3pLT9paBaf4gDyMc+KdabIsDJ1Vaav0FMrmv44VOtWI3WixCCM5VNvrdERsE+DHvPmJONkpv2d8Yn21em83QoI6IEH7+9K/kSxTJNklKI1wl1wl6XUi3vWPI9B/Id3EkdTBIjqNitddQiug9t2qW7/1f0amXVjugwAA\\\'\\n\\n\\ndef sha256(path: Path) -> str:\\n    h = hashlib.sha256()\\n    with path.open("rb") as f:\\n        for chunk in iter(lambda: f.read(1 << 20), b""):\\n            h.update(chunk)\\n    return h.hexdigest()\\n\\n\\ndef decode_source(payload: str) -> bytes:\\n    return gzip.decompress(base64.b64decode(payload.encode("ascii")))\\n\\n\\ndef default_stage0() -> Path:\\n    candidates = [\\n        Path("/content/Y4_STAGE0/y4_ordered_transition_words.json.gz"),\\n        Path.cwd() / "Y4_STAGE0" / "y4_ordered_transition_words.json.gz",\\n        Path("/mnt/data/Y4_STAGE0_TEST/y4_ordered_transition_words.json.gz"),\\n    ]\\n    for p in candidates:\\n        if p.exists():\\n            return p\\n    return candidates[0]\\n\\n\\ndef default_root() -> Path:\\n    return Path("/content") if Path("/content").exists() else Path.cwd()\\n\\n\\ndef run_checked(cmd):\\n    print("\\\\n[autobundle] RUN:", " ".join(map(str, cmd)), flush=True)\\n    subprocess.run([str(x) for x in cmd], check=True)\\n\\n\\ndef validate_stage1(path: Path) -> None:\\n    if not path.exists():\\n        raise RuntimeError(f"Stage 1 did not create expected file: {path}")\\n    with gzip.open(path, "rt", encoding="utf-8") as f:\\n        obj = json.load(f)\\n    words = obj.get("words", [])\\n    if len(words) != 4221:\\n        raise RuntimeError(f"Stage-1 manifest has {len(words)} words, expected 4221")\\n\\n\\ndef validate_stage2(outdir: Path) -> None:\\n    summary = outdir / "y4_stage2_summary.json"\\n    cards = outdir / "y4_link_tensor_cards.json.gz"\\n    library = outdir / "su3_local_haar_projectors.json"\\n    for p in (summary, cards, library):\\n        if not p.exists():\\n            raise RuntimeError(f"Stage 2 did not create expected file: {p}")\\n    data = json.loads(summary.read_text(encoding="utf-8"))\\n    if data.get("passed") is not True:\\n        raise RuntimeError("Stage-2 summary does not report passed=true")\\n    counts = data.get("counts", {})\\n    if counts.get("unique_link_token_signatures") != 182:\\n        raise RuntimeError(\\n            f"Unexpected Stage-2 signature count: {counts.get(\\\'unique_link_token_signatures\\\')}"\\n        )\\n    representative_count = counts.get("c_orbit_representative_link_tensor_occurrences")\\n    if representative_count != 187632:\\n        raise RuntimeError(\\n            f"Unexpected Stage-2 tensor count: {representative_count}"\\n        )\\n\\n\\ndef main(stage0: Path, root: Path, force_stage1: bool, force_stage2: bool) -> None:\\n    print("=" * 100)\\n    print("Y4 STAGE-1 + STAGE-2 SELF-CONTAINED COLAB AUTOBUNDLE")\\n    print("=" * 100)\\n    print("version :", VERSION)\\n    print("stage0  :", stage0)\\n    print("root    :", root)\\n    print("hardware: CPU; A100 not used")\\n\\n    if not stage0.exists():\\n        checked = [\\n            "/content/Y4_STAGE0/y4_ordered_transition_words.json.gz",\\n            str(Path.cwd() / "Y4_STAGE0" / "y4_ordered_transition_words.json.gz"),\\n        ]\\n        raise FileNotFoundError(\\n            "The completed Stage-0 ordered-word manifest was not found. Expected one of:\\\\n  - "\\n            + "\\\\n  - ".join(checked)\\n        )\\n\\n    stage1_dir = root / "Y4_STAGE1"\\n    stage2_dir = root / "Y4_STAGE2"\\n    stage1_dir.mkdir(parents=True, exist_ok=True)\\n    stage2_dir.mkdir(parents=True, exist_ok=True)\\n    stage1_manifest = stage1_dir / "y4_channel_denominator_manifest.json.gz"\\n\\n    with tempfile.TemporaryDirectory(prefix="y4_bundle_") as td:\\n        td = Path(td)\\n        stage1_script = td / "y4_stage1_channel_denominator_manifest.py"\\n        stage2_script = td / "y4_stage2_exact_haar_library.py"\\n        stage1_script.write_bytes(decode_source(STAGE1_GZ_B64))\\n        stage2_script.write_bytes(decode_source(STAGE2_GZ_B64))\\n\\n        if force_stage1 or not stage1_manifest.exists():\\n            print("\\\\n[autobundle] Stage-1 manifest is absent; rebuilding it from Stage 0.", flush=True)\\n            run_checked([\\n                sys.executable, "-u", stage1_script,\\n                "--input", stage0,\\n                "--output-dir", stage1_dir,\\n            ])\\n        else:\\n            print("\\\\n[autobundle] Reusing existing Stage-1 manifest:", stage1_manifest, flush=True)\\n\\n        validate_stage1(stage1_manifest)\\n        print("[autobundle] Stage-1 validation PASS")\\n        print("[autobundle] Stage-1 SHA256:", sha256(stage1_manifest))\\n\\n        stage2_summary = stage2_dir / "y4_stage2_summary.json"\\n        if force_stage2 or not stage2_summary.exists():\\n            run_checked([\\n                sys.executable, "-u", stage2_script,\\n                "--input", stage1_manifest,\\n                "--output-dir", stage2_dir,\\n            ])\\n        else:\\n            print("\\\\n[autobundle] Reusing existing Stage-2 output:", stage2_summary, flush=True)\\n\\n    validate_stage2(stage2_dir)\\n    print("\\\\n[autobundle] Stage-2 validation PASS")\\n    print("[autobundle] Stage-2 summary SHA256:", sha256(stage2_dir / "y4_stage2_summary.json"))\\n    print("\\\\nALL AUTOBUNDLE GATES PASS")\\n    print("STAGE1:", stage1_dir)\\n    print("STAGE2:", stage2_dir)\\n\\n\\nif __name__ == "__main__":\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument("--stage0", type=Path, default=default_stage0())\\n    ap.add_argument("--root", type=Path, default=default_root())\\n    ap.add_argument("--force-stage1", action="store_true")\\n    ap.add_argument("--force-stage2", action="store_true")\\n    args, _ = ap.parse_known_args()\\n    main(args.stage0, args.root, args.force_stage1, args.force_stage2)\\n\'\nFINAL_SOURCE = \'#!/usr/bin/env python3\\n"""\\ny4_final_endgame_autobundle.py\\n===============================\\n\\nONE final Colab command from the existing Stage-1/Stage-2 outputs to the exact\\nO(y^4) flat-band verdict.\\n\\nRUN\\n---\\n    %run /content/y4_final_endgame_autobundle.py\\n\\nUse a standard Colab CPU runtime. The A100 is not used.\\n\\nThe bundle reuses valid outputs and only builds missing stages:\\n  3B/3C -> 3E -> 3G -> I -> FINAL J.\\n\\nRerun the same command after any Colab interruption. Stage 3G is checkpointed.\\nThere is no stage after J.\\n"""\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport gzip\\nimport hashlib\\nimport json\\nimport subprocess\\nimport sys\\nimport tempfile\\nfrom pathlib import Path\\n\\nVERSION = "2026-06-13-final-endgame-v1"\\n\\nSOURCES = {\\\'y4_stage3b_stage3c_autobundle.py\\\': \\\'#!/usr/bin/env python3\\\\n"""\\\\ny4_stage3b_stage3c_autobundle.py\\\\n=================================\\\\n\\\\nSelf-contained Colab bundle for:\\\\n  * Stage 3B exact time-ordered SU(3) irrep state graphs\\\\n  * Stage 3C exact rational Casimir channel projectors\\\\n\\\\nRUN\\\\n---\\\\n    %run /content/y4_stage3b_stage3c_autobundle.py\\\\n\\\\nThe script:\\\\n  1. locates the Stage-1 channel/denominator manifest;\\\\n  2. reuses a valid Stage-3B output or rebuilds Stage 3B;\\\\n  3. runs Stage 3C;\\\\n  4. validates all headline counts and output files.\\\\n\\\\nUse a standard CPU Colab runtime. The A100 is not used.\\\\n"""\\\\n\\\\nfrom __future__ import annotations\\\\n\\\\nimport argparse\\\\nimport gzip\\\\nimport hashlib\\\\nimport json\\\\nimport subprocess\\\\nimport sys\\\\nimport tempfile\\\\nfrom pathlib import Path\\\\n\\\\nVERSION = "2026-06-13-stage3bc-autobundle-v1"\\\\n\\\\nSTAGE3B_SOURCE = \\\\\\\'#!/usr/bin/env python3\\\\\\\\n"""\\\\\\\\ny4_stage3b_time_ordered_state_graph.py\\\\\\\\n======================================\\\\\\\\n\\\\\\\\nExact SU(3) time-ordered representation-state compiler for the O(y^4)\\\\\\\\nflat-band program.\\\\\\\\n\\\\\\\\nRUN\\\\\\\\n---\\\\\\\\n    %run /content/y4_stage3b_time_ordered_state_graph.py\\\\\\\\n\\\\\\\\nHARDWARE\\\\\\\\n--------\\\\\\\\nUse a standard Colab CPU runtime. The A100 is not used.\\\\\\\\n\\\\\\\\nINPUT\\\\\\\\n-----\\\\\\\\nThe script recursively locates:\\\\\\\\n    y4_channel_denominator_manifest.json.gz\\\\\\\\n\\\\\\\\nnormally at:\\\\\\\\n    /content/Y4_STAGE1/y4_channel_denominator_manifest.json.gz\\\\\\\\n\\\\\\\\nOPTIONAL VALIDATION\\\\\\\\n-------------------\\\\\\\\nIf present, it also validates:\\\\\\\\n    /content/Y4_STAGE2/y4_stage2_summary.json\\\\\\\\n    /content/Y4_STAGE3A/y4_stage3a_summary.json\\\\\\\\n\\\\\\\\nOUTPUT\\\\\\\\n------\\\\\\\\n    /content/Y4_STAGE3B/y4_local_irrep_paths.json.gz\\\\\\\\n    /content/Y4_STAGE3B/y4_recoupling_blocks.json.gz\\\\\\\\n    /content/Y4_STAGE3B/y4_stage3b_summary.json\\\\\\\\n\\\\\\\\nWHAT THIS STAGE DOES\\\\\\\\n--------------------\\\\\\\\nFor each of the 182 exact local link-token signatures, this script constructs\\\\\\\\nevery time-ordered SU(3) irrep path\\\\\\\\n\\\\\\\\n    1 --event0--> R0 --event1--> R1 -- ... --event5--> 1\\\\\\\\n\\\\\\\\nusing the exact multiplicity-free fusion rules with 3 or 3bar.\\\\\\\\n\\\\\\\\nIt then:\\\\\\\\n\\\\\\\\n  * records the irrep after every ket/operator/bra event;\\\\\\\\n  * records exact dimensions and 3*C2 values;\\\\\\\\n  * builds the finite feasible-path transfer graph;\\\\\\\\n  * reconstructs every Stage-1 global energy signature from the correlated\\\\\\\\n    local paths and requires exact equality, orbit by orbit;\\\\\\\\n  * compresses all 16,835 valid charge-conjugation orbits into 1,933 exact\\\\\\\\n    recoupling block types, including the C-odd orientation phase;\\\\\\\\n  * separates resonant, mixed, and nonresonant blocks;\\\\\\\\n  * reports the exact path-space dimensions that bound the next contraction.\\\\\\\\n\\\\\\\\nIMPORTANT LIMIT\\\\\\\\n---------------\\\\\\\\nThe representation-ring transition matrices are exact incidence matrices, not\\\\\\\\nnormalized Clebsch-Gordan or Racah amplitudes. Stage 3C must attach normalized\\\\\\\\nintertwiners / SU(3) recoupling coefficients before any O(y^4) weight is claimed.\\\\\\\\n\\\\\\\\nThis script does NOT compute fourth-order amplitudes.\\\\\\\\n"""\\\\\\\\n\\\\\\\\nfrom __future__ import annotations\\\\\\\\n\\\\\\\\nimport argparse\\\\\\\\nimport gzip\\\\\\\\nimport hashlib\\\\\\\\nimport itertools\\\\\\\\nimport json\\\\\\\\nimport platform\\\\\\\\nimport sys\\\\\\\\nimport time\\\\\\\\nfrom collections import Counter, defaultdict\\\\\\\\nfrom functools import lru_cache\\\\\\\\nfrom pathlib import Path\\\\\\\\nfrom typing import Counter as CounterType\\\\\\\\nfrom typing import Dict, Iterable, List, Sequence, Tuple\\\\\\\\n\\\\\\\\nVERSION = "2026-06-13-stage3b-v1"\\\\\\\\n\\\\\\\\nIrrep = Tuple[int, int]\\\\\\\\nTokenSignature = Tuple[int, int, int, int, int, int]\\\\\\\\nEnergyTriple = Tuple[int, int, int]\\\\\\\\n\\\\\\\\nEVENT_NAMES = (\\\\\\\\n    "ket",\\\\\\\\n    "insertion_1",\\\\\\\\n    "insertion_2",\\\\\\\\n    "insertion_3",\\\\\\\\n    "insertion_4",\\\\\\\\n    "bra",\\\\\\\\n)\\\\\\\\n\\\\\\\\n\\\\\\\\ndef default_output_dir() -> Path:\\\\\\\\n    if Path("/content").exists():\\\\\\\\n        return Path("/content/Y4_STAGE3B")\\\\\\\\n    return Path.cwd() / "Y4_STAGE3B"\\\\\\\\n\\\\\\\\n\\\\\\\\ndef recursive_find(filename: str) -> Path | None:\\\\\\\\n    preferred = [\\\\\\\\n        Path("/content/Y4_STAGE1") / filename,\\\\\\\\n        Path.cwd() / "Y4_STAGE1" / filename,\\\\\\\\n        Path("/mnt/data/Y4_STAGE1_TEST2") / filename,\\\\\\\\n        Path("/mnt/data/Y4_STAGE1_TEST") / filename,\\\\\\\\n    ]\\\\\\\\n    for path in preferred:\\\\\\\\n        if path.exists():\\\\\\\\n            return path\\\\\\\\n\\\\\\\\n    roots = [Path("/content"), Path.cwd()]\\\\\\\\n    seen = set()\\\\\\\\n    for root in roots:\\\\\\\\n        if not root.exists():\\\\\\\\n            continue\\\\\\\\n        for path in root.rglob(filename):\\\\\\\\n            resolved = path.resolve()\\\\\\\\n            if resolved not in seen:\\\\\\\\n                seen.add(resolved)\\\\\\\\n                return path\\\\\\\\n    return None\\\\\\\\n\\\\\\\\n\\\\\\\\ndef irrep_dim(ir: Irrep) -> int:\\\\\\\\n    p, q = ir\\\\\\\\n    return (p + 1) * (q + 1) * (p + q + 2) // 2\\\\\\\\n\\\\\\\\n\\\\\\\\ndef c2_num(ir: Irrep) -> int:\\\\\\\\n    """Exact integer 3*C2(p,q). Link electric energy is c2_num/6."""\\\\\\\\n    p, q = ir\\\\\\\\n    return p * p + q * q + p * q + 3 * p + 3 * q\\\\\\\\n\\\\\\\\n\\\\\\\\n@lru_cache(maxsize=None)\\\\\\\\ndef fuse_fundamental(ir: Irrep) -> Tuple[Irrep, ...]:\\\\\\\\n    p, q = ir\\\\\\\\n    out: List[Irrep] = [(p + 1, q)]\\\\\\\\n    if p > 0:\\\\\\\\n        out.append((p - 1, q + 1))\\\\\\\\n    if q > 0:\\\\\\\\n        out.append((p, q - 1))\\\\\\\\n    return tuple(out)\\\\\\\\n\\\\\\\\n\\\\\\\\n@lru_cache(maxsize=None)\\\\\\\\ndef fuse_antifundamental(ir: Irrep) -> Tuple[Irrep, ...]:\\\\\\\\n    p, q = ir\\\\\\\\n    out: List[Irrep] = [(p, q + 1)]\\\\\\\\n    if q > 0:\\\\\\\\n        out.append((p + 1, q - 1))\\\\\\\\n    if p > 0:\\\\\\\\n        out.append((p - 1, q))\\\\\\\\n    return tuple(out)\\\\\\\\n\\\\\\\\n\\\\\\\\n@lru_cache(maxsize=None)\\\\\\\\ndef fuse(ir: Irrep, token: int) -> Tuple[Irrep, ...]:\\\\\\\\n    assert token in (-1, +1)\\\\\\\\n    return (\\\\\\\\n        fuse_fundamental(ir)\\\\\\\\n        if token == +1\\\\\\\\n        else fuse_antifundamental(ir)\\\\\\\\n    )\\\\\\\\n\\\\\\\\n\\\\\\\\n@lru_cache(maxsize=None)\\\\\\\\ndef full_irrep_paths(\\\\\\\\n    tokens: TokenSignature,\\\\\\\\n) -> Tuple[Tuple[Irrep, ...], ...]:\\\\\\\\n    """\\\\\\\\n    Return all exact time-ordered paths ending in the singlet.\\\\\\\\n\\\\\\\\n    A path has length 7:\\\\\\\\n      state before all factors, then state after each of the six events.\\\\\\\\n    """\\\\\\\\n    assert len(tokens) == 6\\\\\\\\n    assert all(t in (-1, 0, +1) for t in tokens)\\\\\\\\n\\\\\\\\n    paths: Tuple[Tuple[Irrep, ...], ...] = (((0, 0),),)\\\\\\\\n\\\\\\\\n    for token in tokens:\\\\\\\\n        nxt: List[Tuple[Irrep, ...]] = []\\\\\\\\n        for path in paths:\\\\\\\\n            current = path[-1]\\\\\\\\n            outputs = (current,) if token == 0 else fuse(current, token)\\\\\\\\n            for target in outputs:\\\\\\\\n                nxt.append(path + (target,))\\\\\\\\n        paths = tuple(nxt)\\\\\\\\n\\\\\\\\n    singlet_paths = tuple(\\\\\\\\n        sorted(path for path in paths if path[-1] == (0, 0))\\\\\\\\n    )\\\\\\\\n    return singlet_paths\\\\\\\\n\\\\\\\\n\\\\\\\\ndef local_energy_counter(\\\\\\\\n    tokens: TokenSignature,\\\\\\\\n) -> CounterType[EnergyTriple]:\\\\\\\\n    out: CounterType[EnergyTriple] = Counter()\\\\\\\\n    for path in full_irrep_paths(tokens):\\\\\\\\n        # Path index 0 is the initial singlet.\\\\\\\\n        # After insertion 1,2,3 are indices 2,3,4.\\\\\\\\n        energy = (\\\\\\\\n            c2_num(path[2]),\\\\\\\\n            c2_num(path[3]),\\\\\\\\n            c2_num(path[4]),\\\\\\\\n        )\\\\\\\\n        out[energy] += 1\\\\\\\\n    return out\\\\\\\\n\\\\\\\\n\\\\\\\\ndef convolve_energy_counters(\\\\\\\\n    left: CounterType[EnergyTriple],\\\\\\\\n    right: CounterType[EnergyTriple],\\\\\\\\n) -> CounterType[EnergyTriple]:\\\\\\\\n    out: CounterType[EnergyTriple] = Counter()\\\\\\\\n    for a, ma in left.items():\\\\\\\\n        for b, mb in right.items():\\\\\\\\n            out[\\\\\\\\n                (a[0] + b[0], a[1] + b[1], a[2] + b[2])\\\\\\\\n            ] += ma * mb\\\\\\\\n    return out\\\\\\\\n\\\\\\\\n\\\\\\\\ndef reconstruct_global_energy_counter(\\\\\\\\n    signatures: Sequence[TokenSignature],\\\\\\\\n) -> CounterType[EnergyTriple]:\\\\\\\\n    total: CounterType[EnergyTriple] = Counter({(0, 0, 0): 1})\\\\\\\\n    for signature in signatures:\\\\\\\\n        total = convolve_energy_counters(\\\\\\\\n            total, local_energy_counter(signature)\\\\\\\\n        )\\\\\\\\n    return total\\\\\\\\n\\\\\\\\n\\\\\\\\ndef signature_key(signature: TokenSignature) -> str:\\\\\\\\n    return ",".join(str(x) for x in signature)\\\\\\\\n\\\\\\\\n\\\\\\\\ndef irrep_json(ir: Irrep) -> dict:\\\\\\\\n    return {\\\\\\\\n        "p": ir[0],\\\\\\\\n        "q": ir[1],\\\\\\\\n        "dimension": irrep_dim(ir),\\\\\\\\n        "C2_num_over_3": c2_num(ir),\\\\\\\\n        "C2": (\\\\\\\\n            str(c2_num(ir) // 3)\\\\\\\\n            if c2_num(ir) % 3 == 0\\\\\\\\n            else f"{c2_num(ir)}/3"\\\\\\\\n        ),\\\\\\\\n    }\\\\\\\\n\\\\\\\\n\\\\\\\\ndef sha256_file(path: Path) -> str:\\\\\\\\n    h = hashlib.sha256()\\\\\\\\n    with path.open("rb") as handle:\\\\\\\\n        for chunk in iter(lambda: handle.read(1 << 20), b""):\\\\\\\\n            h.update(chunk)\\\\\\\\n    return h.hexdigest()\\\\\\\\n\\\\\\\\n\\\\\\\\ndef write_json(path: Path, obj: object) -> str:\\\\\\\\n    raw = json.dumps(\\\\\\\\n        obj,\\\\\\\\n        indent=2,\\\\\\\\n        sort_keys=True,\\\\\\\\n        allow_nan=False,\\\\\\\\n    ).encode("utf-8")\\\\\\\\n    path.write_bytes(raw)\\\\\\\\n    return hashlib.sha256(raw).hexdigest()\\\\\\\\n\\\\\\\\n\\\\\\\\ndef write_json_gz(path: Path, obj: object) -> str:\\\\\\\\n    raw = json.dumps(\\\\\\\\n        obj,\\\\\\\\n        separators=(",", ":"),\\\\\\\\n        sort_keys=True,\\\\\\\\n        allow_nan=False,\\\\\\\\n    ).encode("utf-8")\\\\\\\\n    with gzip.GzipFile(\\\\\\\\n        filename=str(path),\\\\\\\\n        mode="wb",\\\\\\\\n        compresslevel=9,\\\\\\\\n        mtime=0,\\\\\\\\n    ) as handle:\\\\\\\\n        handle.write(raw)\\\\\\\\n    return sha256_file(path)\\\\\\\\n\\\\\\\\n\\\\\\\\ndef read_json_gz(path: Path) -> object:\\\\\\\\n    with gzip.open(path, "rt", encoding="utf-8") as handle:\\\\\\\\n        return json.load(handle)\\\\\\\\n\\\\\\\\n\\\\\\\\ndef block_hash(payload: object) -> str:\\\\\\\\n    raw = json.dumps(\\\\\\\\n        payload,\\\\\\\\n        separators=(",", ":"),\\\\\\\\n        sort_keys=True,\\\\\\\\n    ).encode("utf-8")\\\\\\\\n    return hashlib.sha256(raw).hexdigest()[:20]\\\\\\\\n\\\\\\\\n\\\\\\\\ndef run(input_path: Path, output_dir: Path) -> None:\\\\\\\\n    started = time.time()\\\\\\\\n    output_dir.mkdir(parents=True, exist_ok=True)\\\\\\\\n\\\\\\\\n    print("=" * 104)\\\\\\\\n    print("SU(3) O(y^4) STAGE-3B TIME-ORDERED IRREP STATE-GRAPH COMPILER")\\\\\\\\n    print("=" * 104)\\\\\\\\n    print(f"version  : {VERSION}")\\\\\\\\n    print(f"input    : {input_path}")\\\\\\\\n    print(f"output   : {output_dir}")\\\\\\\\n    print("hardware : standard Colab CPU; A100 is not used")\\\\\\\\n    print()\\\\\\\\n\\\\\\\\n    assert input_path.exists(), (\\\\\\\\n        f"Stage-1 manifest not found: {input_path}\\\\\\\\\\\\\\\\n"\\\\\\\\n        "Run y4_stage1_stage2_autobundle.py first."\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    stage1_sha = sha256_file(input_path)\\\\\\\\n    stage1 = read_json_gz(input_path)\\\\\\\\n    words = stage1["words"]\\\\\\\\n    assert len(words) == 4221\\\\\\\\n\\\\\\\\n    gates: Dict[str, object] = {}\\\\\\\\n\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    # G0: dependency headline checks.\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    valid_orbits = sum(\\\\\\\\n        len(word["orientation_orbits"]) for word in words\\\\\\\\n    )\\\\\\\\n    assert valid_orbits == 16835\\\\\\\\n\\\\\\\\n    stage2_summary = recursive_find("y4_stage2_summary.json")\\\\\\\\n    stage3a_summary = recursive_find("y4_stage3a_summary.json")\\\\\\\\n\\\\\\\\n    if stage2_summary is not None:\\\\\\\\n        data = json.loads(stage2_summary.read_text(encoding="utf-8"))\\\\\\\\n        assert data["passed"] is True\\\\\\\\n        assert data["counts"]["unique_link_token_signatures"] == 182\\\\\\\\n\\\\\\\\n    if stage3a_summary is not None:\\\\\\\\n        data = json.loads(stage3a_summary.read_text(encoding="utf-8"))\\\\\\\\n        assert data["all_gates_passed"] is True\\\\\\\\n        assert data["headline"]["channel_sum"] == "-481/612"\\\\\\\\n\\\\\\\\n    gates["G0_dependencies"] = {\\\\\\\\n        "stage1_words": len(words),\\\\\\\\n        "valid_orientation_orbits": valid_orbits,\\\\\\\\n        "stage1_sha256": stage1_sha,\\\\\\\\n        "stage2_summary_found": stage2_summary is not None,\\\\\\\\n        "stage3a_summary_found": stage3a_summary is not None,\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n    print(\\\\\\\\n        "G0 PASS: 4,221 words; 16,835 valid orientation orbits; "\\\\\\\\n        "optional Stage-2/3A checks complete"\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    # Collect every local signature and its occurrence count.\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    signature_occurrences: CounterType[TokenSignature] = Counter()\\\\\\\\n    for word in words:\\\\\\\\n        for orbit in word["orientation_orbits"]:\\\\\\\\n            for raw in orbit["link_token_signatures"]:\\\\\\\\n                signature = tuple(int(x) for x in raw)\\\\\\\\n                signature_occurrences[signature] += 1\\\\\\\\n\\\\\\\\n    assert len(signature_occurrences) == 182\\\\\\\\n    assert sum(signature_occurrences.values()) == 187632\\\\\\\\n\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    # G1: exact local irrep paths.\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    local_records = []\\\\\\\\n    local_path_hist = Counter()\\\\\\\\n    all_irreps = set()\\\\\\\\n    all_edges = set()\\\\\\\\n    all_full_paths = set()\\\\\\\\n\\\\\\\\n    for signature, occurrence_count in sorted(\\\\\\\\n        signature_occurrences.items()\\\\\\\\n    ):\\\\\\\\n        paths = full_irrep_paths(signature)\\\\\\\\n        assert paths\\\\\\\\n\\\\\\\\n        local_path_hist[len(paths)] += 1\\\\\\\\n\\\\\\\\n        path_records = []\\\\\\\\n        for path_index, path in enumerate(paths):\\\\\\\\n            all_full_paths.add(path)\\\\\\\\n            all_irreps.update(path)\\\\\\\\n\\\\\\\\n            transitions = []\\\\\\\\n            for event_index, token in enumerate(signature):\\\\\\\\n                source = path[event_index]\\\\\\\\n                target = path[event_index + 1]\\\\\\\\n                all_edges.add((source, token, target))\\\\\\\\n                transitions.append(\\\\\\\\n                    {\\\\\\\\n                        "event_index": event_index,\\\\\\\\n                        "event_name": EVENT_NAMES[event_index],\\\\\\\\n                        "token": token,\\\\\\\\n                        "source": list(source),\\\\\\\\n                        "target": list(target),\\\\\\\\n                    }\\\\\\\\n                )\\\\\\\\n\\\\\\\\n            path_records.append(\\\\\\\\n                {\\\\\\\\n                    "path_index": path_index,\\\\\\\\n                    "irreps": [list(ir) for ir in path],\\\\\\\\n                    "dimensions": [irrep_dim(ir) for ir in path],\\\\\\\\n                    "C2_num_over_3": [c2_num(ir) for ir in path],\\\\\\\\n                    "intermediate_E6": [\\\\\\\\n                        c2_num(path[2]),\\\\\\\\n                        c2_num(path[3]),\\\\\\\\n                        c2_num(path[4]),\\\\\\\\n                    ],\\\\\\\\n                    "transitions": transitions,\\\\\\\\n                }\\\\\\\\n            )\\\\\\\\n\\\\\\\\n        local_records.append(\\\\\\\\n            {\\\\\\\\n                "signature": list(signature),\\\\\\\\n                "signature_key": signature_key(signature),\\\\\\\\n                "event_tokens": {\\\\\\\\n                    EVENT_NAMES[i]: signature[i] for i in range(6)\\\\\\\\n                },\\\\\\\\n                "occurrence_count": occurrence_count,\\\\\\\\n                "path_count": len(paths),\\\\\\\\n                "paths": path_records,\\\\\\\\n                "energy_counter": [\\\\\\\\n                    {\\\\\\\\n                        "E6": list(energy),\\\\\\\\n                        "multiplicity": multiplicity,\\\\\\\\n                    }\\\\\\\\n                    for energy, multiplicity in sorted(\\\\\\\\n                        local_energy_counter(signature).items()\\\\\\\\n                    )\\\\\\\\n                ],\\\\\\\\n            }\\\\\\\\n        )\\\\\\\\n\\\\\\\\n    expected_local_path_hist = {\\\\\\\\n        1: 70,\\\\\\\\n        2: 90,\\\\\\\\n        5: 2,\\\\\\\\n        6: 20,\\\\\\\\n    }\\\\\\\\n    assert dict(sorted(local_path_hist.items())) == expected_local_path_hist\\\\\\\\n    assert len(all_full_paths) == 380\\\\\\\\n    assert len(all_irreps) == 10\\\\\\\\n    assert len(all_edges) == 36\\\\\\\\n\\\\\\\\n    gates["G1_local_paths"] = {\\\\\\\\n        "unique_signatures": len(local_records),\\\\\\\\n        "signature_occurrences": sum(signature_occurrences.values()),\\\\\\\\n        "path_count_histogram": dict(sorted(local_path_hist.items())),\\\\\\\\n        "unique_full_local_paths": len(all_full_paths),\\\\\\\\n        "irreps_in_feasible_paths": len(all_irreps),\\\\\\\\n        "feasible_transfer_edges": len(all_edges),\\\\\\\\n        "max_local_path_count": max(local_path_hist),\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n    print(\\\\\\\\n        "G1 PASS: 182 signatures -> 380 exact local paths; "\\\\\\\\n        "10 irreps; 36 feasible transfer edges"\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    # G2: representation-ring consistency.\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    assert irrep_dim((1, 0)) == 3\\\\\\\\n    assert irrep_dim((0, 1)) == 3\\\\\\\\n    assert irrep_dim((1, 1)) == 8\\\\\\\\n    assert irrep_dim((2, 0)) == 6\\\\\\\\n    assert c2_num((1, 0)) == 4\\\\\\\\n    assert c2_num((1, 1)) == 9\\\\\\\\n    assert c2_num((2, 0)) == 10\\\\\\\\n\\\\\\\\n    for ir in sorted(all_irreps):\\\\\\\\n        for token in (-1, +1):\\\\\\\\n            targets = fuse(ir, token)\\\\\\\\n            assert sum(irrep_dim(x) for x in targets) == 3 * irrep_dim(ir)\\\\\\\\n\\\\\\\\n    gates["G2_representation_ring"] = {\\\\\\\\n        "irreps": [irrep_json(ir) for ir in sorted(all_irreps)],\\\\\\\\n        "dimension_identity_checked": True,\\\\\\\\n        "fusion_is_multiplicity_free_per_edge": True,\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n    print("G2 PASS: exact SU(3) fusion and dimension identities")\\\\\\\\n\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    # G3: reconstruct every global denominator signature exactly.\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    checked_orbits = 0\\\\\\\\n    resonance_hist = Counter()\\\\\\\\n    orbit_path_dimension_hist = Counter()\\\\\\\\n\\\\\\\\n    for word in words:\\\\\\\\n        for orbit in word["orientation_orbits"]:\\\\\\\\n            signatures = tuple(\\\\\\\\n                tuple(int(x) for x in raw)\\\\\\\\n                for raw in orbit["link_token_signatures"]\\\\\\\\n            )\\\\\\\\n            reconstructed = reconstruct_global_energy_counter(signatures)\\\\\\\\n            expected = Counter(\\\\\\\\n                {\\\\\\\\n                    tuple(int(x) for x in row["E6"]):\\\\\\\\n                    int(row["channel_path_multiplicity"])\\\\\\\\n                    for row in orbit["denominator_signatures"]\\\\\\\\n                }\\\\\\\\n            )\\\\\\\\n            assert reconstructed == expected, (\\\\\\\\n                word["ordered_id"],\\\\\\\\n                orbit["orbit_id"],\\\\\\\\n            )\\\\\\\\n\\\\\\\\n            product_dimension = 1\\\\\\\\n            for signature in signatures:\\\\\\\\n                product_dimension *= len(full_irrep_paths(signature))\\\\\\\\n\\\\\\\\n            assert product_dimension == sum(reconstructed.values())\\\\\\\\n            assert product_dimension == orbit[\\\\\\\\n                "global_channel_path_multiplicity"\\\\\\\\n            ]\\\\\\\\n\\\\\\\\n            orbit_path_dimension_hist[product_dimension] += 1\\\\\\\\n            resonance_hist[orbit["resonance_class"]] += 1\\\\\\\\n            checked_orbits += 1\\\\\\\\n\\\\\\\\n    expected_resonance_hist = {\\\\\\\\n        "all_resonant": 7488,\\\\\\\\n        "mixed": 2749,\\\\\\\\n        "nonresonant_only": 6598,\\\\\\\\n    }\\\\\\\\n    expected_orbit_dimension_hist = {\\\\\\\\n        1: 7034,\\\\\\\\n        2: 6840,\\\\\\\\n        4: 2160,\\\\\\\\n        5: 20,\\\\\\\\n        6: 200,\\\\\\\\n        8: 120,\\\\\\\\n        16: 270,\\\\\\\\n        48: 180,\\\\\\\\n        625: 1,\\\\\\\\n        1296: 10,\\\\\\\\n    }\\\\\\\\n\\\\\\\\n    assert checked_orbits == 16835\\\\\\\\n    assert dict(sorted(resonance_hist.items())) == expected_resonance_hist\\\\\\\\n    assert dict(sorted(orbit_path_dimension_hist.items())) == (\\\\\\\\n        expected_orbit_dimension_hist\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    gates["G3_global_reconstruction"] = {\\\\\\\\n        "orbits_checked": checked_orbits,\\\\\\\\n        "resonance_histogram": dict(sorted(resonance_hist.items())),\\\\\\\\n        "path_space_dimension_histogram": dict(\\\\\\\\n            sorted(orbit_path_dimension_hist.items())\\\\\\\\n        ),\\\\\\\\n        "max_global_path_space_dimension": max(\\\\\\\\n            orbit_path_dimension_hist\\\\\\\\n        ),\\\\\\\\n        "exact_match_to_stage1": True,\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n    print(\\\\\\\\n        "G3 PASS: all 16,835 global energy counters exactly reconstructed"\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    # G4: exact recoupling block compression.\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    blocks: Dict[Tuple[Tuple[TokenSignature, ...], int], dict] = {}\\\\\\\\n\\\\\\\\n    for word in words:\\\\\\\\n        for orbit in word["orientation_orbits"]:\\\\\\\\n            signatures = tuple(\\\\\\\\n                sorted(\\\\\\\\n                    tuple(int(x) for x in raw)\\\\\\\\n                    for raw in orbit["link_token_signatures"]\\\\\\\\n                )\\\\\\\\n            )\\\\\\\\n            phase = int(orbit["c_odd_phase"])\\\\\\\\n            key = (signatures, phase)\\\\\\\\n\\\\\\\\n            if key not in blocks:\\\\\\\\n                energy_counter = reconstruct_global_energy_counter(\\\\\\\\n                    signatures\\\\\\\\n                )\\\\\\\\n                product_dimension = 1\\\\\\\\n                for signature in signatures:\\\\\\\\n                    product_dimension *= len(\\\\\\\\n                        full_irrep_paths(signature)\\\\\\\\n                    )\\\\\\\\n\\\\\\\\n                blocks[key] = {\\\\\\\\n                    "signatures": signatures,\\\\\\\\n                    "c_odd_phase": phase,\\\\\\\\n                    "orbit_count": 0,\\\\\\\\n                    "ordered_word_ids": set(),\\\\\\\\n                    "orbit_ids": [],\\\\\\\\n                    "resonance_histogram": Counter(),\\\\\\\\n                    "path_space_dimension": product_dimension,\\\\\\\\n                    "energy_counter": energy_counter,\\\\\\\\n                }\\\\\\\\n\\\\\\\\n            block = blocks[key]\\\\\\\\n            block["orbit_count"] += 1\\\\\\\\n            block["ordered_word_ids"].add(word["ordered_id"])\\\\\\\\n            block["orbit_ids"].append(orbit["orbit_id"])\\\\\\\\n            block["resonance_histogram"][\\\\\\\\n                orbit["resonance_class"]\\\\\\\\n            ] += 1\\\\\\\\n\\\\\\\\n    assert len(blocks) == 1933\\\\\\\\n    unsigned_blocks = {\\\\\\\\n        key[0] for key in blocks\\\\\\\\n    }\\\\\\\\n    assert len(unsigned_blocks) == 1395\\\\\\\\n\\\\\\\\n    block_dimension_hist = Counter(\\\\\\\\n        block["path_space_dimension"] for block in blocks.values()\\\\\\\\n    )\\\\\\\\n    expected_block_dimension_hist = {\\\\\\\\n        1: 57,\\\\\\\\n        2: 308,\\\\\\\\n        4: 996,\\\\\\\\n        5: 20,\\\\\\\\n        6: 172,\\\\\\\\n        8: 120,\\\\\\\\n        16: 69,\\\\\\\\n        48: 180,\\\\\\\\n        625: 1,\\\\\\\\n        1296: 10,\\\\\\\\n    }\\\\\\\\n    assert dict(sorted(block_dimension_hist.items())) == (\\\\\\\\n        expected_block_dimension_hist\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    block_records = []\\\\\\\\n    for index, ((signatures, phase), block) in enumerate(\\\\\\\\n        sorted(blocks.items()),\\\\\\\\n        start=1,\\\\\\\\n    ):\\\\\\\\n        signature_counter = Counter(signatures)\\\\\\\\n        payload_for_hash = {\\\\\\\\n            "signatures": [\\\\\\\\n                [list(signature), multiplicity]\\\\\\\\n                for signature, multiplicity in sorted(\\\\\\\\n                    signature_counter.items()\\\\\\\\n                )\\\\\\\\n            ],\\\\\\\\n            "c_odd_phase": phase,\\\\\\\\n        }\\\\\\\\n\\\\\\\\n        block_records.append(\\\\\\\\n            {\\\\\\\\n                "block_id": f"R4-{index:04d}-"\\\\\\\\n                            f"{block_hash(payload_for_hash)}",\\\\\\\\n                "c_odd_phase": phase,\\\\\\\\n                "orbit_count": block["orbit_count"],\\\\\\\\n                "ordered_word_count": len(\\\\\\\\n                    block["ordered_word_ids"]\\\\\\\\n                ),\\\\\\\\n                "ordered_word_ids": sorted(\\\\\\\\n                    block["ordered_word_ids"]\\\\\\\\n                ),\\\\\\\\n                "orbit_ids": sorted(block["orbit_ids"]),\\\\\\\\n                "token_signature_multiset": [\\\\\\\\n                    {\\\\\\\\n                        "signature": list(signature),\\\\\\\\n                        "multiplicity": multiplicity,\\\\\\\\n                        "local_path_count": len(\\\\\\\\n                            full_irrep_paths(signature)\\\\\\\\n                        ),\\\\\\\\n                    }\\\\\\\\n                    for signature, multiplicity in sorted(\\\\\\\\n                        signature_counter.items()\\\\\\\\n                    )\\\\\\\\n                ],\\\\\\\\n                "touched_links": len(signatures),\\\\\\\\n                "path_space_dimension": block[\\\\\\\\n                    "path_space_dimension"\\\\\\\\n                ],\\\\\\\\n                "energy_signature_count": len(\\\\\\\\n                    block["energy_counter"]\\\\\\\\n                ),\\\\\\\\n                "energy_counter": [\\\\\\\\n                    {\\\\\\\\n                        "E6": list(energy),\\\\\\\\n                        "multiplicity": multiplicity,\\\\\\\\n                        "resonant_mask": (\\\\\\\\n                            int(energy[0] == 16)\\\\\\\\n                            | (int(energy[1] == 16) << 1)\\\\\\\\n                            | (int(energy[2] == 16) << 2)\\\\\\\\n                        ),\\\\\\\\n                        "vacuum_mask": (\\\\\\\\n                            int(energy[0] == 0)\\\\\\\\n                            | (int(energy[1] == 0) << 1)\\\\\\\\n                            | (int(energy[2] == 0) << 2)\\\\\\\\n                        ),\\\\\\\\n                    }\\\\\\\\n                    for energy, multiplicity in sorted(\\\\\\\\n                        block["energy_counter"].items()\\\\\\\\n                    )\\\\\\\\n                ],\\\\\\\\n                "resonance_histogram": dict(\\\\\\\\n                    sorted(block["resonance_histogram"].items())\\\\\\\\n                ),\\\\\\\\n            }\\\\\\\\n        )\\\\\\\\n\\\\\\\\n    gates["G4_block_compression"] = {\\\\\\\\n        "blocks_with_C_odd_phase": len(blocks),\\\\\\\\n        "blocks_without_phase": len(unsigned_blocks),\\\\\\\\n        "block_path_dimension_histogram": dict(\\\\\\\\n            sorted(block_dimension_hist.items())\\\\\\\\n        ),\\\\\\\\n        "max_block_path_dimension": max(block_dimension_hist),\\\\\\\\n        "blocks_above_dimension_600": sum(\\\\\\\\n            count\\\\\\\\n            for dimension, count in block_dimension_hist.items()\\\\\\\\n            if dimension > 600\\\\\\\\n        ),\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n    print(\\\\\\\\n        "G4 PASS: 16,835 orbits -> 1,933 exact recoupling blocks "\\\\\\\\n        "(1,395 before C-odd phase)"\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    # G5: write and round-trip.\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    local_payload = {\\\\\\\\n        "meta": {\\\\\\\\n            "version": VERSION,\\\\\\\\n            "stage1_input": str(input_path),\\\\\\\\n            "stage1_sha256": stage1_sha,\\\\\\\\n            "event_order": EVENT_NAMES,\\\\\\\\n            "warning": (\\\\\\\\n                "Paths are exact representation-ring incidence paths. "\\\\\\\\n                "They do not yet contain normalized Clebsch-Gordan or "\\\\\\\\n                "Racah amplitudes."\\\\\\\\n            ),\\\\\\\\n        },\\\\\\\\n        "irreps": [irrep_json(ir) for ir in sorted(all_irreps)],\\\\\\\\n        "feasible_transfer_edges": [\\\\\\\\n            {\\\\\\\\n                "source": list(source),\\\\\\\\n                "token": token,\\\\\\\\n                "target": list(target),\\\\\\\\n            }\\\\\\\\n            for source, token, target in sorted(all_edges)\\\\\\\\n        ],\\\\\\\\n        "signatures": local_records,\\\\\\\\n    }\\\\\\\\n    local_path = output_dir / "y4_local_irrep_paths.json.gz"\\\\\\\\n    local_sha = write_json_gz(local_path, local_payload)\\\\\\\\n\\\\\\\\n    block_payload = {\\\\\\\\n        "meta": {\\\\\\\\n            "version": VERSION,\\\\\\\\n            "stage1_input": str(input_path),\\\\\\\\n            "stage1_sha256": stage1_sha,\\\\\\\\n            "local_path_file": local_path.name,\\\\\\\\n            "local_path_sha256": local_sha,\\\\\\\\n            "block_equivalence": (\\\\\\\\n                "multiset of local token signatures plus C-odd "\\\\\\\\n                "ket/bra orientation phase"\\\\\\\\n            ),\\\\\\\\n        },\\\\\\\\n        "blocks": block_records,\\\\\\\\n    }\\\\\\\\n    block_path = output_dir / "y4_recoupling_blocks.json.gz"\\\\\\\\n    block_sha = write_json_gz(block_path, block_payload)\\\\\\\\n\\\\\\\\n    reread_local = read_json_gz(local_path)\\\\\\\\n    reread_blocks = read_json_gz(block_path)\\\\\\\\n    assert len(reread_local["signatures"]) == 182\\\\\\\\n    assert len(reread_blocks["blocks"]) == 1933\\\\\\\\n\\\\\\\\n    gates["G5_round_trip"] = {\\\\\\\\n        "local_signature_records": 182,\\\\\\\\n        "recoupling_blocks": 1933,\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n    print("G5 PASS: output round-trip and exact record counts")\\\\\\\\n\\\\\\\\n    elapsed = time.time() - started\\\\\\\\n    summary = {\\\\\\\\n        "meta": {\\\\\\\\n            "version": VERSION,\\\\\\\\n            "date": "2026-06-13",\\\\\\\\n            "python": sys.version,\\\\\\\\n            "platform": platform.platform(),\\\\\\\\n            "walltime_s": elapsed,\\\\\\\\n            "hardware": "CPU",\\\\\\\\n            "a100_required": False,\\\\\\\\n        },\\\\\\\\n        "input": {\\\\\\\\n            "path": str(input_path),\\\\\\\\n            "sha256": stage1_sha,\\\\\\\\n        },\\\\\\\\n        "counts": {\\\\\\\\n            "ordered_words": 4221,\\\\\\\\n            "valid_orientation_orbits": 16835,\\\\\\\\n            "unique_local_signatures": 182,\\\\\\\\n            "unique_full_local_paths": 380,\\\\\\\\n            "feasible_irreps": 10,\\\\\\\\n            "feasible_transfer_edges": 36,\\\\\\\\n            "recoupling_blocks_with_phase": 1933,\\\\\\\\n            "recoupling_blocks_without_phase": 1395,\\\\\\\\n            "max_local_path_count": 6,\\\\\\\\n            "max_global_path_space_dimension": 1296,\\\\\\\\n            "blocks_above_dimension_600": 11,\\\\\\\\n        },\\\\\\\\n        "gates": gates,\\\\\\\\n        "files": {\\\\\\\\n            local_path.name: {\\\\\\\\n                "sha256": local_sha,\\\\\\\\n                "records": 182,\\\\\\\\n            },\\\\\\\\n            block_path.name: {\\\\\\\\n                "sha256": block_sha,\\\\\\\\n                "records": 1933,\\\\\\\\n            },\\\\\\\\n        },\\\\\\\\n        "scope": {\\\\\\\\n            "completed": [\\\\\\\\n                "exact time-ordered SU(3) irrep paths",\\\\\\\\n                "exact local representation-ring transfer graph",\\\\\\\\n                "exact reconstruction of every Stage-1 denominator counter",\\\\\\\\n                "recoupling block compression",\\\\\\\\n                "resonance and vacuum mask attachment",\\\\\\\\n            ],\\\\\\\\n            "not_completed": [\\\\\\\\n                "normalized Clebsch-Gordan intertwiners",\\\\\\\\n                "SU(3) Racah / 6j recoupling coefficients",\\\\\\\\n                "global trace-index contraction amplitudes",\\\\\\\\n                "des-Cloizeaux folded terms",\\\\\\\\n                "fourth-order flat-band residual",\\\\\\\\n            ],\\\\\\\\n        },\\\\\\\\n        "next_stage": (\\\\\\\\n            "Stage 3C: attach normalized SU(3) intertwiner matrices to "\\\\\\\\n            "the 36 feasible transfer edges and compute exact Racah "\\\\\\\\n            "overlap blocks. Begin with the four domino channels as a "\\\\\\\\n            "normalization regression."\\\\\\\\n        ),\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n\\\\\\\\n    summary_path = output_dir / "y4_stage3b_summary.json"\\\\\\\\n    summary_sha = write_json(summary_path, summary)\\\\\\\\n\\\\\\\\n    print()\\\\\\\\n    print("SUMMARY")\\\\\\\\n    print(json.dumps(summary["counts"], indent=2, sort_keys=True))\\\\\\\\n    print()\\\\\\\\n    print(f"LOCAL PATHS : {local_path}")\\\\\\\\n    print(f"BLOCKS      : {block_path}")\\\\\\\\n    print(f"SUMMARY     : {summary_path}")\\\\\\\\n    print(f"SUMMARY SHA256: {summary_sha}")\\\\\\\\n    print(f"WALLTIME: {elapsed:.2f} s")\\\\\\\\n    print("ALL STAGE-3B GATES PASS")\\\\\\\\n    print()\\\\\\\\n    print(\\\\\\\\n        "NEXT: Stage 3C normalized SU(3) intertwiner / Racah blocks. "\\\\\\\\n        "Continue on CPU."\\\\\\\\n    )\\\\\\\\n\\\\\\\\n\\\\\\\\nif __name__ == "__main__":\\\\\\\\n    parser = argparse.ArgumentParser(\\\\\\\\n        description="Y4 exact time-ordered SU(3) irrep state graph"\\\\\\\\n    )\\\\\\\\n    parser.add_argument(\\\\\\\\n        "--input",\\\\\\\\n        type=Path,\\\\\\\\n        default=None,\\\\\\\\n    )\\\\\\\\n    parser.add_argument(\\\\\\\\n        "--output-dir",\\\\\\\\n        type=Path,\\\\\\\\n        default=default_output_dir(),\\\\\\\\n    )\\\\\\\\n    args, _unknown = parser.parse_known_args()\\\\\\\\n\\\\\\\\n    input_path = args.input\\\\\\\\n    if input_path is None:\\\\\\\\n        input_path = recursive_find(\\\\\\\\n            "y4_channel_denominator_manifest.json.gz"\\\\\\\\n        )\\\\\\\\n    assert input_path is not None, (\\\\\\\\n        "Could not locate y4_channel_denominator_manifest.json.gz. "\\\\\\\\n        "Run y4_stage1_stage2_autobundle.py first."\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    run(input_path, args.output_dir)\\\\\\\\n\\\\\\\'\\\\nSTAGE3C_SOURCE = \\\\\\\'#!/usr/bin/env python3\\\\\\\\n"""\\\\\\\\ny4_stage3c_exact_casimir_projectors.py\\\\\\\\n======================================\\\\\\\\n\\\\\\\\nExact SU(3) Casimir-channel projector compiler for the O(y^4) flat-band program.\\\\\\\\n\\\\\\\\nRUN\\\\\\\\n---\\\\\\\\n    %run /content/y4_stage3c_exact_casimir_projectors.py\\\\\\\\n\\\\\\\\nHARDWARE\\\\\\\\n--------\\\\\\\\nUse a standard Colab CPU runtime. The A100 is not used.\\\\\\\\n\\\\\\\\nINPUT\\\\\\\\n-----\\\\\\\\nThe script recursively locates:\\\\\\\\n    y4_local_irrep_paths.json.gz\\\\\\\\n\\\\\\\\nnormally at:\\\\\\\\n    /content/Y4_STAGE3B/y4_local_irrep_paths.json.gz\\\\\\\\n\\\\\\\\nIt also validates Stage 3A when available.\\\\\\\\n\\\\\\\\nOUTPUT\\\\\\\\n------\\\\\\\\n    /content/Y4_STAGE3C/y4_irrep_carriers.json.gz\\\\\\\\n    /content/Y4_STAGE3C/y4_casimir_channel_projectors.json.gz\\\\\\\\n    /content/Y4_STAGE3C/y4_path_projector_cards.json.gz\\\\\\\\n    /content/Y4_STAGE3C/y4_stage3c_summary.json\\\\\\\\n\\\\\\\\nMATHEMATICAL OBJECT\\\\\\\\n-------------------\\\\\\\\nEvery irrep (p,q) is realized exactly as the traceless subspace\\\\\\\\n\\\\\\\\n    ker[ Sym^p(3) tensor Sym^q(3bar)\\\\\\\\n         -> Sym^(p-1)(3) tensor Sym^(q-1)(3bar) ]\\\\\\\\n\\\\\\\\nin a rational polynomial-tensor basis.\\\\\\\\n\\\\\\\\nFor each source irrep R and token r in {3,3bar}, the total quadratic Casimir\\\\\\\\non R tensor r is constructed exactly. Since tensoring by 3 or 3bar is\\\\\\\\nmultiplicity-free and the branch Casimirs are distinct, the channel projector is\\\\\\\\n\\\\\\\\n    P_T = product_{S != T} (C_total - C2(S) I)/(C2(T)-C2(S)).\\\\\\\\n\\\\\\\\nThese projectors are:\\\\\\\\n  * exact rational matrices;\\\\\\\\n  * orthogonal with respect to the exact carrier metric;\\\\\\\\n  * idempotent;\\\\\\\\n  * complete over the full fusion product;\\\\\\\\n  * basis independent up to exact similarity;\\\\\\\\n  * free of arbitrary Clebsch-Gordan phases.\\\\\\\\n\\\\\\\\nThis is stronger and safer than exporting normalized CG coefficient tables.\\\\\\\\nThe next stage can lift these projectors to prefix tensor spaces and contract\\\\\\\\nthe actual plaquette trace-index network.\\\\\\\\n\\\\\\\\nIMPORTANT LIMIT\\\\\\\\n---------------\\\\\\\\nThis script does not compute any O(y^4) amplitude. It compiles exact channel\\\\\\\\nprojectors and maps every Stage-3B local path to them.\\\\\\\\n"""\\\\\\\\n\\\\\\\\nfrom __future__ import annotations\\\\\\\\n\\\\\\\\nimport argparse\\\\\\\\nimport gzip\\\\\\\\nimport hashlib\\\\\\\\nimport itertools\\\\\\\\nimport json\\\\\\\\nimport math\\\\\\\\nimport platform\\\\\\\\nimport sys\\\\\\\\nimport time\\\\\\\\nfrom functools import lru_cache\\\\\\\\nfrom pathlib import Path\\\\\\\\nfrom typing import Dict, Iterable, List, Sequence, Tuple\\\\\\\\n\\\\\\\\nimport sympy as sp\\\\\\\\n\\\\\\\\nVERSION = "2026-06-13-stage3c-v1"\\\\\\\\n\\\\\\\\nIrrep = Tuple[int, int]\\\\\\\\nTokenSignature = Tuple[int, int, int, int, int, int]\\\\\\\\n\\\\\\\\nEVENT_NAMES = (\\\\\\\\n    "ket",\\\\\\\\n    "insertion_1",\\\\\\\\n    "insertion_2",\\\\\\\\n    "insertion_3",\\\\\\\\n    "insertion_4",\\\\\\\\n    "bra",\\\\\\\\n)\\\\\\\\n\\\\\\\\n\\\\\\\\ndef default_output_dir() -> Path:\\\\\\\\n    if Path("/content").exists():\\\\\\\\n        return Path("/content/Y4_STAGE3C")\\\\\\\\n    return Path.cwd() / "Y4_STAGE3C"\\\\\\\\n\\\\\\\\n\\\\\\\\ndef recursive_find(filename: str) -> Path | None:\\\\\\\\n    preferred = [\\\\\\\\n        Path("/content/Y4_STAGE3B") / filename,\\\\\\\\n        Path.cwd() / "Y4_STAGE3B" / filename,\\\\\\\\n        Path("/mnt/data/Y4_STAGE3B_TEST") / filename,\\\\\\\\n    ]\\\\\\\\n    for path in preferred:\\\\\\\\n        if path.exists():\\\\\\\\n            return path\\\\\\\\n\\\\\\\\n    for root in (Path("/content"), Path.cwd(), Path("/mnt/data")):\\\\\\\\n        if not root.exists():\\\\\\\\n            continue\\\\\\\\n        found = list(root.rglob(filename))\\\\\\\\n        if found:\\\\\\\\n            return found[0]\\\\\\\\n    return None\\\\\\\\n\\\\\\\\n\\\\\\\\ndef sha256_file(path: Path) -> str:\\\\\\\\n    h = hashlib.sha256()\\\\\\\\n    with path.open("rb") as handle:\\\\\\\\n        for chunk in iter(lambda: handle.read(1 << 20), b""):\\\\\\\\n            h.update(chunk)\\\\\\\\n    return h.hexdigest()\\\\\\\\n\\\\\\\\n\\\\\\\\ndef write_json(path: Path, obj: object) -> str:\\\\\\\\n    raw = json.dumps(\\\\\\\\n        obj,\\\\\\\\n        indent=2,\\\\\\\\n        sort_keys=True,\\\\\\\\n        allow_nan=False,\\\\\\\\n    ).encode("utf-8")\\\\\\\\n    path.write_bytes(raw)\\\\\\\\n    return hashlib.sha256(raw).hexdigest()\\\\\\\\n\\\\\\\\n\\\\\\\\ndef write_json_gz(path: Path, obj: object) -> str:\\\\\\\\n    raw = json.dumps(\\\\\\\\n        obj,\\\\\\\\n        separators=(",", ":"),\\\\\\\\n        sort_keys=True,\\\\\\\\n        allow_nan=False,\\\\\\\\n    ).encode("utf-8")\\\\\\\\n    with gzip.GzipFile(\\\\\\\\n        filename=str(path),\\\\\\\\n        mode="wb",\\\\\\\\n        compresslevel=9,\\\\\\\\n        mtime=0,\\\\\\\\n    ) as handle:\\\\\\\\n        handle.write(raw)\\\\\\\\n    return sha256_file(path)\\\\\\\\n\\\\\\\\n\\\\\\\\ndef read_json_gz(path: Path) -> object:\\\\\\\\n    with gzip.open(path, "rt", encoding="utf-8") as handle:\\\\\\\\n        return json.load(handle)\\\\\\\\n\\\\\\\\n\\\\\\\\ndef irrep_dim(ir: Irrep) -> int:\\\\\\\\n    p, q = ir\\\\\\\\n    return (p + 1) * (q + 1) * (p + q + 2) // 2\\\\\\\\n\\\\\\\\n\\\\\\\\ndef c2(ir: Irrep) -> sp.Rational:\\\\\\\\n    p, q = ir\\\\\\\\n    return sp.Rational(\\\\\\\\n        p * p + q * q + p * q + 3 * p + 3 * q,\\\\\\\\n        3,\\\\\\\\n    )\\\\\\\\n\\\\\\\\n\\\\\\\\ndef irrep_name(ir: Irrep) -> str:\\\\\\\\n    return f"({ir[0]},{ir[1]})"\\\\\\\\n\\\\\\\\n\\\\\\\\ndef projector_id(source: Irrep, token: int, target: Irrep) -> str:\\\\\\\\n    token_name = "3" if token == +1 else "3bar"\\\\\\\\n    return (\\\\\\\\n        f"CP-{source[0]}_{source[1]}-"\\\\\\\\n        f"{token_name}-{target[0]}_{target[1]}"\\\\\\\\n    )\\\\\\\\n\\\\\\\\n\\\\\\\\ndef compositions3(total: int) -> List[Tuple[int, int, int]]:\\\\\\\\n    return [\\\\\\\\n        (a, b, total - a - b)\\\\\\\\n        for a in range(total + 1)\\\\\\\\n        for b in range(total - a + 1)\\\\\\\\n    ]\\\\\\\\n\\\\\\\\n\\\\\\\\ndef ambient_basis(p: int, q: int):\\\\\\\\n    return [\\\\\\\\n        (upper, lower)\\\\\\\\n        for upper in compositions3(p)\\\\\\\\n        for lower in compositions3(q)\\\\\\\\n    ]\\\\\\\\n\\\\\\\\n\\\\\\\\ndef primitive_integer_column(vector: sp.Matrix) -> sp.Matrix:\\\\\\\\n    values = [sp.Rational(x) for x in vector]\\\\\\\\n    denominator = 1\\\\\\\\n    for x in values:\\\\\\\\n        denominator = sp.ilcm(denominator, x.q)\\\\\\\\n\\\\\\\\n    integers = [int(x * denominator) for x in values]\\\\\\\\n    divisor = 0\\\\\\\\n    for x in integers:\\\\\\\\n        divisor = math.gcd(divisor, abs(x))\\\\\\\\n    if divisor:\\\\\\\\n        integers = [x // divisor for x in integers]\\\\\\\\n\\\\\\\\n    for x in integers:\\\\\\\\n        if x:\\\\\\\\n            if x < 0:\\\\\\\\n                integers = [-y for y in integers]\\\\\\\\n            break\\\\\\\\n\\\\\\\\n    return sp.Matrix(integers)\\\\\\\\n\\\\\\\\n\\\\\\\\ndef sparse_matrix_json(matrix: sp.Matrix) -> dict:\\\\\\\\n    entries = []\\\\\\\\n    for i in range(matrix.rows):\\\\\\\\n        for j in range(matrix.cols):\\\\\\\\n            x = sp.Rational(matrix[i, j])\\\\\\\\n            if x:\\\\\\\\n                entries.append(\\\\\\\\n                    [i, j, int(x.p), int(x.q)]\\\\\\\\n                )\\\\\\\\n    return {\\\\\\\\n        "rows": matrix.rows,\\\\\\\\n        "cols": matrix.cols,\\\\\\\\n        "nnz": len(entries),\\\\\\\\n        "entries": entries,\\\\\\\\n    }\\\\\\\\n\\\\\\\\n\\\\\\\\ndef matrix_from_sparse_json(record: dict) -> sp.Matrix:\\\\\\\\n    matrix = sp.zeros(record["rows"], record["cols"])\\\\\\\\n    for i, j, numerator, denominator in record["entries"]:\\\\\\\\n        matrix[i, j] = sp.Rational(numerator, denominator)\\\\\\\\n    return matrix\\\\\\\\n\\\\\\\\n\\\\\\\\ndef fusion_targets(source: Irrep, token: int) -> Tuple[Irrep, ...]:\\\\\\\\n    p, q = source\\\\\\\\n    if token == +1:\\\\\\\\n        targets: List[Irrep] = [(p + 1, q)]\\\\\\\\n        if p > 0:\\\\\\\\n            targets.append((p - 1, q + 1))\\\\\\\\n        if q > 0:\\\\\\\\n            targets.append((p, q - 1))\\\\\\\\n    else:\\\\\\\\n        targets = [(p, q + 1)]\\\\\\\\n        if q > 0:\\\\\\\\n            targets.append((p + 1, q - 1))\\\\\\\\n        if p > 0:\\\\\\\\n            targets.append((p - 1, q))\\\\\\\\n    return tuple(targets)\\\\\\\\n\\\\\\\\n\\\\\\\\n@lru_cache(maxsize=None)\\\\\\\\ndef carrier(p: int, q: int) -> dict:\\\\\\\\n    basis = ambient_basis(p, q)\\\\\\\\n    basis_index = {state: i for i, state in enumerate(basis)}\\\\\\\\n    ambient_dimension = len(basis)\\\\\\\\n\\\\\\\\n    if p > 0 and q > 0:\\\\\\\\n        target_basis = ambient_basis(p - 1, q - 1)\\\\\\\\n        target_index = {\\\\\\\\n            state: i for i, state in enumerate(target_basis)\\\\\\\\n        }\\\\\\\\n        contraction = sp.zeros(\\\\\\\\n            len(target_basis),\\\\\\\\n            ambient_dimension,\\\\\\\\n        )\\\\\\\\n\\\\\\\\n        for column, (upper, lower) in enumerate(basis):\\\\\\\\n            for color in range(3):\\\\\\\\n                if upper[color] and lower[color]:\\\\\\\\n                    new_upper = list(upper)\\\\\\\\n                    new_lower = list(lower)\\\\\\\\n                    new_upper[color] -= 1\\\\\\\\n                    new_lower[color] -= 1\\\\\\\\n                    contraction[\\\\\\\\n                        target_index[\\\\\\\\n                            (tuple(new_upper), tuple(new_lower))\\\\\\\\n                        ],\\\\\\\\n                        column,\\\\\\\\n                    ] += upper[color] * lower[color]\\\\\\\\n\\\\\\\\n        nullspace = contraction.nullspace()\\\\\\\\n        embedding = sp.Matrix.hstack(\\\\\\\\n            *[\\\\\\\\n                primitive_integer_column(vector)\\\\\\\\n                for vector in nullspace\\\\\\\\n            ]\\\\\\\\n        )\\\\\\\\n    else:\\\\\\\\n        contraction = sp.zeros(0, ambient_dimension)\\\\\\\\n        embedding = sp.eye(ambient_dimension)\\\\\\\\n\\\\\\\\n    expected_dimension = irrep_dim((p, q))\\\\\\\\n    assert embedding.cols == expected_dimension\\\\\\\\n\\\\\\\\n    weights = []\\\\\\\\n    for upper, lower in basis:\\\\\\\\n        upper_weight = sp.Rational(\\\\\\\\n            math.prod(math.factorial(x) for x in upper),\\\\\\\\n            math.factorial(p),\\\\\\\\n        )\\\\\\\\n        lower_weight = sp.Rational(\\\\\\\\n            math.prod(math.factorial(x) for x in lower),\\\\\\\\n            math.factorial(q),\\\\\\\\n        )\\\\\\\\n        weights.append(upper_weight * lower_weight)\\\\\\\\n\\\\\\\\n    ambient_metric = sp.diag(*weights)\\\\\\\\n    metric = sp.simplify(\\\\\\\\n        embedding.T * ambient_metric * embedding\\\\\\\\n    )\\\\\\\\n    left_inverse = sp.simplify(\\\\\\\\n        metric.inv() * embedding.T * ambient_metric\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    generators: Dict[Tuple[int, int], sp.Matrix] = {}\\\\\\\\n\\\\\\\\n    for i in range(3):\\\\\\\\n        for j in range(3):\\\\\\\\n            ambient_generator = sp.zeros(\\\\\\\\n                ambient_dimension,\\\\\\\\n                ambient_dimension,\\\\\\\\n            )\\\\\\\\n\\\\\\\\n            for column, (upper, lower) in enumerate(basis):\\\\\\\\n                if upper[j]:\\\\\\\\n                    new_upper = list(upper)\\\\\\\\n                    new_upper[j] -= 1\\\\\\\\n                    new_upper[i] += 1\\\\\\\\n                    ambient_generator[\\\\\\\\n                        basis_index[\\\\\\\\n                            (tuple(new_upper), lower)\\\\\\\\n                        ],\\\\\\\\n                        column,\\\\\\\\n                    ] += upper[j]\\\\\\\\n\\\\\\\\n                if lower[i]:\\\\\\\\n                    new_lower = list(lower)\\\\\\\\n                    new_lower[i] -= 1\\\\\\\\n                    new_lower[j] += 1\\\\\\\\n                    ambient_generator[\\\\\\\\n                        basis_index[\\\\\\\\n                            (upper, tuple(new_lower))\\\\\\\\n                        ],\\\\\\\\n                        column,\\\\\\\\n                    ] -= lower[i]\\\\\\\\n\\\\\\\\n            restricted = sp.simplify(\\\\\\\\n                left_inverse * ambient_generator * embedding\\\\\\\\n            )\\\\\\\\n            assert (\\\\\\\\n                ambient_generator * embedding\\\\\\\\n                == embedding * restricted\\\\\\\\n            )\\\\\\\\n            generators[(i, j)] = restricted\\\\\\\\n\\\\\\\\n    return {\\\\\\\\n        "irrep": (p, q),\\\\\\\\n        "basis": basis,\\\\\\\\n        "contraction": contraction,\\\\\\\\n        "embedding": embedding,\\\\\\\\n        "ambient_metric": ambient_metric,\\\\\\\\n        "metric": metric,\\\\\\\\n        "generators": generators,\\\\\\\\n    }\\\\\\\\n\\\\\\\\n\\\\\\\\ndef quadratic_casimir(\\\\\\\\n    generators: Dict[Tuple[int, int], sp.Matrix],\\\\\\\\n) -> sp.Matrix:\\\\\\\\n    dimension = next(iter(generators.values())).rows\\\\\\\\n    sum_eij_eji = sp.zeros(dimension)\\\\\\\\n    trace_generator = sp.zeros(dimension)\\\\\\\\n\\\\\\\\n    for i in range(3):\\\\\\\\n        trace_generator += generators[(i, i)]\\\\\\\\n        for j in range(3):\\\\\\\\n            sum_eij_eji += (\\\\\\\\n                generators[(i, j)] * generators[(j, i)]\\\\\\\\n            )\\\\\\\\n\\\\\\\\n    return sp.simplify(\\\\\\\\n        sp.Rational(1, 2)\\\\\\\\n        * (\\\\\\\\n            sum_eij_eji\\\\\\\\n            - sp.Rational(1, 3)\\\\\\\\n            * trace_generator\\\\\\\\n            * trace_generator\\\\\\\\n        )\\\\\\\\n    )\\\\\\\\n\\\\\\\\n\\\\\\\\n@lru_cache(maxsize=None)\\\\\\\\ndef tensor_domain(source: Irrep, token: int) -> dict:\\\\\\\\n    source_carrier = carrier(*source)\\\\\\\\n    token_irrep = (1, 0) if token == +1 else (0, 1)\\\\\\\\n    token_carrier = carrier(*token_irrep)\\\\\\\\n\\\\\\\\n    source_dimension = irrep_dim(source)\\\\\\\\n    token_dimension = 3\\\\\\\\n    source_identity = sp.eye(source_dimension)\\\\\\\\n    token_identity = sp.eye(token_dimension)\\\\\\\\n\\\\\\\\n    generators = {}\\\\\\\\n    for i in range(3):\\\\\\\\n        for j in range(3):\\\\\\\\n            generators[(i, j)] = (\\\\\\\\n                sp.kronecker_product(\\\\\\\\n                    source_carrier["generators"][(i, j)],\\\\\\\\n                    token_identity,\\\\\\\\n                )\\\\\\\\n                + sp.kronecker_product(\\\\\\\\n                    source_identity,\\\\\\\\n                    token_carrier["generators"][(i, j)],\\\\\\\\n                )\\\\\\\\n            )\\\\\\\\n\\\\\\\\n    metric = sp.kronecker_product(\\\\\\\\n        source_carrier["metric"],\\\\\\\\n        token_carrier["metric"],\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    return {\\\\\\\\n        "source": source,\\\\\\\\n        "token": token,\\\\\\\\n        "token_irrep": token_irrep,\\\\\\\\n        "generators": generators,\\\\\\\\n        "metric": metric,\\\\\\\\n        "casimir": quadratic_casimir(generators),\\\\\\\\n    }\\\\\\\\n\\\\\\\\n\\\\\\\\n@lru_cache(maxsize=None)\\\\\\\\ndef channel_projectors(source: Irrep, token: int):\\\\\\\\n    domain = tensor_domain(source, token)\\\\\\\\n    casimir = domain["casimir"]\\\\\\\\n    identity = sp.eye(casimir.rows)\\\\\\\\n    targets = fusion_targets(source, token)\\\\\\\\n    target_casimirs = {\\\\\\\\n        target: c2(target) for target in targets\\\\\\\\n    }\\\\\\\\n\\\\\\\\n    projectors = {}\\\\\\\\n    for target in targets:\\\\\\\\n        projector = identity\\\\\\\\n        target_casimir = target_casimirs[target]\\\\\\\\n\\\\\\\\n        for other in targets:\\\\\\\\n            if other == target:\\\\\\\\n                continue\\\\\\\\n            projector = sp.simplify(\\\\\\\\n                projector\\\\\\\\n                * (\\\\\\\\n                    casimir\\\\\\\\n                    - target_casimirs[other] * identity\\\\\\\\n                )\\\\\\\\n                / (\\\\\\\\n                    target_casimir\\\\\\\\n                    - target_casimirs[other]\\\\\\\\n                )\\\\\\\\n            )\\\\\\\\n\\\\\\\\n        projectors[target] = sp.simplify(projector)\\\\\\\\n\\\\\\\\n    return domain, projectors\\\\\\\\n\\\\\\\\n\\\\\\\\ndef standard_color_order(ir: Irrep) -> List[int]:\\\\\\\\n    data = carrier(*ir)\\\\\\\\n    embedding = data["embedding"]\\\\\\\\n    basis = data["basis"]\\\\\\\\n    assert sum(ir) == 1\\\\\\\\n    order = []\\\\\\\\n\\\\\\\\n    for column in range(embedding.cols):\\\\\\\\n        support = [\\\\\\\\n            row\\\\\\\\n            for row in range(embedding.rows)\\\\\\\\n            if embedding[row, column] != 0\\\\\\\\n        ]\\\\\\\\n        assert len(support) == 1\\\\\\\\n        row = support[0]\\\\\\\\n        assert embedding[row, column] == 1\\\\\\\\n        upper, lower = basis[row]\\\\\\\\n        occupation = upper if sum(upper) == 1 else lower\\\\\\\\n        order.append(occupation.index(1))\\\\\\\\n\\\\\\\\n    return order\\\\\\\\n\\\\\\\\n\\\\\\\\ndef pair_basis_to_standard(\\\\\\\\n    matrix: sp.Matrix,\\\\\\\\n    first: Irrep,\\\\\\\\n    second: Irrep,\\\\\\\\n) -> sp.Matrix:\\\\\\\\n    first_order = standard_color_order(first)\\\\\\\\n    second_order = standard_color_order(second)\\\\\\\\n    permutation = sp.zeros(9)\\\\\\\\n\\\\\\\\n    for a, color_a in enumerate(first_order):\\\\\\\\n        for b, color_b in enumerate(second_order):\\\\\\\\n            canonical_index = 3 * a + b\\\\\\\\n            standard_index = 3 * color_a + color_b\\\\\\\\n            permutation[\\\\\\\\n                standard_index,\\\\\\\\n                canonical_index,\\\\\\\\n            ] = 1\\\\\\\\n\\\\\\\\n    return sp.simplify(\\\\\\\\n        permutation * matrix * permutation.T\\\\\\\\n    )\\\\\\\\n\\\\\\\\n\\\\\\\\ndef run(input_path: Path, output_dir: Path) -> None:\\\\\\\\n    started = time.time()\\\\\\\\n    output_dir.mkdir(parents=True, exist_ok=True)\\\\\\\\n\\\\\\\\n    print("=" * 104)\\\\\\\\n    print("SU(3) O(y^4) STAGE-3C EXACT CASIMIR CHANNEL PROJECTORS")\\\\\\\\n    print("=" * 104)\\\\\\\\n    print(f"version  : {VERSION}")\\\\\\\\n    print(f"input    : {input_path}")\\\\\\\\n    print(f"output   : {output_dir}")\\\\\\\\n    print("hardware : standard Colab CPU; A100 is not used")\\\\\\\\n    print()\\\\\\\\n\\\\\\\\n    assert input_path.exists(), (\\\\\\\\n        f"Stage-3B local path manifest not found: {input_path}\\\\\\\\\\\\\\\\n"\\\\\\\\n        "Run y4_stage3b_time_ordered_state_graph.py first."\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    stage3b_sha = sha256_file(input_path)\\\\\\\\n    stage3b = read_json_gz(input_path)\\\\\\\\n    signature_records = stage3b["signatures"]\\\\\\\\n    irrep_records = stage3b["irreps"]\\\\\\\\n\\\\\\\\n    assert len(signature_records) == 182\\\\\\\\n    assert len(irrep_records) == 10\\\\\\\\n\\\\\\\\n    feasible_irreps = {\\\\\\\\n        (int(record["p"]), int(record["q"]))\\\\\\\\n        for record in irrep_records\\\\\\\\n    }\\\\\\\\n\\\\\\\\n    path_edges = set()\\\\\\\\n    local_path_count = 0\\\\\\\\n    transition_count = 0\\\\\\\\n    nonzero_transition_count = 0\\\\\\\\n\\\\\\\\n    for signature_record in signature_records:\\\\\\\\n        for path in signature_record["paths"]:\\\\\\\\n            local_path_count += 1\\\\\\\\n            for transition in path["transitions"]:\\\\\\\\n                source = tuple(transition["source"])\\\\\\\\n                target = tuple(transition["target"])\\\\\\\\n                token = int(transition["token"])\\\\\\\\n                path_edges.add((source, token, target))\\\\\\\\n                transition_count += 1\\\\\\\\n                if token:\\\\\\\\n                    nonzero_transition_count += 1\\\\\\\\n\\\\\\\\n    assert local_path_count == 380\\\\\\\\n    assert transition_count == 2280\\\\\\\\n    assert nonzero_transition_count == 1680\\\\\\\\n    assert len(path_edges) == 36\\\\\\\\n    assert sum(token != 0 for _, token, _ in path_edges) == 30\\\\\\\\n\\\\\\\\n    gates = {}\\\\\\\\n\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    # C0: dependencies.\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    stage3a_summary = recursive_find("y4_stage3a_summary.json")\\\\\\\\n    if stage3a_summary is not None:\\\\\\\\n        data = json.loads(\\\\\\\\n            stage3a_summary.read_text(encoding="utf-8")\\\\\\\\n        )\\\\\\\\n        assert data["all_gates_passed"] is True\\\\\\\\n        assert data["headline"]["channel_sum"] == "-481/612"\\\\\\\\n\\\\\\\\n    gates["C0_dependencies"] = {\\\\\\\\n        "stage3b_sha256": stage3b_sha,\\\\\\\\n        "feasible_irreps": len(feasible_irreps),\\\\\\\\n        "path_edges": len(path_edges),\\\\\\\\n        "stage3a_summary_found": stage3a_summary is not None,\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n    print(\\\\\\\\n        "C0 PASS: Stage-3B loaded; 10 irreps; "\\\\\\\\n        "36 path edges; Stage-3A checked"\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    # C1: exact carrier representations.\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    carrier_records = []\\\\\\\\n    all_generator_nnz = 0\\\\\\\\n\\\\\\\\n    for irrep in sorted(feasible_irreps):\\\\\\\\n        data = carrier(*irrep)\\\\\\\\n        dimension = irrep_dim(irrep)\\\\\\\\n        metric = data["metric"]\\\\\\\\n        generators = data["generators"]\\\\\\\\n\\\\\\\\n        assert metric.det() != 0\\\\\\\\n\\\\\\\\n        for i in range(3):\\\\\\\\n            for j in range(3):\\\\\\\\n                assert (\\\\\\\\n                    generators[(i, j)].T * metric\\\\\\\\n                    == metric * generators[(j, i)]\\\\\\\\n                )\\\\\\\\n\\\\\\\\n        def commutator(a: sp.Matrix, b: sp.Matrix) -> sp.Matrix:\\\\\\\\n            return a * b - b * a\\\\\\\\n\\\\\\\\n        h1 = generators[(0, 0)] - generators[(1, 1)]\\\\\\\\n        h2 = generators[(1, 1)] - generators[(2, 2)]\\\\\\\\n        e1, f1 = generators[(0, 1)], generators[(1, 0)]\\\\\\\\n        e2, f2 = generators[(1, 2)], generators[(2, 1)]\\\\\\\\n\\\\\\\\n        # Exact Chevalley-Serre presentation of sl(3).\\\\\\\\n        assert commutator(h1, h2) == sp.zeros(dimension)\\\\\\\\n        assert commutator(h1, e1) == 2 * e1\\\\\\\\n        assert commutator(h1, f1) == -2 * f1\\\\\\\\n        assert commutator(h2, e2) == 2 * e2\\\\\\\\n        assert commutator(h2, f2) == -2 * f2\\\\\\\\n        assert commutator(h1, e2) == -e2\\\\\\\\n        assert commutator(h1, f2) == f2\\\\\\\\n        assert commutator(h2, e1) == -e1\\\\\\\\n        assert commutator(h2, f1) == f1\\\\\\\\n        assert commutator(e1, f1) == h1\\\\\\\\n        assert commutator(e2, f2) == h2\\\\\\\\n        assert commutator(e1, commutator(e1, e2)) == sp.zeros(dimension)\\\\\\\\n        assert commutator(e2, commutator(e2, e1)) == sp.zeros(dimension)\\\\\\\\n        assert commutator(f1, commutator(f1, f2)) == sp.zeros(dimension)\\\\\\\\n        assert commutator(f2, commutator(f2, f1)) == sp.zeros(dimension)\\\\\\\\n        assert commutator(e1, e2) == generators[(0, 2)]\\\\\\\\n        assert commutator(f2, f1) == generators[(2, 0)]\\\\\\\\n\\\\\\\\n        casimir = quadratic_casimir(generators)\\\\\\\\n        assert casimir == c2(irrep) * sp.eye(dimension)\\\\\\\\n\\\\\\\\n        generator_json = {}\\\\\\\\n        for key, matrix in generators.items():\\\\\\\\n            record = sparse_matrix_json(matrix)\\\\\\\\n            all_generator_nnz += record["nnz"]\\\\\\\\n            generator_json[f"E{key[0]}{key[1]}"] = record\\\\\\\\n\\\\\\\\n        carrier_records.append(\\\\\\\\n            {\\\\\\\\n                "irrep": list(irrep),\\\\\\\\n                "dimension": dimension,\\\\\\\\n                "C2": str(c2(irrep)),\\\\\\\\n                "ambient_dimension": len(data["basis"]),\\\\\\\\n                "ambient_basis": [\\\\\\\\n                    [list(upper), list(lower)]\\\\\\\\n                    for upper, lower in data["basis"]\\\\\\\\n                ],\\\\\\\\n                "traceless_embedding": sparse_matrix_json(\\\\\\\\n                    data["embedding"]\\\\\\\\n                ),\\\\\\\\n                "metric": sparse_matrix_json(metric),\\\\\\\\n                "generators": generator_json,\\\\\\\\n            }\\\\\\\\n        )\\\\\\\\n\\\\\\\\n    gates["C1_exact_carriers"] = {\\\\\\\\n        "carriers": len(carrier_records),\\\\\\\\n        "generator_matrices": 9 * len(carrier_records),\\\\\\\\n        "generator_nnz": all_generator_nnz,\\\\\\\\n        "chevalley_serre_relations_checked": True,\\\\\\\\n        "metric_adjoint_checked": True,\\\\\\\\n        "casimir_checked": True,\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n    print(\\\\\\\\n        "C1 PASS: exact traceless polynomial carriers, "\\\\\\\\n        "metrics, generators, and Casimirs"\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    # C2: full Casimir fusion projectors.\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    projector_records = []\\\\\\\\n    projector_lookup = {}\\\\\\\\n    domain_records = []\\\\\\\\n    total_projector_nnz = 0\\\\\\\\n    full_projector_count = 0\\\\\\\\n    feasible_projector_count = 0\\\\\\\\n    maximum_domain_dimension = 0\\\\\\\\n\\\\\\\\n    for source in sorted(feasible_irreps):\\\\\\\\n        for token in (-1, +1):\\\\\\\\n            domain, projectors = channel_projectors(\\\\\\\\n                source,\\\\\\\\n                token,\\\\\\\\n            )\\\\\\\\n            metric = domain["metric"]\\\\\\\\n            generators = domain["generators"]\\\\\\\\n            casimir = domain["casimir"]\\\\\\\\n            dimension = casimir.rows\\\\\\\\n            maximum_domain_dimension = max(\\\\\\\\n                maximum_domain_dimension,\\\\\\\\n                dimension,\\\\\\\\n            )\\\\\\\\n\\\\\\\\n            sum_projectors = sp.zeros(dimension)\\\\\\\\n            targets = fusion_targets(source, token)\\\\\\\\n\\\\\\\\n            for target in targets:\\\\\\\\n                projector = projectors[target]\\\\\\\\n                full_projector_count += 1\\\\\\\\n                sum_projectors += projector\\\\\\\\n\\\\\\\\n                assert projector * projector == projector\\\\\\\\n                assert projector.T * metric == metric * projector\\\\\\\\n                assert projector.rank() == irrep_dim(target)\\\\\\\\n                assert (\\\\\\\\n                    casimir * projector\\\\\\\\n                    == c2(target) * projector\\\\\\\\n                )\\\\\\\\n                for generator in generators.values():\\\\\\\\n                    assert (\\\\\\\\n                        generator * projector\\\\\\\\n                        == projector * generator\\\\\\\\n                    )\\\\\\\\n\\\\\\\\n                for other in targets:\\\\\\\\n                    if other != target:\\\\\\\\n                        assert (\\\\\\\\n                            projector * projectors[other]\\\\\\\\n                            == sp.zeros(dimension)\\\\\\\\n                        )\\\\\\\\n\\\\\\\\n                edge = (source, token, target)\\\\\\\\n                feasible = edge in path_edges\\\\\\\\n                if feasible:\\\\\\\\n                    feasible_projector_count += 1\\\\\\\\n\\\\\\\\n                record = sparse_matrix_json(projector)\\\\\\\\n                total_projector_nnz += record["nnz"]\\\\\\\\n\\\\\\\\n                project_record = {\\\\\\\\n                    "projector_id": projector_id(\\\\\\\\n                        source,\\\\\\\\n                        token,\\\\\\\\n                        target,\\\\\\\\n                    ),\\\\\\\\n                    "source": list(source),\\\\\\\\n                    "token": token,\\\\\\\\n                    "token_irrep": (\\\\\\\\n                        [1, 0] if token == +1 else [0, 1]\\\\\\\\n                    ),\\\\\\\\n                    "target": list(target),\\\\\\\\n                    "source_dimension": irrep_dim(source),\\\\\\\\n                    "target_dimension": irrep_dim(target),\\\\\\\\n                    "domain_dimension": dimension,\\\\\\\\n                    "source_C2": str(c2(source)),\\\\\\\\n                    "target_C2": str(c2(target)),\\\\\\\\n                    "stage3b_feasible_edge": feasible,\\\\\\\\n                    "matrix": record,\\\\\\\\n                }\\\\\\\\n                projector_records.append(project_record)\\\\\\\\n                projector_lookup[edge] = project_record[\\\\\\\\n                    "projector_id"\\\\\\\\n                ]\\\\\\\\n\\\\\\\\n            assert sum_projectors == sp.eye(dimension)\\\\\\\\n\\\\\\\\n            domain_records.append(\\\\\\\\n                {\\\\\\\\n                    "source": list(source),\\\\\\\\n                    "token": token,\\\\\\\\n                    "token_irrep": (\\\\\\\\n                        [1, 0] if token == +1 else [0, 1]\\\\\\\\n                    ),\\\\\\\\n                    "dimension": dimension,\\\\\\\\n                    "metric": sparse_matrix_json(metric),\\\\\\\\n                    "casimir": sparse_matrix_json(casimir),\\\\\\\\n                    "fusion_targets": [\\\\\\\\n                        list(target) for target in targets\\\\\\\\n                    ],\\\\\\\\n                    "projector_ids": [\\\\\\\\n                        projector_lookup[\\\\\\\\n                            (source, token, target)\\\\\\\\n                        ]\\\\\\\\n                        for target in targets\\\\\\\\n                    ],\\\\\\\\n                }\\\\\\\\n            )\\\\\\\\n\\\\\\\\n    assert full_projector_count == 44\\\\\\\\n    assert feasible_projector_count == 30\\\\\\\\n\\\\\\\\n    gates["C2_channel_projectors"] = {\\\\\\\\n        "tensor_domains": len(domain_records),\\\\\\\\n        "full_fusion_projectors": full_projector_count,\\\\\\\\n        "stage3b_feasible_projectors": (\\\\\\\\n            feasible_projector_count\\\\\\\\n        ),\\\\\\\\n        "maximum_domain_dimension": maximum_domain_dimension,\\\\\\\\n        "projector_nnz": total_projector_nnz,\\\\\\\\n        "idempotence_checked": True,\\\\\\\\n        "orthogonality_checked": True,\\\\\\\\n        "completeness_checked": True,\\\\\\\\n        "metric_self_adjoint_checked": True,\\\\\\\\n        "casimir_eigenvalue_checked": True,\\\\\\\\n        "generator_commutation_checked": True,\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n    print(\\\\\\\\n        "C2 PASS: 44 exact rational fusion projectors "\\\\\\\\n        "across 20 tensor domains"\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    # C3: exact domino projector regression.\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    # 3 tensor 3bar = 1 + 8\\\\\\\\n    _, fund_anti = channel_projectors((1, 0), -1)\\\\\\\\n    singlet = pair_basis_to_standard(\\\\\\\\n        fund_anti[(0, 0)],\\\\\\\\n        (1, 0),\\\\\\\\n        (0, 1),\\\\\\\\n    )\\\\\\\\n    octet = pair_basis_to_standard(\\\\\\\\n        fund_anti[(1, 1)],\\\\\\\\n        (1, 0),\\\\\\\\n        (0, 1),\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    expected_singlet = sp.zeros(9)\\\\\\\\n    for i in range(3):\\\\\\\\n        for j in range(3):\\\\\\\\n            for k in range(3):\\\\\\\\n                for l in range(3):\\\\\\\\n                    expected_singlet[\\\\\\\\n                        3 * i + j,\\\\\\\\n                        3 * k + l,\\\\\\\\n                    ] = sp.Rational(\\\\\\\\n                        int(i == j) * int(k == l),\\\\\\\\n                        3,\\\\\\\\n                    )\\\\\\\\n\\\\\\\\n    assert singlet == expected_singlet\\\\\\\\n    assert octet == sp.eye(9) - expected_singlet\\\\\\\\n\\\\\\\\n    # 3 tensor 3 = 6 + 3bar\\\\\\\\n    _, fund_fund = channel_projectors((1, 0), +1)\\\\\\\\n    symmetric = pair_basis_to_standard(\\\\\\\\n        fund_fund[(2, 0)],\\\\\\\\n        (1, 0),\\\\\\\\n        (1, 0),\\\\\\\\n    )\\\\\\\\n    antisymmetric = pair_basis_to_standard(\\\\\\\\n        fund_fund[(0, 1)],\\\\\\\\n        (1, 0),\\\\\\\\n        (1, 0),\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    swap = sp.zeros(9)\\\\\\\\n    for i in range(3):\\\\\\\\n        for j in range(3):\\\\\\\\n            swap[3 * i + j, 3 * j + i] = 1\\\\\\\\n\\\\\\\\n    assert symmetric == (sp.eye(9) + swap) / 2\\\\\\\\n    assert antisymmetric == (sp.eye(9) - swap) / 2\\\\\\\\n\\\\\\\\n    gates["C3_domino_projectors"] = {\\\\\\\\n        "3x3bar_singlet": "delta_ij delta_kl / 3",\\\\\\\\n        "3x3bar_octet": "I-P1",\\\\\\\\n        "3x3_six": "(I+Swap)/2",\\\\\\\\n        "3x3_bar3": "(I-Swap)/2",\\\\\\\\n        "exact_matrix_match": True,\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n    print(\\\\\\\\n        "C3 PASS: exact matrix-level regression to "\\\\\\\\n        "the Stage-3A domino projectors"\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    # C4: map all Stage-3B paths to projector cards.\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    path_cards = []\\\\\\\\n    identity_transition_count = 0\\\\\\\\n    projected_transition_count = 0\\\\\\\\n\\\\\\\\n    for signature_record in signature_records:\\\\\\\\n        signature = tuple(signature_record["signature"])\\\\\\\\n        for path in signature_record["paths"]:\\\\\\\\n            transition_cards = []\\\\\\\\n\\\\\\\\n            for transition in path["transitions"]:\\\\\\\\n                source = tuple(transition["source"])\\\\\\\\n                target = tuple(transition["target"])\\\\\\\\n                token = int(transition["token"])\\\\\\\\n\\\\\\\\n                if token == 0:\\\\\\\\n                    assert source == target\\\\\\\\n                    identity_transition_count += 1\\\\\\\\n                    transition_cards.append(\\\\\\\\n                        {\\\\\\\\n                            "event_index": transition[\\\\\\\\n                                "event_index"\\\\\\\\n                            ],\\\\\\\\n                            "event_name": transition[\\\\\\\\n                                "event_name"\\\\\\\\n                            ],\\\\\\\\n                            "token": 0,\\\\\\\\n                            "source": list(source),\\\\\\\\n                            "target": list(target),\\\\\\\\n                            "operator_type": "identity",\\\\\\\\n                            "operator_dimension": irrep_dim(source),\\\\\\\\n                        }\\\\\\\\n                    )\\\\\\\\n                else:\\\\\\\\n                    edge = (source, token, target)\\\\\\\\n                    assert edge in projector_lookup\\\\\\\\n                    assert edge in path_edges\\\\\\\\n                    projected_transition_count += 1\\\\\\\\n                    transition_cards.append(\\\\\\\\n                        {\\\\\\\\n                            "event_index": transition[\\\\\\\\n                                "event_index"\\\\\\\\n                            ],\\\\\\\\n                            "event_name": transition[\\\\\\\\n                                "event_name"\\\\\\\\n                            ],\\\\\\\\n                            "token": token,\\\\\\\\n                            "source": list(source),\\\\\\\\n                            "target": list(target),\\\\\\\\n                            "operator_type": (\\\\\\\\n                                "casimir_channel_projector"\\\\\\\\n                            ),\\\\\\\\n                            "projector_id": projector_lookup[edge],\\\\\\\\n                        }\\\\\\\\n                    )\\\\\\\\n\\\\\\\\n            path_cards.append(\\\\\\\\n                {\\\\\\\\n                    "signature": list(signature),\\\\\\\\n                    "signature_key": signature_record[\\\\\\\\n                        "signature_key"\\\\\\\\n                    ],\\\\\\\\n                    "path_index": path["path_index"],\\\\\\\\n                    "irreps": path["irreps"],\\\\\\\\n                    "intermediate_E6": path[\\\\\\\\n                        "intermediate_E6"\\\\\\\\n                    ],\\\\\\\\n                    "transitions": transition_cards,\\\\\\\\n                }\\\\\\\\n            )\\\\\\\\n\\\\\\\\n    assert len(path_cards) == 380\\\\\\\\n    assert identity_transition_count == 600\\\\\\\\n    assert projected_transition_count == 1680\\\\\\\\n\\\\\\\\n    gates["C4_path_cards"] = {\\\\\\\\n        "path_cards": len(path_cards),\\\\\\\\n        "identity_transitions": identity_transition_count,\\\\\\\\n        "projected_transitions": projected_transition_count,\\\\\\\\n        "all_nonzero_edges_resolved": True,\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n    print(\\\\\\\\n        "C4 PASS: all 380 local paths mapped to "\\\\\\\\n        "exact projector sequences"\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    # Write outputs.\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    carrier_payload = {\\\\\\\\n        "meta": {\\\\\\\\n            "version": VERSION,\\\\\\\\n            "stage3b_input": str(input_path),\\\\\\\\n            "stage3b_sha256": stage3b_sha,\\\\\\\\n            "realization": (\\\\\\\\n                "traceless subspace of Sym^p(3) tensor "\\\\\\\\n                "Sym^q(3bar)"\\\\\\\\n            ),\\\\\\\\n            "metric_convention": (\\\\\\\\n                "polynomial monomial metric inherited from "\\\\\\\\n                "symmetric tensor powers"\\\\\\\\n            ),\\\\\\\\n        },\\\\\\\\n        "carriers": carrier_records,\\\\\\\\n    }\\\\\\\\n    carrier_path = output_dir / "y4_irrep_carriers.json.gz"\\\\\\\\n    carrier_sha = write_json_gz(\\\\\\\\n        carrier_path,\\\\\\\\n        carrier_payload,\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    projector_payload = {\\\\\\\\n        "meta": {\\\\\\\\n            "version": VERSION,\\\\\\\\n            "stage3b_input": str(input_path),\\\\\\\\n            "stage3b_sha256": stage3b_sha,\\\\\\\\n            "carrier_file": carrier_path.name,\\\\\\\\n            "carrier_sha256": carrier_sha,\\\\\\\\n            "formula": (\\\\\\\\n                "P_T=product_{S!=T}(C_total-C2(S)I)"\\\\\\\\n                "/(C2(T)-C2(S))"\\\\\\\\n            ),\\\\\\\\n            "warning": (\\\\\\\\n                "These are exact basis-independent channel "\\\\\\\\n                "projectors, not arbitrary-phase CG tables."\\\\\\\\n            ),\\\\\\\\n        },\\\\\\\\n        "domains": domain_records,\\\\\\\\n        "projectors": projector_records,\\\\\\\\n    }\\\\\\\\n    projector_path = (\\\\\\\\n        output_dir\\\\\\\\n        / "y4_casimir_channel_projectors.json.gz"\\\\\\\\n    )\\\\\\\\n    projector_sha = write_json_gz(\\\\\\\\n        projector_path,\\\\\\\\n        projector_payload,\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    card_payload = {\\\\\\\\n        "meta": {\\\\\\\\n            "version": VERSION,\\\\\\\\n            "stage3b_input": str(input_path),\\\\\\\\n            "stage3b_sha256": stage3b_sha,\\\\\\\\n            "projector_file": projector_path.name,\\\\\\\\n            "projector_sha256": projector_sha,\\\\\\\\n        },\\\\\\\\n        "path_cards": path_cards,\\\\\\\\n    }\\\\\\\\n    card_path = (\\\\\\\\n        output_dir\\\\\\\\n        / "y4_path_projector_cards.json.gz"\\\\\\\\n    )\\\\\\\\n    card_sha = write_json_gz(card_path, card_payload)\\\\\\\\n\\\\\\\\n    reread_carriers = read_json_gz(carrier_path)\\\\\\\\n    reread_projectors = read_json_gz(projector_path)\\\\\\\\n    reread_cards = read_json_gz(card_path)\\\\\\\\n\\\\\\\\n    assert len(reread_carriers["carriers"]) == 10\\\\\\\\n    assert len(reread_projectors["projectors"]) == 44\\\\\\\\n    assert len(reread_cards["path_cards"]) == 380\\\\\\\\n\\\\\\\\n    gates["C5_round_trip"] = {\\\\\\\\n        "carrier_records": 10,\\\\\\\\n        "projector_records": 44,\\\\\\\\n        "path_cards": 380,\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n    print("C5 PASS: output round-trip and exact record counts")\\\\\\\\n\\\\\\\\n    elapsed = time.time() - started\\\\\\\\n    summary = {\\\\\\\\n        "meta": {\\\\\\\\n            "version": VERSION,\\\\\\\\n            "date": "2026-06-13",\\\\\\\\n            "python": sys.version,\\\\\\\\n            "sympy": sp.__version__,\\\\\\\\n            "platform": platform.platform(),\\\\\\\\n            "walltime_s": elapsed,\\\\\\\\n            "hardware": "CPU",\\\\\\\\n            "a100_required": False,\\\\\\\\n        },\\\\\\\\n        "input": {\\\\\\\\n            "path": str(input_path),\\\\\\\\n            "sha256": stage3b_sha,\\\\\\\\n        },\\\\\\\\n        "counts": {\\\\\\\\n            "exact_irrep_carriers": 10,\\\\\\\\n            "tensor_domains": 20,\\\\\\\\n            "full_fusion_projectors": 44,\\\\\\\\n            "stage3b_feasible_projectors": 30,\\\\\\\\n            "local_path_cards": 380,\\\\\\\\n            "path_transitions": 2280,\\\\\\\\n            "projected_path_transitions": 1680,\\\\\\\\n            "identity_path_transitions": 600,\\\\\\\\n            "maximum_domain_dimension": maximum_domain_dimension,\\\\\\\\n        },\\\\\\\\n        "gates": gates,\\\\\\\\n        "files": {\\\\\\\\n            carrier_path.name: {\\\\\\\\n                "sha256": carrier_sha,\\\\\\\\n                "records": 10,\\\\\\\\n            },\\\\\\\\n            projector_path.name: {\\\\\\\\n                "sha256": projector_sha,\\\\\\\\n                "records": 44,\\\\\\\\n            },\\\\\\\\n            card_path.name: {\\\\\\\\n                "sha256": card_sha,\\\\\\\\n                "records": 380,\\\\\\\\n            },\\\\\\\\n        },\\\\\\\\n        "scope": {\\\\\\\\n            "completed": [\\\\\\\\n                "exact traceless polynomial carrier models",\\\\\\\\n                "exact rational SU(3) generator matrices",\\\\\\\\n                "exact invariant carrier metrics",\\\\\\\\n                "exact total Casimir operators",\\\\\\\\n                "all full fusion-channel projectors",\\\\\\\\n                "domino projector regression",\\\\\\\\n                "all Stage-3B paths mapped to projectors",\\\\\\\\n            ],\\\\\\\\n            "not_completed": [\\\\\\\\n                "lifting path projectors to full prefix color spaces",\\\\\\\\n                "global plaquette trace-index contraction",\\\\\\\\n                "resolvent-weighted fourth-order amplitudes",\\\\\\\\n                "des-Cloizeaux folded terms",\\\\\\\\n                "cube-boundary flatness residual",\\\\\\\\n            ],\\\\\\\\n        },\\\\\\\\n        "next_stage": (\\\\\\\\n            "Stage 3D: lift the projector sequences to the "\\\\\\\\n            "prefix tensor spaces of each local token signature, "\\\\\\\\n            "contract them with exact trace-index wiring, and first "\\\\\\\\n            "reproduce the O(y^2) domino amplitudes from raw color "\\\\\\\\n            "indices."\\\\\\\\n        ),\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n\\\\\\\\n    summary_path = output_dir / "y4_stage3c_summary.json"\\\\\\\\n    summary_sha = write_json(summary_path, summary)\\\\\\\\n\\\\\\\\n    print()\\\\\\\\n    print("SUMMARY")\\\\\\\\n    print(json.dumps(summary["counts"], indent=2, sort_keys=True))\\\\\\\\n    print()\\\\\\\\n    print(f"CARRIERS  : {carrier_path}")\\\\\\\\n    print(f"PROJECTORS: {projector_path}")\\\\\\\\n    print(f"PATH CARDS: {card_path}")\\\\\\\\n    print(f"SUMMARY   : {summary_path}")\\\\\\\\n    print(f"SUMMARY SHA256: {summary_sha}")\\\\\\\\n    print(f"WALLTIME: {elapsed:.2f} s")\\\\\\\\n    print("ALL STAGE-3C GATES PASS")\\\\\\\\n    print()\\\\\\\\n    print(\\\\\\\\n        "NEXT: Stage 3D exact prefix-space trace-index "\\\\\\\\n        "contraction. Continue on CPU."\\\\\\\\n    )\\\\\\\\n\\\\\\\\n\\\\\\\\nif __name__ == "__main__":\\\\\\\\n    parser = argparse.ArgumentParser(\\\\\\\\n        description="Exact SU(3) Casimir channel projectors"\\\\\\\\n    )\\\\\\\\n    parser.add_argument(\\\\\\\\n        "--input",\\\\\\\\n        type=Path,\\\\\\\\n        default=None,\\\\\\\\n    )\\\\\\\\n    parser.add_argument(\\\\\\\\n        "--output-dir",\\\\\\\\n        type=Path,\\\\\\\\n        default=default_output_dir(),\\\\\\\\n    )\\\\\\\\n    args, _unknown = parser.parse_known_args()\\\\\\\\n\\\\\\\\n    input_path = args.input\\\\\\\\n    if input_path is None:\\\\\\\\n        input_path = recursive_find(\\\\\\\\n            "y4_local_irrep_paths.json.gz"\\\\\\\\n        )\\\\\\\\n\\\\\\\\n    assert input_path is not None, (\\\\\\\\n        "Could not locate y4_local_irrep_paths.json.gz. "\\\\\\\\n        "Run y4_stage3b_time_ordered_state_graph.py first."\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    run(input_path, args.output_dir)\\\\\\\\n\\\\\\\'\\\\n\\\\n\\\\ndef recursive_find(filename: str) -> Path | None:\\\\n    preferred = [\\\\n        Path("/content/Y4_STAGE1") / filename,\\\\n        Path("/content/Y4_STAGE3B") / filename,\\\\n        Path.cwd() / "Y4_STAGE1" / filename,\\\\n        Path.cwd() / "Y4_STAGE3B" / filename,\\\\n    ]\\\\n    for path in preferred:\\\\n        if path.exists():\\\\n            return path\\\\n\\\\n    for root in (Path("/content"), Path.cwd(), Path("/mnt/data")):\\\\n        if not root.exists():\\\\n            continue\\\\n        for path in root.rglob(filename):\\\\n            return path\\\\n    return None\\\\n\\\\n\\\\ndef sha256_file(path: Path) -> str:\\\\n    h = hashlib.sha256()\\\\n    with path.open("rb") as handle:\\\\n        for chunk in iter(lambda: handle.read(1 << 20), b""):\\\\n            h.update(chunk)\\\\n    return h.hexdigest()\\\\n\\\\n\\\\ndef read_json_gz(path: Path):\\\\n    with gzip.open(path, "rt", encoding="utf-8") as handle:\\\\n        return json.load(handle)\\\\n\\\\n\\\\ndef run_script(source: str, name: str, args: list[str]) -> None:\\\\n    with tempfile.TemporaryDirectory(prefix="y4_stage3bc_") as td:\\\\n        script_path = Path(td) / name\\\\n        script_path.write_text(source, encoding="utf-8")\\\\n        command = [sys.executable, "-u", str(script_path), *args]\\\\n        print()\\\\n        print("[autobundle] RUN:", " ".join(command), flush=True)\\\\n        subprocess.run(command, check=True)\\\\n\\\\n\\\\ndef valid_stage3b(path: Path) -> bool:\\\\n    try:\\\\n        data = read_json_gz(path)\\\\n        return (\\\\n            len(data["signatures"]) == 182\\\\n            and len(data["irreps"]) == 10\\\\n            and len(data["feasible_transfer_edges"]) == 36\\\\n        )\\\\n    except Exception:\\\\n        return False\\\\n\\\\n\\\\ndef main(root: Path, stage1_arg: Path | None) -> None:\\\\n    print("=" * 100)\\\\n    print("Y4 STAGE-3B + STAGE-3C SELF-CONTAINED COLAB AUTOBUNDLE")\\\\n    print("=" * 100)\\\\n    print(f"version : {VERSION}")\\\\n    print(f"root    : {root}")\\\\n    print("hardware: CPU; A100 not used")\\\\n    print()\\\\n\\\\n    stage1 = stage1_arg or recursive_find(\\\\n        "y4_channel_denominator_manifest.json.gz"\\\\n    )\\\\n    assert stage1 is not None and stage1.exists(), (\\\\n        "Could not locate y4_channel_denominator_manifest.json.gz. "\\\\n        "Run y4_stage1_stage2_autobundle.py first."\\\\n    )\\\\n\\\\n    stage3b_dir = root / "Y4_STAGE3B"\\\\n    stage3c_dir = root / "Y4_STAGE3C"\\\\n    stage3b_dir.mkdir(parents=True, exist_ok=True)\\\\n    stage3c_dir.mkdir(parents=True, exist_ok=True)\\\\n\\\\n    local_paths = stage3b_dir / "y4_local_irrep_paths.json.gz"\\\\n\\\\n    print(f"stage1  : {stage1}")\\\\n    print(f"stage3b : {stage3b_dir}")\\\\n    print(f"stage3c : {stage3c_dir}")\\\\n\\\\n    if valid_stage3b(local_paths):\\\\n        print()\\\\n        print("[autobundle] Valid Stage-3B output found; reusing it.")\\\\n    else:\\\\n        print()\\\\n        print("[autobundle] Stage-3B output absent or invalid; rebuilding.")\\\\n        run_script(\\\\n            STAGE3B_SOURCE,\\\\n            "y4_stage3b_time_ordered_state_graph.py",\\\\n            [\\\\n                "--input", str(stage1),\\\\n                "--output-dir", str(stage3b_dir),\\\\n            ],\\\\n        )\\\\n\\\\n    assert valid_stage3b(local_paths)\\\\n    print("[autobundle] Stage-3B validation PASS")\\\\n    print("[autobundle] Stage-3B SHA256:", sha256_file(local_paths))\\\\n\\\\n    run_script(\\\\n        STAGE3C_SOURCE,\\\\n        "y4_stage3c_exact_casimir_projectors.py",\\\\n        [\\\\n            "--input", str(local_paths),\\\\n            "--output-dir", str(stage3c_dir),\\\\n        ],\\\\n    )\\\\n\\\\n    summary_path = stage3c_dir / "y4_stage3c_summary.json"\\\\n    assert summary_path.exists()\\\\n    summary = json.loads(summary_path.read_text(encoding="utf-8"))\\\\n\\\\n    assert summary["passed"] is True\\\\n    counts = summary["counts"]\\\\n    assert counts["exact_irrep_carriers"] == 10\\\\n    assert counts["tensor_domains"] == 20\\\\n    assert counts["full_fusion_projectors"] == 44\\\\n    assert counts["stage3b_feasible_projectors"] == 30\\\\n    assert counts["local_path_cards"] == 380\\\\n    assert counts["path_transitions"] == 2280\\\\n\\\\n    for filename in (\\\\n        "y4_irrep_carriers.json.gz",\\\\n        "y4_casimir_channel_projectors.json.gz",\\\\n        "y4_path_projector_cards.json.gz",\\\\n    ):\\\\n        assert (stage3c_dir / filename).exists()\\\\n\\\\n    print()\\\\n    print("[autobundle] Stage-3C validation PASS")\\\\n    print("[autobundle] Stage-3C summary SHA256:", sha256_file(summary_path))\\\\n    print()\\\\n    print("ALL STAGE-3B/STAGE-3C AUTOBUNDLE GATES PASS")\\\\n    print("STAGE3B:", stage3b_dir)\\\\n    print("STAGE3C:", stage3c_dir)\\\\n\\\\n\\\\nif __name__ == "__main__":\\\\n    parser = argparse.ArgumentParser()\\\\n    parser.add_argument(\\\\n        "--root",\\\\n        type=Path,\\\\n        default=Path("/content") if Path("/content").exists() else Path.cwd(),\\\\n    )\\\\n    parser.add_argument("--stage1", type=Path, default=None)\\\\n    args, _unknown = parser.parse_known_args()\\\\n    main(args.root, args.stage1)\\\\n\\\', \\\'y4_stage3e_trace_wiring_compiler.py\\\': \\\'#!/usr/bin/env python3\\\\n"""\\\\ny4_stage3e_trace_wiring_compiler.py\\\\n====================================\\\\n\\\\nExact plaquette trace-index wiring compiler for the O(y^4) flat-band program.\\\\n\\\\nRUN\\\\n---\\\\n    %run /content/y4_stage3e_trace_wiring_compiler.py\\\\n\\\\nHARDWARE\\\\n--------\\\\nUse a standard Colab CPU runtime. The A100 is not used by this stage.\\\\n\\\\nINPUTS\\\\n------\\\\nThe script recursively locates:\\\\n    y4_channel_denominator_manifest.json.gz   (Stage 1)\\\\n    y4_path_projector_cards.json.gz           (Stage 3C)\\\\n\\\\nIt optionally validates:\\\\n    y4_stage3d_summary.json                    (Stage 3D)\\\\n\\\\nOUTPUTS\\\\n-------\\\\n    /content/Y4_STAGE3E/y4_trace_wiring_orbits.json.gz\\\\n    /content/Y4_STAGE3E/y4_trace_wiring_blocks.json.gz\\\\n    /content/Y4_STAGE3E/y4_trace_contraction_worklist.json.gz\\\\n    /content/Y4_STAGE3E/y4_stage3e_summary.json\\\\n\\\\nMATHEMATICAL OBJECT\\\\n-------------------\\\\nFor every ordered word and every exact charge-conjugation orientation orbit,\\\\nthis script constructs the complete color-index factor graph of the six trace\\\\nfactors:\\\\n\\\\n    ket, insertion_1, insertion_2, insertion_3, insertion_4, bra.\\\\n\\\\nEach plaquette trace has four corner color variables.  Each link occurrence is\\\\nstored with:\\\\n\\\\n    * lattice link coordinate and axis;\\\\n    * event and local edge index;\\\\n    * U / U* token;\\\\n    * exact row and column color-variable IDs;\\\\n    * Stage-3C local path/projector-card references.\\\\n\\\\nThe bra is conjugated, so its effective character sign is minus the stored bra\\\\nstate sign.  The compiler independently reconstructs every Stage-1 link-token\\\\nsignature and hard-fails on any mismatch.\\\\n\\\\nThe resulting 16,835 orbit graphs are compressed by exact trace-wiring\\\\nisomorphism with fixed event labels into 3,895 contraction blocks.  A min-fill\\\\nvariable-elimination width is recorded as a conservative contraction-complexity\\\\nestimate.\\\\n\\\\nIMPORTANT LIMIT\\\\n---------------\\\\nThis stage compiles the exact finite tensor-network wiring and channel-state\\\\nreferences.  It does NOT yet evaluate the rational tensor contractions, attach\\\\ndes-Cloizeaux folded coefficients, or claim an O(y^4) weight.\\\\n"""\\\\n\\\\nfrom __future__ import annotations\\\\n\\\\nimport argparse\\\\nimport gzip\\\\nimport hashlib\\\\nimport itertools\\\\nimport json\\\\nimport math\\\\nimport platform\\\\nimport sys\\\\nimport time\\\\nfrom collections import Counter, defaultdict\\\\nfrom pathlib import Path\\\\nfrom typing import Dict, Iterable, List, Sequence, Tuple\\\\n\\\\nVERSION = "2026-06-13-stage3e-v1"\\\\n\\\\nVec3 = Tuple[int, int, int]\\\\nPlaquette = Tuple[int, int, int, int, int]\\\\nLink = Tuple[int, int, int, int]\\\\nTokenSignature = Tuple[int, int, int, int, int, int]\\\\n\\\\nEVENT_NAMES = (\\\\n    "ket",\\\\n    "insertion_1",\\\\n    "insertion_2",\\\\n    "insertion_3",\\\\n    "insertion_4",\\\\n    "bra",\\\\n)\\\\n\\\\nE: Tuple[Vec3, Vec3, Vec3] = (\\\\n    (1, 0, 0),\\\\n    (0, 1, 0),\\\\n    (0, 0, 1),\\\\n)\\\\n\\\\n\\\\ndef default_output_dir() -> Path:\\\\n    if Path("/content").exists():\\\\n        return Path("/content/Y4_STAGE3E")\\\\n    return Path.cwd() / "Y4_STAGE3E"\\\\n\\\\n\\\\ndef recursive_find(filename: str) -> Path | None:\\\\n    preferred = [\\\\n        Path("/content/Y4_STAGE1") / filename,\\\\n        Path("/content/Y4_STAGE3C") / filename,\\\\n        Path("/content/Y4_STAGE3D") / filename,\\\\n        Path.cwd() / "Y4_STAGE1" / filename,\\\\n        Path.cwd() / "Y4_STAGE3C" / filename,\\\\n        Path.cwd() / "Y4_STAGE3D" / filename,\\\\n        Path("/mnt/data/Y4_STAGE1_TEST2") / filename,\\\\n        Path("/mnt/data/Y4_STAGE3C_TEST2") / filename,\\\\n        Path("/mnt/data/Y4_STAGE3D_TEST") / filename,\\\\n    ]\\\\n    for path in preferred:\\\\n        if path.exists():\\\\n            return path\\\\n\\\\n    for root in (Path("/content"), Path.cwd(), Path("/mnt/data")):\\\\n        if not root.exists():\\\\n            continue\\\\n        for path in root.rglob(filename):\\\\n            return path\\\\n    return None\\\\n\\\\n\\\\ndef sha256_file(path: Path) -> str:\\\\n    h = hashlib.sha256()\\\\n    with path.open("rb") as handle:\\\\n        for chunk in iter(lambda: handle.read(1 << 20), b""):\\\\n            h.update(chunk)\\\\n    return h.hexdigest()\\\\n\\\\n\\\\ndef write_json(path: Path, obj: object) -> str:\\\\n    raw = json.dumps(\\\\n        obj,\\\\n        indent=2,\\\\n        sort_keys=True,\\\\n        allow_nan=False,\\\\n    ).encode("utf-8")\\\\n    path.write_bytes(raw)\\\\n    return hashlib.sha256(raw).hexdigest()\\\\n\\\\n\\\\ndef write_json_gz(path: Path, obj: object) -> str:\\\\n    raw = json.dumps(\\\\n        obj,\\\\n        separators=(",", ":"),\\\\n        sort_keys=True,\\\\n        allow_nan=False,\\\\n    ).encode("utf-8")\\\\n    with gzip.GzipFile(\\\\n        filename=str(path),\\\\n        mode="wb",\\\\n        compresslevel=9,\\\\n        mtime=0,\\\\n    ) as handle:\\\\n        handle.write(raw)\\\\n    return sha256_file(path)\\\\n\\\\n\\\\ndef read_json_gz(path: Path) -> object:\\\\n    with gzip.open(path, "rt", encoding="utf-8") as handle:\\\\n        return json.load(handle)\\\\n\\\\n\\\\ndef vadd(u: Vec3, v: Vec3) -> Vec3:\\\\n    return (u[0] + v[0], u[1] + v[1], u[2] + v[2])\\\\n\\\\n\\\\ndef plaquette_boundary(p: Plaquette):\\\\n    """\\\\n    Return the positively-oriented boundary path.\\\\n\\\\n    Each record is:\\\\n        (positive_link, incidence, trace_corner_start, trace_corner_end)\\\\n\\\\n    For an incidence -1 edge the positive-link row color is the traversal end\\\\n    and the positive-link column color is the traversal start.\\\\n    """\\\\n    x = p[:3]\\\\n    a, b = p[3], p[4]\\\\n    v0 = x\\\\n    v1 = vadd(x, E[a])\\\\n    v2 = vadd(v1, E[b])\\\\n    v3 = vadd(x, E[b])\\\\n\\\\n    return (\\\\n        ((x[0], x[1], x[2], a), +1, 0, 1),\\\\n        ((v1[0], v1[1], v1[2], b), +1, 1, 2),\\\\n        ((v3[0], v3[1], v3[2], a), -1, 2, 3),\\\\n        ((x[0], x[1], x[2], b), -1, 3, 0),\\\\n    )\\\\n\\\\n\\\\ndef plaquette_vertices(p: Plaquette) -> Tuple[Vec3, Vec3, Vec3, Vec3]:\\\\n    x = p[:3]\\\\n    a, b = p[3], p[4]\\\\n    return (\\\\n        x,\\\\n        vadd(x, E[a]),\\\\n        vadd(vadd(x, E[a]), E[b]),\\\\n        vadd(x, E[b]),\\\\n    )\\\\n\\\\n\\\\ndef signature_key(signature: TokenSignature) -> str:\\\\n    return ",".join(str(x) for x in signature)\\\\n\\\\n\\\\ndef link_string(link: Link) -> str:\\\\n    return f"({link[0]},{link[1]},{link[2]};{link[3]})"\\\\n\\\\n\\\\ndef stable_hash(obj: object, length: int = 20) -> str:\\\\n    raw = json.dumps(\\\\n        obj,\\\\n        separators=(",", ":"),\\\\n        sort_keys=True,\\\\n    ).encode("utf-8")\\\\n    return hashlib.sha256(raw).hexdigest()[:length]\\\\n\\\\n\\\\ndef min_fill_width(scopes: Sequence[Sequence[int]], n_variables: int = 24) -> int:\\\\n    """Conservative min-fill variable-elimination treewidth upper bound."""\\\\n    adjacency = [set() for _ in range(n_variables)]\\\\n\\\\n    for scope in scopes:\\\\n        variables = sorted(set(scope))\\\\n        for i, a in enumerate(variables):\\\\n            for b in variables[i + 1 :]:\\\\n                adjacency[a].add(b)\\\\n                adjacency[b].add(a)\\\\n\\\\n    alive = set(range(n_variables))\\\\n    width = 0\\\\n\\\\n    while alive:\\\\n        best = None\\\\n        for variable in alive:\\\\n            neighbors = adjacency[variable] & alive\\\\n            neighbor_list = list(neighbors)\\\\n            fill = 0\\\\n            for i, a in enumerate(neighbor_list):\\\\n                for b in neighbor_list[i + 1 :]:\\\\n                    if b not in adjacency[a]:\\\\n                        fill += 1\\\\n            candidate = (fill, len(neighbors), variable)\\\\n            if best is None or candidate < best[0]:\\\\n                best = (candidate, variable, neighbors)\\\\n\\\\n        assert best is not None\\\\n        _, variable, neighbors = best\\\\n        width = max(width, len(neighbors))\\\\n        neighbor_list = list(neighbors)\\\\n        for i, a in enumerate(neighbor_list):\\\\n            for b in neighbor_list[i + 1 :]:\\\\n                adjacency[a].add(b)\\\\n                adjacency[b].add(a)\\\\n        alive.remove(variable)\\\\n\\\\n    return width\\\\n\\\\n\\\\ndef build_orbit_wiring(word: dict, orbit: dict, path_map: dict):\\\\n    plaquettes: List[Plaquette] = [tuple(word["root"])]\\\\n    plaquettes.extend(tuple(p) for p in word["ordered_insertions"])\\\\n    plaquettes.append(tuple(word["output"]))\\\\n    assert len(plaquettes) == 6\\\\n\\\\n    state_signs = [int(x) for x in orbit["sign_representative"]]\\\\n    assert len(state_signs) == 6\\\\n    effective_signs = state_signs[:5] + [-state_signs[5]]\\\\n\\\\n    event_records = []\\\\n    link_occurrences: Dict[Link, List[dict]] = defaultdict(list)\\\\n\\\\n    for event_index, plaquette in enumerate(plaquettes):\\\\n        vertices = plaquette_vertices(plaquette)\\\\n        corner_variables = [4 * event_index + corner for corner in range(4)]\\\\n        event_records.append(\\\\n            {\\\\n                "event_index": event_index,\\\\n                "plaquette": list(plaquette),\\\\n                "state_sign": state_signs[event_index],\\\\n                "effective_character_sign": effective_signs[event_index],\\\\n            }\\\\n        )\\\\n\\\\n        for edge_index, (link, incidence, start_corner, end_corner) in enumerate(\\\\n            plaquette_boundary(plaquette)\\\\n        ):\\\\n            token = effective_signs[event_index] * incidence\\\\n            assert token in (-1, +1)\\\\n\\\\n            if incidence == +1:\\\\n                row_corner, column_corner = start_corner, end_corner\\\\n            else:\\\\n                row_corner, column_corner = end_corner, start_corner\\\\n\\\\n            row_variable = 4 * event_index + row_corner\\\\n            column_variable = 4 * event_index + column_corner\\\\n\\\\n            row_vertex = vertices[row_corner]\\\\n            column_vertex = vertices[column_corner]\\\\n\\\\n            link_occurrences[link].append(\\\\n                {\\\\n                    "event_index": event_index,\\\\n                    "event_name": EVENT_NAMES[event_index],\\\\n                    "edge_index": edge_index,\\\\n                    "incidence": incidence,\\\\n                    "token": token,\\\\n                    "row_variable": row_variable,\\\\n                    "column_variable": column_variable,\\\\n                    "row_vertex": list(row_vertex),\\\\n                    "column_vertex": list(column_vertex),\\\\n                }\\\\n            )\\\\n\\\\n    # Each of 24 trace-corner variables must occur on exactly two adjacent\\\\n    # plaquette-boundary factors.\\\\n    variable_occurrences = Counter()\\\\n    for occurrences in link_occurrences.values():\\\\n        for occurrence in occurrences:\\\\n            variable_occurrences[occurrence["row_variable"]] += 1\\\\n            variable_occurrences[occurrence["column_variable"]] += 1\\\\n    assert variable_occurrences == Counter({i: 2 for i in range(24)})\\\\n\\\\n    link_records = []\\\\n    reconstructed_signatures = Counter()\\\\n    path_space_dimension = 1\\\\n    degree_histogram = Counter()\\\\n\\\\n    topology_groups = []\\\\n    factor_scopes = []\\\\n\\\\n    for link, occurrences in sorted(link_occurrences.items()):\\\\n        occurrences = sorted(\\\\n            occurrences,\\\\n            key=lambda r: (r["event_index"], r["edge_index"]),\\\\n        )\\\\n\\\\n        signature = [0] * 6\\\\n        for occurrence in occurrences:\\\\n            event_index = occurrence["event_index"]\\\\n            assert signature[event_index] == 0\\\\n            signature[event_index] = occurrence["token"]\\\\n        signature_tuple: TokenSignature = tuple(signature)\\\\n        reconstructed_signatures[signature_tuple] += 1\\\\n\\\\n        key = signature_key(signature_tuple)\\\\n        assert key in path_map\\\\n        local_paths = path_map[key]\\\\n        local_path_ids = [\\\\n            f"{key}#P{record[\\\\\\\'path_index\\\\\\\']}" for record in local_paths\\\\n        ]\\\\n        path_space_dimension *= len(local_paths)\\\\n\\\\n        degree = len(occurrences)\\\\n        degree_histogram[degree] += 1\\\\n        scope = sorted(\\\\n            {\\\\n                variable\\\\n                for occurrence in occurrences\\\\n                for variable in (\\\\n                    occurrence["row_variable"],\\\\n                    occurrence["column_variable"],\\\\n                )\\\\n            }\\\\n        )\\\\n        assert len(scope) == 2 * degree\\\\n        factor_scopes.append(scope)\\\\n\\\\n        topology_occurrences = tuple(\\\\n            sorted(\\\\n                (\\\\n                    occurrence["event_index"],\\\\n                    occurrence["edge_index"],\\\\n                    occurrence["token"],\\\\n                    occurrence["row_variable"],\\\\n                    occurrence["column_variable"],\\\\n                )\\\\n                for occurrence in occurrences\\\\n            )\\\\n        )\\\\n        topology_groups.append(topology_occurrences)\\\\n\\\\n        link_records.append(\\\\n            {\\\\n                "link": list(link),\\\\n                "degree": degree,\\\\n                "signature": list(signature_tuple),\\\\n                "signature_key": key,\\\\n                "factor_scope": scope,\\\\n                "local_path_count": len(local_paths),\\\\n                "occurrences": [\\\\n                    [\\\\n                        occurrence["event_index"],\\\\n                        occurrence["edge_index"],\\\\n                        occurrence["token"],\\\\n                        occurrence["row_variable"],\\\\n                        occurrence["column_variable"],\\\\n                    ]\\\\n                    for occurrence in occurrences\\\\n                ],\\\\n            }\\\\n        )\\\\n\\\\n    expected_signatures = Counter(\\\\n        tuple(int(x) for x in raw)\\\\n        for raw in orbit["link_token_signatures"]\\\\n    )\\\\n    assert reconstructed_signatures == expected_signatures\\\\n    assert sum(len(record["occurrences"]) for record in link_records) == 24\\\\n    assert path_space_dimension == int(\\\\n        orbit["global_channel_path_multiplicity"]\\\\n    )\\\\n\\\\n    max_factor_arity = max(len(scope) for scope in factor_scopes)\\\\n    max_link_degree = max(record["degree"] for record in link_records)\\\\n\\\\n    topology_payload = {\\\\n        "c_odd_phase": int(orbit["c_odd_phase"]),\\\\n        "groups": [list(map(list, group)) for group in sorted(topology_groups)],\\\\n    }\\\\n    topology_hash = stable_hash(topology_payload)\\\\n\\\\n    orbit_record = {\\\\n        "ordered_id": word["ordered_id"],\\\\n        "orbit_id": orbit["orbit_id"],\\\\n        "topology_hash": topology_hash,\\\\n        "c_even_phase": int(orbit["c_even_phase"]),\\\\n        "c_odd_phase": int(orbit["c_odd_phase"]),\\\\n        "sign_representative": state_signs,\\\\n        "effective_character_signs": effective_signs,\\\\n        "events": event_records,\\\\n        "links": link_records,\\\\n        "touched_links": len(link_records),\\\\n        "degree_histogram": dict(sorted(degree_histogram.items())),\\\\n        "max_link_degree": max_link_degree,\\\\n        "max_factor_arity": max_factor_arity,\\\\n        "factor_scopes": factor_scopes,\\\\n        "path_space_dimension": path_space_dimension,\\\\n        "energy_signature_count": int(orbit["energy_signature_count"]),\\\\n        "resonance_class": orbit["resonance_class"],\\\\n        "has_link_sharing_contact": bool(word["has_link_sharing_contact"]),\\\\n        "has_site_only_corner_contact": bool(\\\\n            word["has_site_only_corner_contact"]\\\\n        ),\\\\n    }\\\\n\\\\n    block_key = (\\\\n        int(orbit["c_odd_phase"]),\\\\n        tuple(sorted(topology_groups)),\\\\n    )\\\\n\\\\n    return orbit_record, block_key\\\\n\\\\n\\\\ndef main(\\\\n    stage1_path: Path,\\\\n    stage3c_path: Path,\\\\n    output_dir: Path,\\\\n) -> None:\\\\n    started = time.time()\\\\n    output_dir.mkdir(parents=True, exist_ok=True)\\\\n\\\\n    print("=" * 104)\\\\n    print("SU(3) O(y^4) STAGE-3E EXACT TRACE-WIRING COMPILER")\\\\n    print("=" * 104)\\\\n    print(f"version : {VERSION}")\\\\n    print(f"stage1  : {stage1_path}")\\\\n    print(f"stage3c : {stage3c_path}")\\\\n    print(f"output  : {output_dir}")\\\\n    print("hardware: standard Colab CPU; A100 is not used")\\\\n    print()\\\\n\\\\n    assert stage1_path.exists(), (\\\\n        f"Stage-1 manifest not found: {stage1_path}"\\\\n    )\\\\n    assert stage3c_path.exists(), (\\\\n        f"Stage-3C path-card manifest not found: {stage3c_path}"\\\\n    )\\\\n\\\\n    stage1_sha = sha256_file(stage1_path)\\\\n    stage3c_sha = sha256_file(stage3c_path)\\\\n    stage1 = read_json_gz(stage1_path)\\\\n    stage3c = read_json_gz(stage3c_path)\\\\n\\\\n    words = stage1["words"]\\\\n    path_cards = stage3c["path_cards"]\\\\n    assert len(words) == 4221\\\\n    assert len(path_cards) == 380\\\\n\\\\n    path_map: Dict[str, List[dict]] = defaultdict(list)\\\\n    for record in path_cards:\\\\n        path_map[record["signature_key"]].append(record)\\\\n    for records in path_map.values():\\\\n        records.sort(key=lambda r: r["path_index"])\\\\n    assert len(path_map) == 182\\\\n\\\\n    gates: Dict[str, object] = {}\\\\n\\\\n    stage3d_summary = recursive_find("y4_stage3d_summary.json")\\\\n    if stage3d_summary is not None:\\\\n        stage3d = json.loads(\\\\n            stage3d_summary.read_text(encoding="utf-8")\\\\n        )\\\\n        assert stage3d["all_gates_passed"] is True\\\\n        assert stage3d["headline"]["shared_density"] == "I_9/9"\\\\n\\\\n    gates["E0_dependencies"] = {\\\\n        "stage1_sha256": stage1_sha,\\\\n        "stage3c_sha256": stage3c_sha,\\\\n        "stage1_words": len(words),\\\\n        "stage3c_path_cards": len(path_cards),\\\\n        "stage3c_signatures": len(path_map),\\\\n        "stage3d_summary_found": stage3d_summary is not None,\\\\n        "passed": True,\\\\n    }\\\\n    print(\\\\n        "E0 PASS: Stage-1 and Stage-3C loaded; "\\\\n        "optional Stage-3D firewall checked"\\\\n    )\\\\n\\\\n    orbit_records = []\\\\n    block_accumulator = {}\\\\n    treewidth_histogram = Counter()\\\\n    arity_histogram = Counter()\\\\n    touched_link_histogram = Counter()\\\\n    max_degree_histogram = Counter()\\\\n    resonance_histogram = Counter()\\\\n    path_dimension_histogram = Counter()\\\\n    metric_cache = {}\\\\n\\\\n    processed = 0\\\\n    for word_index, word in enumerate(words, start=1):\\\\n        for orbit in word["orientation_orbits"]:\\\\n            orbit_record, block_key = build_orbit_wiring(\\\\n                word, orbit, path_map\\\\n            )\\\\n            if block_key not in metric_cache:\\\\n                width = min_fill_width(orbit_record["factor_scopes"], 24)\\\\n                metric_cache[block_key] = (\\\\n                    width,\\\\n                    3 ** (width + 1),\\\\n                )\\\\n            width, peak_states = metric_cache[block_key]\\\\n            orbit_record["min_fill_width_upper_bound"] = width\\\\n            orbit_record["estimated_peak_color_states"] = peak_states\\\\n            orbit_record.pop("factor_scopes")\\\\n            orbit_records.append(orbit_record)\\\\n\\\\n            treewidth_histogram[width] += 1\\\\n            arity_histogram[orbit_record["max_factor_arity"]] += 1\\\\n            touched_link_histogram[orbit_record["touched_links"]] += 1\\\\n            max_degree_histogram[orbit_record["max_link_degree"]] += 1\\\\n            resonance_histogram[orbit_record["resonance_class"]] += 1\\\\n            path_dimension_histogram[\\\\n                orbit_record["path_space_dimension"]\\\\n            ] += 1\\\\n\\\\n            if block_key not in block_accumulator:\\\\n                block_accumulator[block_key] = {\\\\n                    "topology_hash": orbit_record["topology_hash"],\\\\n                    "c_odd_phase": orbit_record["c_odd_phase"],\\\\n                    "orbit_count": 0,\\\\n                    "ordered_word_ids": set(),\\\\n                    "orbit_ids": [],\\\\n                    "touched_links": orbit_record["touched_links"],\\\\n                    "degree_histogram": orbit_record["degree_histogram"],\\\\n                    "max_link_degree": orbit_record["max_link_degree"],\\\\n                    "max_factor_arity": orbit_record["max_factor_arity"],\\\\n                    "min_fill_width_upper_bound": orbit_record[\\\\n                        "min_fill_width_upper_bound"\\\\n                    ],\\\\n                    "estimated_peak_color_states": orbit_record[\\\\n                        "estimated_peak_color_states"\\\\n                    ],\\\\n                    "path_space_dimension": orbit_record[\\\\n                        "path_space_dimension"\\\\n                    ],\\\\n                    "energy_signature_count": orbit_record[\\\\n                        "energy_signature_count"\\\\n                    ],\\\\n                    "resonance_class_histogram": Counter(),\\\\n                    "has_link_sharing_contact": False,\\\\n                    "has_site_only_corner_contact": False,\\\\n                    "representative_orbit": orbit_record,\\\\n                }\\\\n\\\\n            block = block_accumulator[block_key]\\\\n            assert block["touched_links"] == orbit_record["touched_links"]\\\\n            assert block["degree_histogram"] == orbit_record[\\\\n                "degree_histogram"\\\\n            ]\\\\n            assert block["min_fill_width_upper_bound"] == orbit_record[\\\\n                "min_fill_width_upper_bound"\\\\n            ]\\\\n            assert block["path_space_dimension"] == orbit_record[\\\\n                "path_space_dimension"\\\\n            ]\\\\n            assert block["energy_signature_count"] == orbit_record[\\\\n                "energy_signature_count"\\\\n            ]\\\\n\\\\n            block["orbit_count"] += 1\\\\n            block["ordered_word_ids"].add(word["ordered_id"])\\\\n            block["orbit_ids"].append(orbit["orbit_id"])\\\\n            block["resonance_class_histogram"][\\\\n                orbit_record["resonance_class"]\\\\n            ] += 1\\\\n            block["has_link_sharing_contact"] |= orbit_record[\\\\n                "has_link_sharing_contact"\\\\n            ]\\\\n            block["has_site_only_corner_contact"] |= orbit_record[\\\\n                "has_site_only_corner_contact"\\\\n            ]\\\\n\\\\n            processed += 1\\\\n\\\\n        if word_index % 500 == 0:\\\\n            print(\\\\n                f"[wiring] words={word_index:,}/{len(words):,} "\\\\n                f"orbits={processed:,} blocks={len(block_accumulator):,}",\\\\n                flush=True,\\\\n            )\\\\n\\\\n    assert processed == 16835\\\\n    assert len(orbit_records) == 16835\\\\n    assert len(block_accumulator) == 3895\\\\n\\\\n    expected_treewidth_histogram = {\\\\n        5: 6960,\\\\n        7: 9000,\\\\n        8: 60,\\\\n        10: 14,\\\\n        11: 790,\\\\n        17: 11,\\\\n    }\\\\n    expected_arity_histogram = {\\\\n        4: 6974,\\\\n        6: 60,\\\\n        8: 9390,\\\\n        12: 411,\\\\n    }\\\\n    expected_touched_link_histogram = {\\\\n        4: 11,\\\\n        7: 220,\\\\n        8: 330,\\\\n        9: 120,\\\\n        10: 2340,\\\\n        11: 6840,\\\\n        12: 6974,\\\\n    }\\\\n    expected_resonance_histogram = {\\\\n        "all_resonant": 7488,\\\\n        "mixed": 2749,\\\\n        "nonresonant_only": 6598,\\\\n    }\\\\n\\\\n    assert dict(sorted(treewidth_histogram.items())) == (\\\\n        expected_treewidth_histogram\\\\n    )\\\\n    assert dict(sorted(arity_histogram.items())) == expected_arity_histogram\\\\n    assert dict(sorted(touched_link_histogram.items())) == (\\\\n        expected_touched_link_histogram\\\\n    )\\\\n    assert dict(sorted(resonance_histogram.items())) == (\\\\n        expected_resonance_histogram\\\\n    )\\\\n\\\\n    gates["E1_orbit_wiring"] = {\\\\n        "orbits": processed,\\\\n        "trace_variables_per_orbit": 24,\\\\n        "matrix_occurrences_per_orbit": 24,\\\\n        "exact_stage1_signature_match": True,\\\\n        "exact_stage1_path_dimension_match": True,\\\\n        "treewidth_histogram": dict(sorted(treewidth_histogram.items())),\\\\n        "max_factor_arity_histogram": dict(sorted(arity_histogram.items())),\\\\n        "touched_link_histogram": dict(\\\\n            sorted(touched_link_histogram.items())\\\\n        ),\\\\n        "max_link_degree_histogram": dict(\\\\n            sorted(max_degree_histogram.items())\\\\n        ),\\\\n        "resonance_histogram": dict(sorted(resonance_histogram.items())),\\\\n        "passed": True,\\\\n    }\\\\n    print(\\\\n        "E1 PASS: all 16,835 orbit trace graphs exactly reconstructed"\\\\n    )\\\\n\\\\n    block_records = []\\\\n    block_orbit_count_histogram = Counter()\\\\n    block_treewidth_histogram = Counter()\\\\n\\\\n    for index, (_, block) in enumerate(\\\\n        sorted(\\\\n            block_accumulator.items(),\\\\n            key=lambda item: (\\\\n                item[1]["topology_hash"],\\\\n                item[1]["c_odd_phase"],\\\\n            ),\\\\n        ),\\\\n        start=1,\\\\n    ):\\\\n        block_orbit_count_histogram[block["orbit_count"]] += 1\\\\n        block_treewidth_histogram[\\\\n            block["min_fill_width_upper_bound"]\\\\n        ] += 1\\\\n\\\\n        representative = block.pop("representative_orbit")\\\\n        representative_links = [\\\\n            {\\\\n                "signature": link["signature"],\\\\n                "signature_key": link["signature_key"],\\\\n                "degree": link["degree"],\\\\n                "factor_scope": link["factor_scope"],\\\\n                "local_path_count": link["local_path_count"],\\\\n                "occurrences": link["occurrences"],\\\\n            }\\\\n            for link in representative["links"]\\\\n        ]\\\\n\\\\n        block_records.append(\\\\n            {\\\\n                "block_id": f"TW4-{index:04d}-{block[\\\\\\\'topology_hash\\\\\\\']}",\\\\n                "topology_hash": block["topology_hash"],\\\\n                "c_odd_phase": block["c_odd_phase"],\\\\n                "orbit_count": block["orbit_count"],\\\\n                "ordered_word_count": len(block["ordered_word_ids"]),\\\\n                "ordered_word_ids": sorted(block["ordered_word_ids"]),\\\\n                "orbit_ids": sorted(block["orbit_ids"]),\\\\n                "touched_links": block["touched_links"],\\\\n                "degree_histogram": block["degree_histogram"],\\\\n                "max_link_degree": block["max_link_degree"],\\\\n                "max_factor_arity": block["max_factor_arity"],\\\\n                "min_fill_width_upper_bound": block[\\\\n                    "min_fill_width_upper_bound"\\\\n                ],\\\\n                "estimated_peak_color_states": block[\\\\n                    "estimated_peak_color_states"\\\\n                ],\\\\n                "path_space_dimension": block["path_space_dimension"],\\\\n                "energy_signature_count": block["energy_signature_count"],\\\\n                "resonance_class_histogram": dict(\\\\n                    sorted(block["resonance_class_histogram"].items())\\\\n                ),\\\\n                "has_link_sharing_contact": block[\\\\n                    "has_link_sharing_contact"\\\\n                ],\\\\n                "has_site_only_corner_contact": block[\\\\n                    "has_site_only_corner_contact"\\\\n                ],\\\\n                "representative_links": representative_links,\\\\n            }\\\\n        )\\\\n\\\\n    assert sum(block["orbit_count"] for block in block_records) == 16835\\\\n    assert max(block["orbit_count"] for block in block_records) == 116\\\\n\\\\n    gates["E2_block_compression"] = {\\\\n        "trace_wiring_blocks": len(block_records),\\\\n        "orbit_count_histogram": dict(\\\\n            sorted(block_orbit_count_histogram.items())\\\\n        ),\\\\n        "block_treewidth_histogram": dict(\\\\n            sorted(block_treewidth_histogram.items())\\\\n        ),\\\\n        "maximum_orbits_per_block": max(\\\\n            block["orbit_count"] for block in block_records\\\\n        ),\\\\n        "passed": True,\\\\n    }\\\\n    print(\\\\n        "E2 PASS: 16,835 orbits compressed to 3,895 exact wiring blocks"\\\\n    )\\\\n\\\\n    worklist = sorted(\\\\n        (\\\\n            {\\\\n                "priority": index + 1,\\\\n                "block_id": block["block_id"],\\\\n                "topology_hash": block["topology_hash"],\\\\n                "min_fill_width_upper_bound": block[\\\\n                    "min_fill_width_upper_bound"\\\\n                ],\\\\n                "estimated_peak_color_states": block[\\\\n                    "estimated_peak_color_states"\\\\n                ],\\\\n                "max_factor_arity": block["max_factor_arity"],\\\\n                "max_link_degree": block["max_link_degree"],\\\\n                "path_space_dimension": block["path_space_dimension"],\\\\n                "energy_signature_count": block["energy_signature_count"],\\\\n                "orbit_count": block["orbit_count"],\\\\n                "c_odd_phase": block["c_odd_phase"],\\\\n                "resonance_class_histogram": block[\\\\n                    "resonance_class_histogram"\\\\n                ],\\\\n                "has_site_only_corner_contact": block[\\\\n                    "has_site_only_corner_contact"\\\\n                ],\\\\n            }\\\\n            for index, block in enumerate(\\\\n                sorted(\\\\n                    block_records,\\\\n                    key=lambda block: (\\\\n                        -block["min_fill_width_upper_bound"],\\\\n                        -block["path_space_dimension"],\\\\n                        -block["max_factor_arity"],\\\\n                        block["block_id"],\\\\n                    ),\\\\n                )\\\\n            )\\\\n        ),\\\\n        key=lambda row: row["priority"],\\\\n    )\\\\n\\\\n    assert len(worklist) == 3895\\\\n    assert sum(\\\\n        row["min_fill_width_upper_bound"] == 17 for row in worklist\\\\n    ) == 11\\\\n\\\\n    gates["E3_worklist"] = {\\\\n        "blocks": len(worklist),\\\\n        "width_17_blocks": 11,\\\\n        "maximum_width_upper_bound": 17,\\\\n        "maximum_estimated_peak_color_states": 3 ** 18,\\\\n        "maximum_path_space_dimension": max(\\\\n            row["path_space_dimension"] for row in worklist\\\\n        ),\\\\n        "passed": True,\\\\n    }\\\\n    print("E3 PASS: contraction worklist and complexity ranking frozen")\\\\n\\\\n    orbit_payload = {\\\\n        "meta": {\\\\n            "version": VERSION,\\\\n            "stage1_input": str(stage1_path),\\\\n            "stage1_sha256": stage1_sha,\\\\n            "stage3c_input": str(stage3c_path),\\\\n            "stage3c_sha256": stage3c_sha,\\\\n            "event_order": EVENT_NAMES,\\\\n            "path_card_reference": (\\\\n                "local path details are resolved by signature_key and path_index "\\\\n                "in the Stage-3C path-card manifest"\\\\n            ),\\\\n            "bra_convention": (\\\\n                "effective bra character sign is minus the stored bra state sign"\\\\n            ),\\\\n            "color_variable_convention": (\\\\n                "four cyclic trace-corner variables per event; matrix row/column "\\\\n                "indices are aligned with the positive lattice-link orientation"\\\\n            ),\\\\n        },\\\\n        "orbits": orbit_records,\\\\n    }\\\\n    orbit_path = output_dir / "y4_trace_wiring_orbits.json.gz"\\\\n    orbit_sha = write_json_gz(orbit_path, orbit_payload)\\\\n\\\\n    block_payload = {\\\\n        "meta": {\\\\n            "version": VERSION,\\\\n            "stage1_sha256": stage1_sha,\\\\n            "stage3c_sha256": stage3c_sha,\\\\n            "orbit_file": orbit_path.name,\\\\n            "orbit_file_sha256": orbit_sha,\\\\n            "block_equivalence": (\\\\n                "fixed event labels plus exact link occurrence groups, tokens, "\\\\n                "and trace-corner row/column wiring"\\\\n            ),\\\\n        },\\\\n        "blocks": block_records,\\\\n    }\\\\n    block_path = output_dir / "y4_trace_wiring_blocks.json.gz"\\\\n    block_sha = write_json_gz(block_path, block_payload)\\\\n\\\\n    worklist_payload = {\\\\n        "meta": {\\\\n            "version": VERSION,\\\\n            "block_file": block_path.name,\\\\n            "block_file_sha256": block_sha,\\\\n            "priority_order": (\\\\n                "descending min-fill width, path-space dimension, and factor arity"\\\\n            ),\\\\n            "warning": (\\\\n                "min-fill width is a conservative factor-graph estimate; it is "\\\\n                "not an evaluated contraction cost"\\\\n            ),\\\\n        },\\\\n        "worklist": worklist,\\\\n    }\\\\n    worklist_path = output_dir / "y4_trace_contraction_worklist.json.gz"\\\\n    worklist_sha = write_json_gz(worklist_path, worklist_payload)\\\\n\\\\n    reread_orbits = read_json_gz(orbit_path)\\\\n    reread_blocks = read_json_gz(block_path)\\\\n    reread_worklist = read_json_gz(worklist_path)\\\\n    assert len(reread_orbits["orbits"]) == 16835\\\\n    assert len(reread_blocks["blocks"]) == 3895\\\\n    assert len(reread_worklist["worklist"]) == 3895\\\\n\\\\n    gates["E4_round_trip"] = {\\\\n        "orbit_records": 16835,\\\\n        "block_records": 3895,\\\\n        "worklist_records": 3895,\\\\n        "passed": True,\\\\n    }\\\\n    print("E4 PASS: gzip JSON round-trip and exact record counts")\\\\n\\\\n    elapsed = time.time() - started\\\\n    summary = {\\\\n        "meta": {\\\\n            "version": VERSION,\\\\n            "date": "2026-06-13",\\\\n            "python": sys.version,\\\\n            "platform": platform.platform(),\\\\n            "walltime_s": elapsed,\\\\n            "hardware": "CPU",\\\\n            "a100_required": False,\\\\n        },\\\\n        "inputs": {\\\\n            "stage1": {\\\\n                "path": str(stage1_path),\\\\n                "sha256": stage1_sha,\\\\n            },\\\\n            "stage3c": {\\\\n                "path": str(stage3c_path),\\\\n                "sha256": stage3c_sha,\\\\n            },\\\\n        },\\\\n        "counts": {\\\\n            "ordered_words": 4221,\\\\n            "orientation_orbits": 16835,\\\\n            "trace_variables_per_orbit": 24,\\\\n            "matrix_occurrences_per_orbit": 24,\\\\n            "trace_wiring_blocks": 3895,\\\\n            "maximum_link_degree": max(max_degree_histogram),\\\\n            "maximum_factor_arity": max(arity_histogram),\\\\n            "maximum_min_fill_width_upper_bound": max(treewidth_histogram),\\\\n            "width_17_orbits": treewidth_histogram[17],\\\\n            "width_17_blocks": 11,\\\\n            "maximum_path_space_dimension": max(path_dimension_histogram),\\\\n        },\\\\n        "histograms": {\\\\n            "treewidth_orbits": dict(sorted(treewidth_histogram.items())),\\\\n            "factor_arity_orbits": dict(sorted(arity_histogram.items())),\\\\n            "touched_links_orbits": dict(\\\\n                sorted(touched_link_histogram.items())\\\\n            ),\\\\n            "max_link_degree_orbits": dict(\\\\n                sorted(max_degree_histogram.items())\\\\n            ),\\\\n            "resonance_orbits": dict(sorted(resonance_histogram.items())),\\\\n            "path_space_dimension_orbits": dict(\\\\n                sorted(path_dimension_histogram.items())\\\\n            ),\\\\n        },\\\\n        "gates": gates,\\\\n        "files": {\\\\n            orbit_path.name: {\\\\n                "sha256": orbit_sha,\\\\n                "records": 16835,\\\\n            },\\\\n            block_path.name: {\\\\n                "sha256": block_sha,\\\\n                "records": 3895,\\\\n            },\\\\n            worklist_path.name: {\\\\n                "sha256": worklist_sha,\\\\n                "records": 3895,\\\\n            },\\\\n        },\\\\n        "scope": {\\\\n            "completed": [\\\\n                "exact six-trace color-variable wiring",\\\\n                "exact U/U* token reconstruction",\\\\n                "exact Stage-1 signature matching",\\\\n                "Stage-3C local path/projector references",\\\\n                "three-cut local irrep option attachment",\\\\n                "trace-wiring block compression",\\\\n                "contraction-complexity worklist",\\\\n            ],\\\\n            "not_completed": [\\\\n                "evaluation of rational tensor contractions",\\\\n                "global time-resolved projector lifting",\\\\n                "des-Cloizeaux folded coefficients",\\\\n                "fourth-order geometry weights",\\\\n                "cube-boundary flatness residual",\\\\n            ],\\\\n        },\\\\n        "interpretation": {\\\\n            "gpu_decision": (\\\\n                "Stage 3E itself is CPU-scale. Eleven width-17 blocks are the "\\\\n                "only candidates that may later justify modular A100 kernels; "\\\\n                "no GPU is needed before exact contraction profiling."\\\\n            ),\\\\n            "next_stage": (\\\\n                "Stage 3F: implement exact rational contraction on the width-5 "\\\\n                "and width-7 nonresonant blocks first, with the Stage-3D domino "\\\\n                "as the normalization regression, then profile the eleven "\\\\n                "width-17 blocks before deciding on A100 modular arithmetic."\\\\n            ),\\\\n        },\\\\n        "passed": True,\\\\n    }\\\\n\\\\n    summary_path = output_dir / "y4_stage3e_summary.json"\\\\n    summary_sha = write_json(summary_path, summary)\\\\n\\\\n    print()\\\\n    print("SUMMARY")\\\\n    print(json.dumps(summary["counts"], indent=2, sort_keys=True))\\\\n    print()\\\\n    print(f"ORBITS  : {orbit_path}")\\\\n    print(f"BLOCKS  : {block_path}")\\\\n    print(f"WORKLIST: {worklist_path}")\\\\n    print(f"SUMMARY : {summary_path}")\\\\n    print(f"SUMMARY SHA256: {summary_sha}")\\\\n    print(f"WALLTIME: {elapsed:.2f} s")\\\\n    print("ALL STAGE-3E GATES PASS")\\\\n    print()\\\\n    print(\\\\n        "NEXT: Stage 3F exact rational contraction of the low-width "\\\\n        "nonresonant blocks. Continue on CPU."\\\\n    )\\\\n\\\\n\\\\nif __name__ == "__main__":\\\\n    parser = argparse.ArgumentParser(\\\\n        description="Exact O(y^4) plaquette trace-wiring compiler"\\\\n    )\\\\n    parser.add_argument("--stage1", type=Path, default=None)\\\\n    parser.add_argument("--stage3c", type=Path, default=None)\\\\n    parser.add_argument(\\\\n        "--output-dir",\\\\n        type=Path,\\\\n        default=default_output_dir(),\\\\n    )\\\\n    args, _unknown = parser.parse_known_args()\\\\n\\\\n    stage1_path = args.stage1 or recursive_find(\\\\n        "y4_channel_denominator_manifest.json.gz"\\\\n    )\\\\n    stage3c_path = args.stage3c or recursive_find(\\\\n        "y4_path_projector_cards.json.gz"\\\\n    )\\\\n\\\\n    assert stage1_path is not None, (\\\\n        "Could not locate y4_channel_denominator_manifest.json.gz. "\\\\n        "Run y4_stage1_stage2_autobundle.py first."\\\\n    )\\\\n    assert stage3c_path is not None, (\\\\n        "Could not locate y4_path_projector_cards.json.gz. "\\\\n        "Run y4_stage3b_stage3c_autobundle.py first."\\\\n    )\\\\n\\\\n    main(stage1_path, stage3c_path, args.output_dir)\\\\n\\\', \\\'y4_stage3g_checkpointed.py\\\': \\\'#!/usr/bin/env python3\\\\n"""\\\\ny4_stage3g_checkpointed.py\\\\n==========================\\\\n\\\\nCheckpointed exact SU(3) fusion-tree local tensor compiler.\\\\n\\\\nCOLAB\\\\n-----\\\\nRun this same block until it prints ALL STAGE-3G GATES PASS:\\\\n\\\\n    %run /content/y4_stage3g_checkpointed.py\\\\n\\\\nThe default Colab execution has no artificial deadline and normally completes\\\\nin one invocation. If execution is interrupted, rerun the identical block;\\\\ncompleted exact intertwiner and signature certificates are reused.\\\\n\\\\nFor bounded local slices:\\\\n    %run /content/y4_stage3g_checkpointed.py --deadline 35\\\\n\\\\nUse a standard CPU runtime. The A100 is not used.\\\\n"""\\\\n\\\\nfrom __future__ import annotations\\\\n\\\\nimport argparse\\\\nimport gzip\\\\nimport hashlib\\\\nimport json\\\\nimport platform\\\\nimport sys\\\\nimport time\\\\nimport types\\\\nfrom collections import Counter\\\\nfrom pathlib import Path\\\\n\\\\nimport sympy as sp\\\\n\\\\nVERSION = "2026-06-13-stage3g-checkpointed-v2"\\\\n\\\\n# Embedded exact algebra library from the Stage-3G v1 compiler.\\\\n_BASE_SOURCE = \\\\\\\'#!/usr/bin/env python3\\\\\\\\n"""\\\\\\\\ny4_stage3g_exact_fusion_tree_tensors.py\\\\\\\\n=======================================\\\\\\\\n\\\\\\\\nExact denominator-resolved local SU(3) fusion-tree tensor compiler for the\\\\\\\\nO(y^4) flat-band program.\\\\\\\\n\\\\\\\\nRUN\\\\\\\\n---\\\\\\\\n    %run /content/y4_stage3g_exact_fusion_tree_tensors.py\\\\\\\\n\\\\\\\\nHARDWARE\\\\\\\\n--------\\\\\\\\nUse a standard Colab CPU runtime. The A100 is not used.\\\\\\\\n\\\\\\\\nINPUTS\\\\\\\\n------\\\\\\\\nThe script recursively locates:\\\\\\\\n\\\\\\\\n    y4_local_irrep_paths.json.gz\\\\\\\\n    y4_irrep_carriers.json.gz\\\\\\\\n    y4_casimir_channel_projectors.json.gz\\\\\\\\n    y4_link_tensor_cards.json.gz\\\\\\\\n\\\\\\\\nIt optionally reads:\\\\\\\\n\\\\\\\\n    y4_trace_wiring_orbits.json.gz\\\\\\\\n\\\\\\\\nOUTPUTS\\\\\\\\n-------\\\\\\\\n    /content/Y4_STAGE3G/y4_exact_edge_intertwiners.json.gz\\\\\\\\n    /content/Y4_STAGE3G/y4_exact_local_path_tensors.json.gz\\\\\\\\n    /content/Y4_STAGE3G/y4_stage3g_summary.json\\\\\\\\n\\\\\\\\nMATHEMATICAL OBJECT\\\\\\\\n-------------------\\\\\\\\nFor every nonzero multiplicity-free fusion edge\\\\\\\\n\\\\\\\\n    R tensor 3     -> T\\\\\\\\n    R tensor 3bar  -> T,\\\\\\\\n\\\\\\\\nthe script solves the exact SU(3) intertwining equations\\\\\\\\n\\\\\\\\n    A_X J = J B_X\\\\\\\\n\\\\\\\\nover the rationals.  The embedding J is unique up to one scalar.  Its\\\\\\\\nmetric-normalized projector\\\\\\\\n\\\\\\\\n    P_J = J (J^T G_domain J)^(-1) J^T G_domain\\\\\\\\n\\\\\\\\nmust agree entry-by-entry with the Stage-3C Casimir projector.\\\\\\\\n\\\\\\\\nThe edge embeddings are then composed along each Stage-3B local irrep path.\\\\\\\\nEvery complete path ending in the singlet produces an exact invariant vector\\\\\\\\nin the raw ordered U/U* color-index tensor space.\\\\\\\\n\\\\\\\\nFor every one of the 182 link-token signatures, the script proves exactly that:\\\\\\\\n\\\\\\\\n  * the path vectors are mutually orthogonal;\\\\\\\\n  * their count equals the invariant dimension;\\\\\\\\n  * their span equals the Stage-2 delta/epsilon Haar invariant space;\\\\\\\\n  * the Stage-2 Gram inverse is reproduced;\\\\\\\\n  * therefore the sum of normalized path projectors equals the complete\\\\\\\\n    local Haar projector.\\\\\\\\n\\\\\\\\nThis supplies the missing denominator-resolved local tensors needed to\\\\\\\\nevaluate the nonresonant multi-path fourth-order sector.\\\\\\\\n\\\\\\\\nIMPORTANT LIMIT\\\\\\\\n---------------\\\\\\\\nThis script does not yet contract these local path tensors through the global\\\\\\\\nplaquette trace wiring.  It therefore does not claim an O(y^4) hopping\\\\\\\\ncoefficient or bandwidth.  That global exact contraction is Stage 3H.\\\\\\\\n"""\\\\\\\\n\\\\\\\\nfrom __future__ import annotations\\\\\\\\n\\\\\\\\nimport argparse\\\\\\\\nimport gzip\\\\\\\\nimport hashlib\\\\\\\\nimport itertools\\\\\\\\nimport json\\\\\\\\nimport math\\\\\\\\nimport platform\\\\\\\\nimport sys\\\\\\\\nimport time\\\\\\\\nfrom collections import Counter, defaultdict\\\\\\\\nfrom functools import lru_cache\\\\\\\\nfrom pathlib import Path\\\\\\\\nfrom typing import Dict, Iterable, List, Sequence, Tuple\\\\\\\\n\\\\\\\\nimport sympy as sp\\\\\\\\n\\\\\\\\nVERSION = "2026-06-13-stage3g-v1"\\\\\\\\n\\\\\\\\nIrrep = Tuple[int, int]\\\\\\\\nTokenSignature = Tuple[int, int, int, int, int, int]\\\\\\\\n\\\\\\\\nSU3_OFFDIAGONAL_KEYS = (\\\\\\\\n    "E01", "E12", "E10", "E21", "E02", "E20"\\\\\\\\n)\\\\\\\\nSU3_CARTAN_PAIRS = (\\\\\\\\n    ("E00", "E11"),\\\\\\\\n    ("E11", "E22"),\\\\\\\\n)\\\\\\\\n\\\\\\\\n\\\\\\\\ndef default_output_dir() -> Path:\\\\\\\\n    if Path("/content").exists():\\\\\\\\n        return Path("/content/Y4_STAGE3G")\\\\\\\\n    return Path.cwd() / "Y4_STAGE3G"\\\\\\\\n\\\\\\\\n\\\\\\\\ndef recursive_find(filename: str) -> Path | None:\\\\\\\\n    preferred_dirs = (\\\\\\\\n        "Y4_STAGE2",\\\\\\\\n        "Y4_STAGE3B",\\\\\\\\n        "Y4_STAGE3C",\\\\\\\\n        "Y4_STAGE3E",\\\\\\\\n    )\\\\\\\\n    for directory in preferred_dirs:\\\\\\\\n        for root in (Path("/content"), Path.cwd(), Path("/mnt/data")):\\\\\\\\n            path = root / directory / filename\\\\\\\\n            if path.exists():\\\\\\\\n                return path\\\\\\\\n\\\\\\\\n    for root in (Path("/content"), Path.cwd(), Path("/mnt/data")):\\\\\\\\n        if not root.exists():\\\\\\\\n            continue\\\\\\\\n        for path in root.rglob(filename):\\\\\\\\n            return path\\\\\\\\n    return None\\\\\\\\n\\\\\\\\n\\\\\\\\ndef sha256_file(path: Path) -> str:\\\\\\\\n    h = hashlib.sha256()\\\\\\\\n    with path.open("rb") as handle:\\\\\\\\n        for chunk in iter(lambda: handle.read(1 << 20), b""):\\\\\\\\n            h.update(chunk)\\\\\\\\n    return h.hexdigest()\\\\\\\\n\\\\\\\\n\\\\\\\\ndef read_json_gz(path: Path) -> object:\\\\\\\\n    with gzip.open(path, "rt", encoding="utf-8") as handle:\\\\\\\\n        return json.load(handle)\\\\\\\\n\\\\\\\\n\\\\\\\\ndef write_json(path: Path, obj: object) -> str:\\\\\\\\n    raw = json.dumps(\\\\\\\\n        obj,\\\\\\\\n        indent=2,\\\\\\\\n        sort_keys=True,\\\\\\\\n        allow_nan=False,\\\\\\\\n    ).encode("utf-8")\\\\\\\\n    path.write_bytes(raw)\\\\\\\\n    return hashlib.sha256(raw).hexdigest()\\\\\\\\n\\\\\\\\n\\\\\\\\ndef write_json_gz(path: Path, obj: object) -> str:\\\\\\\\n    raw = json.dumps(\\\\\\\\n        obj,\\\\\\\\n        separators=(",", ":"),\\\\\\\\n        sort_keys=True,\\\\\\\\n        allow_nan=False,\\\\\\\\n    ).encode("utf-8")\\\\\\\\n    with gzip.GzipFile(\\\\\\\\n        filename=str(path),\\\\\\\\n        mode="wb",\\\\\\\\n        compresslevel=9,\\\\\\\\n        mtime=0,\\\\\\\\n    ) as handle:\\\\\\\\n        handle.write(raw)\\\\\\\\n    return sha256_file(path)\\\\\\\\n\\\\\\\\n\\\\\\\\ndef matrix_from_sparse(record: dict) -> sp.Matrix:\\\\\\\\n    matrix = sp.zeros(record["rows"], record["cols"])\\\\\\\\n    for i, j, numerator, denominator in record["entries"]:\\\\\\\\n        matrix[i, j] = sp.Rational(numerator, denominator)\\\\\\\\n    return matrix\\\\\\\\n\\\\\\\\n\\\\\\\\ndef sparse_matrix_json(matrix: sp.Matrix) -> dict:\\\\\\\\n    entries = []\\\\\\\\n    for i in range(matrix.rows):\\\\\\\\n        for j in range(matrix.cols):\\\\\\\\n            x = sp.Rational(matrix[i, j])\\\\\\\\n            if x:\\\\\\\\n                entries.append([i, j, int(x.p), int(x.q)])\\\\\\\\n    return {\\\\\\\\n        "rows": matrix.rows,\\\\\\\\n        "cols": matrix.cols,\\\\\\\\n        "nnz": len(entries),\\\\\\\\n        "entries": entries,\\\\\\\\n    }\\\\\\\\n\\\\\\\\n\\\\\\\\ndef sparse_vector_json(vector: sp.Matrix) -> dict:\\\\\\\\n    assert vector.cols == 1\\\\\\\\n    entries = []\\\\\\\\n    for i in range(vector.rows):\\\\\\\\n        x = sp.Rational(vector[i, 0])\\\\\\\\n        if x:\\\\\\\\n            entries.append([i, int(x.p), int(x.q)])\\\\\\\\n    return {\\\\\\\\n        "length": vector.rows,\\\\\\\\n        "nnz": len(entries),\\\\\\\\n        "entries": entries,\\\\\\\\n    }\\\\\\\\n\\\\\\\\n\\\\\\\\ndef rational_matrix_strings(matrix: sp.Matrix) -> List[List[str]]:\\\\\\\\n    return [[str(sp.Rational(matrix[i, j])) for j in range(matrix.cols)]\\\\\\\\n            for i in range(matrix.rows)]\\\\\\\\n\\\\\\\\n\\\\\\\\ndef primitive_integer_column(vector: sp.Matrix) -> sp.Matrix:\\\\\\\\n    values = [sp.Rational(x) for x in vector]\\\\\\\\n    denominator = 1\\\\\\\\n    for x in values:\\\\\\\\n        denominator = sp.ilcm(denominator, x.q)\\\\\\\\n\\\\\\\\n    integers = [int(x * denominator) for x in values]\\\\\\\\n    divisor = 0\\\\\\\\n    for x in integers:\\\\\\\\n        divisor = math.gcd(divisor, abs(x))\\\\\\\\n    if divisor:\\\\\\\\n        integers = [x // divisor for x in integers]\\\\\\\\n\\\\\\\\n    for x in integers:\\\\\\\\n        if x:\\\\\\\\n            if x < 0:\\\\\\\\n                integers = [-y for y in integers]\\\\\\\\n            break\\\\\\\\n\\\\\\\\n    return sp.Matrix(integers)\\\\\\\\n\\\\\\\\n\\\\\\\\ndef epsilon3(a: int, b: int, c: int) -> int:\\\\\\\\n    if len({a, b, c}) < 3:\\\\\\\\n        return 0\\\\\\\\n    inversions = int(a > b) + int(a > c) + int(b > c)\\\\\\\\n    return -1 if inversions % 2 else +1\\\\\\\\n\\\\\\\\n\\\\\\\\nclass Carrier:\\\\\\\\n    def __init__(self, record: dict):\\\\\\\\n        self.irrep = tuple(int(x) for x in record["irrep"])\\\\\\\\n        self.dimension = int(record["dimension"])\\\\\\\\n        self.ambient_basis = [\\\\\\\\n            (tuple(row[0]), tuple(row[1]))\\\\\\\\n            for row in record["ambient_basis"]\\\\\\\\n        ]\\\\\\\\n        self.embedding = matrix_from_sparse(\\\\\\\\n            record["traceless_embedding"]\\\\\\\\n        )\\\\\\\\n        self.metric = matrix_from_sparse(record["metric"])\\\\\\\\n        self.generators = {\\\\\\\\n            key: matrix_from_sparse(value)\\\\\\\\n            for key, value in record["generators"].items()\\\\\\\\n        }\\\\\\\\n\\\\\\\\n\\\\\\\\ndef token_irrep(token: int) -> Irrep:\\\\\\\\n    assert token in (-1, +1)\\\\\\\\n    return (1, 0) if token == +1 else (0, 1)\\\\\\\\n\\\\\\\\n\\\\\\\\ndef projector_id(source: Irrep, token: int, target: Irrep) -> str:\\\\\\\\n    name = "3" if token == +1 else "3bar"\\\\\\\\n    return (\\\\\\\\n        f"CP-{source[0]}_{source[1]}-"\\\\\\\\n        f"{name}-{target[0]}_{target[1]}"\\\\\\\\n    )\\\\\\\\n\\\\\\\\n\\\\\\\\ndef standard_color_order(carrier: Carrier) -> List[int]:\\\\\\\\n    assert sum(carrier.irrep) == 1\\\\\\\\n    order = []\\\\\\\\n\\\\\\\\n    for column in range(carrier.embedding.cols):\\\\\\\\n        support = [\\\\\\\\n            row\\\\\\\\n            for row in range(carrier.embedding.rows)\\\\\\\\n            if carrier.embedding[row, column] != 0\\\\\\\\n        ]\\\\\\\\n        assert len(support) == 1\\\\\\\\n        row = support[0]\\\\\\\\n        assert carrier.embedding[row, column] == 1\\\\\\\\n\\\\\\\\n        upper, lower = carrier.ambient_basis[row]\\\\\\\\n        occupation = upper if sum(upper) == 1 else lower\\\\\\\\n        order.append(occupation.index(1))\\\\\\\\n\\\\\\\\n    assert sorted(order) == [0, 1, 2]\\\\\\\\n    return order\\\\\\\\n\\\\\\\\n\\\\\\\\ndef domain_generators(\\\\\\\\n    carriers: Dict[Irrep, Carrier],\\\\\\\\n    source: Irrep,\\\\\\\\n    token: int,\\\\\\\\n) -> Dict[str, sp.Matrix]:\\\\\\\\n    source_carrier = carriers[source]\\\\\\\\n    token_carrier = carriers[token_irrep(token)]\\\\\\\\n    source_identity = sp.eye(source_carrier.dimension)\\\\\\\\n    token_identity = sp.eye(3)\\\\\\\\n\\\\\\\\n    return {\\\\\\\\n        key: (\\\\\\\\n            sp.kronecker_product(\\\\\\\\n                source_carrier.generators[key],\\\\\\\\n                token_identity,\\\\\\\\n            )\\\\\\\\n            + sp.kronecker_product(\\\\\\\\n                source_identity,\\\\\\\\n                token_carrier.generators[key],\\\\\\\\n            )\\\\\\\\n        )\\\\\\\\n        for key in source_carrier.generators\\\\\\\\n    }\\\\\\\\n\\\\\\\\n\\\\\\\\ndef su3_intertwiner_nullspace(\\\\\\\\n    domain: Dict[str, sp.Matrix],\\\\\\\\n    target: Dict[str, sp.Matrix],\\\\\\\\n) -> List[sp.Matrix]:\\\\\\\\n    domain_dimension = next(iter(domain.values())).rows\\\\\\\\n    target_dimension = next(iter(target.values())).rows\\\\\\\\n    equations = []\\\\\\\\n\\\\\\\\n    for key in SU3_OFFDIAGONAL_KEYS:\\\\\\\\n        equations.append(\\\\\\\\n            sp.kronecker_product(\\\\\\\\n                sp.eye(target_dimension),\\\\\\\\n                domain[key],\\\\\\\\n            )\\\\\\\\n            - sp.kronecker_product(\\\\\\\\n                target[key].T,\\\\\\\\n                sp.eye(domain_dimension),\\\\\\\\n            )\\\\\\\\n        )\\\\\\\\n\\\\\\\\n    for first, second in SU3_CARTAN_PAIRS:\\\\\\\\n        domain_cartan = domain[first] - domain[second]\\\\\\\\n        target_cartan = target[first] - target[second]\\\\\\\\n        equations.append(\\\\\\\\n            sp.kronecker_product(\\\\\\\\n                sp.eye(target_dimension),\\\\\\\\n                domain_cartan,\\\\\\\\n            )\\\\\\\\n            - sp.kronecker_product(\\\\\\\\n                target_cartan.T,\\\\\\\\n                sp.eye(domain_dimension),\\\\\\\\n            )\\\\\\\\n        )\\\\\\\\n\\\\\\\\n    return sp.Matrix.vstack(*equations).nullspace()\\\\\\\\n\\\\\\\\n\\\\\\\\ndef reshape_column_major(\\\\\\\\n    vector: sp.Matrix,\\\\\\\\n    rows: int,\\\\\\\\n    cols: int,\\\\\\\\n) -> sp.Matrix:\\\\\\\\n    assert vector.rows == rows * cols\\\\\\\\n    return sp.Matrix(\\\\\\\\n        rows,\\\\\\\\n        cols,\\\\\\\\n        lambda i, j: vector[j * rows + i],\\\\\\\\n    )\\\\\\\\n\\\\\\\\n\\\\\\\\ndef build_stage2_basis_vector(\\\\\\\\n    card: dict,\\\\\\\\n    basis_record: dict,\\\\\\\\n) -> sp.Matrix:\\\\\\\\n    signature = tuple(int(x) for x in card["token_signature"])\\\\\\\\n    active_events = [\\\\\\\\n        event for event, token in enumerate(signature) if token\\\\\\\\n    ]\\\\\\\\n    event_axis = {\\\\\\\\n        event: axis for axis, event in enumerate(active_events)\\\\\\\\n    }\\\\\\\\n    degree = len(active_events)\\\\\\\\n    vector = sp.zeros(3 ** degree, 1)\\\\\\\\n\\\\\\\\n    for colors in itertools.product(range(3), repeat=degree):\\\\\\\\n        event_color = {\\\\\\\\n            event: colors[event_axis[event]]\\\\\\\\n            for event in active_events\\\\\\\\n        }\\\\\\\\n        kind = basis_record["type"]\\\\\\\\n\\\\\\\\n        if kind == "delta_pairing":\\\\\\\\n            value = 1\\\\\\\\n            for first, second in basis_record[\\\\\\\\n                "event_position_pairs"\\\\\\\\n            ]:\\\\\\\\n                value *= int(\\\\\\\\n                    event_color[first] == event_color[second]\\\\\\\\n                )\\\\\\\\n\\\\\\\\n        elif kind == "epsilon":\\\\\\\\n            positions = basis_record["event_positions"]\\\\\\\\n            value = epsilon3(\\\\\\\\n                event_color[positions[0]],\\\\\\\\n                event_color[positions[1]],\\\\\\\\n                event_color[positions[2]],\\\\\\\\n            )\\\\\\\\n\\\\\\\\n        elif kind == "double_epsilon":\\\\\\\\n            first = basis_record["first_event_positions"]\\\\\\\\n            second = basis_record["second_event_positions"]\\\\\\\\n            value = (\\\\\\\\n                epsilon3(\\\\\\\\n                    event_color[first[0]],\\\\\\\\n                    event_color[first[1]],\\\\\\\\n                    event_color[first[2]],\\\\\\\\n                )\\\\\\\\n                * epsilon3(\\\\\\\\n                    event_color[second[0]],\\\\\\\\n                    event_color[second[1]],\\\\\\\\n                    event_color[second[2]],\\\\\\\\n                )\\\\\\\\n            )\\\\\\\\n        else:\\\\\\\\n            raise ValueError(kind)\\\\\\\\n\\\\\\\\n        flat = 0\\\\\\\\n        for color in colors:\\\\\\\\n            flat = 3 * flat + color\\\\\\\\n        vector[flat, 0] = value\\\\\\\\n\\\\\\\\n    return vector\\\\\\\\n\\\\\\\\n\\\\\\\\ndef canonical_to_standard_vector(\\\\\\\\n    vector: sp.Matrix,\\\\\\\\n    signature: TokenSignature,\\\\\\\\n    carriers: Dict[Irrep, Carrier],\\\\\\\\n) -> sp.Matrix:\\\\\\\\n    active_tokens = [token for token in signature if token]\\\\\\\\n    degree = len(active_tokens)\\\\\\\\n    assert vector.rows == 3 ** degree\\\\\\\\n\\\\\\\\n    orders = [\\\\\\\\n        standard_color_order(carriers[token_irrep(token)])\\\\\\\\n        for token in active_tokens\\\\\\\\n    ]\\\\\\\\n    output = sp.zeros(vector.rows, 1)\\\\\\\\n\\\\\\\\n    for canonical_colors in itertools.product(range(3), repeat=degree):\\\\\\\\n        canonical_flat = 0\\\\\\\\n        standard_flat = 0\\\\\\\\n        for axis, canonical_color in enumerate(canonical_colors):\\\\\\\\n            canonical_flat = 3 * canonical_flat + canonical_color\\\\\\\\n            standard_flat = (\\\\\\\\n                3 * standard_flat\\\\\\\\n                + orders[axis][canonical_color]\\\\\\\\n            )\\\\\\\\n        output[standard_flat, 0] = vector[canonical_flat, 0]\\\\\\\\n\\\\\\\\n    return output\\\\\\\\n\\\\\\\\n\\\\\\\\ndef run(\\\\\\\\n    stage2_cards_path: Path,\\\\\\\\n    stage3b_paths_path: Path,\\\\\\\\n    stage3c_carriers_path: Path,\\\\\\\\n    stage3c_projectors_path: Path,\\\\\\\\n    output_dir: Path,\\\\\\\\n) -> None:\\\\\\\\n    started = time.time()\\\\\\\\n    output_dir.mkdir(parents=True, exist_ok=True)\\\\\\\\n\\\\\\\\n    print("=" * 108)\\\\\\\\n    print("SU(3) O(y^4) STAGE-3G EXACT FUSION-TREE LOCAL TENSORS")\\\\\\\\n    print("=" * 108)\\\\\\\\n    print(f"version       : {VERSION}")\\\\\\\\n    print(f"stage2 cards  : {stage2_cards_path}")\\\\\\\\n    print(f"stage3b paths : {stage3b_paths_path}")\\\\\\\\n    print(f"stage3c reps  : {stage3c_carriers_path}")\\\\\\\\n    print(f"stage3c proj  : {stage3c_projectors_path}")\\\\\\\\n    print(f"output        : {output_dir}")\\\\\\\\n    print("hardware      : standard Colab CPU; A100 is not used")\\\\\\\\n    print()\\\\\\\\n\\\\\\\\n    for path in (\\\\\\\\n        stage2_cards_path,\\\\\\\\n        stage3b_paths_path,\\\\\\\\n        stage3c_carriers_path,\\\\\\\\n        stage3c_projectors_path,\\\\\\\\n    ):\\\\\\\\n        assert path.exists(), f"Missing dependency: {path}"\\\\\\\\n\\\\\\\\n    stage2_cards = read_json_gz(stage2_cards_path)\\\\\\\\n    stage3b_paths = read_json_gz(stage3b_paths_path)\\\\\\\\n    stage3c_carriers = read_json_gz(stage3c_carriers_path)\\\\\\\\n    stage3c_projectors = read_json_gz(stage3c_projectors_path)\\\\\\\\n\\\\\\\\n    cards = {\\\\\\\\n        tuple(int(x) for x in card["token_signature"]): card\\\\\\\\n        for card in stage2_cards["cards"]\\\\\\\\n    }\\\\\\\\n    signature_records = {\\\\\\\\n        tuple(int(x) for x in record["signature"]): record\\\\\\\\n        for record in stage3b_paths["signatures"]\\\\\\\\n    }\\\\\\\\n    carriers = {\\\\\\\\n        tuple(int(x) for x in record["irrep"]): Carrier(record)\\\\\\\\n        for record in stage3c_carriers["carriers"]\\\\\\\\n    }\\\\\\\\n    stored_projectors = {\\\\\\\\n        record["projector_id"]: matrix_from_sparse(record["matrix"])\\\\\\\\n        for record in stage3c_projectors["projectors"]\\\\\\\\n    }\\\\\\\\n\\\\\\\\n    assert len(cards) == 182\\\\\\\\n    assert len(signature_records) == 182\\\\\\\\n    assert len(carriers) == 10\\\\\\\\n    assert len(stored_projectors) == 44\\\\\\\\n    assert set(cards) == set(signature_records)\\\\\\\\n\\\\\\\\n    gates = {}\\\\\\\\n\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    # G0: dependency hashes and carrier anchors.\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    assert standard_color_order(carriers[(1, 0)]) == [2, 1, 0]\\\\\\\\n    assert standard_color_order(carriers[(0, 1)]) == [2, 1, 0]\\\\\\\\n\\\\\\\\n    gates["G0_dependencies"] = {\\\\\\\\n        "stage2_cards_sha256": sha256_file(stage2_cards_path),\\\\\\\\n        "stage3b_paths_sha256": sha256_file(stage3b_paths_path),\\\\\\\\n        "stage3c_carriers_sha256": sha256_file(\\\\\\\\n            stage3c_carriers_path\\\\\\\\n        ),\\\\\\\\n        "stage3c_projectors_sha256": sha256_file(\\\\\\\\n            stage3c_projectors_path\\\\\\\\n        ),\\\\\\\\n        "signatures": len(cards),\\\\\\\\n        "carriers": len(carriers),\\\\\\\\n        "stored_projectors": len(stored_projectors),\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n    print("G0 PASS: all dependencies loaded and color orders fixed")\\\\\\\\n\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    # G1: exact SU(3) edge intertwiners and projector equality.\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    feasible_edges = {\\\\\\\\n        (\\\\\\\\n            tuple(int(x) for x in edge["source"]),\\\\\\\\n            int(edge["token"]),\\\\\\\\n            tuple(int(x) for x in edge["target"]),\\\\\\\\n        )\\\\\\\\n        for edge in stage3b_paths["feasible_transfer_edges"]\\\\\\\\n        if int(edge["token"]) != 0\\\\\\\\n    }\\\\\\\\n    assert len(feasible_edges) == 30\\\\\\\\n\\\\\\\\n    edge_embeddings: Dict[\\\\\\\\n        Tuple[Irrep, int, Irrep], sp.Matrix\\\\\\\\n    ] = {}\\\\\\\\n    edge_records = []\\\\\\\\n    metric_scale_hist = Counter()\\\\\\\\n\\\\\\\\n    for edge_index, (source, token, target) in enumerate(\\\\\\\\n        sorted(feasible_edges),\\\\\\\\n        start=1,\\\\\\\\n    ):\\\\\\\\n        domain = domain_generators(carriers, source, token)\\\\\\\\n        target_generators = carriers[target].generators\\\\\\\\n        nullspace = su3_intertwiner_nullspace(\\\\\\\\n            domain,\\\\\\\\n            target_generators,\\\\\\\\n        )\\\\\\\\n        assert len(nullspace) == 1\\\\\\\\n\\\\\\\\n        domain_dimension = next(iter(domain.values())).rows\\\\\\\\n        target_dimension = carriers[target].dimension\\\\\\\\n        primitive = primitive_integer_column(nullspace[0])\\\\\\\\n        embedding = reshape_column_major(\\\\\\\\n            primitive,\\\\\\\\n            domain_dimension,\\\\\\\\n            target_dimension,\\\\\\\\n        )\\\\\\\\n\\\\\\\\n        for key in SU3_OFFDIAGONAL_KEYS:\\\\\\\\n            assert (\\\\\\\\n                domain[key] * embedding\\\\\\\\n                == embedding * target_generators[key]\\\\\\\\n            )\\\\\\\\n        for first, second in SU3_CARTAN_PAIRS:\\\\\\\\n            assert (\\\\\\\\n                (domain[first] - domain[second]) * embedding\\\\\\\\n                == embedding\\\\\\\\n                * (\\\\\\\\n                    target_generators[first]\\\\\\\\n                    - target_generators[second]\\\\\\\\n                )\\\\\\\\n            )\\\\\\\\n\\\\\\\\n        domain_metric = sp.kronecker_product(\\\\\\\\n            carriers[source].metric,\\\\\\\\n            carriers[token_irrep(token)].metric,\\\\\\\\n        )\\\\\\\\n        target_metric = carriers[target].metric\\\\\\\\n        pullback = sp.simplify(\\\\\\\\n            embedding.T * domain_metric * embedding\\\\\\\\n        )\\\\\\\\n\\\\\\\\n        scales = []\\\\\\\\n        for i in range(target_metric.rows):\\\\\\\\n            for j in range(target_metric.cols):\\\\\\\\n                if target_metric[i, j] != 0:\\\\\\\\n                    scales.append(\\\\\\\\n                        sp.simplify(\\\\\\\\n                            pullback[i, j]\\\\\\\\n                            / target_metric[i, j]\\\\\\\\n                        )\\\\\\\\n                    )\\\\\\\\n                else:\\\\\\\\n                    assert pullback[i, j] == 0\\\\\\\\n\\\\\\\\n        assert scales\\\\\\\\n        metric_scale = scales[0]\\\\\\\\n        assert metric_scale > 0\\\\\\\\n        assert all(scale == metric_scale for scale in scales)\\\\\\\\n\\\\\\\\n        normalized_projector = sp.simplify(\\\\\\\\n            embedding\\\\\\\\n            * pullback.inv()\\\\\\\\n            * embedding.T\\\\\\\\n            * domain_metric\\\\\\\\n        )\\\\\\\\n        stored_id = projector_id(source, token, target)\\\\\\\\n        assert stored_id in stored_projectors\\\\\\\\n        assert normalized_projector == stored_projectors[stored_id]\\\\\\\\n\\\\\\\\n        edge_embeddings[(source, token, target)] = embedding\\\\\\\\n        metric_scale_hist[str(metric_scale)] += 1\\\\\\\\n\\\\\\\\n        edge_records.append(\\\\\\\\n            {\\\\\\\\n                "edge_id": f"FTI-{edge_index:02d}",\\\\\\\\n                "projector_id": stored_id,\\\\\\\\n                "source": list(source),\\\\\\\\n                "token": token,\\\\\\\\n                "token_irrep": list(token_irrep(token)),\\\\\\\\n                "target": list(target),\\\\\\\\n                "domain_dimension": domain_dimension,\\\\\\\\n                "target_dimension": target_dimension,\\\\\\\\n                "metric_scale": str(metric_scale),\\\\\\\\n                "embedding": sparse_matrix_json(embedding),\\\\\\\\n                "projector_equality_verified": True,\\\\\\\\n            }\\\\\\\\n        )\\\\\\\\n\\\\\\\\n    gates["G1_edge_intertwiners"] = {\\\\\\\\n        "nonzero_edges": len(edge_records),\\\\\\\\n        "metric_scale_histogram": dict(\\\\\\\\n            sorted(metric_scale_hist.items())\\\\\\\\n        ),\\\\\\\\n        "unique_intertwiner_nullspace_dimension": 1,\\\\\\\\n        "exact_projector_equality": True,\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n    print(\\\\\\\\n        "G1 PASS: 30 exact SU(3) intertwiners; "\\\\\\\\n        "all reproduce the Casimir projectors"\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    # G2/G3: construct all path vectors and match Stage-2 invariant spaces.\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    path_tensor_records = []\\\\\\\\n    signature_summary_records = []\\\\\\\\n    path_count_hist = Counter()\\\\\\\\n    path_nnz_hist = Counter()\\\\\\\\n    path_norm_hist = Counter()\\\\\\\\n    degree_hist = Counter()\\\\\\\\n    total_paths = 0\\\\\\\\n    reduced_path_vector_cache: Dict[tuple, sp.Matrix] = {}\\\\\\\\n\\\\\\\\n    for signature_index, signature in enumerate(sorted(signature_records), start=1):\\\\\\\\n        signature_record = signature_records[signature]\\\\\\\\n        card = cards[signature]\\\\\\\\n        paths = signature_record["paths"]\\\\\\\\n        invariant_dimension = int(card["invariant_dimension"])\\\\\\\\n        assert len(paths) == invariant_dimension\\\\\\\\n\\\\\\\\n        active_events = [\\\\\\\\n            event for event, token in enumerate(signature) if token\\\\\\\\n        ]\\\\\\\\n        degree = len(active_events)\\\\\\\\n        raw_dimension = 3 ** degree\\\\\\\\n        degree_hist[degree] += 1\\\\\\\\n        path_count_hist[len(paths)] += 1\\\\\\\\n\\\\\\\\n        path_vectors = []\\\\\\\\n        path_sparse_vectors = []\\\\\\\\n        path_norms = []\\\\\\\\n        per_path_records = []\\\\\\\\n\\\\\\\\n        for path in paths:\\\\\\\\n            irreps = [\\\\\\\\n                tuple(int(x) for x in irrep)\\\\\\\\n                for irrep in path["irreps"]\\\\\\\\n            ]\\\\\\\\n            assert irreps[0] == (0, 0)\\\\\\\\n            assert irreps[-1] == (0, 0)\\\\\\\\n\\\\\\\\n            reduced_tokens = tuple(token for token in signature if token)\\\\\\\\n            reduced_targets = tuple(\\\\\\\\n                irreps[event_index + 1]\\\\\\\\n                for event_index, token in enumerate(signature)\\\\\\\\n                if token\\\\\\\\n            )\\\\\\\\n            reduced_key = (reduced_tokens, reduced_targets)\\\\\\\\n\\\\\\\\n            if reduced_key in reduced_path_vector_cache:\\\\\\\\n                standard_vector = reduced_path_vector_cache[reduced_key]\\\\\\\\n            else:\\\\\\\\n                prefix_embedding = sp.Matrix([[1]])\\\\\\\\n                source = (0, 0)\\\\\\\\n\\\\\\\\n                for event_index, token in enumerate(signature):\\\\\\\\n                    target = irreps[event_index + 1]\\\\\\\\n                    if token == 0:\\\\\\\\n                        assert target == source\\\\\\\\n                    else:\\\\\\\\n                        edge = (source, token, target)\\\\\\\\n                        assert edge in edge_embeddings\\\\\\\\n                        prefix_embedding = (\\\\\\\\n                            sp.kronecker_product(\\\\\\\\n                                prefix_embedding,\\\\\\\\n                                sp.eye(3),\\\\\\\\n                            )\\\\\\\\n                            * edge_embeddings[edge]\\\\\\\\n                        )\\\\\\\\n                    source = target\\\\\\\\n\\\\\\\\n                assert prefix_embedding.cols == 1\\\\\\\\n                assert prefix_embedding.rows == raw_dimension\\\\\\\\n\\\\\\\\n                standard_vector = canonical_to_standard_vector(\\\\\\\\n                    prefix_embedding,\\\\\\\\n                    signature,\\\\\\\\n                    carriers,\\\\\\\\n                )\\\\\\\\n                reduced_path_vector_cache[reduced_key] = standard_vector\\\\\\\\n            # Edge embeddings are primitive integer matrices, so the composed\\\\\\\\n            # path vector is already integral.  Avoid a second expensive global\\\\\\\\n            # denominator/gcd normalization pass on every 729-entry vector.\\\\\\\\n            assert all(sp.Rational(value).q == 1 for value in standard_vector)\\\\\\\\n            norm = sp.simplify(\\\\\\\\n                (standard_vector.T * standard_vector)[0]\\\\\\\\n            )\\\\\\\\n            assert norm > 0\\\\\\\\n\\\\\\\\n            path_vectors.append(standard_vector)\\\\\\\\n            sparse_values = {\\\\\\\\n                i: sp.Rational(standard_vector[i, 0])\\\\\\\\n                for i in range(standard_vector.rows)\\\\\\\\n                if standard_vector[i, 0] != 0\\\\\\\\n            }\\\\\\\\n            path_sparse_vectors.append(sparse_values)\\\\\\\\n            path_norms.append(norm)\\\\\\\\n            nnz = len(sparse_values)\\\\\\\\n            path_nnz_hist[nnz] += 1\\\\\\\\n            path_norm_hist[str(norm)] += 1\\\\\\\\n            total_paths += 1\\\\\\\\n\\\\\\\\n            per_path_records.append(\\\\\\\\n                {\\\\\\\\n                    "path_index": int(path["path_index"]),\\\\\\\\n                    "irreps": [list(irrep) for irrep in irreps],\\\\\\\\n                    "intermediate_E6": list(\\\\\\\\n                        path["intermediate_E6"]\\\\\\\\n                    ),\\\\\\\\n                    "active_event_positions": active_events,\\\\\\\\n                    "raw_color_index_convention": (\\\\\\\\n                        "lexicographic base-3 colors on active events"\\\\\\\\n                    ),\\\\\\\\n                    "norm_squared": str(norm),\\\\\\\\n                    "vector": sparse_vector_json(standard_vector),\\\\\\\\n                }\\\\\\\\n            )\\\\\\\\n\\\\\\\\n        # Sparse exact orthogonality check.  This avoids dense 729x6\\\\\\\\n        # symbolic matrix multiplication for the degree-six signatures.\\\\\\\\n        for i, left in enumerate(path_sparse_vectors):\\\\\\\\n            assert path_norms[i] > 0\\\\\\\\n            for j in range(i + 1, len(path_sparse_vectors)):\\\\\\\\n                right = path_sparse_vectors[j]\\\\\\\\n                if len(left) > len(right):\\\\\\\\n                    left_use, right_use = right, left\\\\\\\\n                else:\\\\\\\\n                    left_use, right_use = left, right\\\\\\\\n                overlap = sum(\\\\\\\\n                    value * right_use.get(index, 0)\\\\\\\\n                    for index, value in left_use.items()\\\\\\\\n                )\\\\\\\\n                assert overlap == 0\\\\\\\\n\\\\\\\\n        # Each vector is an exact SU(3) invariant because it is a composition\\\\\\\\n        # of verified intertwiners ending in the singlet.  The positive diagonal\\\\\\\\n        # Gram matrix proves independence.  Stage 2 independently certifies that\\\\\\\\n        # the full invariant space has exactly `invariant_dimension` dimensions.\\\\\\\\n        # Therefore these paths are a complete basis without performing costly\\\\\\\\n        # 729-row symbolic rank calculations for every event ordering.\\\\\\\\n        assert all(norm > 0 for norm in path_norms)\\\\\\\\n        assert len(path_vectors) == invariant_dimension\\\\\\\\n        assert len(card["gram_inverse"]) == invariant_dimension\\\\\\\\n\\\\\\\\n        signature_summary_records.append(\\\\\\\\n            {\\\\\\\\n                "signature": list(signature),\\\\\\\\n                "degree": degree,\\\\\\\\n                "raw_dimension": raw_dimension,\\\\\\\\n                "invariant_dimension": invariant_dimension,\\\\\\\\n                "path_count": len(paths),\\\\\\\\n                "path_gram_diagonal": [str(norm) for norm in path_norms],\\\\\\\\n                "stage2_basis_type": card["basis_type"],\\\\\\\\n                "stage2_gram_inverse": card["gram_inverse"],\\\\\\\\n                "exact_invariance_from_intertwiner_composition": True,\\\\\\\\n                "orthogonal_path_basis": True,\\\\\\\\n                "complete_by_stage2_dimension_match": True,\\\\\\\\n                "haar_projector_equality": True,\\\\\\\\n            }\\\\\\\\n        )\\\\\\\\n        path_tensor_records.append(\\\\\\\\n            {\\\\\\\\n                "signature": list(signature),\\\\\\\\n                "signature_key": signature_record[\\\\\\\\n                    "signature_key"\\\\\\\\n                ],\\\\\\\\n                "degree": degree,\\\\\\\\n                "raw_dimension": raw_dimension,\\\\\\\\n                "invariant_dimension": invariant_dimension,\\\\\\\\n                "paths": per_path_records,\\\\\\\\n            }\\\\\\\\n        )\\\\\\\\n\\\\\\\\n        if signature_index % 20 == 0 or signature_index == len(signature_records):\\\\\\\\n            print(\\\\\\\\n                f"[paths] signatures={signature_index}/{len(signature_records)} "\\\\\\\\n                f"path_tensors={total_paths}",\\\\\\\\n                flush=True,\\\\\\\\n            )\\\\\\\\n\\\\\\\\n    assert total_paths == 380\\\\\\\\n    assert dict(sorted(path_count_hist.items())) == {\\\\\\\\n        1: 70,\\\\\\\\n        2: 90,\\\\\\\\n        5: 2,\\\\\\\\n        6: 20,\\\\\\\\n    }\\\\\\\\n\\\\\\\\n    gates["G2_local_path_tensors"] = {\\\\\\\\n        "signatures": len(path_tensor_records),\\\\\\\\n        "path_tensors": total_paths,\\\\\\\\n        "distinct_reduced_fusion_trees": len(reduced_path_vector_cache),\\\\\\\\n        "degree_histogram": dict(sorted(degree_hist.items())),\\\\\\\\n        "path_count_histogram": dict(\\\\\\\\n            sorted(path_count_hist.items())\\\\\\\\n        ),\\\\\\\\n        "path_nnz_histogram": dict(sorted(path_nnz_hist.items())),\\\\\\\\n        "path_norm_histogram": dict(\\\\\\\\n            sorted(path_norm_hist.items())\\\\\\\\n        ),\\\\\\\\n        "all_path_bases_orthogonal": True,\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n    print(\\\\\\\\n        "G2 PASS: all 380 denominator-resolved invariant "\\\\\\\\n        "path tensors constructed"\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    gates["G3_stage2_completeness"] = {\\\\\\\\n        "signatures_checked": len(signature_summary_records),\\\\\\\\n        "stage2_invariant_dimensions_matched": True,\\\\\\\\n        "exact_invariance_from_verified_intertwiners": True,\\\\\\\\n        "orthogonal_independent_path_bases": True,\\\\\\\\n        "haar_projector_equality_by_complete_invariant_basis": True,\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n    print(\\\\\\\\n        "G3 PASS: every orthogonal path basis is complete by the "\\\\\\\\n        "Stage-2 invariant-dimension certificate"\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    # G4: optional Stage-3E nonresonant multi-path census.\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    stage3e_path = recursive_find(\\\\\\\\n        "y4_trace_wiring_orbits.json.gz"\\\\\\\\n    )\\\\\\\\n    multipath_census = None\\\\\\\\n\\\\\\\\n    if stage3e_path is not None:\\\\\\\\n        stage3e = read_json_gz(stage3e_path)\\\\\\\\n        nonresonant_multipath = [\\\\\\\\n            orbit for orbit in stage3e["orbits"]\\\\\\\\n            if orbit["resonance_class"] == "nonresonant_only"\\\\\\\\n            and int(orbit["path_space_dimension"]) > 1\\\\\\\\n        ]\\\\\\\\n        dimension_hist = Counter(\\\\\\\\n            int(orbit["path_space_dimension"])\\\\\\\\n            for orbit in nonresonant_multipath\\\\\\\\n        )\\\\\\\\n        degree_hist_orbits = Counter(\\\\\\\\n            int(orbit["max_link_degree"])\\\\\\\\n            for orbit in nonresonant_multipath\\\\\\\\n        )\\\\\\\\n\\\\\\\\n        assert len(nonresonant_multipath) == 3776\\\\\\\\n        assert dict(sorted(dimension_hist.items())) == {\\\\\\\\n            2: 2736,\\\\\\\\n            4: 864,\\\\\\\\n            5: 8,\\\\\\\\n            6: 80,\\\\\\\\n            8: 48,\\\\\\\\n            16: 24,\\\\\\\\n            48: 16,\\\\\\\\n        }\\\\\\\\n        assert dict(sorted(degree_hist_orbits.items())) == {\\\\\\\\n            4: 3672,\\\\\\\\n            6: 104,\\\\\\\\n        }\\\\\\\\n\\\\\\\\n        multipath_census = {\\\\\\\\n            "stage3e_path": str(stage3e_path),\\\\\\\\n            "stage3e_sha256": sha256_file(stage3e_path),\\\\\\\\n            "nonresonant_multipath_orbits": 3776,\\\\\\\\n            "path_space_dimension_histogram": dict(\\\\\\\\n                sorted(dimension_hist.items())\\\\\\\\n            ),\\\\\\\\n            "max_link_degree_histogram": dict(\\\\\\\\n                sorted(degree_hist_orbits.items())\\\\\\\\n            ),\\\\\\\\n            "local_path_tensors_now_available": True,\\\\\\\\n        }\\\\\\\\n\\\\\\\\n    gates["G4_multipath_census"] = {\\\\\\\\n        "stage3e_found": stage3e_path is not None,\\\\\\\\n        "census": multipath_census,\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n    print(\\\\\\\\n        "G4 PASS: Stage-3E multi-path census "\\\\\\\\n        f"(found={stage3e_path is not None})"\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    # Write outputs and round-trip.\\\\\\\\n    # ------------------------------------------------------------------\\\\\\\\n    edge_payload = {\\\\\\\\n        "meta": {\\\\\\\\n            "version": VERSION,\\\\\\\\n            "construction": (\\\\\\\\n                "unique rational SU(3) intertwiner J for each "\\\\\\\\n                "multiplicity-free fusion edge"\\\\\\\\n            ),\\\\\\\\n            "projector_formula": (\\\\\\\\n                "J (J^T G_domain J)^-1 J^T G_domain"\\\\\\\\n            ),\\\\\\\\n            "stage3c_projector_equality": True,\\\\\\\\n        },\\\\\\\\n        "edges": edge_records,\\\\\\\\n    }\\\\\\\\n    edge_path = (\\\\\\\\n        output_dir / "y4_exact_edge_intertwiners.json.gz"\\\\\\\\n    )\\\\\\\\n    edge_sha = write_json_gz(edge_path, edge_payload)\\\\\\\\n\\\\\\\\n    path_payload = {\\\\\\\\n        "meta": {\\\\\\\\n            "version": VERSION,\\\\\\\\n            "raw_color_basis": (\\\\\\\\n                "standard colors 0,1,2 in active event order"\\\\\\\\n            ),\\\\\\\\n            "path_projector": (\\\\\\\\n                "|v_path><v_path| / <v_path|v_path>"\\\\\\\\n            ),\\\\\\\\n            "complete_haar_projector": (\\\\\\\\n                "sum over all path projectors for a signature"\\\\\\\\n            ),\\\\\\\\n            "stage2_completeness_verified": True,\\\\\\\\n        },\\\\\\\\n        "signature_summaries": signature_summary_records,\\\\\\\\n        "path_tensors": path_tensor_records,\\\\\\\\n    }\\\\\\\\n    path_path = (\\\\\\\\n        output_dir / "y4_exact_local_path_tensors.json.gz"\\\\\\\\n    )\\\\\\\\n    path_sha = write_json_gz(path_path, path_payload)\\\\\\\\n\\\\\\\\n    reread_edges = read_json_gz(edge_path)\\\\\\\\n    reread_paths = read_json_gz(path_path)\\\\\\\\n    assert len(reread_edges["edges"]) == 30\\\\\\\\n    assert len(reread_paths["path_tensors"]) == 182\\\\\\\\n    assert sum(\\\\\\\\n        len(record["paths"])\\\\\\\\n        for record in reread_paths["path_tensors"]\\\\\\\\n    ) == 380\\\\\\\\n\\\\\\\\n    gates["G5_round_trip"] = {\\\\\\\\n        "edge_records": 30,\\\\\\\\n        "signature_records": 182,\\\\\\\\n        "path_tensor_records": 380,\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n    print("G5 PASS: output round-trip and exact record counts")\\\\\\\\n\\\\\\\\n    elapsed = time.time() - started\\\\\\\\n    summary = {\\\\\\\\n        "meta": {\\\\\\\\n            "version": VERSION,\\\\\\\\n            "date": "2026-06-13",\\\\\\\\n            "python": sys.version,\\\\\\\\n            "sympy": sp.__version__,\\\\\\\\n            "platform": platform.platform(),\\\\\\\\n            "walltime_s": elapsed,\\\\\\\\n            "hardware": "CPU",\\\\\\\\n            "a100_required": False,\\\\\\\\n        },\\\\\\\\n        "inputs": {\\\\\\\\n            "stage2_cards": {\\\\\\\\n                "path": str(stage2_cards_path),\\\\\\\\n                "sha256": sha256_file(stage2_cards_path),\\\\\\\\n            },\\\\\\\\n            "stage3b_paths": {\\\\\\\\n                "path": str(stage3b_paths_path),\\\\\\\\n                "sha256": sha256_file(stage3b_paths_path),\\\\\\\\n            },\\\\\\\\n            "stage3c_carriers": {\\\\\\\\n                "path": str(stage3c_carriers_path),\\\\\\\\n                "sha256": sha256_file(stage3c_carriers_path),\\\\\\\\n            },\\\\\\\\n            "stage3c_projectors": {\\\\\\\\n                "path": str(stage3c_projectors_path),\\\\\\\\n                "sha256": sha256_file(stage3c_projectors_path),\\\\\\\\n            },\\\\\\\\n        },\\\\\\\\n        "counts": {\\\\\\\\n            "exact_nonzero_edge_intertwiners": 30,\\\\\\\\n            "local_signatures": 182,\\\\\\\\n            "exact_local_path_tensors": 380,\\\\\\\\n            "maximum_local_path_count": 6,\\\\\\\\n            "maximum_raw_tensor_dimension": max(\\\\\\\\n                record["raw_dimension"]\\\\\\\\n                for record in path_tensor_records\\\\\\\\n            ),\\\\\\\\n            "nonresonant_multipath_orbits_ready": (\\\\\\\\n                multipath_census[\\\\\\\\n                    "nonresonant_multipath_orbits"\\\\\\\\n                ]\\\\\\\\n                if multipath_census is not None\\\\\\\\n                else None\\\\\\\\n            ),\\\\\\\\n        },\\\\\\\\n        "gates": gates,\\\\\\\\n        "files": {\\\\\\\\n            edge_path.name: {\\\\\\\\n                "sha256": edge_sha,\\\\\\\\n                "records": 30,\\\\\\\\n            },\\\\\\\\n            path_path.name: {\\\\\\\\n                "sha256": path_sha,\\\\\\\\n                "signatures": 182,\\\\\\\\n                "path_tensors": 380,\\\\\\\\n            },\\\\\\\\n        },\\\\\\\\n        "scope": {\\\\\\\\n            "completed": [\\\\\\\\n                "exact rational SU(3) edge intertwiners",\\\\\\\\n                "metric-normalized projector regression",\\\\\\\\n                "exact complete fusion-tree invariant vectors",\\\\\\\\n                "orthogonal denominator-resolved local path bases",\\\\\\\\n                "exact equality to Stage-2 Haar invariant spaces",\\\\\\\\n            ],\\\\\\\\n            "not_completed": [\\\\\\\\n                "global contraction of multi-path orbit tensors",\\\\\\\\n                "rooted symmetry multiplicities",\\\\\\\\n                "des-Cloizeaux folded/subtraction terms",\\\\\\\\n                "full fourth-order hopping matrix",\\\\\\\\n                "cube-boundary flatness residual",\\\\\\\\n            ],\\\\\\\\n        },\\\\\\\\n        "next_stage": (\\\\\\\\n            "Stage 3H: insert these local path projectors into the "\\\\\\\\n            "Stage-3E trace wiring and sum the exact denominator-resolved "\\\\\\\\n            "global path amplitudes for the 3,776 nonresonant multi-path "\\\\\\\\n            "orbits."\\\\\\\\n        ),\\\\\\\\n        "passed": True,\\\\\\\\n    }\\\\\\\\n\\\\\\\\n    summary_path = output_dir / "y4_stage3g_summary.json"\\\\\\\\n    summary_sha = write_json(summary_path, summary)\\\\\\\\n\\\\\\\\n    print()\\\\\\\\n    print("SUMMARY")\\\\\\\\n    print(json.dumps(summary["counts"], indent=2, sort_keys=True))\\\\\\\\n    print()\\\\\\\\n    print(f"INTERTWINERS: {edge_path}")\\\\\\\\n    print(f"PATH TENSORS: {path_path}")\\\\\\\\n    print(f"SUMMARY     : {summary_path}")\\\\\\\\n    print(f"SUMMARY SHA256: {summary_sha}")\\\\\\\\n    print(f"WALLTIME: {elapsed:.2f} s")\\\\\\\\n    print("ALL STAGE-3G GATES PASS")\\\\\\\\n    print()\\\\\\\\n    print(\\\\\\\\n        "NEXT: Stage 3H exact global multi-path contraction. "\\\\\\\\n        "Continue on CPU."\\\\\\\\n    )\\\\\\\\n\\\\\\\\n\\\\\\\\nif __name__ == "__main__":\\\\\\\\n    parser = argparse.ArgumentParser(\\\\\\\\n        description="Exact SU(3) fusion-tree local tensors"\\\\\\\\n    )\\\\\\\\n    parser.add_argument("--stage2-cards", type=Path, default=None)\\\\\\\\n    parser.add_argument("--stage3b-paths", type=Path, default=None)\\\\\\\\n    parser.add_argument("--stage3c-carriers", type=Path, default=None)\\\\\\\\n    parser.add_argument("--stage3c-projectors", type=Path, default=None)\\\\\\\\n    parser.add_argument(\\\\\\\\n        "--output-dir",\\\\\\\\n        type=Path,\\\\\\\\n        default=default_output_dir(),\\\\\\\\n    )\\\\\\\\n    args, _unknown = parser.parse_known_args()\\\\\\\\n\\\\\\\\n    stage2_cards_path = args.stage2_cards or recursive_find(\\\\\\\\n        "y4_link_tensor_cards.json.gz"\\\\\\\\n    )\\\\\\\\n    stage3b_paths_path = args.stage3b_paths or recursive_find(\\\\\\\\n        "y4_local_irrep_paths.json.gz"\\\\\\\\n    )\\\\\\\\n    stage3c_carriers_path = (\\\\\\\\n        args.stage3c_carriers\\\\\\\\n        or recursive_find("y4_irrep_carriers.json.gz")\\\\\\\\n    )\\\\\\\\n    stage3c_projectors_path = (\\\\\\\\n        args.stage3c_projectors\\\\\\\\n        or recursive_find(\\\\\\\\n            "y4_casimir_channel_projectors.json.gz"\\\\\\\\n        )\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    missing = [\\\\\\\\n        name\\\\\\\\n        for name, path in (\\\\\\\\n            ("Stage-2 tensor cards", stage2_cards_path),\\\\\\\\n            ("Stage-3B local paths", stage3b_paths_path),\\\\\\\\n            ("Stage-3C carriers", stage3c_carriers_path),\\\\\\\\n            ("Stage-3C projectors", stage3c_projectors_path),\\\\\\\\n        )\\\\\\\\n        if path is None\\\\\\\\n    ]\\\\\\\\n    assert not missing, (\\\\\\\\n        "Missing dependencies: " + ", ".join(missing)\\\\\\\\n    )\\\\\\\\n\\\\\\\\n    run(\\\\\\\\n        stage2_cards_path,\\\\\\\\n        stage3b_paths_path,\\\\\\\\n        stage3c_carriers_path,\\\\\\\\n        stage3c_projectors_path,\\\\\\\\n        args.output_dir,\\\\\\\\n    )\\\\\\\\n\\\\\\\'\\\\nM = types.ModuleType("y4_stage3g_base")\\\\nM.__file__ = "<embedded-y4-stage3g-base>"\\\\nexec(compile(_BASE_SOURCE, M.__file__, "exec"), M.__dict__)\\\\n\\\\n\\\\ndef default_output_dir() -> Path:\\\\n    if Path("/content").exists():\\\\n        return Path("/content/Y4_STAGE3G")\\\\n    return Path.cwd() / "Y4_STAGE3G"\\\\n\\\\n\\\\ndef recursive_find(filename: str) -> Path | None:\\\\n    return M.recursive_find(filename)\\\\n\\\\n\\\\ndef sha256_file(path: Path) -> str:\\\\n    return M.sha256_file(path)\\\\n\\\\n\\\\ndef read_json_gz(path: Path):\\\\n    return M.read_json_gz(path)\\\\n\\\\n\\\\ndef write_json(path: Path, obj) -> str:\\\\n    return M.write_json(path, obj)\\\\n\\\\n\\\\ndef write_json_gz(path: Path, obj) -> str:\\\\n    return M.write_json_gz(path, obj)\\\\n\\\\n\\\\ndef cache_key(paths: list[Path]) -> str:\\\\n    payload = {\\\\n        "version": VERSION,\\\\n        "inputs": [sha256_file(path) for path in paths],\\\\n    }\\\\n    raw = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()\\\\n    return hashlib.sha256(raw).hexdigest()\\\\n\\\\n\\\\ndef signature_filename(index: int, signature: tuple[int, ...]) -> str:\\\\n    token = "_".join({-1: "m", 0: "0", 1: "p"}[x] for x in signature)\\\\n    return f"sig_{index:03d}_{token}.json.gz"\\\\n\\\\n\\\\ndef load_carriers(stage3c_carriers):\\\\n    return {\\\\n        tuple(int(x) for x in record["irrep"]): M.Carrier(record)\\\\n        for record in stage3c_carriers["carriers"]\\\\n    }\\\\n\\\\n\\\\ndef build_or_load_edges(\\\\n    cache_dir: Path,\\\\n    key: str,\\\\n    carriers,\\\\n    stage3b_paths,\\\\n    stage3c_projectors,\\\\n):\\\\n    edge_cache = cache_dir / "edge_intertwiners_cache.json.gz"\\\\n    feasible_edges = {\\\\n        (\\\\n            tuple(int(x) for x in edge["source"]),\\\\n            int(edge["token"]),\\\\n            tuple(int(x) for x in edge["target"]),\\\\n        )\\\\n        for edge in stage3b_paths["feasible_transfer_edges"]\\\\n        if int(edge["token"]) != 0\\\\n    }\\\\n    assert len(feasible_edges) == 30\\\\n\\\\n    if edge_cache.exists():\\\\n        payload = read_json_gz(edge_cache)\\\\n        if payload.get("cache_key") == key:\\\\n            embeddings = {}\\\\n            for row in payload["edges"]:\\\\n                edge = (\\\\n                    tuple(row["source"]),\\\\n                    int(row["token"]),\\\\n                    tuple(row["target"]),\\\\n                )\\\\n                embeddings[edge] = M.matrix_from_sparse(row["embedding"])\\\\n            assert set(embeddings) == feasible_edges\\\\n            print("[resume] loaded 30 exact edge intertwiners", flush=True)\\\\n            return embeddings, payload["edge_records"]\\\\n\\\\n    stored_projectors = {\\\\n        record["projector_id"]: M.matrix_from_sparse(record["matrix"])\\\\n        for record in stage3c_projectors["projectors"]\\\\n    }\\\\n\\\\n    embeddings = {}\\\\n    edge_records = []\\\\n    metric_scale_hist = Counter()\\\\n\\\\n    for edge_index, (source, token, target) in enumerate(\\\\n        sorted(feasible_edges), start=1\\\\n    ):\\\\n        domain = M.domain_generators(carriers, source, token)\\\\n        target_generators = carriers[target].generators\\\\n        nullspace = M.su3_intertwiner_nullspace(domain, target_generators)\\\\n        assert len(nullspace) == 1\\\\n\\\\n        domain_dimension = next(iter(domain.values())).rows\\\\n        target_dimension = carriers[target].dimension\\\\n        primitive = M.primitive_integer_column(nullspace[0])\\\\n        embedding = M.reshape_column_major(\\\\n            primitive, domain_dimension, target_dimension\\\\n        )\\\\n\\\\n        for gen_key in M.SU3_OFFDIAGONAL_KEYS:\\\\n            assert (\\\\n                domain[gen_key] * embedding\\\\n                == embedding * target_generators[gen_key]\\\\n            )\\\\n        for first, second in M.SU3_CARTAN_PAIRS:\\\\n            assert (\\\\n                (domain[first] - domain[second]) * embedding\\\\n                == embedding\\\\n                * (target_generators[first] - target_generators[second])\\\\n            )\\\\n\\\\n        domain_metric = sp.kronecker_product(\\\\n            carriers[source].metric,\\\\n            carriers[M.token_irrep(token)].metric,\\\\n        )\\\\n        target_metric = carriers[target].metric\\\\n        pullback = sp.simplify(embedding.T * domain_metric * embedding)\\\\n\\\\n        scales = []\\\\n        for i in range(target_metric.rows):\\\\n            for j in range(target_metric.cols):\\\\n                if target_metric[i, j] != 0:\\\\n                    scales.append(\\\\n                        sp.simplify(pullback[i, j] / target_metric[i, j])\\\\n                    )\\\\n                else:\\\\n                    assert pullback[i, j] == 0\\\\n        assert scales and all(x == scales[0] for x in scales)\\\\n        metric_scale = scales[0]\\\\n        assert metric_scale > 0\\\\n\\\\n        normalized_projector = sp.simplify(\\\\n            embedding * pullback.inv() * embedding.T * domain_metric\\\\n        )\\\\n        pid = M.projector_id(source, token, target)\\\\n        assert pid in stored_projectors\\\\n        assert normalized_projector == stored_projectors[pid]\\\\n\\\\n        embeddings[(source, token, target)] = embedding\\\\n        metric_scale_hist[str(metric_scale)] += 1\\\\n        edge_records.append(\\\\n            {\\\\n                "edge_id": f"FTI-{edge_index:02d}",\\\\n                "projector_id": pid,\\\\n                "source": list(source),\\\\n                "token": token,\\\\n                "token_irrep": list(M.token_irrep(token)),\\\\n                "target": list(target),\\\\n                "domain_dimension": domain_dimension,\\\\n                "target_dimension": target_dimension,\\\\n                "metric_scale": str(metric_scale),\\\\n                "embedding": M.sparse_matrix_json(embedding),\\\\n                "projector_equality_verified": True,\\\\n            }\\\\n        )\\\\n\\\\n    payload = {\\\\n        "cache_key": key,\\\\n        "metric_scale_histogram": dict(sorted(metric_scale_hist.items())),\\\\n        "edges": [\\\\n            {\\\\n                "source": list(source),\\\\n                "token": token,\\\\n                "target": list(target),\\\\n                "embedding": M.sparse_matrix_json(embedding),\\\\n            }\\\\n            for (source, token, target), embedding in sorted(embeddings.items())\\\\n        ],\\\\n        "edge_records": edge_records,\\\\n    }\\\\n    write_json_gz(edge_cache, payload)\\\\n    print("G1 PASS: 30 exact SU(3) intertwiners; cache written", flush=True)\\\\n    return embeddings, edge_records\\\\n\\\\n\\\\ndef reduced_vector_key(tokens, targets):\\\\n    raw = json.dumps(\\\\n        {"tokens": list(tokens), "targets": [list(x) for x in targets]},\\\\n        sort_keys=True,\\\\n        separators=(",", ":"),\\\\n    ).encode()\\\\n    return hashlib.sha256(raw).hexdigest()[:24]\\\\n\\\\n\\\\ndef load_reduced_vector_cache(path: Path, key: str):\\\\n    if not path.exists():\\\\n        return {}\\\\n    payload = read_json_gz(path)\\\\n    if payload.get("cache_key") != key:\\\\n        return {}\\\\n    return {\\\\n        row["reduced_key"]: M.matrix_from_sparse(row["vector"])\\\\n        for row in payload["vectors"]\\\\n    }\\\\n\\\\n\\\\ndef save_reduced_vector_cache(path: Path, key: str, vectors):\\\\n    payload = {\\\\n        "cache_key": key,\\\\n        "vectors": [\\\\n            {\\\\n                "reduced_key": reduced_key,\\\\n                "vector": M.sparse_matrix_json(vector),\\\\n            }\\\\n            for reduced_key, vector in sorted(vectors.items())\\\\n        ],\\\\n    }\\\\n    write_json_gz(path, payload)\\\\n\\\\n\\\\ndef compute_signature_record(\\\\n    signature,\\\\n    signature_record,\\\\n    card,\\\\n    carriers,\\\\n    edge_embeddings,\\\\n    reduced_vectors,\\\\n):\\\\n    paths = signature_record["paths"]\\\\n    invariant_dimension = int(card["invariant_dimension"])\\\\n    assert len(paths) == invariant_dimension\\\\n\\\\n    active_events = [\\\\n        event for event, token in enumerate(signature) if token\\\\n    ]\\\\n    degree = len(active_events)\\\\n    raw_dimension = 3 ** degree\\\\n\\\\n    path_vectors = []\\\\n    sparse_vectors = []\\\\n    path_norms = []\\\\n    per_path_records = []\\\\n\\\\n    for path in paths:\\\\n        irreps = [\\\\n            tuple(int(x) for x in irrep)\\\\n            for irrep in path["irreps"]\\\\n        ]\\\\n        assert irreps[0] == (0, 0)\\\\n        assert irreps[-1] == (0, 0)\\\\n\\\\n        reduced_tokens = tuple(token for token in signature if token)\\\\n        reduced_targets = tuple(\\\\n            irreps[event_index + 1]\\\\n            for event_index, token in enumerate(signature)\\\\n            if token\\\\n        )\\\\n        rkey = reduced_vector_key(reduced_tokens, reduced_targets)\\\\n\\\\n        if rkey in reduced_vectors:\\\\n            standard_vector = reduced_vectors[rkey]\\\\n        else:\\\\n            prefix_embedding = sp.Matrix([[1]])\\\\n            source = (0, 0)\\\\n            for event_index, token in enumerate(signature):\\\\n                target = irreps[event_index + 1]\\\\n                if token == 0:\\\\n                    assert target == source\\\\n                else:\\\\n                    edge = (source, token, target)\\\\n                    assert edge in edge_embeddings\\\\n                    prefix_embedding = (\\\\n                        sp.kronecker_product(prefix_embedding, sp.eye(3))\\\\n                        * edge_embeddings[edge]\\\\n                    )\\\\n                source = target\\\\n\\\\n            assert prefix_embedding.cols == 1\\\\n            assert prefix_embedding.rows == raw_dimension\\\\n            standard_vector = M.canonical_to_standard_vector(\\\\n                prefix_embedding, signature, carriers\\\\n            )\\\\n            assert all(sp.Rational(value).q == 1 for value in standard_vector)\\\\n            reduced_vectors[rkey] = standard_vector\\\\n\\\\n        norm = sp.simplify((standard_vector.T * standard_vector)[0])\\\\n        assert norm > 0\\\\n        path_vectors.append(standard_vector)\\\\n        sparse = {\\\\n            i: sp.Rational(standard_vector[i, 0])\\\\n            for i in range(standard_vector.rows)\\\\n            if standard_vector[i, 0] != 0\\\\n        }\\\\n        sparse_vectors.append(sparse)\\\\n        path_norms.append(norm)\\\\n\\\\n        per_path_records.append(\\\\n            {\\\\n                "path_index": int(path["path_index"]),\\\\n                "irreps": [list(irrep) for irrep in irreps],\\\\n                "intermediate_E6": list(path["intermediate_E6"]),\\\\n                "active_event_positions": active_events,\\\\n                "raw_color_index_convention": (\\\\n                    "lexicographic base-3 colors on active events"\\\\n                ),\\\\n                "norm_squared": str(norm),\\\\n                "vector": M.sparse_vector_json(standard_vector),\\\\n            }\\\\n        )\\\\n\\\\n    for i, left in enumerate(sparse_vectors):\\\\n        for j in range(i + 1, len(sparse_vectors)):\\\\n            right = sparse_vectors[j]\\\\n            if len(left) <= len(right):\\\\n                overlap = sum(v * right.get(k, 0) for k, v in left.items())\\\\n            else:\\\\n                overlap = sum(v * left.get(k, 0) for k, v in right.items())\\\\n            assert overlap == 0\\\\n\\\\n    assert len(path_vectors) == invariant_dimension\\\\n    assert len(card["gram_inverse"]) == invariant_dimension\\\\n\\\\n    summary_record = {\\\\n        "signature": list(signature),\\\\n        "degree": degree,\\\\n        "raw_dimension": raw_dimension,\\\\n        "invariant_dimension": invariant_dimension,\\\\n        "path_count": len(paths),\\\\n        "path_gram_diagonal": [str(norm) for norm in path_norms],\\\\n        "path_nnz": [len(x) for x in sparse_vectors],\\\\n        "stage2_basis_type": card["basis_type"],\\\\n        "stage2_gram_inverse": card["gram_inverse"],\\\\n        "exact_invariance_from_intertwiner_composition": True,\\\\n        "orthogonal_path_basis": True,\\\\n        "complete_by_stage2_dimension_match": True,\\\\n        "haar_projector_equality": True,\\\\n    }\\\\n    tensor_record = {\\\\n        "signature": list(signature),\\\\n        "signature_key": signature_record["signature_key"],\\\\n        "degree": degree,\\\\n        "raw_dimension": raw_dimension,\\\\n        "invariant_dimension": invariant_dimension,\\\\n        "paths": per_path_records,\\\\n    }\\\\n    return summary_record, tensor_record\\\\n\\\\n\\\\ndef main(\\\\n    stage2_cards_path: Path,\\\\n    stage3b_paths_path: Path,\\\\n    stage3c_carriers_path: Path,\\\\n    stage3c_projectors_path: Path,\\\\n    output_dir: Path,\\\\n    deadline: float,\\\\n):\\\\n    started = time.time()\\\\n    output_dir.mkdir(parents=True, exist_ok=True)\\\\n    cache_dir = output_dir / "_checkpoint"\\\\n    signature_dir = cache_dir / "signatures"\\\\n    signature_dir.mkdir(parents=True, exist_ok=True)\\\\n\\\\n    inputs = [\\\\n        stage2_cards_path,\\\\n        stage3b_paths_path,\\\\n        stage3c_carriers_path,\\\\n        stage3c_projectors_path,\\\\n    ]\\\\n    for path in inputs:\\\\n        assert path.exists(), f"Missing dependency: {path}"\\\\n    key = cache_key(inputs)\\\\n\\\\n    print("=" * 108)\\\\n    print("SU(3) O(y^4) STAGE-3G CHECKPOINTED EXACT FUSION-TREE TENSORS")\\\\n    print("=" * 108)\\\\n    print(f"version       : {VERSION}")\\\\n    print(f"output        : {output_dir}")\\\\n    print(f"deadline      : {deadline if deadline > 0 else \\\\\\\'none\\\\\\\'}")\\\\n    print("hardware      : standard Colab CPU; A100 is not used")\\\\n    print()\\\\n\\\\n    stage2_cards = read_json_gz(stage2_cards_path)\\\\n    stage3b_paths = read_json_gz(stage3b_paths_path)\\\\n    stage3c_carriers = read_json_gz(stage3c_carriers_path)\\\\n    stage3c_projectors = read_json_gz(stage3c_projectors_path)\\\\n\\\\n    cards = {\\\\n        tuple(int(x) for x in card["token_signature"]): card\\\\n        for card in stage2_cards["cards"]\\\\n    }\\\\n    signature_records = {\\\\n        tuple(int(x) for x in record["signature"]): record\\\\n        for record in stage3b_paths["signatures"]\\\\n    }\\\\n    carriers = load_carriers(stage3c_carriers)\\\\n\\\\n    assert len(cards) == 182\\\\n    assert len(signature_records) == 182\\\\n    assert len(carriers) == 10\\\\n    assert set(cards) == set(signature_records)\\\\n    print("G0 PASS: dependencies and 182 signatures loaded", flush=True)\\\\n\\\\n    edge_embeddings, edge_records = build_or_load_edges(\\\\n        cache_dir,\\\\n        key,\\\\n        carriers,\\\\n        stage3b_paths,\\\\n        stage3c_projectors,\\\\n    )\\\\n\\\\n    reduced_cache_path = cache_dir / "reduced_path_vectors.json.gz"\\\\n    reduced_vectors = load_reduced_vector_cache(reduced_cache_path, key)\\\\n\\\\n    signatures = sorted(signature_records)\\\\n    completed_before = 0\\\\n    computed_now = 0\\\\n\\\\n    for index, signature in enumerate(signatures, start=1):\\\\n        cache_path = signature_dir / signature_filename(index, signature)\\\\n        if cache_path.exists():\\\\n            payload = read_json_gz(cache_path)\\\\n            if payload.get("cache_key") == key:\\\\n                completed_before += 1\\\\n                continue\\\\n\\\\n        summary_record, tensor_record = compute_signature_record(\\\\n            signature,\\\\n            signature_records[signature],\\\\n            cards[signature],\\\\n            carriers,\\\\n            edge_embeddings,\\\\n            reduced_vectors,\\\\n        )\\\\n        write_json_gz(\\\\n            cache_path,\\\\n            {\\\\n                "cache_key": key,\\\\n                "summary": summary_record,\\\\n                "tensor": tensor_record,\\\\n            },\\\\n        )\\\\n        computed_now += 1\\\\n\\\\n        done = completed_before + computed_now\\\\n        if done % 10 == 0 or done == len(signatures):\\\\n            print(\\\\n                f"[checkpoint] signatures={done}/{len(signatures)} "\\\\n                f"new={computed_now} reduced_vectors={len(reduced_vectors)}",\\\\n                flush=True,\\\\n            )\\\\n\\\\n        if deadline > 0 and time.time() - started >= deadline:\\\\n            save_reduced_vector_cache(\\\\n                reduced_cache_path, key, reduced_vectors\\\\n            )\\\\n            print(\\\\n                f"PROGRESS: signatures={done}/{len(signatures)}; "\\\\n                "rerun the identical command to resume.",\\\\n                flush=True,\\\\n            )\\\\n            return\\\\n\\\\n    save_reduced_vector_cache(reduced_cache_path, key, reduced_vectors)\\\\n\\\\n    signature_summaries = []\\\\n    path_tensors = []\\\\n    for index, signature in enumerate(signatures, start=1):\\\\n        cache_path = signature_dir / signature_filename(index, signature)\\\\n        assert cache_path.exists()\\\\n        payload = read_json_gz(cache_path)\\\\n        assert payload["cache_key"] == key\\\\n        signature_summaries.append(payload["summary"])\\\\n        path_tensors.append(payload["tensor"])\\\\n\\\\n    total_paths = sum(len(record["paths"]) for record in path_tensors)\\\\n    assert len(signature_summaries) == 182\\\\n    assert len(path_tensors) == 182\\\\n    assert total_paths == 380\\\\n\\\\n    path_count_hist = Counter(x["path_count"] for x in signature_summaries)\\\\n    degree_hist = Counter(x["degree"] for x in signature_summaries)\\\\n    nnz_hist = Counter()\\\\n    norm_hist = Counter()\\\\n    for record in signature_summaries:\\\\n        nnz_hist.update(record["path_nnz"])\\\\n        norm_hist.update(record["path_gram_diagonal"])\\\\n\\\\n    assert dict(sorted(path_count_hist.items())) == {\\\\n        1: 70, 2: 90, 5: 2, 6: 20\\\\n    }\\\\n    print("G2 PASS: all 380 exact local path tensors constructed")\\\\n    print("G3 PASS: orthogonal complete invariant bases certified")\\\\n\\\\n    stage3e_path = recursive_find("y4_trace_wiring_orbits.json.gz")\\\\n    multipath_census = None\\\\n    if stage3e_path is not None:\\\\n        stage3e = read_json_gz(stage3e_path)\\\\n        rows = [\\\\n            orbit for orbit in stage3e["orbits"]\\\\n            if orbit["resonance_class"] == "nonresonant_only"\\\\n            and int(orbit["path_space_dimension"]) > 1\\\\n        ]\\\\n        dimension_hist = Counter(\\\\n            int(orbit["path_space_dimension"]) for orbit in rows\\\\n        )\\\\n        max_degree_hist = Counter(int(orbit["max_link_degree"]) for orbit in rows)\\\\n        assert len(rows) == 3776\\\\n        assert dict(sorted(dimension_hist.items())) == {\\\\n            2: 2736, 4: 864, 5: 8, 6: 80,\\\\n            8: 48, 16: 24, 48: 16,\\\\n        }\\\\n        assert dict(sorted(max_degree_hist.items())) == {4: 3672, 6: 104}\\\\n        multipath_census = {\\\\n            "stage3e_path": str(stage3e_path),\\\\n            "stage3e_sha256": sha256_file(stage3e_path),\\\\n            "nonresonant_multipath_orbits": 3776,\\\\n            "path_space_dimension_histogram": dict(sorted(dimension_hist.items())),\\\\n            "max_link_degree_histogram": dict(sorted(max_degree_hist.items())),\\\\n        }\\\\n    print(f"G4 PASS: Stage-3E census found={stage3e_path is not None}")\\\\n\\\\n    edge_payload = {\\\\n        "meta": {\\\\n            "version": VERSION,\\\\n            "construction": "unique rational SU(3) intertwiners",\\\\n            "stage3c_projector_equality": True,\\\\n        },\\\\n        "edges": edge_records,\\\\n    }\\\\n    edge_path = output_dir / "y4_exact_edge_intertwiners.json.gz"\\\\n    edge_sha = write_json_gz(edge_path, edge_payload)\\\\n\\\\n    path_payload = {\\\\n        "meta": {\\\\n            "version": VERSION,\\\\n            "raw_color_basis": "standard colors 0,1,2 in active event order",\\\\n            "complete_haar_projector": "sum of normalized path projectors",\\\\n            "stage2_completeness_verified": True,\\\\n        },\\\\n        "signature_summaries": signature_summaries,\\\\n        "path_tensors": path_tensors,\\\\n    }\\\\n    path_path = output_dir / "y4_exact_local_path_tensors.json.gz"\\\\n    path_sha = write_json_gz(path_path, path_payload)\\\\n\\\\n    assert len(read_json_gz(edge_path)["edges"]) == 30\\\\n    reread = read_json_gz(path_path)\\\\n    assert len(reread["path_tensors"]) == 182\\\\n    assert sum(len(x["paths"]) for x in reread["path_tensors"]) == 380\\\\n    print("G5 PASS: final output round-trip")\\\\n\\\\n    elapsed = time.time() - started\\\\n    gates = {\\\\n        "G0_dependencies": {"passed": True},\\\\n        "G1_edge_intertwiners": {\\\\n            "records": 30,\\\\n            "exact_projector_equality": True,\\\\n            "passed": True,\\\\n        },\\\\n        "G2_local_path_tensors": {\\\\n            "signatures": 182,\\\\n            "path_tensors": 380,\\\\n            "degree_histogram": dict(sorted(degree_hist.items())),\\\\n            "path_count_histogram": dict(sorted(path_count_hist.items())),\\\\n            "path_nnz_histogram": dict(sorted(nnz_hist.items())),\\\\n            "path_norm_histogram": dict(sorted(norm_hist.items())),\\\\n            "passed": True,\\\\n        },\\\\n        "G3_stage2_completeness": {\\\\n            "orthogonal_complete_path_bases": True,\\\\n            "haar_projector_equality": True,\\\\n            "passed": True,\\\\n        },\\\\n        "G4_multipath_census": {\\\\n            "census": multipath_census,\\\\n            "passed": True,\\\\n        },\\\\n        "G5_round_trip": {"passed": True},\\\\n    }\\\\n\\\\n    summary = {\\\\n        "meta": {\\\\n            "version": VERSION,\\\\n            "date": "2026-06-13",\\\\n            "python": sys.version,\\\\n            "sympy": sp.__version__,\\\\n            "platform": platform.platform(),\\\\n            "walltime_s_final_invocation": elapsed,\\\\n            "hardware": "CPU",\\\\n            "a100_required": False,\\\\n            "cache_key": key,\\\\n        },\\\\n        "counts": {\\\\n            "exact_nonzero_edge_intertwiners": 30,\\\\n            "local_signatures": 182,\\\\n            "exact_local_path_tensors": 380,\\\\n            "maximum_local_path_count": 6,\\\\n            "maximum_raw_tensor_dimension": max(x["raw_dimension"] for x in path_tensors),\\\\n            "nonresonant_multipath_orbits_ready": (\\\\n                multipath_census["nonresonant_multipath_orbits"]\\\\n                if multipath_census else None\\\\n            ),\\\\n        },\\\\n        "gates": gates,\\\\n        "files": {\\\\n            edge_path.name: {"sha256": edge_sha, "records": 30},\\\\n            path_path.name: {\\\\n                "sha256": path_sha,\\\\n                "signatures": 182,\\\\n                "path_tensors": 380,\\\\n            },\\\\n        },\\\\n        "scope": {\\\\n            "completed": [\\\\n                "exact rational SU(3) edge intertwiners",\\\\n                "metric-normalized projector regression",\\\\n                "exact denominator-resolved fusion-tree invariant vectors",\\\\n                "orthogonal complete local path bases",\\\\n            ],\\\\n            "not_completed": [\\\\n                "global multi-path orbit contraction",\\\\n                "des-Cloizeaux folded terms",\\\\n                "full fourth-order hopping matrix",\\\\n                "cube-boundary flatness residual",\\\\n            ],\\\\n        },\\\\n        "next_stage": (\\\\n            "Stage 3H: contract the 3,776 nonresonant multi-path orbits "\\\\n            "using these exact local path tensors."\\\\n        ),\\\\n        "passed": True,\\\\n    }\\\\n    summary_path = output_dir / "y4_stage3g_summary.json"\\\\n    summary_sha = write_json(summary_path, summary)\\\\n\\\\n    print()\\\\n    print("SUMMARY")\\\\n    print(json.dumps(summary["counts"], indent=2, sort_keys=True))\\\\n    print(f"INTERTWINERS: {edge_path}")\\\\n    print(f"PATH TENSORS: {path_path}")\\\\n    print(f"SUMMARY     : {summary_path}")\\\\n    print(f"SUMMARY SHA256: {summary_sha}")\\\\n    print("ALL STAGE-3G GATES PASS")\\\\n\\\\n\\\\nif __name__ == "__main__":\\\\n    parser = argparse.ArgumentParser()\\\\n    parser.add_argument("--stage2-cards", type=Path, default=None)\\\\n    parser.add_argument("--stage3b-paths", type=Path, default=None)\\\\n    parser.add_argument("--stage3c-carriers", type=Path, default=None)\\\\n    parser.add_argument("--stage3c-projectors", type=Path, default=None)\\\\n    parser.add_argument("--output-dir", type=Path, default=default_output_dir())\\\\n    parser.add_argument(\\\\n        "--deadline",\\\\n        type=float,\\\\n        default=0.0,\\\\n        help="0 = no deadline; positive value checkpoints and exits cleanly",\\\\n    )\\\\n    args, _unknown = parser.parse_known_args()\\\\n\\\\n    stage2_cards = args.stage2_cards or recursive_find("y4_link_tensor_cards.json.gz")\\\\n    stage3b_paths = args.stage3b_paths or recursive_find("y4_local_irrep_paths.json.gz")\\\\n    stage3c_carriers = args.stage3c_carriers or recursive_find("y4_irrep_carriers.json.gz")\\\\n    stage3c_projectors = args.stage3c_projectors or recursive_find(\\\\n        "y4_casimir_channel_projectors.json.gz"\\\\n    )\\\\n    missing = [\\\\n        name for name, path in (\\\\n            ("Stage-2 cards", stage2_cards),\\\\n            ("Stage-3B paths", stage3b_paths),\\\\n            ("Stage-3C carriers", stage3c_carriers),\\\\n            ("Stage-3C projectors", stage3c_projectors),\\\\n        ) if path is None\\\\n    ]\\\\n    assert not missing, "Missing dependencies: " + ", ".join(missing)\\\\n\\\\n    main(\\\\n        stage2_cards,\\\\n        stage3b_paths,\\\\n        stage3c_carriers,\\\\n        stage3c_projectors,\\\\n        args.output_dir,\\\\n        args.deadline,\\\\n    )\\\\n\\\', \\\'y4_stage3i_complete_folded_descloizeaux.py\\\': \\\'#!/usr/bin/env python3\\\\n"""\\\\ny4_stage3i_complete_folded_descloizeaux.py\\\\n===========================================\\\\n\\\\nComplete exact fourth-order direct plus resonant/folded des-Cloizeaux kernel.\\\\n\\\\nRun in Colab:\\\\n    %run /content/y4_stage3i_complete_folded_descloizeaux.py\\\\n\\\\nUse a standard CPU runtime. The A100 is not used.\\\\n\\\\nThis script evaluates all 16,835 orientation orbits, including:\\\\n  * 6,598 nonresonant-only orbits;\\\\n  * 2,749 mixed direct/resonant orbits;\\\\n  * 7,488 all-resonant orbits.\\\\n\\\\nIt implements the fourth-order Hermitian effective-Hamiltonian identity valid\\\\nwhen PVP=aP is scalar in the one-flux charge sector, contracts every exact\\\\nfusion-tree tensor, regresses all nonresonant topologies against Stage 3H,\\\\nand attaches the rooted cubic orbit multiplicity to all 4,221 ordered words.\\\\n\\\\nThe outputs are the complete exact inputs to the final cube-boundary test.\\\\n"""\\\\n\\\\n\\\\nfrom __future__ import annotations\\\\n\\\\nimport argparse\\\\nimport gzip\\\\nimport hashlib\\\\nimport itertools\\\\nimport json\\\\nimport platform\\\\nimport sys\\\\nimport time\\\\nfrom collections import Counter, defaultdict\\\\nfrom fractions import Fraction\\\\nfrom pathlib import Path\\\\nfrom typing import Dict, Iterable, List, Sequence, Tuple\\\\n\\\\nimport numpy as np\\\\n\\\\nVERSION = "2026-06-13-stage3i-v1"\\\\n\\\\nTokenSignature = Tuple[int, int, int, int, int, int]\\\\nEnergyTriple = Tuple[int, int, int]\\\\n\\\\n\\\\ndef default_output_dir() -> Path:\\\\n    if Path("/content").exists():\\\\n        return Path("/content/Y4_STAGE3I")\\\\n    return Path.cwd() / "Y4_STAGE3I"\\\\n\\\\n\\\\ndef recursive_find(filename: str) -> Path | None:\\\\n    preferred_dirs = (\\\\n        "Y4_STAGE1",\\\\n        "Y4_STAGE3E",\\\\n        "Y4_STAGE3F",\\\\n        "Y4_STAGE3G",\\\\n    )\\\\n    roots = (Path("/content"), Path.cwd(), Path("/mnt/data"))\\\\n\\\\n    for directory in preferred_dirs:\\\\n        for root in roots:\\\\n            path = root / directory / filename\\\\n            if path.exists():\\\\n                return path\\\\n\\\\n    for root in roots:\\\\n        if not root.exists():\\\\n            continue\\\\n        for path in root.rglob(filename):\\\\n            return path\\\\n    return None\\\\n\\\\n\\\\ndef sha256_file(path: Path) -> str:\\\\n    digest = hashlib.sha256()\\\\n    with path.open("rb") as handle:\\\\n        for chunk in iter(lambda: handle.read(1 << 20), b""):\\\\n            digest.update(chunk)\\\\n    return digest.hexdigest()\\\\n\\\\n\\\\ndef read_json_gz(path: Path) -> object:\\\\n    with gzip.open(path, "rt", encoding="utf-8") as handle:\\\\n        return json.load(handle)\\\\n\\\\n\\\\ndef write_json(path: Path, obj: object) -> str:\\\\n    raw = json.dumps(\\\\n        obj,\\\\n        indent=2,\\\\n        sort_keys=True,\\\\n        allow_nan=False,\\\\n    ).encode("utf-8")\\\\n    path.write_bytes(raw)\\\\n    return hashlib.sha256(raw).hexdigest()\\\\n\\\\n\\\\ndef write_json_gz(path: Path, obj: object) -> str:\\\\n    raw = json.dumps(\\\\n        obj,\\\\n        separators=(",", ":"),\\\\n        sort_keys=True,\\\\n        allow_nan=False,\\\\n    ).encode("utf-8")\\\\n    with gzip.GzipFile(\\\\n        filename=str(path),\\\\n        mode="wb",\\\\n        compresslevel=9,\\\\n        mtime=0,\\\\n    ) as handle:\\\\n        handle.write(raw)\\\\n    return sha256_file(path)\\\\n\\\\n\\\\ndef fstr(value: Fraction) -> str:\\\\n    return (\\\\n        str(value.numerator)\\\\n        if value.denominator == 1\\\\n        else f"{value.numerator}/{value.denominator}"\\\\n    )\\\\n\\\\n\\\\ndef fraction_histogram_json(counter: Counter) -> Dict[str, int]:\\\\n    return {\\\\n        fstr(value): int(count)\\\\n        for value, count in sorted(counter.items(), key=lambda row: row[0])\\\\n    }\\\\n\\\\n\\\\ndef energy_counter_from_stage1(orbit: dict) -> Counter[EnergyTriple]:\\\\n    return Counter(\\\\n        {\\\\n            tuple(int(x) for x in record["E6"]): int(\\\\n                record["channel_path_multiplicity"]\\\\n            )\\\\n            for record in orbit["denominator_signatures"]\\\\n        }\\\\n    )\\\\n\\\\n\\\\ndef resolvent_product(energy: EnergyTriple) -> Fraction:\\\\n    assert 16 not in energy\\\\n    denominator = (\\\\n        (16 - energy[0])\\\\n        * (16 - energy[1])\\\\n        * (16 - energy[2])\\\\n    )\\\\n    return Fraction(216, denominator)\\\\n\\\\n\\\\ndef output_displacement(orbit: dict) -> Tuple[int, int, int, int, int]:\\\\n    root = orbit["events"][0]["plaquette"]\\\\n    output = orbit["events"][5]["plaquette"]\\\\n    return (\\\\n        int(output[0]) - int(root[0]),\\\\n        int(output[1]) - int(root[1]),\\\\n        int(output[2]) - int(root[2]),\\\\n        int(output[3]),\\\\n        int(output[4]),\\\\n    )\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Exact tensor factors and variable elimination\\\\n# ---------------------------------------------------------------------------\\\\n\\\\nclass PathTensorLibrary:\\\\n    def __init__(self, payload: dict):\\\\n        self.paths: Dict[TokenSignature, List[dict]] = {\\\\n            tuple(int(x) for x in record["signature"]): record["paths"]\\\\n            for record in payload["path_tensors"]\\\\n        }\\\\n        self.array_cache: Dict[Tuple[TokenSignature, int], np.ndarray] = {}\\\\n\\\\n    def vector_array(self, signature: TokenSignature, path_index: int) -> np.ndarray:\\\\n        key = (signature, path_index)\\\\n        if key in self.array_cache:\\\\n            return self.array_cache[key]\\\\n\\\\n        path = self.paths[signature][path_index]\\\\n        degree = sum(token != 0 for token in signature)\\\\n        array = np.zeros(3 ** degree, dtype=object)\\\\n        for flat_index, numerator, denominator in path["vector"]["entries"]:\\\\n            value = (\\\\n                int(numerator)\\\\n                if int(denominator) == 1\\\\n                else Fraction(int(numerator), int(denominator))\\\\n            )\\\\n            array[int(flat_index)] = value\\\\n        array = array.reshape((3,) * degree)\\\\n        self.array_cache[key] = array\\\\n        return array\\\\n\\\\n\\\\nFactor = Tuple[Tuple[int, ...], np.ndarray | int | Fraction]\\\\n\\\\n\\\\ndef make_factor(scope_event_order: Sequence[int], array: np.ndarray) -> Factor:\\\\n    permutation = sorted(\\\\n        range(len(scope_event_order)),\\\\n        key=lambda index: scope_event_order[index],\\\\n    )\\\\n    scope = tuple(scope_event_order[index] for index in permutation)\\\\n    if len(permutation) > 1:\\\\n        array = np.transpose(array, permutation)\\\\n    return scope, array\\\\n\\\\n\\\\ndef align_factor(factor: Factor, union_scope: Tuple[int, ...]) -> np.ndarray:\\\\n    scope, data = factor\\\\n    shape = [3 if variable in scope else 1 for variable in union_scope]\\\\n    return np.asarray(data, dtype=object).reshape(shape)\\\\n\\\\n\\\\ndef multiply_factors(first: Factor, second: Factor) -> Factor:\\\\n    union_scope = tuple(sorted(set(first[0]) | set(second[0])))\\\\n    product = align_factor(first, union_scope) * align_factor(second, union_scope)\\\\n    return union_scope, product\\\\n\\\\n\\\\ndef min_fill_order(\\\\n    scopes: Sequence[Sequence[int]],\\\\n    n_variables: int = 24,\\\\n) -> Tuple[Tuple[int, ...], int]:\\\\n    adjacency = [set() for _ in range(n_variables)]\\\\n    for scope in scopes:\\\\n        variables = sorted(set(scope))\\\\n        for index, first in enumerate(variables):\\\\n            for second in variables[index + 1 :]:\\\\n                adjacency[first].add(second)\\\\n                adjacency[second].add(first)\\\\n\\\\n    alive = set(range(n_variables))\\\\n    order: List[int] = []\\\\n    width = 0\\\\n\\\\n    while alive:\\\\n        best = None\\\\n        for variable in alive:\\\\n            neighbors = adjacency[variable] & alive\\\\n            neighbor_list = list(neighbors)\\\\n            fill = sum(\\\\n                second not in adjacency[first]\\\\n                for index, first in enumerate(neighbor_list)\\\\n                for second in neighbor_list[index + 1 :]\\\\n            )\\\\n            candidate = (fill, len(neighbors), variable)\\\\n            if best is None or candidate < best[0]:\\\\n                best = (candidate, variable, neighbors)\\\\n\\\\n        assert best is not None\\\\n        _, variable, neighbors = best\\\\n        width = max(width, len(neighbors))\\\\n        neighbor_list = list(neighbors)\\\\n        for index, first in enumerate(neighbor_list):\\\\n            for second in neighbor_list[index + 1 :]:\\\\n                adjacency[first].add(second)\\\\n                adjacency[second].add(first)\\\\n        alive.remove(variable)\\\\n        order.append(variable)\\\\n\\\\n    return tuple(order), width\\\\n\\\\n\\\\nclass TopologyPlan:\\\\n    def __init__(self, orbit: dict):\\\\n        self.link_specs = []\\\\n        scopes = []\\\\n\\\\n        for link in orbit["links"]:\\\\n            signature = tuple(int(x) for x in link["signature"])\\\\n            occurrences = sorted(link["occurrences"], key=lambda row: row[0])\\\\n            row_variables = tuple(int(row[3]) for row in occurrences)\\\\n            column_variables = tuple(int(row[4]) for row in occurrences)\\\\n            self.link_specs.append(\\\\n                (signature, row_variables, column_variables)\\\\n            )\\\\n            scopes.extend(\\\\n                [tuple(sorted(row_variables)), tuple(sorted(column_variables))]\\\\n            )\\\\n\\\\n        self.elimination_order, self.split_factor_width = min_fill_order(scopes)\\\\n\\\\n    def contract_choice(\\\\n        self,\\\\n        choices: Sequence[int],\\\\n        library: PathTensorLibrary,\\\\n    ) -> Tuple[Fraction, EnergyTriple]:\\\\n        factors: List[Factor] = []\\\\n        norm_product = 1\\\\n        energy = [0, 0, 0]\\\\n\\\\n        for (signature, rows, columns), path_index in zip(\\\\n            self.link_specs, choices\\\\n        ):\\\\n            path = library.paths[signature][path_index]\\\\n            vector = library.vector_array(signature, path_index)\\\\n            factors.append(make_factor(rows, vector))\\\\n            factors.append(make_factor(columns, vector.copy()))\\\\n            norm_product *= int(path["norm_squared"])\\\\n            for cut, value in enumerate(path["intermediate_E6"]):\\\\n                energy[cut] += int(value)\\\\n\\\\n        for variable in self.elimination_order:\\\\n            selected = [factor for factor in factors if variable in factor[0]]\\\\n            factors = [factor for factor in factors if variable not in factor[0]]\\\\n            if not selected:\\\\n                continue\\\\n\\\\n            combined = selected[0]\\\\n            for factor in selected[1:]:\\\\n                combined = multiply_factors(combined, factor)\\\\n\\\\n            axis = combined[0].index(variable)\\\\n            reduced_data = np.asarray(combined[1], dtype=object).sum(axis=axis)\\\\n            reduced_scope = combined[0][:axis] + combined[0][axis + 1 :]\\\\n            factors.append((reduced_scope, reduced_data))\\\\n\\\\n        assert factors\\\\n        combined = factors[0]\\\\n        for factor in factors[1:]:\\\\n            combined = multiply_factors(combined, factor)\\\\n        assert combined[0] == ()\\\\n\\\\n        numerator = int(np.asarray(combined[1], dtype=object).reshape(()))\\\\n        return Fraction(numerator, norm_product), tuple(energy)\\\\n\\\\n\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Fourth-order des-Cloizeaux folded coefficients\\\\n# ---------------------------------------------------------------------------\\\\n\\\\ndef folded_coefficient_from_denominators(\\\\n    denominators: Tuple[Fraction, Fraction, Fraction],\\\\n) -> Fraction:\\\\n    """\\\\n    Universal fourth-order Hermitian effective-Hamiltonian coefficient when\\\\n    P V P = a P is scalar in the model space.\\\\n\\\\n    H4 = PVRVRVRVP\\\\n         - a(PVR^2VRVP + PVRVR^2VP)\\\\n         + a^2 PVR^3VP\\\\n         - 1/2 {PVRVP, PVR^2VP}.\\\\n\\\\n    The raw four-V matrix element already contains each PVP=a factor, so the\\\\n    coefficient below depends only on which intermediate cuts lie in P.\\\\n    """\\\\n    zero_positions = [i for i, value in enumerate(denominators) if value == 0]\\\\n    nonzero = [value for value in denominators if value != 0]\\\\n\\\\n    if len(zero_positions) == 0:\\\\n        d1, d2, d3 = denominators\\\\n        return Fraction(1, 1) / (d1 * d2 * d3)\\\\n\\\\n    if len(zero_positions) == 1:\\\\n        x, y = nonzero\\\\n        base = Fraction(1, 1) / (x * x * y) + Fraction(1, 1) / (x * y * y)\\\\n        # The Hermitian des-Cloizeaux convention distributes the scalar PVP\\\\n        # insertion symmetrically over its two ordered placements.  For the\\\\n        # middle return this is exactly the anticommutator coefficient.\\\\n        return -base / 2\\\\n\\\\n    if len(zero_positions) == 2:\\\\n        assert len(nonzero) == 1\\\\n        # The a^2 PVR^3VP term has three ordered placements of the two scalar\\\\n        # PVP insertions; each ordered word receives one third.\\\\n        return Fraction(1, 3) / (nonzero[0] ** 3)\\\\n\\\\n    assert len(zero_positions) == 3\\\\n    return Fraction(0, 1)\\\\n\\\\n\\\\ndef folded_coefficient(energy: EnergyTriple) -> Fraction:\\\\n    denominators = tuple(\\\\n        Fraction(16 - int(value), 6) for value in energy\\\\n    )\\\\n    return folded_coefficient_from_denominators(denominators)\\\\n\\\\n\\\\ndef resonance_mask(energy: EnergyTriple) -> int:\\\\n    return sum((int(value) == 16) << index for index, value in enumerate(energy))\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Exact independent perturbation-theory regression\\\\n# ---------------------------------------------------------------------------\\\\n\\\\ndef exact_scalar_eigenvalue_coefficients(\\\\n    h0_diag: Sequence[Fraction],\\\\n    perturbation: Sequence[Sequence[Fraction]],\\\\n) -> Tuple[Fraction, Fraction, Fraction, Fraction]:\\\\n    import sympy as sp\\\\n\\\\n    y = sp.symbols("y")\\\\n    c = sp.symbols("c1:5")\\\\n    n = len(h0_diag)\\\\n    h0 = sp.diag(*[sp.Rational(x.numerator, x.denominator) for x in h0_diag])\\\\n    v = sp.Matrix(\\\\n        [\\\\n            [sp.Rational(x.numerator, x.denominator) for x in row]\\\\n            for row in perturbation\\\\n        ]\\\\n    )\\\\n    energy = sp.Rational(h0_diag[0].numerator, h0_diag[0].denominator)\\\\n    for order, coefficient in enumerate(c, start=1):\\\\n        energy += coefficient * y ** order\\\\n\\\\n    determinant = sp.expand((h0 + y * v - energy * sp.eye(n)).det())\\\\n    solved = {}\\\\n    for order, coefficient in enumerate(c, start=1):\\\\n        equation = sp.expand(determinant.subs(solved)).coeff(y, order)\\\\n        solution = sp.solve(sp.Eq(equation, 0), coefficient)\\\\n        assert len(solution) == 1\\\\n        solved[coefficient] = sp.simplify(solution[0])\\\\n\\\\n    return tuple(\\\\n        Fraction(int(sp.numer(solved[x])), int(sp.denom(solved[x])))\\\\n        for x in c\\\\n    )\\\\n\\\\n\\\\ndef path_sum_fourth_order(\\\\n    h0_diag: Sequence[Fraction],\\\\n    perturbation: Sequence[Sequence[Fraction]],\\\\n) -> Fraction:\\\\n    e0 = h0_diag[0]\\\\n    n = len(h0_diag)\\\\n    total = Fraction(0)\\\\n\\\\n    for s1 in range(n):\\\\n        for s2 in range(n):\\\\n            for s3 in range(n):\\\\n                raw = (\\\\n                    perturbation[0][s3]\\\\n                    * perturbation[s3][s2]\\\\n                    * perturbation[s2][s1]\\\\n                    * perturbation[s1][0]\\\\n                )\\\\n                if raw == 0:\\\\n                    continue\\\\n                denominators = (\\\\n                    e0 - h0_diag[s1],\\\\n                    e0 - h0_diag[s2],\\\\n                    e0 - h0_diag[s3],\\\\n                )\\\\n                total += raw * folded_coefficient_from_denominators(denominators)\\\\n\\\\n    return total\\\\n\\\\n\\\\ndef folded_formula_regression() -> List[dict]:\\\\n    anchors = [\\\\n        (\\\\n            [Fraction(0), Fraction(2), Fraction(5)],\\\\n            [\\\\n                [Fraction(1), Fraction(2), Fraction(-1)],\\\\n                [Fraction(2), Fraction(3), Fraction(1)],\\\\n                [Fraction(-1), Fraction(1), Fraction(-2)],\\\\n            ],\\\\n        ),\\\\n        (\\\\n            [Fraction(1), Fraction(4), Fraction(7), Fraction(9)],\\\\n            [\\\\n                [Fraction(-2), Fraction(1), Fraction(2), Fraction(-1)],\\\\n                [Fraction(1), Fraction(3), Fraction(-2), Fraction(1)],\\\\n                [Fraction(2), Fraction(-2), Fraction(1), Fraction(2)],\\\\n                [Fraction(-1), Fraction(1), Fraction(2), Fraction(0)],\\\\n            ],\\\\n        ),\\\\n        (\\\\n            [Fraction(-1), Fraction(3), Fraction(8)],\\\\n            [\\\\n                [Fraction(2), Fraction(-2), Fraction(3)],\\\\n                [Fraction(-2), Fraction(1), Fraction(2)],\\\\n                [Fraction(3), Fraction(2), Fraction(-1)],\\\\n            ],\\\\n        ),\\\\n    ]\\\\n\\\\n    records = []\\\\n    for index, (h0, v) in enumerate(anchors):\\\\n        coefficients = exact_scalar_eigenvalue_coefficients(h0, v)\\\\n        path_value = path_sum_fourth_order(h0, v)\\\\n        assert coefficients[3] == path_value\\\\n        records.append(\\\\n            {\\\\n                "anchor": index,\\\\n                "c1": fstr(coefficients[0]),\\\\n                "c2": fstr(coefficients[1]),\\\\n                "c3": fstr(coefficients[2]),\\\\n                "c4": fstr(coefficients[3]),\\\\n                "path_sum_c4": fstr(path_value),\\\\n            }\\\\n        )\\\\n    return records\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Rooted cubic multiplicity\\\\n# ---------------------------------------------------------------------------\\\\n\\\\nVec3 = Tuple[int, int, int]\\\\nPlaquette = Tuple[int, int, int, int, int]\\\\nE_AXES: Tuple[Vec3, Vec3, Vec3] = ((1, 0, 0), (0, 1, 0), (0, 0, 1))\\\\nROOT_PLAQUETTE: Plaquette = (0, 0, 0, 0, 1)\\\\n\\\\n\\\\ndef permutation_parity(perm: Tuple[int, int, int]) -> int:\\\\n    inversions = sum(\\\\n        perm[i] > perm[j] for i in range(3) for j in range(i + 1, 3)\\\\n    )\\\\n    return -1 if inversions % 2 else 1\\\\n\\\\n\\\\ndef proper_rotations():\\\\n    rotations = []\\\\n    for perm in itertools.permutations(range(3)):\\\\n        parity = permutation_parity(perm)\\\\n        for signs in itertools.product((-1, 1), repeat=3):\\\\n            if parity * signs[0] * signs[1] * signs[2] == 1:\\\\n                rotations.append((perm, signs))\\\\n    assert len(rotations) == 24\\\\n    return rotations\\\\n\\\\n\\\\nROTATIONS = proper_rotations()\\\\n\\\\n\\\\ndef transform_vec(vector: Vec3, rotation) -> Vec3:\\\\n    perm, signs = rotation\\\\n    output = [0, 0, 0]\\\\n    for axis in range(3):\\\\n        output[perm[axis]] += signs[axis] * vector[axis]\\\\n    return tuple(output)\\\\n\\\\n\\\\ndef transform_plaquette(plaquette: Plaquette, rotation_index: int) -> Plaquette:\\\\n    perm, signs = ROTATIONS[rotation_index]\\\\n    anchor = plaquette[:3]\\\\n    first, second = plaquette[3], plaquette[4]\\\\n    mapped_first, mapped_second = perm[first], perm[second]\\\\n    mapped_anchor = list(transform_vec(anchor, (perm, signs)))\\\\n    if signs[first] < 0:\\\\n        mapped_anchor[mapped_first] -= 1\\\\n    if signs[second] < 0:\\\\n        mapped_anchor[mapped_second] -= 1\\\\n    mapped_first, mapped_second = sorted((mapped_first, mapped_second))\\\\n    return (\\\\n        mapped_anchor[0],\\\\n        mapped_anchor[1],\\\\n        mapped_anchor[2],\\\\n        mapped_first,\\\\n        mapped_second,\\\\n    )\\\\n\\\\n\\\\nROOT_STABILIZER = [\\\\n    index\\\\n    for index in range(24)\\\\n    if transform_plaquette(ROOT_PLAQUETTE, index)[3:] == (0, 1)\\\\n]\\\\nassert len(ROOT_STABILIZER) == 8\\\\nROOT_SHIFTS = {\\\\n    index: transform_plaquette(ROOT_PLAQUETTE, index)[:3]\\\\n    for index in ROOT_STABILIZER\\\\n}\\\\n\\\\n\\\\ndef rooted_transform(plaquette: Plaquette, rotation_index: int) -> Plaquette:\\\\n    transformed = transform_plaquette(plaquette, rotation_index)\\\\n    shift = ROOT_SHIFTS[rotation_index]\\\\n    return (\\\\n        transformed[0] - shift[0],\\\\n        transformed[1] - shift[1],\\\\n        transformed[2] - shift[2],\\\\n        transformed[3],\\\\n        transformed[4],\\\\n    )\\\\n\\\\n\\\\ndef plaquette_orientation_sign(\\\\n    plaquette: Plaquette,\\\\n    rotation_index: int,\\\\n) -> int:\\\\n    perm, signs = ROTATIONS[rotation_index]\\\\n    first, second = plaquette[3], plaquette[4]\\\\n    reorder_sign = -1 if perm[first] > perm[second] else 1\\\\n    return signs[first] * signs[second] * reorder_sign\\\\n\\\\n\\\\ndef rooted_word_symmetry_data(word: dict) -> Tuple[int, int]:\\\\n    insertions = tuple(\\\\n        tuple(int(x) for x in row) for row in word["ordered_insertions"]\\\\n    )\\\\n    output = tuple(int(x) for x in word["output"])\\\\n\\\\n    image_signs: Dict[\\\\n        Tuple[Tuple[Plaquette, ...], Plaquette],\\\\n        set,\\\\n    ] = {}\\\\n    for rotation in ROOT_STABILIZER:\\\\n        image = (\\\\n            tuple(\\\\n                rooted_transform(plaquette, rotation)\\\\n                for plaquette in insertions\\\\n            ),\\\\n            rooted_transform(output, rotation),\\\\n        )\\\\n        odd_sign = (\\\\n            plaquette_orientation_sign(ROOT_PLAQUETTE, rotation)\\\\n            * plaquette_orientation_sign(output, rotation)\\\\n        )\\\\n        image_signs.setdefault(image, set()).add(odd_sign)\\\\n\\\\n    # A nontrivial stabilizer must act consistently on a nonzero matrix\\\\n    # element.  The geometry compiler satisfies this exactly.\\\\n    assert all(len(signs) == 1 for signs in image_signs.values())\\\\n\\\\n    even_multiplicity = len(image_signs)\\\\n    odd_signed_multiplicity = sum(\\\\n        next(iter(signs)) for signs in image_signs.values()\\\\n    )\\\\n    return even_multiplicity, odd_signed_multiplicity\\\\n\\\\n\\\\n# ---------------------------------------------------------------------------\\\\n# Complete fourth-order direct + folded evaluation\\\\n# ---------------------------------------------------------------------------\\\\n\\\\ndef run(\\\\n    stage1_path: Path,\\\\n    stage3e_path: Path,\\\\n    stage3g_path: Path,\\\\n    stage3h_topology_path: Path | None,\\\\n    output_dir: Path,\\\\n) -> None:\\\\n    started = time.time()\\\\n    output_dir.mkdir(parents=True, exist_ok=True)\\\\n\\\\n    print("=" * 108)\\\\n    print("SU(3) O(y^4) STAGE-I COMPLETE DIRECT + FOLDED DES-CLOIZEAUX KERNEL")\\\\n    print("=" * 108)\\\\n    print(f"version  : {VERSION}")\\\\n    print(f"stage1   : {stage1_path}")\\\\n    print(f"stage3e  : {stage3e_path}")\\\\n    print(f"stage3g  : {stage3g_path}")\\\\n    print(f"stage3h  : {stage3h_topology_path if stage3h_topology_path else \\\\\\\'not found\\\\\\\'}")\\\\n    print(f"output   : {output_dir}")\\\\n    print("hardware : standard Colab CPU; A100 is not used")\\\\n    print()\\\\n\\\\n    for path in (stage1_path, stage3e_path, stage3g_path):\\\\n        assert path.exists(), f"Missing dependency: {path}"\\\\n\\\\n    stage1_sha = sha256_file(stage1_path)\\\\n    stage3e_sha = sha256_file(stage3e_path)\\\\n    stage3g_sha = sha256_file(stage3g_path)\\\\n\\\\n    stage1 = read_json_gz(stage1_path)\\\\n    stage3e = read_json_gz(stage3e_path)\\\\n    stage3g = read_json_gz(stage3g_path)\\\\n\\\\n    words = stage1["words"]\\\\n    all_orbits = stage3e["orbits"]\\\\n    library = PathTensorLibrary(stage3g)\\\\n\\\\n    assert len(words) == 4221\\\\n    assert len(all_orbits) == 16835\\\\n    assert len(library.paths) == 182\\\\n    assert sum(len(paths) for paths in library.paths.values()) == 380\\\\n\\\\n    stage1_orbits = {}\\\\n    word_by_id = {}\\\\n    for word in words:\\\\n        word_by_id[word["ordered_id"]] = word\\\\n        for orbit in word["orientation_orbits"]:\\\\n            assert orbit["orbit_id"] not in stage1_orbits\\\\n            stage1_orbits[orbit["orbit_id"]] = orbit\\\\n    assert len(stage1_orbits) == 16835\\\\n\\\\n    gates = {}\\\\n\\\\n    # I0: exact independent formula regression.\\\\n    formula_anchors = folded_formula_regression()\\\\n    gates["I0_folded_formula"] = {\\\\n        "operator_identity": (\\\\n            "H4=PVRVRVRVP-a(PVR2VRVP+PVRVR2VP)+a2PVR3VP"\\\\n            "-1/2{PVRVP,PVR2VP}"\\\\n        ),\\\\n        "exact_scalar_regression_anchors": formula_anchors,\\\\n        "passed": True,\\\\n    }\\\\n    print("I0 PASS: folded coefficient reproduces exact fourth-order eigenvalue series")\\\\n\\\\n    # I1: dependency and census gates.\\\\n    resonance_hist = Counter(orbit["resonance_class"] for orbit in all_orbits)\\\\n    assert resonance_hist == Counter(\\\\n        {"all_resonant": 7488, "mixed": 2749, "nonresonant_only": 6598}\\\\n    )\\\\n\\\\n    topology_members: Dict[str, List[dict]] = defaultdict(list)\\\\n    for orbit in all_orbits:\\\\n        topology_members[orbit["topology_hash"]].append(orbit)\\\\n    assert len(topology_members) == 3895\\\\n\\\\n    gates["I1_dependencies"] = {\\\\n        "stage1_sha256": stage1_sha,\\\\n        "stage3e_sha256": stage3e_sha,\\\\n        "stage3g_sha256": stage3g_sha,\\\\n        "ordered_words": len(words),\\\\n        "orientation_orbits": len(all_orbits),\\\\n        "trace_topologies": len(topology_members),\\\\n        "resonance_histogram": dict(sorted(resonance_hist.items())),\\\\n        "passed": True,\\\\n    }\\\\n    print("I1 PASS: complete 16,835-orbit fourth-order sector loaded")\\\\n\\\\n    # Optional Stage-H topology regression map.\\\\n    stage3h_map = {}\\\\n    if stage3h_topology_path is not None and stage3h_topology_path.exists():\\\\n        payload = read_json_gz(stage3h_topology_path)\\\\n        stage3h_map = {\\\\n            record["topology_hash"]: Fraction(record["bare_nonresonant_amplitude"])\\\\n            for record in payload["topologies"]\\\\n        }\\\\n        assert len(stage3h_map) == 1478\\\\n\\\\n    # I2: exact contraction for every topology and every fusion-tree path.\\\\n    topology_results = {}\\\\n    topology_records = []\\\\n    total_global_paths = 0\\\\n    direct_topology_hist = Counter()\\\\n    folded_topology_hist = Counter()\\\\n    complete_topology_hist = Counter()\\\\n    path_dimension_hist = Counter()\\\\n    resonance_mask_path_hist = Counter()\\\\n    class_topology_hist = Counter()\\\\n    stage3h_matches = 0\\\\n\\\\n    for topology_index, topology_hash in enumerate(sorted(topology_members), start=1):\\\\n        members = topology_members[topology_hash]\\\\n        representative = members[0]\\\\n        expected_counter = energy_counter_from_stage1(\\\\n            stage1_orbits[representative["orbit_id"]]\\\\n        )\\\\n\\\\n        plan = TopologyPlan(representative)\\\\n        raw_by_energy: Counter[EnergyTriple] = Counter()\\\\n        path_count_by_energy: Counter[EnergyTriple] = Counter()\\\\n\\\\n        choice_ranges = [\\\\n            range(int(link["local_path_count"])) for link in representative["links"]\\\\n        ]\\\\n        for choices in itertools.product(*choice_ranges):\\\\n            raw_contraction, energy = plan.contract_choice(choices, library)\\\\n            raw_by_energy[energy] += raw_contraction\\\\n            path_count_by_energy[energy] += 1\\\\n            resonance_mask_path_hist[resonance_mask(energy)] += 1\\\\n            total_global_paths += 1\\\\n\\\\n        assert path_count_by_energy == expected_counter\\\\n        assert sum(path_count_by_energy.values()) == int(\\\\n            representative["path_space_dimension"]\\\\n        )\\\\n\\\\n        for member in members[1:]:\\\\n            assert energy_counter_from_stage1(\\\\n                stage1_orbits[member["orbit_id"]]\\\\n            ) == expected_counter\\\\n\\\\n        direct = Fraction(0)\\\\n        folded = Fraction(0)\\\\n        raw_total = Fraction(0)\\\\n        energy_records = []\\\\n\\\\n        for energy in sorted(raw_by_energy):\\\\n            raw_sum = raw_by_energy[energy]\\\\n            coefficient = folded_coefficient(energy)\\\\n            contribution = raw_sum * coefficient\\\\n            mask = resonance_mask(energy)\\\\n            raw_total += raw_sum\\\\n            if mask == 0:\\\\n                direct += contribution\\\\n            else:\\\\n                folded += contribution\\\\n            energy_records.append(\\\\n                {\\\\n                    "E6": list(energy),\\\\n                    "resonance_mask": mask,\\\\n                    "global_path_count": int(path_count_by_energy[energy]),\\\\n                    "raw_haar_sum": fstr(raw_sum),\\\\n                    "des_cloizeaux_coefficient": fstr(coefficient),\\\\n                    "amplitude_contribution": fstr(contribution),\\\\n                }\\\\n            )\\\\n\\\\n        complete = direct + folded\\\\n        topology_results[topology_hash] = (direct, folded, complete, raw_total)\\\\n        path_dimension_hist[int(representative["path_space_dimension"])] += 1\\\\n        class_topology_hist[representative["resonance_class"]] += 1\\\\n        direct_topology_hist[direct] += 1\\\\n        folded_topology_hist[folded] += 1\\\\n        complete_topology_hist[complete] += 1\\\\n\\\\n        if topology_hash in stage3h_map:\\\\n            assert folded == 0\\\\n            assert complete == stage3h_map[topology_hash]\\\\n            stage3h_matches += 1\\\\n\\\\n        topology_records.append(\\\\n            {\\\\n                "topology_hash": topology_hash,\\\\n                "representative_orbit_id": representative["orbit_id"],\\\\n                "resonance_class": representative["resonance_class"],\\\\n                "orbit_count": len(members),\\\\n                "ordered_word_count": len({row["ordered_id"] for row in members}),\\\\n                "path_space_dimension": int(representative["path_space_dimension"]),\\\\n                "energy_signature_count": len(raw_by_energy),\\\\n                "raw_complete_haar_integral": fstr(raw_total),\\\\n                "direct_nonresonant_amplitude": fstr(direct),\\\\n                "folded_resonant_amplitude": fstr(folded),\\\\n                "complete_des_cloizeaux_amplitude": fstr(complete),\\\\n                "energy_contributions": energy_records,\\\\n            }\\\\n        )\\\\n\\\\n        if topology_index % 250 == 0:\\\\n            print(\\\\n                f"[contract] topologies={topology_index:,}/{len(topology_members):,} "\\\\n                f"global_paths={total_global_paths:,}",\\\\n                flush=True,\\\\n            )\\\\n\\\\n    assert total_global_paths == 35979\\\\n    assert stage3h_matches == len(stage3h_map)\\\\n\\\\n    gates["I2_complete_topology_contraction"] = {\\\\n        "topologies_evaluated": len(topology_records),\\\\n        "global_fusion_tree_paths": total_global_paths,\\\\n        "path_dimension_histogram": dict(sorted(path_dimension_hist.items())),\\\\n        "resonance_class_topology_histogram": dict(sorted(class_topology_hist.items())),\\\\n        "resonance_mask_path_histogram": {\\\\n            str(key): value for key, value in sorted(resonance_mask_path_hist.items())\\\\n        },\\\\n        "distinct_direct_amplitudes": len(direct_topology_hist),\\\\n        "distinct_folded_amplitudes": len(folded_topology_hist),\\\\n        "distinct_complete_amplitudes": len(complete_topology_hist),\\\\n        "stage3h_topologies_regressed": stage3h_matches,\\\\n        "passed": True,\\\\n    }\\\\n    print(\\\\n        "I2 PASS: all 3,895 topologies and 35,979 fusion-tree paths contracted"\\\\n    )\\\\n\\\\n    # I3: expand to every orientation orbit and aggregate canonical words.\\\\n    orbit_records = []\\\\n    word_accumulator = defaultdict(\\\\n        lambda: {\\\\n            "direct_even": Fraction(0),\\\\n            "direct_odd": Fraction(0),\\\\n            "folded_even": Fraction(0),\\\\n            "folded_odd": Fraction(0),\\\\n            "complete_even": Fraction(0),\\\\n            "complete_odd": Fraction(0),\\\\n            "orbit_count": 0,\\\\n            "topologies": set(),\\\\n            "output": None,\\\\n        }\\\\n    )\\\\n    orbit_complete_hist = Counter()\\\\n    orbit_folded_hist = Counter()\\\\n\\\\n    for orbit in sorted(all_orbits, key=lambda row: row["orbit_id"]):\\\\n        direct, folded, complete, raw_total = topology_results[orbit["topology_hash"]]\\\\n        even_phase = int(orbit["c_even_phase"])\\\\n        odd_phase = int(orbit["c_odd_phase"])\\\\n        displacement = output_displacement(orbit)\\\\n\\\\n        accumulator = word_accumulator[orbit["ordered_id"]]\\\\n        accumulator["direct_even"] += even_phase * direct\\\\n        accumulator["direct_odd"] += odd_phase * direct\\\\n        accumulator["folded_even"] += even_phase * folded\\\\n        accumulator["folded_odd"] += odd_phase * folded\\\\n        accumulator["complete_even"] += even_phase * complete\\\\n        accumulator["complete_odd"] += odd_phase * complete\\\\n        accumulator["orbit_count"] += 1\\\\n        accumulator["topologies"].add(orbit["topology_hash"])\\\\n        if accumulator["output"] is None:\\\\n            accumulator["output"] = displacement\\\\n        else:\\\\n            assert accumulator["output"] == displacement\\\\n\\\\n        orbit_complete_hist[complete] += 1\\\\n        orbit_folded_hist[folded] += 1\\\\n        orbit_records.append(\\\\n            {\\\\n                "orbit_id": orbit["orbit_id"],\\\\n                "ordered_id": orbit["ordered_id"],\\\\n                "topology_hash": orbit["topology_hash"],\\\\n                "resonance_class": orbit["resonance_class"],\\\\n                "path_space_dimension": int(orbit["path_space_dimension"]),\\\\n                "output_displacement_plane": list(displacement),\\\\n                "c_even_phase": even_phase,\\\\n                "c_odd_phase": odd_phase,\\\\n                "raw_complete_haar_integral": fstr(raw_total),\\\\n                "direct_nonresonant_amplitude": fstr(direct),\\\\n                "folded_resonant_amplitude": fstr(folded),\\\\n                "complete_des_cloizeaux_amplitude": fstr(complete),\\\\n                "C_even_complete_kernel": fstr(even_phase * complete),\\\\n                "C_odd_complete_kernel": fstr(odd_phase * complete),\\\\n            }\\\\n        )\\\\n\\\\n    assert len(orbit_records) == 16835\\\\n\\\\n    # Sixteen Stage-0 ordered words have no exact-singlet orientation orbit.\\\\n    # They remain explicit zero rows in the complete operator table.\\\\n    for ordered_id, word in word_by_id.items():\\\\n        accumulator = word_accumulator[ordered_id]\\\\n        if accumulator["output"] is None:\\\\n            output = tuple(int(x) for x in word["output"])\\\\n            accumulator["output"] = (\\\\n                output[0],\\\\n                output[1],\\\\n                output[2],\\\\n                output[3],\\\\n                output[4],\\\\n            )\\\\n\\\\n    assert len(word_accumulator) == 4221\\\\n\\\\n    word_records = []\\\\n    rooted_multiplicity_hist = Counter()\\\\n    rooted_odd_multiplicity_hist = Counter()\\\\n    zero_complete_odd = 0\\\\n    zero_complete_even = 0\\\\n\\\\n    for ordered_id in sorted(word_accumulator):\\\\n        accumulator = word_accumulator[ordered_id]\\\\n        word = word_by_id[ordered_id]\\\\n        even_multiplicity, odd_signed_multiplicity = (\\\\n            rooted_word_symmetry_data(word)\\\\n        )\\\\n        rooted_multiplicity_hist[even_multiplicity] += 1\\\\n        rooted_odd_multiplicity_hist[odd_signed_multiplicity] += 1\\\\n\\\\n        rooted_even = even_multiplicity * accumulator["complete_even"]\\\\n        rooted_odd = odd_signed_multiplicity * accumulator["complete_odd"]\\\\n\\\\n        if rooted_odd == 0:\\\\n            zero_complete_odd += 1\\\\n        if rooted_even == 0:\\\\n            zero_complete_even += 1\\\\n\\\\n        word_records.append(\\\\n            {\\\\n                "ordered_id": ordered_id,\\\\n                "ordered_insertions": word["ordered_insertions"],\\\\n                "output": word["output"],\\\\n                "output_displacement_plane": list(accumulator["output"]),\\\\n                "rooted_cubic_multiplicity_even": even_multiplicity,\\\\n                "rooted_cubic_signed_multiplicity_odd": (\\\\n                    odd_signed_multiplicity\\\\n                ),\\\\n                "orientation_orbit_count": accumulator["orbit_count"],\\\\n                "topology_count": len(accumulator["topologies"]),\\\\n                "canonical_direct_sum_even": fstr(accumulator["direct_even"]),\\\\n                "canonical_direct_sum_odd": fstr(accumulator["direct_odd"]),\\\\n                "canonical_folded_sum_even": fstr(accumulator["folded_even"]),\\\\n                "canonical_folded_sum_odd": fstr(accumulator["folded_odd"]),\\\\n                "canonical_complete_sum_even": fstr(accumulator["complete_even"]),\\\\n                "canonical_complete_sum_odd": fstr(accumulator["complete_odd"]),\\\\n                "rooted_complete_weight_even": fstr(rooted_even),\\\\n                "rooted_complete_weight_odd": fstr(rooted_odd),\\\\n            }\\\\n        )\\\\n\\\\n    gates["I3_orbit_word_assembly"] = {\\\\n        "orientation_orbit_records": len(orbit_records),\\\\n        "ordered_word_records": len(word_records),\\\\n        "rooted_even_multiplicity_histogram": dict(\\\\n            sorted(rooted_multiplicity_hist.items())\\\\n        ),\\\\n        "rooted_odd_signed_multiplicity_histogram": dict(\\\\n            sorted(rooted_odd_multiplicity_hist.items())\\\\n        ),\\\\n        "words_with_zero_complete_even_sum": zero_complete_even,\\\\n        "words_with_zero_complete_odd_sum": zero_complete_odd,\\\\n        "distinct_orbit_complete_amplitudes": len(orbit_complete_hist),\\\\n        "distinct_orbit_folded_amplitudes": len(orbit_folded_hist),\\\\n        "passed": True,\\\\n    }\\\\n    print("I3 PASS: complete kernels expanded to all orbits and 4,221 words")\\\\n\\\\n    # I4: write outputs and round-trip.\\\\n    topology_payload = {\\\\n        "meta": {\\\\n            "version": VERSION,\\\\n            "stage1_sha256": stage1_sha,\\\\n            "stage3e_sha256": stage3e_sha,\\\\n            "stage3g_sha256": stage3g_sha,\\\\n            "formula": (\\\\n                "H4=PVRVRVRVP-a(PVR2VRVP+PVRVR2VP)+a2PVR3VP"\\\\n                "-1/2{PVRVP,PVR2VP}"\\\\n            ),\\\\n        },\\\\n        "topologies": topology_records,\\\\n    }\\\\n    topology_path = output_dir / "y4_complete_folded_topology_amplitudes.json.gz"\\\\n    topology_sha = write_json_gz(topology_path, topology_payload)\\\\n\\\\n    orbit_payload = {\\\\n        "meta": {\\\\n            "version": VERSION,\\\\n            "topology_file": topology_path.name,\\\\n            "topology_sha256": topology_sha,\\\\n        },\\\\n        "orbits": orbit_records,\\\\n    }\\\\n    orbit_path = output_dir / "y4_complete_folded_orbit_amplitudes.json.gz"\\\\n    orbit_sha = write_json_gz(orbit_path, orbit_payload)\\\\n\\\\n    word_payload = {\\\\n        "meta": {\\\\n            "version": VERSION,\\\\n            "orbit_file": orbit_path.name,\\\\n            "orbit_sha256": orbit_sha,\\\\n            "rooted_multiplicity": (\\\\n                "orbit size under the 8-element proper cubic stabilizer "\\\\n                "of the rooted input plaquette"\\\\n            ),\\\\n        },\\\\n        "words": word_records,\\\\n    }\\\\n    word_path = output_dir / "y4_complete_folded_word_weights.json.gz"\\\\n    word_sha = write_json_gz(word_path, word_payload)\\\\n\\\\n    assert len(read_json_gz(topology_path)["topologies"]) == 3895\\\\n    assert len(read_json_gz(orbit_path)["orbits"]) == 16835\\\\n    assert len(read_json_gz(word_path)["words"]) == 4221\\\\n\\\\n    gates["I4_round_trip"] = {\\\\n        "topology_records": 3895,\\\\n        "orbit_records": 16835,\\\\n        "word_records": 4221,\\\\n        "passed": True,\\\\n    }\\\\n    print("I4 PASS: output round-trip and exact record counts")\\\\n\\\\n    elapsed = time.time() - started\\\\n    summary = {\\\\n        "meta": {\\\\n            "version": VERSION,\\\\n            "date": "2026-06-13",\\\\n            "python": sys.version,\\\\n            "platform": platform.platform(),\\\\n            "walltime_s": elapsed,\\\\n            "hardware": "CPU",\\\\n            "a100_required": False,\\\\n        },\\\\n        "counts": {\\\\n            "complete_topologies": 3895,\\\\n            "complete_orientation_orbits": 16835,\\\\n            "complete_ordered_words": 4221,\\\\n            "global_fusion_tree_paths": total_global_paths,\\\\n            "stage3h_regression_topologies": stage3h_matches,\\\\n            "words_with_zero_complete_C_odd_sum": zero_complete_odd,\\\\n            "words_with_zero_complete_C_even_sum": zero_complete_even,\\\\n        },\\\\n        "exact_ranges": {\\\\n            "complete_topology_min": fstr(min(complete_topology_hist)),\\\\n            "complete_topology_max": fstr(max(complete_topology_hist)),\\\\n            "folded_topology_min": fstr(min(folded_topology_hist)),\\\\n            "folded_topology_max": fstr(max(folded_topology_hist)),\\\\n        },\\\\n        "gates": gates,\\\\n        "files": {\\\\n            topology_path.name: {\\\\n                "sha256": topology_sha,\\\\n                "records": 3895,\\\\n            },\\\\n            orbit_path.name: {\\\\n                "sha256": orbit_sha,\\\\n                "records": 16835,\\\\n            },\\\\n            word_path.name: {\\\\n                "sha256": word_sha,\\\\n                "records": 4221,\\\\n            },\\\\n        },\\\\n        "scope": {\\\\n            "completed": [\\\\n                "complete nonresonant direct fourth-order term",\\\\n                "all resonant des-Cloizeaux folded/subtraction terms",\\\\n                "mixed-orbit direct and folded completion",\\\\n                "exact rooted cubic multiplicities",\\\\n                "complete ordered-word kernels",\\\\n            ],\\\\n            "not_completed": [\\\\n                "assembly of the real-space H4 orientation matrix",\\\\n                "cube-boundary residual and fourth-order bandwidth verdict",\\\\n            ],\\\\n        },\\\\n        "endgame": (\\\\n            "Final Stage J: assemble H4 from the rooted word weights and "\\\\n            "evaluate H4 psi_cube - c4 psi_cube exactly."\\\\n        ),\\\\n        "passed": True,\\\\n    }\\\\n    summary_path = output_dir / "y4_stage3i_summary.json"\\\\n    summary_sha = write_json(summary_path, summary)\\\\n\\\\n    print()\\\\n    print("SUMMARY")\\\\n    print(json.dumps(summary["counts"], indent=2, sort_keys=True))\\\\n    print()\\\\n    print(f"TOPOLOGIES: {topology_path}")\\\\n    print(f"ORBITS    : {orbit_path}")\\\\n    print(f"WORDS     : {word_path}")\\\\n    print(f"SUMMARY   : {summary_path}")\\\\n    print(f"SUMMARY SHA256: {summary_sha}")\\\\n    print(f"WALLTIME: {elapsed:.2f} s")\\\\n    print("ALL STAGE-I GATES PASS")\\\\n    print()\\\\n    print("NEXT AND FINAL: Stage J exact H4 cube-boundary flatness verdict.")\\\\n\\\\n\\\\nif __name__ == "__main__":\\\\n    parser = argparse.ArgumentParser(\\\\n        description="Complete direct + folded O(y^4) des-Cloizeaux kernel"\\\\n    )\\\\n    parser.add_argument("--stage1", type=Path, default=None)\\\\n    parser.add_argument("--stage3e", type=Path, default=None)\\\\n    parser.add_argument("--stage3g", type=Path, default=None)\\\\n    parser.add_argument("--stage3h-topologies", type=Path, default=None)\\\\n    parser.add_argument(\\\\n        "--output-dir",\\\\n        type=Path,\\\\n        default=default_output_dir(),\\\\n    )\\\\n    args, _unknown = parser.parse_known_args()\\\\n\\\\n    stage1_path = args.stage1 or recursive_find(\\\\n        "y4_channel_denominator_manifest.json.gz"\\\\n    )\\\\n    stage3e_path = args.stage3e or recursive_find(\\\\n        "y4_trace_wiring_orbits.json.gz"\\\\n    )\\\\n    stage3g_path = args.stage3g or recursive_find(\\\\n        "y4_exact_local_path_tensors.json.gz"\\\\n    )\\\\n    stage3h_topology_path = args.stage3h_topologies or recursive_find(\\\\n        "y4_nonresonant_topology_amplitudes.json.gz"\\\\n    )\\\\n\\\\n    missing = [\\\\n        name\\\\n        for name, path in (\\\\n            ("Stage 1", stage1_path),\\\\n            ("Stage 3E", stage3e_path),\\\\n            ("Stage 3G", stage3g_path),\\\\n        )\\\\n        if path is None\\\\n    ]\\\\n    assert not missing, "Missing dependencies: " + ", ".join(missing)\\\\n\\\\n    run(\\\\n        stage1_path,\\\\n        stage3e_path,\\\\n        stage3g_path,\\\\n        stage3h_topology_path,\\\\n        args.output_dir,\\\\n    )\\\\n\\\', \\\'y4_stage3j_final_flatband_verdict.py\\\': \\\'#!/usr/bin/env python3\\\\n"""\\\\ny4_stage3j_final_flatband_verdict.py\\\\n=====================================\\\\n\\\\nFINAL exact O(y^4) flat-band verdict for the SU(3) one-flux C-odd band.\\\\n\\\\nRUN IN COLAB\\\\n-------------\\\\n    %run /content/y4_stage3j_final_flatband_verdict.py\\\\n\\\\nHARDWARE\\\\n--------\\\\nUse a standard Colab CPU runtime. The A100 is not used.\\\\n\\\\nINPUT\\\\n-----\\\\nThe script recursively locates:\\\\n\\\\n    y4_complete_folded_word_weights.json.gz\\\\n\\\\nnormally at:\\\\n\\\\n    /content/Y4_STAGE3I/y4_complete_folded_word_weights.json.gz\\\\n\\\\nOUTPUT\\\\n------\\\\n    /content/Y4_STAGE3J/DATA_Y4_full_real_space_h4_kernel.json.gz\\\\n    /content/Y4_STAGE3J/y4_cube_boundary_residual.json\\\\n    /content/Y4_STAGE3J/CERT_Y4_stage3j_verdict.json\\\\n\\\\nWHAT THIS PROVES\\\\n----------------\\\\nThe script:\\\\n\\\\n  1. expands every canonical ordered word over the proper cubic stabilizer,\\\\n     including the C-odd sign caused by plaquette-orientation reversal;\\\\n\\\\n  2. constructs the complete translation-invariant 3x3 real-space H4 kernel;\\\\n\\\\n  3. proves exact Hermiticity;\\\\n\\\\n  4. applies H4 to the consistently oriented boundary of one elementary cube;\\\\n\\\\n  5. tests whether H4 psi_cube = c4 psi_cube exactly;\\\\n\\\\n  6. evaluates the first-order-in-H4 correction to the former flat branch at\\\\n     parity momenta, giving an exact nonzero dispersion difference.\\\\n\\\\nThe result is binary.  A nonzero residual proves that the C-odd flat band is\\\\nNOT protected at O(y^4) in the computed des-Cloizeaux convention.\\\\n\\\\nSCOPE\\\\n-----\\\\nThis is an order-by-order strong-coupling statement.  It is not a continuum\\\\nYang-Mills claim.\\\\n"""\\\\n\\\\nfrom __future__ import annotations\\\\n\\\\nimport argparse\\\\nimport gzip\\\\nimport hashlib\\\\nimport itertools\\\\nimport json\\\\nimport platform\\\\nimport sys\\\\nimport time\\\\nfrom collections import Counter, defaultdict\\\\nfrom fractions import Fraction\\\\nfrom pathlib import Path\\\\nfrom typing import Dict, List, Sequence, Tuple\\\\n\\\\nVERSION = "2026-06-13-stage3j-v1"\\\\n\\\\nVec3 = Tuple[int, int, int]\\\\nPlane = Tuple[int, int]\\\\nPlaquette = Tuple[int, int, int, int, int]\\\\nKernelKey = Tuple[Plane, Plane, Vec3]\\\\n\\\\nROOT: Plaquette = (0, 0, 0, 0, 1)\\\\nPLANES: Tuple[Plane, Plane, Plane] = ((0, 1), (0, 2), (1, 2))\\\\nPLANE_INDEX = {plane: index for index, plane in enumerate(PLANES)}\\\\n\\\\n\\\\ndef default_output_dir() -> Path:\\\\n    if Path("/content").exists():\\\\n        return Path("/content/Y4_STAGE3J")\\\\n    return Path.cwd() / "Y4_STAGE3J"\\\\n\\\\n\\\\ndef recursive_find(filename: str) -> Path | None:\\\\n    preferred = [\\\\n        Path("/content/Y4_STAGE3I") / filename,\\\\n        Path.cwd() / "Y4_STAGE3I" / filename,\\\\n        Path("/mnt/data/Y4_STAGE3I_TEST") / filename,\\\\n    ]\\\\n    for path in preferred:\\\\n        if path.exists():\\\\n            return path\\\\n\\\\n    for root in (Path("/content"), Path.cwd(), Path("/mnt/data")):\\\\n        if not root.exists():\\\\n            continue\\\\n        for path in root.rglob(filename):\\\\n            return path\\\\n    return None\\\\n\\\\n\\\\ndef sha256_file(path: Path) -> str:\\\\n    digest = hashlib.sha256()\\\\n    with path.open("rb") as handle:\\\\n        for chunk in iter(lambda: handle.read(1 << 20), b""):\\\\n            digest.update(chunk)\\\\n    return digest.hexdigest()\\\\n\\\\n\\\\ndef read_json_gz(path: Path) -> object:\\\\n    with gzip.open(path, "rt", encoding="utf-8") as handle:\\\\n        return json.load(handle)\\\\n\\\\n\\\\ndef write_json(path: Path, obj: object) -> str:\\\\n    raw = json.dumps(\\\\n        obj,\\\\n        indent=2,\\\\n        sort_keys=True,\\\\n        allow_nan=False,\\\\n    ).encode("utf-8")\\\\n    path.write_bytes(raw)\\\\n    return hashlib.sha256(raw).hexdigest()\\\\n\\\\n\\\\ndef write_json_gz(path: Path, obj: object) -> str:\\\\n    raw = json.dumps(\\\\n        obj,\\\\n        separators=(",", ":"),\\\\n        sort_keys=True,\\\\n        allow_nan=False,\\\\n    ).encode("utf-8")\\\\n    with gzip.GzipFile(\\\\n        filename=str(path),\\\\n        mode="wb",\\\\n        compresslevel=9,\\\\n        mtime=0,\\\\n    ) as handle:\\\\n        handle.write(raw)\\\\n    return sha256_file(path)\\\\n\\\\n\\\\ndef fstr(value: Fraction) -> str:\\\\n    return (\\\\n        str(value.numerator)\\\\n        if value.denominator == 1\\\\n        else f"{value.numerator}/{value.denominator}"\\\\n    )\\\\n\\\\n\\\\ndef permutation_parity(perm: Tuple[int, int, int]) -> int:\\\\n    inversions = sum(\\\\n        perm[i] > perm[j]\\\\n        for i in range(3)\\\\n        for j in range(i + 1, 3)\\\\n    )\\\\n    return -1 if inversions % 2 else 1\\\\n\\\\n\\\\ndef proper_rotations():\\\\n    rotations = []\\\\n    for perm in itertools.permutations(range(3)):\\\\n        parity = permutation_parity(perm)\\\\n        for signs in itertools.product((-1, 1), repeat=3):\\\\n            if parity * signs[0] * signs[1] * signs[2] == 1:\\\\n                rotations.append((perm, signs))\\\\n    assert len(rotations) == 24\\\\n    return rotations\\\\n\\\\n\\\\nROTATIONS = proper_rotations()\\\\n\\\\n\\\\ndef transform_vec(vector: Vec3, rotation) -> Vec3:\\\\n    perm, signs = rotation\\\\n    output = [0, 0, 0]\\\\n    for axis in range(3):\\\\n        output[perm[axis]] += signs[axis] * vector[axis]\\\\n    return tuple(output)\\\\n\\\\n\\\\ndef transform_plaquette(\\\\n    plaquette: Plaquette,\\\\n    rotation_index: int,\\\\n) -> Plaquette:\\\\n    perm, signs = ROTATIONS[rotation_index]\\\\n    anchor = plaquette[:3]\\\\n    first, second = plaquette[3], plaquette[4]\\\\n\\\\n    mapped_first, mapped_second = perm[first], perm[second]\\\\n    mapped_anchor = list(transform_vec(anchor, (perm, signs)))\\\\n\\\\n    if signs[first] < 0:\\\\n        mapped_anchor[mapped_first] -= 1\\\\n    if signs[second] < 0:\\\\n        mapped_anchor[mapped_second] -= 1\\\\n\\\\n    mapped_first, mapped_second = sorted(\\\\n        (mapped_first, mapped_second)\\\\n    )\\\\n    return (\\\\n        mapped_anchor[0],\\\\n        mapped_anchor[1],\\\\n        mapped_anchor[2],\\\\n        mapped_first,\\\\n        mapped_second,\\\\n    )\\\\n\\\\n\\\\ndef orientation_sign(\\\\n    plaquette: Plaquette,\\\\n    rotation_index: int,\\\\n) -> int:\\\\n    perm, signs = ROTATIONS[rotation_index]\\\\n    first, second = plaquette[3], plaquette[4]\\\\n    reorder = -1 if perm[first] > perm[second] else 1\\\\n    return signs[first] * signs[second] * reorder\\\\n\\\\n\\\\nROOT_STABILIZER = [\\\\n    index\\\\n    for index in range(24)\\\\n    if transform_plaquette(ROOT, index)[3:] == (0, 1)\\\\n]\\\\nassert len(ROOT_STABILIZER) == 8\\\\n\\\\nROOT_SHIFTS = {\\\\n    index: transform_plaquette(ROOT, index)[:3]\\\\n    for index in ROOT_STABILIZER\\\\n}\\\\n\\\\n\\\\ndef rooted_transform(\\\\n    plaquette: Plaquette,\\\\n    rotation_index: int,\\\\n) -> Plaquette:\\\\n    transformed = transform_plaquette(plaquette, rotation_index)\\\\n    shift = ROOT_SHIFTS[rotation_index]\\\\n    return (\\\\n        transformed[0] - shift[0],\\\\n        transformed[1] - shift[1],\\\\n        transformed[2] - shift[2],\\\\n        transformed[3],\\\\n        transformed[4],\\\\n    )\\\\n\\\\n\\\\ndef subtract_vec(first: Vec3, second: Vec3) -> Vec3:\\\\n    return tuple(first[i] - second[i] for i in range(3))\\\\n\\\\n\\\\ndef add_vec(first: Vec3, second: Vec3) -> Vec3:\\\\n    return tuple(first[i] + second[i] for i in range(3))\\\\n\\\\n\\\\ndef build_root_kernel(words: Sequence[dict]) -> Dict[Plaquette, Fraction]:\\\\n    root_kernel: Dict[Plaquette, Fraction] = defaultdict(Fraction)\\\\n\\\\n    for word in words:\\\\n        amplitude = Fraction(word["canonical_complete_sum_odd"])\\\\n        if amplitude == 0:\\\\n            continue\\\\n\\\\n        insertions = tuple(\\\\n            tuple(int(x) for x in row)\\\\n            for row in word["ordered_insertions"]\\\\n        )\\\\n        output = tuple(int(x) for x in word["output"])\\\\n\\\\n        image_signs = {}\\\\n        for rotation in ROOT_STABILIZER:\\\\n            image = (\\\\n                tuple(\\\\n                    rooted_transform(plaquette, rotation)\\\\n                    for plaquette in insertions\\\\n                ),\\\\n                rooted_transform(output, rotation),\\\\n            )\\\\n            sign = (\\\\n                orientation_sign(ROOT, rotation)\\\\n                * orientation_sign(output, rotation)\\\\n            )\\\\n            if image in image_signs:\\\\n                assert image_signs[image] == sign\\\\n            image_signs[image] = sign\\\\n\\\\n        for (_, transformed_output), sign in image_signs.items():\\\\n            root_kernel[transformed_output] += sign * amplitude\\\\n\\\\n    return {\\\\n        plaquette: value\\\\n        for plaquette, value in root_kernel.items()\\\\n        if value\\\\n    }\\\\n\\\\n\\\\ndef build_full_kernel(\\\\n    root_kernel: Dict[Plaquette, Fraction],\\\\n) -> Dict[KernelKey, Fraction]:\\\\n    candidates: Dict[KernelKey, set] = defaultdict(set)\\\\n\\\\n    for output, amplitude in root_kernel.items():\\\\n        for rotation in range(24):\\\\n            transformed_input = transform_plaquette(ROOT, rotation)\\\\n            transformed_output = transform_plaquette(output, rotation)\\\\n\\\\n            input_plane = transformed_input[3:]\\\\n            output_plane = transformed_output[3:]\\\\n            displacement = subtract_vec(\\\\n                transformed_output[:3],\\\\n                transformed_input[:3],\\\\n            )\\\\n            sign = (\\\\n                orientation_sign(ROOT, rotation)\\\\n                * orientation_sign(output, rotation)\\\\n            )\\\\n\\\\n            candidates[\\\\n                (input_plane, output_plane, displacement)\\\\n            ].add(sign * amplitude)\\\\n\\\\n    assert all(len(values) == 1 for values in candidates.values())\\\\n    kernel = {\\\\n        key: next(iter(values))\\\\n        for key, values in candidates.items()\\\\n        if next(iter(values))\\\\n    }\\\\n\\\\n    for (input_plane, output_plane, displacement), value in kernel.items():\\\\n        reverse = (\\\\n            output_plane,\\\\n            input_plane,\\\\n            tuple(-x for x in displacement),\\\\n        )\\\\n        assert kernel.get(reverse, Fraction(0)) == value\\\\n\\\\n    return kernel\\\\n\\\\n\\\\ndef cube_boundary_state() -> Dict[Plaquette, Fraction]:\\\\n    # ∂[x,y,z] = (T_x-1)[y,z] - (T_y-1)[x,z]\\\\n    #              + (T_z-1)[x,y].\\\\n    return {\\\\n        (1, 0, 0, 1, 2): Fraction(+1),\\\\n        (0, 0, 0, 1, 2): Fraction(-1),\\\\n        (0, 1, 0, 0, 2): Fraction(-1),\\\\n        (0, 0, 0, 0, 2): Fraction(+1),\\\\n        (0, 0, 1, 0, 1): Fraction(+1),\\\\n        (0, 0, 0, 0, 1): Fraction(-1),\\\\n    }\\\\n\\\\n\\\\ndef apply_kernel(\\\\n    kernel: Dict[KernelKey, Fraction],\\\\n    state: Dict[Plaquette, Fraction],\\\\n) -> Dict[Plaquette, Fraction]:\\\\n    output: Dict[Plaquette, Fraction] = defaultdict(Fraction)\\\\n\\\\n    by_input_plane: Dict[Plane, List[Tuple[Plane, Vec3, Fraction]]] = (\\\\n        defaultdict(list)\\\\n    )\\\\n    for (input_plane, output_plane, displacement), value in kernel.items():\\\\n        by_input_plane[input_plane].append(\\\\n            (output_plane, displacement, value)\\\\n        )\\\\n\\\\n    for plaquette, coefficient in state.items():\\\\n        anchor = plaquette[:3]\\\\n        input_plane = plaquette[3:]\\\\n        for output_plane, displacement, value in by_input_plane[\\\\n            input_plane\\\\n        ]:\\\\n            output_anchor = add_vec(anchor, displacement)\\\\n            output[\\\\n                output_anchor + output_plane\\\\n            ] += coefficient * value\\\\n\\\\n    return {\\\\n        plaquette: value\\\\n        for plaquette, value in output.items()\\\\n        if value\\\\n    }\\\\n\\\\n\\\\ndef symbol_at_parity(\\\\n    kernel: Dict[KernelKey, Fraction],\\\\n    phases: Tuple[int, int, int],\\\\n) -> List[List[Fraction]]:\\\\n    matrix = [\\\\n        [Fraction(0) for _ in range(3)]\\\\n        for _ in range(3)\\\\n    ]\\\\n\\\\n    for (input_plane, output_plane, displacement), value in kernel.items():\\\\n        phase = 1\\\\n        for axis in range(3):\\\\n            if displacement[axis] % 2:\\\\n                phase *= phases[axis]\\\\n        matrix[\\\\n            PLANE_INDEX[output_plane]\\\\n        ][\\\\n            PLANE_INDEX[input_plane]\\\\n        ] += phase * value\\\\n\\\\n    return matrix\\\\n\\\\n\\\\ndef flat_vector_at_parity(\\\\n    phases: Tuple[int, int, int],\\\\n) -> Tuple[int, int, int]:\\\\n    # Basis order: xy, xz, yz.\\\\n    return (\\\\n        phases[2] - 1,\\\\n        -(phases[1] - 1),\\\\n        phases[0] - 1,\\\\n    )\\\\n\\\\n\\\\ndef rayleigh(\\\\n    vector: Sequence[int],\\\\n    matrix: Sequence[Sequence[Fraction]],\\\\n) -> Fraction:\\\\n    norm = sum(value * value for value in vector)\\\\n    assert norm > 0\\\\n    numerator = sum(\\\\n        vector[i] * matrix[i][j] * vector[j]\\\\n        for i in range(3)\\\\n        for j in range(3)\\\\n    )\\\\n    return Fraction(numerator, norm)\\\\n\\\\n\\\\ndef run(input_path: Path, output_dir: Path) -> None:\\\\n    started = time.time()\\\\n    output_dir.mkdir(parents=True, exist_ok=True)\\\\n\\\\n    print("=" * 108)\\\\n    print("SU(3) O(y^4) FINAL EXACT FLAT-BAND VERDICT")\\\\n    print("=" * 108)\\\\n    print(f"version  : {VERSION}")\\\\n    print(f"input    : {input_path}")\\\\n    print(f"output   : {output_dir}")\\\\n    print("hardware : standard Colab CPU; A100 is not used")\\\\n    print()\\\\n\\\\n    assert input_path.exists(), f"Missing Stage-I word file: {input_path}"\\\\n    input_sha = sha256_file(input_path)\\\\n    payload = read_json_gz(input_path)\\\\n    words = payload["words"]\\\\n    assert len(words) == 4221\\\\n\\\\n    gates = {}\\\\n\\\\n    # J0: exact real-space kernel.\\\\n    root_kernel = build_root_kernel(words)\\\\n    full_kernel = build_full_kernel(root_kernel)\\\\n\\\\n    assert len(root_kernel) == 63\\\\n    assert len(full_kernel) == 189\\\\n\\\\n    gates["J0_real_space_kernel"] = {\\\\n        "stage3i_sha256": input_sha,\\\\n        "ordered_words": len(words),\\\\n        "nonzero_root_kernel_entries": len(root_kernel),\\\\n        "nonzero_full_kernel_entries": len(full_kernel),\\\\n        "proper_cubic_covariance": True,\\\\n        "exact_hermiticity": True,\\\\n        "passed": True,\\\\n    }\\\\n    print("J0 PASS: complete cubic-covariant Hermitian H4 kernel")\\\\n\\\\n    # J1: exact cube-boundary residual.\\\\n    cube = cube_boundary_state()\\\\n    image = apply_kernel(full_kernel, cube)\\\\n\\\\n    ratios = {\\\\n        image.get(plaquette, Fraction(0)) / coefficient\\\\n        for plaquette, coefficient in cube.items()\\\\n    }\\\\n    assert len(ratios) == 1\\\\n    rigid_component = next(iter(ratios))\\\\n\\\\n    support = set(image) | set(cube)\\\\n    residual = {\\\\n        plaquette: (\\\\n            image.get(plaquette, Fraction(0))\\\\n            - rigid_component * cube.get(plaquette, Fraction(0))\\\\n        )\\\\n        for plaquette in support\\\\n    }\\\\n    residual = {\\\\n        plaquette: value\\\\n        for plaquette, value in residual.items()\\\\n        if value\\\\n    }\\\\n\\\\n    assert len(image) == 36\\\\n    assert len(residual) == 30\\\\n    assert residual\\\\n\\\\n    residual_abs_hist = Counter(abs(value) for value in residual.values())\\\\n    assert residual_abs_hist[Fraction(5, 48)] == 6\\\\n    assert max(residual_abs_hist) == Fraction(5, 48)\\\\n\\\\n    dominant = sorted(\\\\n        (\\\\n            (plaquette, value)\\\\n            for plaquette, value in residual.items()\\\\n            if abs(value) == Fraction(5, 48)\\\\n        ),\\\\n        key=lambda row: row[0],\\\\n    )\\\\n    assert len(dominant) == 6\\\\n\\\\n    gates["J1_cube_boundary"] = {\\\\n        "cube_faces": len(cube),\\\\n        "H4_cube_support": len(image),\\\\n        "rigid_component_c4": fstr(rigid_component),\\\\n        "nonzero_residual_plaquettes": len(residual),\\\\n        "maximum_residual_magnitude": "5/48",\\\\n        "dominant_leakage_count": len(dominant),\\\\n        "flatness_through_O_y4": False,\\\\n        "passed": True,\\\\n    }\\\\n    print("J1 PASS: cube-boundary residual is nonzero; flatness breaks at O(y^4)")\\\\n\\\\n    # J2: exact high-symmetry dispersion witness.\\\\n    parity_points = {\\\\n        "one_pi_x": (-1, +1, +1),\\\\n        "one_pi_y": (+1, -1, +1),\\\\n        "one_pi_z": (+1, +1, -1),\\\\n        "two_pi_xy": (-1, -1, +1),\\\\n        "two_pi_xz": (-1, +1, -1),\\\\n        "two_pi_yz": (+1, -1, -1),\\\\n        "three_pi": (-1, -1, -1),\\\\n    }\\\\n\\\\n    corrections = {}\\\\n    matrices = {}\\\\n    for name, phases in parity_points.items():\\\\n        matrix = symbol_at_parity(full_kernel, phases)\\\\n        vector = flat_vector_at_parity(phases)\\\\n        correction = rayleigh(vector, matrix)\\\\n        corrections[name] = correction\\\\n        matrices[name] = matrix\\\\n\\\\n    one_pi_values = {\\\\n        corrections["one_pi_x"],\\\\n        corrections["one_pi_y"],\\\\n        corrections["one_pi_z"],\\\\n    }\\\\n    two_pi_values = {\\\\n        corrections["two_pi_xy"],\\\\n        corrections["two_pi_xz"],\\\\n        corrections["two_pi_yz"],\\\\n    }\\\\n    assert len(one_pi_values) == 1\\\\n    assert len(two_pi_values) == 1\\\\n\\\\n    one_pi = next(iter(one_pi_values))\\\\n    two_pi = next(iter(two_pi_values))\\\\n    three_pi = corrections["three_pi"]\\\\n\\\\n    assert one_pi == Fraction(\\\\n        -17700498622147435111,\\\\n        7250590288602460800,\\\\n    )\\\\n    assert two_pi == Fraction(\\\\n        -4367164159624988707,\\\\n        1812647572150615200,\\\\n    )\\\\n    assert three_pi == Fraction(\\\\n        -3447362930970494909,\\\\n        1450118057720492160,\\\\n    )\\\\n\\\\n    witness_difference = three_pi - one_pi\\\\n    assert witness_difference == Fraction(\\\\n        17607806155349,\\\\n        275331901291200,\\\\n    )\\\\n    assert witness_difference > 0\\\\n\\\\n    gates["J2_dispersion_witness"] = {\\\\n        "one_pi_correction": fstr(one_pi),\\\\n        "two_pi_correction": fstr(two_pi),\\\\n        "three_pi_correction": fstr(three_pi),\\\\n        "three_pi_minus_one_pi": fstr(witness_difference),\\\\n        "decimal_lower_bound_on_bandwidth_coefficient": float(\\\\n            witness_difference\\\\n        ),\\\\n        "nonzero_dispersion": True,\\\\n        "passed": True,\\\\n    }\\\\n    print(\\\\n        "J2 PASS: exact momentum-space dispersion witness "\\\\n        f"Δc4={fstr(witness_difference)}"\\\\n    )\\\\n\\\\n    # J3: write final certificate.\\\\n    kernel_records = [\\\\n        {\\\\n            "input_plane": list(input_plane),\\\\n            "output_plane": list(output_plane),\\\\n            "displacement": list(displacement),\\\\n            "weight": fstr(value),\\\\n        }\\\\n        for (\\\\n            input_plane,\\\\n            output_plane,\\\\n            displacement,\\\\n        ), value in sorted(full_kernel.items())\\\\n    ]\\\\n    kernel_payload = {\\\\n        "meta": {\\\\n            "version": VERSION,\\\\n            "stage3i_input": str(input_path),\\\\n            "stage3i_sha256": input_sha,\\\\n            "basis_planes": [list(plane) for plane in PLANES],\\\\n        },\\\\n        "kernel": kernel_records,\\\\n    }\\\\n    kernel_path = output_dir / "DATA_Y4_full_real_space_h4_kernel.json.gz"\\\\n    kernel_sha = write_json_gz(kernel_path, kernel_payload)\\\\n\\\\n    residual_payload = {\\\\n        "meta": {\\\\n            "version": VERSION,\\\\n            "kernel_file": kernel_path.name,\\\\n            "kernel_sha256": kernel_sha,\\\\n        },\\\\n        "cube_state": [\\\\n            {\\\\n                "plaquette": list(plaquette),\\\\n                "coefficient": fstr(value),\\\\n            }\\\\n            for plaquette, value in sorted(cube.items())\\\\n        ],\\\\n        "rigid_component_c4": fstr(rigid_component),\\\\n        "H4_cube_image": [\\\\n            {\\\\n                "plaquette": list(plaquette),\\\\n                "coefficient": fstr(value),\\\\n            }\\\\n            for plaquette, value in sorted(image.items())\\\\n        ],\\\\n        "residual": [\\\\n            {\\\\n                "plaquette": list(plaquette),\\\\n                "coefficient": fstr(value),\\\\n            }\\\\n            for plaquette, value in sorted(residual.items())\\\\n        ],\\\\n        "dominant_leakage": [\\\\n            {\\\\n                "plaquette": list(plaquette),\\\\n                "coefficient": fstr(value),\\\\n            }\\\\n            for plaquette, value in dominant\\\\n        ],\\\\n    }\\\\n    residual_path = output_dir / "y4_cube_boundary_residual.json"\\\\n    residual_sha = write_json(residual_path, residual_payload)\\\\n\\\\n    elapsed = time.time() - started\\\\n    verdict = {\\\\n        "meta": {\\\\n            "version": VERSION,\\\\n            "date": "2026-06-13",\\\\n            "python": sys.version,\\\\n            "platform": platform.platform(),\\\\n            "walltime_s": elapsed,\\\\n            "hardware": "CPU",\\\\n            "a100_required": False,\\\\n        },\\\\n        "verdict": {\\\\n            "flat_through_order_y4": False,\\\\n            "first_nonzero_bandwidth_order": "y^4",\\\\n            "cube_boundary_residual_nonzero": True,\\\\n            "maximum_exact_leakage": "5/48",\\\\n            "rigid_component_c4": fstr(rigid_component),\\\\n            "exact_dispersion_witness": fstr(witness_difference),\\\\n            "dispersion_witness_decimal": float(witness_difference),\\\\n            "statement": (\\\\n                "The T1^{+-} one-flux band is exactly flat through O(y^3) "\\\\n                "but acquires nonzero dispersion at O(y^4)."\\\\n            ),\\\\n        },\\\\n        "high_symmetry_corrections": {\\\\n            name: fstr(value)\\\\n            for name, value in corrections.items()\\\\n        },\\\\n        "gates": gates,\\\\n        "files": {\\\\n            kernel_path.name: {\\\\n                "sha256": kernel_sha,\\\\n                "records": len(kernel_records),\\\\n            },\\\\n            residual_path.name: {\\\\n                "sha256": residual_sha,\\\\n                "residual_records": len(residual),\\\\n            },\\\\n        },\\\\n        "scope": {\\\\n            "claim": "strong-coupling one-flux effective Hamiltonian through O(y^4)",\\\\n            "not_claimed": [\\\\n                "all-orders behavior",\\\\n                "continuum glueball bandwidth",\\\\n                "continuum Yang-Mills mass gap",\\\\n            ],\\\\n        },\\\\n        "passed": True,\\\\n    }\\\\n    verdict_path = output_dir / "CERT_Y4_stage3j_verdict.json"\\\\n    verdict_sha = write_json(verdict_path, verdict)\\\\n\\\\n    assert len(read_json_gz(kernel_path)["kernel"]) == 189\\\\n    assert json.loads(residual_path.read_text())["rigid_component_c4"] == (\\\\n        fstr(rigid_component)\\\\n    )\\\\n    assert json.loads(verdict_path.read_text())["passed"] is True\\\\n\\\\n    gates["J3_round_trip"] = {\\\\n        "kernel_records": 189,\\\\n        "residual_records": len(residual),\\\\n        "passed": True,\\\\n    }\\\\n\\\\n    # Rewrite verdict with the final round-trip gate included.\\\\n    verdict["gates"] = gates\\\\n    verdict_sha = write_json(verdict_path, verdict)\\\\n\\\\n    print("J3 PASS: final certificate round-trip")\\\\n    print()\\\\n    print("FINAL VERDICT")\\\\n    print(json.dumps(verdict["verdict"], indent=2, sort_keys=True))\\\\n    print()\\\\n    print(f"KERNEL  : {kernel_path}")\\\\n    print(f"RESIDUAL: {residual_path}")\\\\n    print(f"VERDICT : {verdict_path}")\\\\n    print(f"VERDICT SHA256: {verdict_sha}")\\\\n    print(f"WALLTIME: {elapsed:.3f} s")\\\\n    print("ALL STAGE-J GATES PASS")\\\\n\\\\n\\\\nif __name__ == "__main__":\\\\n    parser = argparse.ArgumentParser(\\\\n        description="Final exact O(y^4) flat-band verdict"\\\\n    )\\\\n    parser.add_argument("--input", type=Path, default=None)\\\\n    parser.add_argument(\\\\n        "--output-dir",\\\\n        type=Path,\\\\n        default=default_output_dir(),\\\\n    )\\\\n    args, _unknown = parser.parse_known_args()\\\\n\\\\n    input_path = args.input or recursive_find(\\\\n        "y4_complete_folded_word_weights.json.gz"\\\\n    )\\\\n    assert input_path is not None, (\\\\n        "Could not locate y4_complete_folded_word_weights.json.gz. "\\\\n        "Run Stage I first."\\\\n    )\\\\n\\\\n    run(input_path, args.output_dir)\\\\n\\\'}\\n\\n\\ndef recursive_find(filename: str) -> Path | None:\\n    preferred_dirs = (\\n        "Y4_STAGE1", "Y4_STAGE2", "Y4_STAGE3B", "Y4_STAGE3C",\\n        "Y4_STAGE3E", "Y4_STAGE3G", "Y4_STAGE3I", "Y4_STAGE3J",\\n    )\\n    roots = (Path("/content"), Path.cwd(), Path("/mnt/data"))\\n    for directory in preferred_dirs:\\n        for root in roots:\\n            path = root / directory / filename\\n            if path.exists():\\n                return path\\n    for root in roots:\\n        if root.exists():\\n            for path in root.rglob(filename):\\n                return path\\n    return None\\n\\n\\ndef sha256_file(path: Path) -> str:\\n    digest = hashlib.sha256()\\n    with path.open("rb") as handle:\\n        for chunk in iter(lambda: handle.read(1 << 20), b""):\\n            digest.update(chunk)\\n    return digest.hexdigest()\\n\\n\\ndef read_json(path: Path):\\n    return json.loads(path.read_text(encoding="utf-8"))\\n\\n\\ndef read_json_gz(path: Path):\\n    with gzip.open(path, "rt", encoding="utf-8") as handle:\\n        return json.load(handle)\\n\\n\\ndef run_source(source_name: str, arguments: list[str]) -> None:\\n    with tempfile.TemporaryDirectory(prefix="y4_final_") as td:\\n        script_path = Path(td) / source_name\\n        script_path.write_text(SOURCES[source_name], encoding="utf-8")\\n        command = [sys.executable, "-u", str(script_path), *arguments]\\n        print()\\n        print("[endgame] RUN:", " ".join(command), flush=True)\\n        subprocess.run(command, check=True)\\n\\n\\ndef valid_summary(path: Path, passed_key: str = "passed") -> bool:\\n    try:\\n        return path.exists() and bool(read_json(path)[passed_key])\\n    except Exception:\\n        return False\\n\\n\\ndef main(root: Path, stage1: Path | None, stage2_cards: Path | None) -> None:\\n    print("=" * 104)\\n    print("Y4 FINAL ENDGAME AUTOBUNDLE")\\n    print("=" * 104)\\n    print(f"version : {VERSION}")\\n    print(f"root    : {root}")\\n    print("hardware: CPU; A100 not used")\\n    print()\\n\\n    root.mkdir(parents=True, exist_ok=True)\\n    stage1 = stage1 or recursive_find("y4_channel_denominator_manifest.json.gz")\\n    stage2_cards = stage2_cards or recursive_find("y4_link_tensor_cards.json.gz")\\n\\n    assert stage1 is not None and stage1.exists(), (\\n        "Missing Stage-1 manifest. Run y4_stage1_stage2_autobundle.py first."\\n    )\\n    assert stage2_cards is not None and stage2_cards.exists(), (\\n        "Missing Stage-2 tensor cards. Run y4_stage1_stage2_autobundle.py first."\\n    )\\n\\n    print("stage1      :", stage1)\\n    print("stage2 cards:", stage2_cards)\\n\\n    # ------------------------------------------------------------------\\n    # 3B / 3C\\n    # ------------------------------------------------------------------\\n    stage3b_dir = root / "Y4_STAGE3B"\\n    stage3c_dir = root / "Y4_STAGE3C"\\n    stage3b_paths = stage3b_dir / "y4_local_irrep_paths.json.gz"\\n    stage3c_summary = stage3c_dir / "y4_stage3c_summary.json"\\n    stage3c_cards = stage3c_dir / "y4_path_projector_cards.json.gz"\\n    stage3c_carriers = stage3c_dir / "y4_irrep_carriers.json.gz"\\n    stage3c_projectors = stage3c_dir / "y4_casimir_channel_projectors.json.gz"\\n\\n    valid_3bc = (\\n        stage3b_paths.exists()\\n        and valid_summary(stage3c_summary)\\n        and stage3c_cards.exists()\\n        and stage3c_carriers.exists()\\n        and stage3c_projectors.exists()\\n    )\\n    if valid_3bc:\\n        print("[endgame] Stage 3B/3C valid; reusing.")\\n    else:\\n        run_source(\\n            "y4_stage3b_stage3c_autobundle.py",\\n            ["--root", str(root), "--stage1", str(stage1)],\\n        )\\n    assert stage3b_paths.exists() and valid_summary(stage3c_summary)\\n\\n    # ------------------------------------------------------------------\\n    # 3E\\n    # ------------------------------------------------------------------\\n    stage3e_dir = root / "Y4_STAGE3E"\\n    stage3e_summary = stage3e_dir / "y4_stage3e_summary.json"\\n    stage3e_orbits = stage3e_dir / "y4_trace_wiring_orbits.json.gz"\\n    if valid_summary(stage3e_summary) and stage3e_orbits.exists():\\n        print("[endgame] Stage 3E valid; reusing.")\\n    else:\\n        run_source(\\n            "y4_stage3e_trace_wiring_compiler.py",\\n            [\\n                "--stage1", str(stage1),\\n                "--stage3c", str(stage3c_cards),\\n                "--output-dir", str(stage3e_dir),\\n            ],\\n        )\\n    assert valid_summary(stage3e_summary) and stage3e_orbits.exists()\\n\\n    # ------------------------------------------------------------------\\n    # 3G (checkpointed)\\n    # ------------------------------------------------------------------\\n    stage3g_dir = root / "Y4_STAGE3G"\\n    stage3g_summary = stage3g_dir / "y4_stage3g_summary.json"\\n    stage3g_tensors = stage3g_dir / "y4_exact_local_path_tensors.json.gz"\\n    if valid_summary(stage3g_summary) and stage3g_tensors.exists():\\n        print("[endgame] Stage 3G valid; reusing.")\\n    else:\\n        run_source(\\n            "y4_stage3g_checkpointed.py",\\n            [\\n                "--stage2-cards", str(stage2_cards),\\n                "--stage3b-paths", str(stage3b_paths),\\n                "--stage3c-carriers", str(stage3c_carriers),\\n                "--stage3c-projectors", str(stage3c_projectors),\\n                "--output-dir", str(stage3g_dir),\\n                "--deadline", "0",\\n            ],\\n        )\\n    assert valid_summary(stage3g_summary) and stage3g_tensors.exists()\\n\\n    # ------------------------------------------------------------------\\n    # I: complete direct + folded kernel\\n    # ------------------------------------------------------------------\\n    stage3i_dir = root / "Y4_STAGE3I"\\n    stage3i_summary = stage3i_dir / "y4_stage3i_summary.json"\\n    stage3i_words = stage3i_dir / "y4_complete_folded_word_weights.json.gz"\\n\\n    run_source(\\n        "y4_stage3i_complete_folded_descloizeaux.py",\\n        [\\n            "--stage1", str(stage1),\\n            "--stage3e", str(stage3e_orbits),\\n            "--stage3g", str(stage3g_tensors),\\n            "--output-dir", str(stage3i_dir),\\n        ],\\n    )\\n    assert valid_summary(stage3i_summary) and stage3i_words.exists()\\n\\n    # ------------------------------------------------------------------\\n    # FINAL J\\n    # ------------------------------------------------------------------\\n    stage3j_dir = root / "Y4_STAGE3J"\\n    verdict_path = stage3j_dir / "CERT_Y4_stage3j_verdict.json"\\n    run_source(\\n        "y4_stage3j_final_flatband_verdict.py",\\n        [\\n            "--input", str(stage3i_words),\\n            "--output-dir", str(stage3j_dir),\\n        ],\\n    )\\n\\n    verdict = read_json(verdict_path)\\n    assert verdict["passed"] is True\\n    result = verdict["verdict"]\\n\\n    print()\\n    print("=" * 104)\\n    print("FINAL RESULT")\\n    print("=" * 104)\\n    print(json.dumps(result, indent=2, sort_keys=True))\\n    print()\\n    print("VERDICT FILE:", verdict_path)\\n    print("SHA256      :", sha256_file(verdict_path))\\n    print("ENDGAME COMPLETE — NO FURTHER STAGES")\\n\\n\\nif __name__ == "__main__":\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\n        "--root",\\n        type=Path,\\n        default=Path("/content") if Path("/content").exists() else Path.cwd(),\\n    )\\n    parser.add_argument("--stage1", type=Path, default=None)\\n    parser.add_argument("--stage2-cards", type=Path, default=None)\\n    args, _unknown = parser.parse_known_args()\\n    main(args.root, args.stage1, args.stage2_cards)\\n\'\n\n\ndef sha256_file(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open(\'rb\') as f:\n        for chunk in iter(lambda: f.read(1 << 20), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef read_json(path: Path):\n    return json.loads(path.read_text(encoding=\'utf-8\'))\n\n\ndef read_json_gz(path: Path):\n    with gzip.open(path, \'rt\', encoding=\'utf-8\') as f:\n        return json.load(f)\n\n\ndef run_source(source: str, name: str, args: list[str]) -> None:\n    with tempfile.TemporaryDirectory(prefix=\'y4_all_\') as td:\n        path = Path(td) / name\n        path.write_text(source, encoding=\'utf-8\')\n        cmd = [sys.executable, \'-u\', str(path), *args]\n        print(\'\\n[complete] RUN:\', \' \'.join(cmd), flush=True)\n        subprocess.run(cmd, check=True)\n\n\ndef valid_stage0(root: Path) -> bool:\n    summary = root / \'Y4_STAGE0\' / \'y4_stage0_summary.json\'\n    words = root / \'Y4_STAGE0\' / \'y4_ordered_transition_words.json.gz\'\n    try:\n        return (\n            summary.exists()\n            and words.exists()\n            and read_json(summary).get(\'passed\') is True\n            and len(read_json_gz(words).get(\'words\', [])) == 4221\n        )\n    except Exception:\n        return False\n\n\ndef valid_stage12(root: Path) -> bool:\n    stage1 = root / \'Y4_STAGE1\' / \'y4_channel_denominator_manifest.json.gz\'\n    stage2 = root / \'Y4_STAGE2\' / \'y4_stage2_summary.json\'\n    cards = root / \'Y4_STAGE2\' / \'y4_link_tensor_cards.json.gz\'\n    try:\n        return (\n            stage1.exists()\n            and stage2.exists()\n            and cards.exists()\n            and len(read_json_gz(stage1).get(\'words\', [])) == 4221\n            and read_json(stage2).get(\'passed\') is True\n            and read_json(stage2)[\'counts\'][\'unique_link_token_signatures\'] == 182\n        )\n    except Exception:\n        return False\n\n\ndef main(root: Path) -> None:\n    print(\'=\' * 104)\n    print(\'Y4 COMPLETE FROM-SCRATCH COLAB PIPELINE\')\n    print(\'=\' * 104)\n    print(\'version :\', VERSION)\n    print(\'root    :\', root)\n    print(\'hardware: CPU; A100 not used\')\n    root.mkdir(parents=True, exist_ok=True)\n\n    stage0_dir = root / \'Y4_STAGE0\'\n    stage0_words = stage0_dir / \'y4_ordered_transition_words.json.gz\'\n    if valid_stage0(root):\n        print(\'[complete] Stage 0 valid; reusing.\')\n    else:\n        run_source(\n            STAGE0_SOURCE,\n            \'y4_stage0_geometry_manifest.py\',\n            [\'--output-dir\', str(stage0_dir)],\n        )\n    assert valid_stage0(root)\n    print(\'[complete] Stage 0 validation PASS\')\n    print(\'[complete] Stage 0 words SHA256:\', sha256_file(stage0_words))\n\n    if valid_stage12(root):\n        print(\'[complete] Stage 1/2 valid; reusing.\')\n    else:\n        run_source(\n            STAGE12_SOURCE,\n            \'y4_stage1_stage2_autobundle.py\',\n            [\'--stage0\', str(stage0_words), \'--root\', str(root)],\n        )\n    assert valid_stage12(root)\n    print(\'[complete] Stage 1/2 validation PASS\')\n\n    stage1 = root / \'Y4_STAGE1\' / \'y4_channel_denominator_manifest.json.gz\'\n    cards = root / \'Y4_STAGE2\' / \'y4_link_tensor_cards.json.gz\'\n    run_source(\n        FINAL_SOURCE,\n        \'y4_final_endgame_autobundle.py\',\n        [\'--root\', str(root), \'--stage1\', str(stage1), \'--stage2-cards\', str(cards)],\n    )\n\n    verdict_path = root / \'Y4_STAGE3J\' / \'CERT_Y4_stage3j_verdict.json\'\n    verdict = read_json(verdict_path)\n    assert verdict.get(\'passed\') is True\n    print(\'\\n\' + \'=\' * 104)\n    print(\'COMPLETE PIPELINE PASSED\')\n    print(\'=\' * 104)\n    print(json.dumps(verdict[\'verdict\'], indent=2, sort_keys=True))\n    print(\'VERDICT FILE:\', verdict_path)\n    print(\'SHA256      :\', sha256_file(verdict_path))\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\n        \'--root\', type=Path,\n        default=Path(\'/content\') if Path(\'/content\').exists() else Path.cwd(),\n    )\n    args, _unknown = parser.parse_known_args()\n    main(args.root)\n'

pipeline_path = Path('/content/y4_complete_from_scratch.py')
pipeline_path.write_text(PIPELINE_SOURCE, encoding='utf-8')
print(f'Wrote {pipeline_path} ({pipeline_path.stat().st_size:,} bytes)')


In [ ]:
%run /content/y4_complete_from_scratch.py